In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Sel 1 — Mount Google Drive dan Cek File Zip Dataset

from google.colab import drive
from pathlib import Path
import json
from datetime import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Path utama proyek
PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
RAW_DIR = PROJECT_DIR / 'data_raw'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

# Path file zip dataset
ZIP_PATH = RAW_DIR / 'TEM virus dataset.zip'

# Pastikan folder outputs ada
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cek file zip
zip_exists = ZIP_PATH.exists()
zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024) if zip_exists else None
zip_size_gb = ZIP_PATH.stat().st_size / (1024 * 1024 * 1024) if zip_exists else None

print("=== HASIL CEK SEL 1 ===")
print(f"Project dir     : {PROJECT_DIR}")
print(f"Raw data dir    : {RAW_DIR}")
print(f"Output dir      : {OUTPUT_DIR}")
print(f"Zip path        : {ZIP_PATH}")
print(f"Zip ditemukan   : {zip_exists}")

if zip_exists:
    print(f"Ukuran zip      : {zip_size_mb:.2f} MB")
    print(f"Ukuran zip      : {zip_size_gb:.2f} GB")
else:
    print("PERINGATAN: File zip tidak ditemukan. Cek nama file atau lokasi di Google Drive.")

# Simpan hasil cek awal
sel1_result = {
    "step": "Sel 1 — Mount Google Drive dan Cek File Zip Dataset",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "project_dir": str(PROJECT_DIR),
    "raw_data_dir": str(RAW_DIR),
    "output_dir": str(OUTPUT_DIR),
    "zip_path": str(ZIP_PATH),
    "zip_exists": zip_exists,
    "zip_size_mb": zip_size_mb,
    "zip_size_gb": zip_size_gb
}

sel1_output_path = OUTPUT_DIR / "sel1_zip_check.json"

with open(sel1_output_path, "w", encoding="utf-8") as f:
    json.dump(sel1_result, f, indent=4, ensure_ascii=False)

print(f"\nHasil Sel 1 disimpan ke: {sel1_output_path}")

In [ ]:
# Sel 2 — Baca Isi Zip Tanpa Ekstraksi

from pathlib import Path
import zipfile
import json
import csv
from collections import Counter
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
RAW_DIR = PROJECT_DIR / 'data_raw'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
ZIP_PATH = RAW_DIR / 'TEM virus dataset.zip'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 2 ===")
print(f"Zip path: {ZIP_PATH}")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"File zip tidak ditemukan: {ZIP_PATH}")

try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        infos = zf.infolist()

        total_items = len(infos)
        total_files = sum(1 for info in infos if not info.is_dir())
        total_dirs = sum(1 for info in infos if info.is_dir())
        total_uncompressed_mb = sum(info.file_size for info in infos) / (1024 * 1024)
        total_compressed_mb = sum(info.compress_size for info in infos) / (1024 * 1024)

        # Ambil struktur top-level
        top_level = []
        for info in infos:
            name = info.filename.strip("/")
            if name:
                top_level.append(name.split("/")[0])
        top_level_counter = Counter(top_level)

        # Hitung ekstensi file
        extensions = []
        for info in infos:
            if not info.is_dir():
                suffix = Path(info.filename).suffix.lower()
                extensions.append(suffix if suffix else "[no_extension]")
        extension_counter = Counter(extensions)

        # Sampel nama file/folder awal
        sample_entries = [info.filename for info in infos[:50]]

        print(f"Zip bisa dibaca        : True")
        print(f"Total item dalam zip   : {total_items}")
        print(f"Total file             : {total_files}")
        print(f"Total folder           : {total_dirs}")
        print(f"Total ukuran asli      : {total_uncompressed_mb:.2f} MB")
        print(f"Total ukuran compressed: {total_compressed_mb:.2f} MB")

        print("\nTop-level folder/file terdeteksi:")
        for name, count in top_level_counter.most_common(20):
            print(f"- {name}: {count} item")

        print("\nEkstensi file terdeteksi:")
        for ext, count in extension_counter.most_common(30):
            print(f"- {ext}: {count} file")

        print("\nContoh 50 entry pertama:")
        for entry in sample_entries:
            print(f"- {entry}")

        # Simpan ringkasan Sel 2
        sel2_summary = {
            "step": "Sel 2 — Baca Isi Zip Tanpa Ekstraksi",
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "zip_path": str(ZIP_PATH),
            "zip_readable": True,
            "total_items": total_items,
            "total_files": total_files,
            "total_dirs": total_dirs,
            "total_uncompressed_mb": total_uncompressed_mb,
            "total_compressed_mb": total_compressed_mb,
            "top_level_counter": dict(top_level_counter),
            "extension_counter": dict(extension_counter),
            "sample_entries_first_50": sample_entries
        }

        summary_path = OUTPUT_DIR / "sel2_zip_inventory_summary.json"
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(sel2_summary, f, indent=4, ensure_ascii=False)

        # Simpan manifest lengkap isi zip
        manifest_path = OUTPUT_DIR / "sel2_zip_manifest.csv"
        with open(manifest_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "filename",
                "is_dir",
                "file_size_bytes",
                "compress_size_bytes",
                "extension"
            ])
            for info in infos:
                writer.writerow([
                    info.filename,
                    info.is_dir(),
                    info.file_size,
                    info.compress_size,
                    Path(info.filename).suffix.lower() if not info.is_dir() else ""
                ])

        print(f"\nRingkasan Sel 2 disimpan ke: {summary_path}")
        print(f"Manifest lengkap disimpan ke: {manifest_path}")

except zipfile.BadZipFile:
    print("Zip bisa dibaca        : False")
    print("ERROR: File bukan ZIP valid atau ZIP rusak.")

In [ ]:
# Sel 3 — Ekstraksi Zip Dataset ke /content

from pathlib import Path
import zipfile
import json
import shutil
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
RAW_DIR = PROJECT_DIR / 'data_raw'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
ZIP_PATH = RAW_DIR / 'TEM virus dataset.zip'

# Lokasi ekstraksi sementara di runtime Colab
EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 3 ===")
print(f"Zip path          : {ZIP_PATH}")
print(f"Extract dir       : {EXTRACT_DIR}")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"File zip tidak ditemukan: {ZIP_PATH}")

# Cek ruang kosong /content
disk_total, disk_used, disk_free = shutil.disk_usage('/content')
disk_free_gb = disk_free / (1024**3)

print(f"Ruang kosong /content: {disk_free_gb:.2f} GB")

# Cek apakah sudah pernah diekstrak di runtime ini
existing_items = list(EXTRACT_DIR.rglob("*"))
already_extracted = len(existing_items) > 1000

if already_extracted:
    print("Status ekstraksi  : Folder ekstraksi sudah berisi banyak file. Ekstraksi dilewati.")
else:
    print("Status ekstraksi  : Mulai ekstraksi zip ke /content ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("Status ekstraksi  : Selesai")

# Cek hasil ekstraksi
all_extracted_items = list(EXTRACT_DIR.rglob("*"))
extracted_files = [p for p in all_extracted_items if p.is_file()]
extracted_dirs = [p for p in all_extracted_items if p.is_dir()]

top_level_after_extract = [p.name for p in EXTRACT_DIR.iterdir()]

print(f"Total item hasil ekstraksi  : {len(all_extracted_items)}")
print(f"Total file hasil ekstraksi  : {len(extracted_files)}")
print(f"Total folder hasil ekstraksi: {len(extracted_dirs)}")

print("\nTop-level folder/file di extract dir:")
for item in top_level_after_extract:
    print(f"- {item}")

# Simpan status Sel 3
sel3_result = {
    "step": "Sel 3 — Ekstraksi Zip Dataset ke /content",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "zip_path": str(ZIP_PATH),
    "extract_dir": str(EXTRACT_DIR),
    "disk_free_content_gb": disk_free_gb,
    "already_extracted_before_cell": already_extracted,
    "total_items_extracted": len(all_extracted_items),
    "total_files_extracted": len(extracted_files),
    "total_dirs_extracted": len(extracted_dirs),
    "top_level_after_extract": top_level_after_extract
}

sel3_output_path = OUTPUT_DIR / "sel3_extract_status.json"

with open(sel3_output_path, "w", encoding="utf-8") as f:
    json.dump(sel3_result, f, indent=4, ensure_ascii=False)

print(f"\nHasil Sel 3 disimpan ke: {sel3_output_path}")

In [ ]:
# Sel 4 — Pemetaan Struktur Folder Setelah Ekstraksi

from pathlib import Path
import json
import csv
from collections import Counter, defaultdict
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 4 ===")
print(f"Extract dir  : {EXTRACT_DIR}")
print(f"Dataset root : {DATASET_ROOT}")
print(f"Dataset root ada: {DATASET_ROOT.exists()}")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Folder dataset root tidak ditemukan: {DATASET_ROOT}")

# Ambil semua folder dan file
all_dirs = [p for p in DATASET_ROOT.rglob("*") if p.is_dir()]
all_files = [p for p in DATASET_ROOT.rglob("*") if p.is_file()]

print(f"\nTotal folder di dataset root: {len(all_dirs)}")
print(f"Total file di dataset root  : {len(all_files)}")

# Immediate children di dataset root
immediate_children = sorted(DATASET_ROOT.iterdir(), key=lambda x: x.name.lower())

print("\nIsi langsung di dalam DATASET_ROOT:")
for p in immediate_children:
    tipe = "folder" if p.is_dir() else "file"
    print(f"- [{tipe}] {p.name}")

# Hitung struktur berdasarkan level relatif
level_counter = Counter()
folder_rows = []

for d in all_dirs:
    rel_parts = d.relative_to(DATASET_ROOT).parts
    level = len(rel_parts)
    level_counter[level] += 1
    folder_rows.append({
        "relative_path": str(d.relative_to(DATASET_ROOT)),
        "level": level,
        "folder_name": d.name
    })

print("\nJumlah folder berdasarkan level:")
for level, count in sorted(level_counter.items()):
    print(f"- Level {level}: {count} folder")

# Tampilkan contoh folder per level, maksimal 20 per level
folders_by_level = defaultdict(list)
for row in folder_rows:
    folders_by_level[row["level"]].append(row["relative_path"])

print("\nContoh struktur folder per level:")
for level in sorted(folders_by_level.keys()):
    print(f"\nLevel {level}:")
    for path in sorted(folders_by_level[level])[:20]:
        print(f"- {path}")
    if len(folders_by_level[level]) > 20:
        print(f"... dan {len(folders_by_level[level]) - 20} folder lainnya")

# Deteksi folder split resmi berdasarkan nama folder
split_keywords = ["train", "training", "val", "valid", "validation", "test", "testing"]
detected_split_dirs = []

for d in all_dirs:
    name_lower = d.name.lower()
    if any(keyword == name_lower for keyword in split_keywords):
        detected_split_dirs.append(str(d.relative_to(DATASET_ROOT)))

print("\nFolder split yang terdeteksi berdasarkan nama:")
if detected_split_dirs:
    for d in sorted(detected_split_dirs):
        print(f"- {d}")
else:
    print("- Tidak ditemukan folder split dengan nama umum train/validation/test")

# Simpan CSV struktur folder
folder_structure_csv = OUTPUT_DIR / "sel4_folder_structure.csv"
with open(folder_structure_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["relative_path", "level", "folder_name"])
    writer.writeheader()
    writer.writerows(folder_rows)

# Simpan ringkasan JSON
sel4_summary = {
    "step": "Sel 4 — Pemetaan Struktur Folder Setelah Ekstraksi",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "extract_dir": str(EXTRACT_DIR),
    "dataset_root": str(DATASET_ROOT),
    "dataset_root_exists": DATASET_ROOT.exists(),
    "total_dirs": len(all_dirs),
    "total_files": len(all_files),
    "immediate_children": [
        {
            "name": p.name,
            "type": "folder" if p.is_dir() else "file"
        }
        for p in immediate_children
    ],
    "level_counter": dict(level_counter),
    "detected_split_dirs": detected_split_dirs
}

sel4_summary_json = OUTPUT_DIR / "sel4_folder_structure_summary.json"
with open(sel4_summary_json, "w", encoding="utf-8") as f:
    json.dump(sel4_summary, f, indent=4, ensure_ascii=False)

print(f"\nStruktur folder CSV disimpan ke: {folder_structure_csv}")
print(f"Ringkasan Sel 4 disimpan ke: {sel4_summary_json}")

In [ ]:
# Sel 5 — Hitung File Gambar per Split dan Kelas

from pathlib import Path
import json
import csv
import pandas as pd
from collections import defaultdict
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 5 ===")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Dataset root ada: {DATASET_ROOT.exists()}")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Folder dataset root tidak ditemukan: {DATASET_ROOT}")

# Ambil semua file .tif
tif_files = sorted(DATASET_ROOT.rglob("*.tif"))

print(f"\nTotal file .tif terdeteksi: {len(tif_files)}")

rows = []

for p in tif_files:
    rel = p.relative_to(DATASET_ROOT)
    parts = rel.parts

    # Format umum yang diharapkan:
    # context_virus_1nm_256x256/train/Adenovirus/xxx.tif
    # context_virus_1nm_256x256/train/_EXCLUDED/Dengue/xxx.tif
    # context_virus_1nm_256x256/test/Adenovirus/crop_outlines/xxx...

    dataset_version = parts[0] if len(parts) > 0 else None
    split = parts[1] if len(parts) > 1 else None

    class_status = "unknown"
    class_name = None
    in_crop_outlines = "crop_outlines" in parts

    if len(parts) >= 3:
        if parts[2] == "_EXCLUDED":
            class_status = "excluded"
            class_name = parts[3] if len(parts) >= 4 else None
        else:
            class_status = "main"
            class_name = parts[2]

    rows.append({
        "relative_path": str(rel),
        "dataset_version": dataset_version,
        "split": split,
        "class_status": class_status,
        "class_name": class_name,
        "in_crop_outlines": in_crop_outlines,
        "filename": p.name,
        "stem": p.stem,
        "suffix": p.suffix.lower(),
        "file_size_bytes": p.stat().st_size
    })

df = pd.DataFrame(rows)

# Simpan inventory lengkap .tif
tif_inventory_path = OUTPUT_DIR / "sel5_tif_inventory.csv"
df.to_csv(tif_inventory_path, index=False)

# Ringkasan hitung per versi, split, status, kelas
summary_df = (
    df.groupby(["dataset_version", "split", "class_status", "class_name", "in_crop_outlines"])
      .size()
      .reset_index(name="tif_count")
      .sort_values(["dataset_version", "split", "class_status", "class_name", "in_crop_outlines"])
)

summary_path = OUTPUT_DIR / "sel5_tif_count_by_split_class.csv"
summary_df.to_csv(summary_path, index=False)

print("\nRingkasan jumlah .tif per dataset_version, split, dan class_status:")
version_split_status = (
    df.groupby(["dataset_version", "split", "class_status"])
      .agg(
          tif_count=("relative_path", "count"),
          class_count=("class_name", "nunique")
      )
      .reset_index()
      .sort_values(["dataset_version", "split", "class_status"])
)

print(version_split_status.to_string(index=False))

print("\nDaftar kelas MAIN per dataset_version dan split:")
main_df = df[df["class_status"] == "main"]

for (version, split), group in main_df.groupby(["dataset_version", "split"]):
    classes = sorted(group["class_name"].dropna().unique().tolist())
    print(f"\n[{version} / {split}]")
    print(f"Jumlah kelas main: {len(classes)}")
    print(classes)

print("\nDaftar kelas EXCLUDED per dataset_version dan split:")
excluded_df = df[df["class_status"] == "excluded"]

if len(excluded_df) == 0:
    print("- Tidak ada kelas excluded terdeteksi pada file .tif")
else:
    for (version, split), group in excluded_df.groupby(["dataset_version", "split"]):
        classes = sorted(group["class_name"].dropna().unique().tolist())
        print(f"\n[{version} / {split}]")
        print(f"Jumlah kelas excluded: {len(classes)}")
        print(classes)

# Gabungan kelas main dan excluded per versi
print("\nGabungan kelas MAIN + EXCLUDED per dataset_version:")
for version, group in df.groupby("dataset_version"):
    main_classes = sorted(group[group["class_status"] == "main"]["class_name"].dropna().unique().tolist())
    excluded_classes = sorted(group[group["class_status"] == "excluded"]["class_name"].dropna().unique().tolist())
    all_classes = sorted(set(main_classes + excluded_classes))

    print(f"\n[{version}]")
    print(f"Jumlah kelas main     : {len(main_classes)}")
    print(f"Jumlah kelas excluded : {len(excluded_classes)}")
    print(f"Jumlah kelas gabungan : {len(all_classes)}")
    print(f"Kelas gabungan        : {all_classes}")

# Simpan ringkasan JSON
summary_json = {
    "step": "Sel 5 — Hitung File Gambar per Split dan Kelas",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_root": str(DATASET_ROOT),
    "total_tif_files": int(len(tif_files)),
    "version_split_status_summary": version_split_status.to_dict(orient="records"),
    "main_classes_by_version_split": {},
    "excluded_classes_by_version_split": {},
    "combined_classes_by_version": {}
}

for (version, split), group in main_df.groupby(["dataset_version", "split"]):
    summary_json["main_classes_by_version_split"][f"{version}/{split}"] = sorted(
        group["class_name"].dropna().unique().tolist()
    )

for (version, split), group in excluded_df.groupby(["dataset_version", "split"]):
    summary_json["excluded_classes_by_version_split"][f"{version}/{split}"] = sorted(
        group["class_name"].dropna().unique().tolist()
    )

for version, group in df.groupby("dataset_version"):
    main_classes = sorted(group[group["class_status"] == "main"]["class_name"].dropna().unique().tolist())
    excluded_classes = sorted(group[group["class_status"] == "excluded"]["class_name"].dropna().unique().tolist())
    all_classes = sorted(set(main_classes + excluded_classes))

    summary_json["combined_classes_by_version"][version] = {
        "main_class_count": len(main_classes),
        "excluded_class_count": len(excluded_classes),
        "combined_class_count": len(all_classes),
        "main_classes": main_classes,
        "excluded_classes": excluded_classes,
        "combined_classes": all_classes
    }

summary_json_path = OUTPUT_DIR / "sel5_tif_count_summary.json"

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=4, ensure_ascii=False)

print(f"\nInventory .tif lengkap disimpan ke: {tif_inventory_path}")
print(f"Ringkasan count .tif disimpan ke: {summary_path}")
print(f"Ringkasan JSON Sel 5 disimpan ke: {summary_json_path}")

In [ ]:
# Sel 6 — Cek File Metadata, CSV, TXT, dan ZIP Tambahan

from pathlib import Path
import json
import csv
import pandas as pd
from collections import Counter
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 6 ===")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Dataset root ada: {DATASET_ROOT.exists()}")

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Folder dataset root tidak ditemukan: {DATASET_ROOT}")

# Ambil semua file non-tif
all_files = [p for p in DATASET_ROOT.rglob("*") if p.is_file()]
non_tif_files = [p for p in all_files if p.suffix.lower() != ".tif"]

print(f"\nTotal semua file       : {len(all_files)}")
print(f"Total file non-.tif    : {len(non_tif_files)}")

# Ringkasan ekstensi non-tif
ext_counter = Counter([p.suffix.lower() if p.suffix else "[no_extension]" for p in non_tif_files])

print("\nEkstensi non-.tif terdeteksi:")
for ext, count in ext_counter.most_common():
    print(f"- {ext}: {count} file")

# Buat inventory non-tif
rows = []

for p in non_tif_files:
    rel = p.relative_to(DATASET_ROOT)
    parts = rel.parts

    dataset_version = parts[0] if len(parts) > 0 else None
    split = parts[1] if len(parts) > 1 else None

    class_status = "unknown"
    class_name = None

    if len(parts) >= 3:
        if parts[2] == "_EXCLUDED":
            class_status = "excluded"
            class_name = parts[3] if len(parts) >= 4 else None
        else:
            class_status = "main"
            class_name = parts[2]

    name_lower = p.name.lower()
    path_lower = str(rel).lower()

    possible_metadata_keyword = any(
        keyword in path_lower
        for keyword in [
            "meta", "metadata", "label", "labels", "annotation", "annotations",
            "source", "sample", "grid", "session", "code", "readme", "info"
        ]
    )

    rows.append({
        "relative_path": str(rel),
        "dataset_version": dataset_version,
        "split": split,
        "class_status": class_status,
        "class_name": class_name,
        "filename": p.name,
        "suffix": p.suffix.lower() if p.suffix else "[no_extension]",
        "file_size_bytes": p.stat().st_size,
        "possible_metadata_keyword": possible_metadata_keyword
    })

non_tif_df = pd.DataFrame(rows)

inventory_path = OUTPUT_DIR / "sel6_non_tif_inventory.csv"
non_tif_df.to_csv(inventory_path, index=False)

print("\nRingkasan non-.tif berdasarkan ekstensi dan dataset_version:")
if len(non_tif_df) > 0:
    summary_ext = (
        non_tif_df.groupby(["dataset_version", "suffix"])
        .size()
        .reset_index(name="file_count")
        .sort_values(["dataset_version", "suffix"])
    )
    print(summary_ext.to_string(index=False))
else:
    print("- Tidak ada file non-.tif")

# Tampilkan file yang punya keyword metadata/code/label/source/sample/grid/session
keyword_df = non_tif_df[non_tif_df["possible_metadata_keyword"] == True].copy()

print("\nFile yang terdeteksi mengandung keyword metadata/label/source/sample/grid/session/code/readme/info:")
if len(keyword_df) == 0:
    print("- Tidak ditemukan berdasarkan keyword sederhana")
else:
    for path in keyword_df["relative_path"].head(50).tolist():
        print(f"- {path}")
    if len(keyword_df) > 50:
        print(f"... dan {len(keyword_df) - 50} file lainnya")

# Tampilkan contoh file non-tif
print("\nContoh 30 file non-.tif:")
for path in non_tif_df["relative_path"].head(30).tolist():
    print(f"- {path}")

# Preview beberapa CSV
csv_files = [p for p in non_tif_files if p.suffix.lower() == ".csv"]
txt_files = [p for p in non_tif_files if p.suffix.lower() == ".txt"]
zip_files = [p for p in non_tif_files if p.suffix.lower() == ".zip"]

print("\nFile .zip tambahan:")
if len(zip_files) == 0:
    print("- Tidak ada file .zip tambahan")
else:
    for p in zip_files:
        print(f"- {p.relative_to(DATASET_ROOT)} | size: {p.stat().st_size / (1024*1024):.2f} MB")

print("\nPreview contoh file CSV:")
csv_previews = []

for p in csv_files[:5]:
    rel = str(p.relative_to(DATASET_ROOT))
    print(f"\n--- CSV: {rel} ---")
    try:
        temp_df = pd.read_csv(p)
        print(f"Shape: {temp_df.shape}")
        print(f"Columns: {list(temp_df.columns)}")
        print(temp_df.head(3).to_string(index=False))

        csv_previews.append({
            "relative_path": rel,
            "readable": True,
            "shape": list(temp_df.shape),
            "columns": list(temp_df.columns),
            "head_records": temp_df.head(3).astype(str).to_dict(orient="records")
        })
    except Exception as e:
        print(f"Gagal membaca CSV: {repr(e)}")
        csv_previews.append({
            "relative_path": rel,
            "readable": False,
            "error": repr(e)
        })

print("\nPreview contoh file TXT:")
txt_previews = []

for p in txt_files[:5]:
    rel = str(p.relative_to(DATASET_ROOT))
    print(f"\n--- TXT: {rel} ---")
    try:
        with open(p, "r", encoding="utf-8", errors="replace") as f:
            lines = [line.rstrip("\n") for line in f.readlines()[:10]]

        for line in lines:
            print(line)

        txt_previews.append({
            "relative_path": rel,
            "readable": True,
            "first_10_lines": lines
        })
    except Exception as e:
        print(f"Gagal membaca TXT: {repr(e)}")
        txt_previews.append({
            "relative_path": rel,
            "readable": False,
            "error": repr(e)
        })

# Simpan ringkasan JSON
sel6_summary = {
    "step": "Sel 6 — Cek File Metadata, CSV, TXT, dan ZIP Tambahan",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_root": str(DATASET_ROOT),
    "total_all_files": len(all_files),
    "total_non_tif_files": len(non_tif_files),
    "extension_counter_non_tif": dict(ext_counter),
    "keyword_detected_files_count": int(len(keyword_df)),
    "keyword_detected_files_sample": keyword_df["relative_path"].head(100).tolist() if len(keyword_df) > 0 else [],
    "zip_files": [
        {
            "relative_path": str(p.relative_to(DATASET_ROOT)),
            "file_size_mb": p.stat().st_size / (1024 * 1024)
        }
        for p in zip_files
    ],
    "csv_preview_count": len(csv_previews),
    "csv_previews": csv_previews,
    "txt_preview_count": len(txt_previews),
    "txt_previews": txt_previews
}

summary_json_path = OUTPUT_DIR / "sel6_metadata_file_check_summary.json"

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(sel6_summary, f, indent=4, ensure_ascii=False)

print(f"\nInventory non-.tif disimpan ke: {inventory_path}")
print(f"Ringkasan Sel 6 disimpan ke: {summary_json_path}")

In [ ]:
# Sel 7 — Analisis Pola Nama File (Grouping & Potensi Leakage)

from pathlib import Path
import pandas as pd
import json
from collections import Counter
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 7 ===")

# Ambil semua tif dari versi classification (1nm_256x256 saja)
tif_files = list((DATASET_ROOT / "context_virus_1nm_256x256").rglob("*.tif"))

print(f"Total tif (1nm_256x256): {len(tif_files)}")

rows = []

for p in tif_files:
    rel = p.relative_to(DATASET_ROOT)
    parts = rel.parts

    split = parts[1] if len(parts) > 1 else None

    if parts[2] == "_EXCLUDED":
        class_name = parts[3]
    else:
        class_name = parts[2]

    filename = p.name
    stem = p.stem

    # coba split pola seperti "10_1"
    if "_" in stem:
        group_id = stem.split("_")[0]
    else:
        group_id = stem

    rows.append({
        "relative_path": str(rel),
        "split": split,
        "class_name": class_name,
        "filename": filename,
        "stem": stem,
        "group_id": group_id
    })

df = pd.DataFrame(rows)

# Hitung berapa banyak group_id unik
total_groups = df["group_id"].nunique()
print(f"\nJumlah group_id unik: {total_groups}")

# Cek distribusi group_id per split
group_split = (
    df.groupby(["group_id", "split"])
    .size()
    .reset_index(name="count")
)

# cek apakah ada group muncul di lebih dari 1 split
group_counts = group_split.groupby("group_id")["split"].nunique().reset_index()
leakage_groups = group_counts[group_counts["split"] > 1]

print(f"\nJumlah group muncul di lebih dari 1 split (potensi leakage): {len(leakage_groups)}")

if len(leakage_groups) > 0:
    print("\nContoh group yang muncul di multiple split:")
    print(leakage_groups.head(20).to_string(index=False))

# distribusi jumlah file per group
group_size = df.groupby("group_id").size().reset_index(name="file_count")

print("\nDistribusi ukuran group:")
print(group_size["file_count"].describe())

# simpan hasil
output_path = OUTPUT_DIR / "sel7_grouping_analysis.csv"
df.to_csv(output_path, index=False)

summary_json = {
    "step": "Sel 7 — Analisis Pola Nama File",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_files": len(df),
    "total_groups": int(total_groups),
    "groups_with_multi_split": int(len(leakage_groups)),
}

summary_path = OUTPUT_DIR / "sel7_grouping_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=4)

print(f"\nHasil grouping disimpan ke: {output_path}")
print(f"Ringkasan disimpan ke: {summary_path}")

In [ ]:
# Sel 8 — Cek Grouping Lebih Presisi antar Split Resmi

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'
DATA_1NM = DATASET_ROOT / "context_virus_1nm_256x256"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 8 ===")
print(f"Folder 1nm: {DATA_1NM}")
print(f"Folder ada: {DATA_1NM.exists()}")

if not DATA_1NM.exists():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_1NM}")

tif_files = sorted(DATA_1NM.rglob("*.tif"))

rows = []

for p in tif_files:
    rel = p.relative_to(DATA_1NM)
    parts = rel.parts

    split = parts[0] if len(parts) > 0 else None

    class_status = "unknown"
    class_name = None

    if len(parts) >= 2:
        if parts[1] == "_EXCLUDED":
            class_status = "excluded"
            class_name = parts[2] if len(parts) >= 3 else None
        else:
            class_status = "main"
            class_name = parts[1]

    in_crop_outlines = "crop_outlines" in parts

    filename = p.name
    stem = p.stem

    # Base ID lebih hati-hati:
    # contoh 10_1.tif -> base_id = 10
    # contoh 8291.tif -> base_id = 8291
    if "_" in stem:
        base_id = stem.split("_")[0]
    else:
        base_id = stem

    # Group key yang lebih aman: status + kelas + base_id
    group_key = f"{class_status}__{class_name}__{base_id}"

    rows.append({
        "relative_path": str(rel),
        "split": split,
        "class_status": class_status,
        "class_name": class_name,
        "in_crop_outlines": in_crop_outlines,
        "filename": filename,
        "stem": stem,
        "base_id": base_id,
        "group_key": group_key,
        "file_size_bytes": p.stat().st_size
    })

df = pd.DataFrame(rows)

print(f"\nTotal file .tif di 1nm_256x256: {len(df)}")

# Ringkasan semua file
summary_all = (
    df.groupby(["split", "class_status", "in_crop_outlines"])
      .size()
      .reset_index(name="file_count")
      .sort_values(["split", "class_status", "in_crop_outlines"])
)

print("\nRingkasan semua file berdasarkan split, status kelas, dan crop_outlines:")
print(summary_all.to_string(index=False))

# Fokus split resmi saja, tidak termasuk augmented_train
official_splits = ["train", "validation", "test"]

official_df = df[
    (df["split"].isin(official_splits)) &
    (df["in_crop_outlines"] == False)
].copy()

print(f"\nTotal file pada split resmi train/validation/test tanpa crop_outlines: {len(official_df)}")

official_summary = (
    official_df.groupby(["split", "class_status"])
      .agg(
          file_count=("relative_path", "count"),
          class_count=("class_name", "nunique"),
          group_count=("group_key", "nunique")
      )
      .reset_index()
      .sort_values(["split", "class_status"])
)

print("\nRingkasan split resmi tanpa crop_outlines:")
print(official_summary.to_string(index=False))

# Cek group_key muncul di lebih dari satu split resmi
group_split = (
    official_df.groupby(["group_key", "split"])
      .size()
      .reset_index(name="file_count")
)

group_multi_split = (
    group_split.groupby("group_key")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("file_count", "sum")
      )
      .reset_index()
)

potential_leak_groups = group_multi_split[group_multi_split["split_count"] > 1].copy()

print(f"\nJumlah group_key muncul di lebih dari 1 split resmi: {len(potential_leak_groups)}")

if len(potential_leak_groups) > 0:
    print("\nContoh group_key yang muncul di multiple split resmi:")
    print(potential_leak_groups.head(30).to_string(index=False))
else:
    print("Tidak ada group_key yang muncul di lebih dari 1 split resmi berdasarkan parsing ini.")

# Cek overlap filename exact dalam kelas yang sama antar split resmi
exact_key_df = official_df.copy()
exact_key_df["exact_key"] = (
    exact_key_df["class_status"] + "__" +
    exact_key_df["class_name"] + "__" +
    exact_key_df["filename"]
)

exact_split = (
    exact_key_df.groupby(["exact_key", "split"])
      .size()
      .reset_index(name="file_count")
)

exact_multi = (
    exact_split.groupby("exact_key")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("file_count", "sum")
      )
      .reset_index()
)

exact_overlap = exact_multi[exact_multi["split_count"] > 1].copy()

print(f"\nJumlah exact filename dalam kelas yang sama muncul di >1 split resmi: {len(exact_overlap)}")

if len(exact_overlap) > 0:
    print("\nContoh exact overlap:")
    print(exact_overlap.head(30).to_string(index=False))
else:
    print("Tidak ada exact filename overlap dalam kelas yang sama antar split resmi.")

# Analisis augmented_train terpisah
aug_df = df[
    (df["split"] == "augmented_train") &
    (df["in_crop_outlines"] == False)
].copy()

train_df = official_df[official_df["split"] == "train"].copy()

aug_group_set = set(aug_df["group_key"])
train_group_set = set(train_df["group_key"])

aug_overlap_train = aug_group_set.intersection(train_group_set)

print(f"\nJumlah group_key augmented_train: {len(aug_group_set)}")
print(f"Jumlah group_key train resmi    : {len(train_group_set)}")
print(f"Overlap group_key augmented_train dengan train resmi: {len(aug_overlap_train)}")

# Simpan hasil lengkap
parsed_path = OUTPUT_DIR / "sel8_1nm_precise_grouping_inventory.csv"
df.to_csv(parsed_path, index=False)

official_path = OUTPUT_DIR / "sel8_official_split_no_crop_inventory.csv"
official_df.to_csv(official_path, index=False)

leak_path = OUTPUT_DIR / "sel8_potential_group_overlap_official_splits.csv"
potential_leak_groups.to_csv(leak_path, index=False)

exact_overlap_path = OUTPUT_DIR / "sel8_exact_filename_overlap_official_splits.csv"
exact_overlap.to_csv(exact_overlap_path, index=False)

summary_json = {
    "step": "Sel 8 — Cek Grouping Lebih Presisi antar Split Resmi",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_1nm_tif_files": int(len(df)),
    "total_official_no_crop_files": int(len(official_df)),
    "official_summary": official_summary.to_dict(orient="records"),
    "potential_group_overlap_official_split_count": int(len(potential_leak_groups)),
    "exact_filename_overlap_official_split_count": int(len(exact_overlap)),
    "augmented_train_group_count": int(len(aug_group_set)),
    "train_group_count": int(len(train_group_set)),
    "augmented_train_overlap_train_group_count": int(len(aug_overlap_train)),
    "output_files": {
        "parsed_inventory": str(parsed_path),
        "official_split_inventory": str(official_path),
        "potential_group_overlap": str(leak_path),
        "exact_filename_overlap": str(exact_overlap_path)
    }
}

summary_path = OUTPUT_DIR / "sel8_precise_grouping_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=4, ensure_ascii=False)

print(f"\nInventory parsing lengkap disimpan ke: {parsed_path}")
print(f"Inventory split resmi tanpa crop_outlines disimpan ke: {official_path}")
print(f"Potensi overlap group resmi disimpan ke: {leak_path}")
print(f"Exact filename overlap resmi disimpan ke: {exact_overlap_path}")
print(f"Ringkasan Sel 8 disimpan ke: {summary_path}")

In [ ]:
# Sel 9 — Cek Exact Duplicate Gambar antar Split Resmi dengan Hash File

from pathlib import Path
import pandas as pd
import hashlib
import json
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'
DATA_1NM = DATASET_ROOT / "context_virus_1nm_256x256"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 9 ===")
print(f"Folder 1nm: {DATA_1NM}")
print(f"Folder ada: {DATA_1NM.exists()}")

if not DATA_1NM.exists():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_1NM}")

official_splits = ["train", "validation", "test"]

def md5_hash_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

rows = []

tif_files = sorted(DATA_1NM.rglob("*.tif"))

for p in tif_files:
    rel = p.relative_to(DATA_1NM)
    parts = rel.parts

    split = parts[0] if len(parts) > 0 else None

    # Hanya split resmi
    if split not in official_splits:
        continue

    # Skip crop_outlines
    if "crop_outlines" in parts:
        continue

    class_status = "unknown"
    class_name = None

    if len(parts) >= 2:
        if parts[1] == "_EXCLUDED":
            class_status = "excluded"
            class_name = parts[2] if len(parts) >= 3 else None
        else:
            class_status = "main"
            class_name = parts[1]

    stem = p.stem
    base_id = stem.split("_")[0] if "_" in stem else stem

    rows.append({
        "path_obj": p,
        "relative_path": str(rel),
        "split": split,
        "class_status": class_status,
        "class_name": class_name,
        "filename": p.name,
        "stem": stem,
        "base_id": base_id,
        "file_size_bytes": p.stat().st_size
    })

df = pd.DataFrame(rows)

print(f"\nTotal file resmi yang dicek hash: {len(df)}")
print("Mulai menghitung hash file. Ini mungkin butuh beberapa menit...")

hashes = []
for i, p in enumerate(df["path_obj"].tolist(), start=1):
    hashes.append(md5_hash_file(p))
    if i % 1000 == 0:
        print(f"Progress hash: {i}/{len(df)} file")

df["md5"] = hashes
df = df.drop(columns=["path_obj"])

print("Hash selesai.")

# Cek hash yang muncul di lebih dari satu split
hash_split = (
    df.groupby(["md5", "split"])
      .size()
      .reset_index(name="file_count")
)

hash_multi_split = (
    hash_split.groupby("md5")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("file_count", "sum")
      )
      .reset_index()
)

duplicate_hashes_multi_split = hash_multi_split[hash_multi_split["split_count"] > 1].copy()

print(f"\nJumlah hash gambar yang muncul di >1 split resmi: {len(duplicate_hashes_multi_split)}")

if len(duplicate_hashes_multi_split) > 0:
    print("\nContoh hash duplicate antar split:")
    print(duplicate_hashes_multi_split.head(20).to_string(index=False))
else:
    print("Tidak ditemukan exact duplicate berdasarkan isi file antar split resmi.")

# Detail file untuk hash duplicate
if len(duplicate_hashes_multi_split) > 0:
    duplicate_detail = df[df["md5"].isin(duplicate_hashes_multi_split["md5"])].copy()
    duplicate_detail = duplicate_detail.sort_values(["md5", "split", "class_status", "class_name", "filename"])
else:
    duplicate_detail = pd.DataFrame(columns=df.columns)

# Ringkasan duplicate berdasarkan main/excluded
if len(duplicate_detail) > 0:
    dup_summary = (
        duplicate_detail.groupby(["class_status", "class_name", "md5"])
          .agg(
              split_count=("split", "nunique"),
              splits=("split", lambda x: sorted(list(set(x)))),
              file_count=("relative_path", "count")
          )
          .reset_index()
          .sort_values(["class_status", "class_name", "split_count", "file_count"], ascending=[True, True, False, False])
    )
else:
    dup_summary = pd.DataFrame(columns=["class_status", "class_name", "md5", "split_count", "splits", "file_count"])

print("\nRingkasan duplicate berdasarkan class_status:")
if len(duplicate_detail) > 0:
    print(
        duplicate_detail.groupby(["class_status"])
        .agg(
            duplicate_files=("relative_path", "count"),
            duplicate_hashes=("md5", "nunique"),
            class_count=("class_name", "nunique")
        )
        .reset_index()
        .to_string(index=False)
    )
else:
    print("- Tidak ada duplicate detail karena tidak ada hash yang overlap antar split.")

print("\nContoh detail file duplicate:")
if len(duplicate_detail) > 0:
    print(duplicate_detail.head(40)[["split", "class_status", "class_name", "filename", "md5", "relative_path"]].to_string(index=False))
else:
    print("- Tidak ada detail duplicate.")

# Simpan output
hash_inventory_path = OUTPUT_DIR / "sel9_official_split_file_hash_inventory.csv"
df.to_csv(hash_inventory_path, index=False)

duplicate_hash_path = OUTPUT_DIR / "sel9_duplicate_hashes_across_official_splits.csv"
duplicate_hashes_multi_split.to_csv(duplicate_hash_path, index=False)

duplicate_detail_path = OUTPUT_DIR / "sel9_duplicate_file_details_across_official_splits.csv"
duplicate_detail.to_csv(duplicate_detail_path, index=False)

duplicate_summary_path = OUTPUT_DIR / "sel9_duplicate_summary_by_class.csv"
dup_summary.to_csv(duplicate_summary_path, index=False)

summary_json = {
    "step": "Sel 9 — Cek Exact Duplicate Gambar antar Split Resmi dengan Hash File",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_files_checked": int(len(df)),
    "unique_hash_count": int(df["md5"].nunique()),
    "duplicate_hashes_across_official_splits": int(len(duplicate_hashes_multi_split)),
    "duplicate_file_detail_count": int(len(duplicate_detail)),
    "output_files": {
        "hash_inventory": str(hash_inventory_path),
        "duplicate_hashes": str(duplicate_hash_path),
        "duplicate_file_details": str(duplicate_detail_path),
        "duplicate_summary_by_class": str(duplicate_summary_path)
    }
}

summary_json_path = OUTPUT_DIR / "sel9_exact_duplicate_hash_summary.json"

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=4, ensure_ascii=False)

print(f"\nInventory hash disimpan ke: {hash_inventory_path}")
print(f"Daftar hash duplicate antar split disimpan ke: {duplicate_hash_path}")
print(f"Detail file duplicate disimpan ke: {duplicate_detail_path}")
print(f"Ringkasan duplicate per kelas disimpan ke: {duplicate_summary_path}")
print(f"Ringkasan Sel 9 disimpan ke: {summary_json_path}")

In [ ]:
# Sel 10 — Cek Isi dataset code.zip

from pathlib import Path
import zipfile
import json
import csv
from collections import Counter
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'

CODE_ZIP_PATH = DATASET_ROOT / "context_virus_1nm_256x256" / "dataset code.zip"
CODE_EXTRACT_DIR = Path('/content/tem_virus_dataset_code')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CODE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 10 ===")
print(f"Code zip path : {CODE_ZIP_PATH}")
print(f"Code zip ada  : {CODE_ZIP_PATH.exists()}")

if not CODE_ZIP_PATH.exists():
    raise FileNotFoundError(f"dataset code.zip tidak ditemukan: {CODE_ZIP_PATH}")

try:
    with zipfile.ZipFile(CODE_ZIP_PATH, 'r') as zf:
        infos = zf.infolist()

        total_items = len(infos)
        total_files = sum(1 for info in infos if not info.is_dir())
        total_dirs = sum(1 for info in infos if info.is_dir())

        extensions = []
        for info in infos:
            if not info.is_dir():
                suffix = Path(info.filename).suffix.lower()
                extensions.append(suffix if suffix else "[no_extension]")

        ext_counter = Counter(extensions)

        print(f"\nCode zip bisa dibaca : True")
        print(f"Total item           : {total_items}")
        print(f"Total file           : {total_files}")
        print(f"Total folder         : {total_dirs}")

        print("\nEkstensi file dalam code zip:")
        for ext, count in ext_counter.most_common():
            print(f"- {ext}: {count}")

        print("\nDaftar isi code zip:")
        for info in infos[:100]:
            tipe = "folder" if info.is_dir() else "file"
            print(f"- [{tipe}] {info.filename}")

        if total_items > 100:
            print(f"... dan {total_items - 100} item lainnya")

        # Ekstrak code zip
        zf.extractall(CODE_EXTRACT_DIR)

    print(f"\nCode zip diekstrak ke: {CODE_EXTRACT_DIR}")

    # Inventaris file hasil ekstraksi
    extracted_files = sorted([p for p in CODE_EXTRACT_DIR.rglob("*") if p.is_file()])

    rows = []
    for p in extracted_files:
        rel = p.relative_to(CODE_EXTRACT_DIR)
        rows.append({
            "relative_path": str(rel),
            "suffix": p.suffix.lower() if p.suffix else "[no_extension]",
            "file_size_bytes": p.stat().st_size
        })

    manifest_path = OUTPUT_DIR / "sel10_dataset_code_manifest.csv"
    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["relative_path", "suffix", "file_size_bytes"])
        writer.writeheader()
        writer.writerows(rows)

    # Preview file teks/kode yang ukurannya masuk akal
    text_like_exts = {
        ".py", ".ipynb", ".txt", ".md", ".csv", ".json", ".yaml", ".yml",
        ".sh", ".bat", ".r", ".m"
    }

    preview_files = [
        p for p in extracted_files
        if p.suffix.lower() in text_like_exts and p.stat().st_size < 300_000
    ]

    previews = []

    print("\nPreview file teks/kode penting:")

    if len(preview_files) == 0:
        print("- Tidak ada file teks/kode kecil yang bisa dipreview.")
    else:
        for p in preview_files[:8]:
            rel = str(p.relative_to(CODE_EXTRACT_DIR))
            print(f"\n--- FILE: {rel} ---")

            try:
                with open(p, "r", encoding="utf-8", errors="replace") as f:
                    lines = [line.rstrip("\n") for line in f.readlines()[:80]]

                for line in lines[:40]:
                    print(line)

                if len(lines) > 40:
                    print("... preview dipotong ...")

                previews.append({
                    "relative_path": rel,
                    "readable": True,
                    "preview_first_80_lines": lines
                })

            except Exception as e:
                print(f"Gagal membaca file: {repr(e)}")
                previews.append({
                    "relative_path": rel,
                    "readable": False,
                    "error": repr(e)
                })

    # Cari keyword penting di file teks/kode
    keywords = [
        "train", "validation", "val", "test",
        "augmented", "augment",
        "excluded", "_EXCLUDED",
        "class", "classes",
        "14", "22",
        "crop", "particle", "position",
        "metadata", "label", "tags",
        "source", "sample", "grid", "session"
    ]

    keyword_hits = []

    for p in preview_files:
        try:
            text = p.read_text(encoding="utf-8", errors="replace")
            text_lower = text.lower()
            hits = [kw for kw in keywords if kw.lower() in text_lower]

            if hits:
                keyword_hits.append({
                    "relative_path": str(p.relative_to(CODE_EXTRACT_DIR)),
                    "hits": hits
                })
        except Exception:
            pass

    print("\nFile yang mengandung keyword penting:")
    if len(keyword_hits) == 0:
        print("- Tidak ada keyword penting yang terdeteksi pada file preview.")
    else:
        for item in keyword_hits[:30]:
            print(f"- {item['relative_path']} -> {item['hits']}")

    # Simpan ringkasan
    summary_json = {
        "step": "Sel 10 — Cek Isi dataset code.zip",
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "code_zip_path": str(CODE_ZIP_PATH),
        "code_zip_exists": CODE_ZIP_PATH.exists(),
        "code_extract_dir": str(CODE_EXTRACT_DIR),
        "total_items_in_code_zip": total_items,
        "total_files_in_code_zip": total_files,
        "total_dirs_in_code_zip": total_dirs,
        "extension_counter": dict(ext_counter),
        "manifest_path": str(manifest_path),
        "preview_count": len(previews),
        "previews": previews,
        "keyword_hits": keyword_hits
    }

    summary_path = OUTPUT_DIR / "sel10_dataset_code_check_summary.json"

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary_json, f, indent=4, ensure_ascii=False)

    print(f"\nManifest code zip disimpan ke: {manifest_path}")
    print(f"Ringkasan Sel 10 disimpan ke: {summary_path}")

except zipfile.BadZipFile:
    print("Code zip bisa dibaca : False")
    print("ERROR: dataset code.zip rusak atau bukan file ZIP valid.")

In [ ]:
# Sel 11 — Analisis Isi Kode MATLAB Dataset

from pathlib import Path
import re
import json
import pandas as pd
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

CODE_EXTRACT_DIR = Path('/content/tem_virus_dataset_code')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== HASIL CEK SEL 11 ===")
print(f"Code extract dir: {CODE_EXTRACT_DIR}")
print(f"Folder ada      : {CODE_EXTRACT_DIR.exists()}")

if not CODE_EXTRACT_DIR.exists():
    raise FileNotFoundError(f"Folder kode tidak ditemukan: {CODE_EXTRACT_DIR}")

m_files = sorted(CODE_EXTRACT_DIR.glob("*.m"))

print(f"\nJumlah file .m ditemukan: {len(m_files)}")
for p in m_files:
    print(f"- {p.name}")

# ------------------------------------------------------------
# 1. Parse getAllVirusData.m
# ------------------------------------------------------------

virus_file = CODE_EXTRACT_DIR / "getAllVirusData.m"

virus_rows = []

if virus_file.exists():
    text = virus_file.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    current = {}

    for line in lines:
        # contoh:
        # viruses(1).name = 'Adenovirus';
        m = re.search(r"viruses\((\d+)\)\.(\w+)\s*=\s*(.*?);", line)
        if m:
            idx = int(m.group(1))
            field = m.group(2)
            value_raw = m.group(3).strip()

            # Bersihkan string MATLAB
            value = value_raw.strip()
            if value.startswith("'") and value.endswith("'"):
                value = value[1:-1]

            if idx not in [r.get("index") for r in virus_rows]:
                virus_rows.append({"index": idx})

            for r in virus_rows:
                if r["index"] == idx:
                    r[field] = value
                    break

    virus_df = pd.DataFrame(virus_rows).sort_values("index")
else:
    virus_df = pd.DataFrame()

print("\nDaftar virus dari getAllVirusData.m:")
if len(virus_df) > 0:
    print(virus_df.to_string(index=False))
else:
    print("- Tidak berhasil membaca daftar virus.")

virus_list_path = OUTPUT_DIR / "sel11_getAllVirusData_parsed.csv"
virus_df.to_csv(virus_list_path, index=False)

# ------------------------------------------------------------
# 2. Cari keyword penting di semua file .m
# ------------------------------------------------------------

keywords = [
    "train",
    "validation",
    "test",
    "augment",
    "augmented",
    "imageSize",
    "samplesNum",
    "particleposition",
    "particle_positions",
    "tags",
    "readtable",
    "Delimiter",
    "copyfile",
    "mkdir",
    "rand",
    "randperm",
    "virusSize",
    "_EXCLUDED",
    "exclude",
    "excluded",
    "14",
    "22"
]

keyword_rows = []

for p in m_files:
    text = p.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    for i, line in enumerate(lines, start=1):
        line_lower = line.lower()
        matched = [kw for kw in keywords if kw.lower() in line_lower]

        if matched:
            keyword_rows.append({
                "file": p.name,
                "line_number": i,
                "matched_keywords": ", ".join(matched),
                "line_text": line.strip()
            })

keyword_df = pd.DataFrame(keyword_rows)

keyword_path = OUTPUT_DIR / "sel11_matlab_keyword_hits.csv"
keyword_df.to_csv(keyword_path, index=False)

print("\nJumlah baris kode yang mengandung keyword penting:", len(keyword_df))

print("\nContoh keyword hits paling penting:")
important_files = [
    "getAllVirusData.m",
    "selectContextVirusSubsets.m",
    "prepareContextVirusImages.m",
    "augmentContextVirusImages.m",
    "getImageTagInfo.m"
]

for fname in important_files:
    sub = keyword_df[keyword_df["file"] == fname].head(25)
    if len(sub) > 0:
        print(f"\n--- {fname} ---")
        print(sub[["line_number", "matched_keywords", "line_text"]].to_string(index=False))

# ------------------------------------------------------------
# 3. Print potongan kode sekitar bagian penting
# ------------------------------------------------------------

def print_context(file_path, search_terms, context=4, max_hits=8):
    text = file_path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()
    hits = []

    for i, line in enumerate(lines):
        if any(term.lower() in line.lower() for term in search_terms):
            hits.append(i)

    printed = set()
    shown = 0

    for idx in hits:
        if shown >= max_hits:
            break

        start = max(0, idx - context)
        end = min(len(lines), idx + context + 1)
        key = (start, end)

        if key in printed:
            continue

        printed.add(key)
        shown += 1

        print(f"\n--- {file_path.name}, sekitar line {idx+1} ---")
        for j in range(start, end):
            print(f"{j+1:04d}: {lines[j]}")

print("\n\nPotongan kode penting dari selectContextVirusSubsets.m:")
select_file = CODE_EXTRACT_DIR / "selectContextVirusSubsets.m"
if select_file.exists():
    print_context(
        select_file,
        search_terms=["train", "validation", "test", "copyfile", "particle_positions", "tags", "randperm"],
        context=3,
        max_hits=10
    )
else:
    print("- selectContextVirusSubsets.m tidak ditemukan.")

print("\n\nPotongan kode penting dari prepareContextVirusImages.m:")
prepare_file = CODE_EXTRACT_DIR / "prepareContextVirusImages.m"
if prepare_file.exists():
    print_context(
        prepare_file,
        search_terms=["imageSize", "outputScale", "particleposition", "crop_outlines", "imresize", "imwrite"],
        context=3,
        max_hits=10
    )
else:
    print("- prepareContextVirusImages.m tidak ditemukan.")

print("\n\nPotongan kode penting dari augmentContextVirusImages.m:")
augment_file = CODE_EXTRACT_DIR / "augmentContextVirusImages.m"
if augment_file.exists():
    print_context(
        augment_file,
        search_terms=["augmentSet", "samplesNum", "augmentation", "flip", "rotat", "outputPath"],
        context=3,
        max_hits=10
    )
else:
    print("- augmentContextVirusImages.m tidak ditemukan.")

# ------------------------------------------------------------
# 4. Simpan ringkasan JSON
# ------------------------------------------------------------

summary = {
    "step": "Sel 11 — Analisis Isi Kode MATLAB Dataset",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "code_extract_dir": str(CODE_EXTRACT_DIR),
    "m_files": [p.name for p in m_files],
    "virus_count_from_getAllVirusData": int(len(virus_df)),
    "virus_list": virus_df.astype(str).to_dict(orient="records") if len(virus_df) > 0 else [],
    "keyword_hits_count": int(len(keyword_df)),
    "output_files": {
        "parsed_virus_list": str(virus_list_path),
        "matlab_keyword_hits": str(keyword_path)
    }
}

summary_path = OUTPUT_DIR / "sel11_matlab_code_analysis_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print(f"\nDaftar virus hasil parsing disimpan ke: {virus_list_path}")
print(f"Keyword hits kode MATLAB disimpan ke: {keyword_path}")
print(f"Ringkasan Sel 11 disimpan ke: {summary_path}")

In [ ]:
# Sel 12 — Cek Kelengkapan Metadata RAW

from pathlib import Path
import pandas as pd
import json
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'
RAW_ROOT = DATASET_ROOT / "context_virus_RAW"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ringkasan verifikasi metadata RAW")
print(f"Folder RAW: {RAW_ROOT}")
print(f"Folder tersedia: {RAW_ROOT.exists()}")

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"Folder RAW tidak ditemukan: {RAW_ROOT}")

official_splits = ["train", "validation", "test"]

rows = []

for split in official_splits:
    split_dir = RAW_ROOT / split

    if not split_dir.exists():
        continue

    class_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])

    for class_dir in class_dirs:
        class_name = class_dir.name

        image_files = sorted(class_dir.glob("*.tif"))

        for img_path in image_files:
            stem = img_path.stem

            tag_path = class_dir / "tags" / f"{img_path.name}_tags.csv"
            particle_path = class_dir / "particle_positions" / f"{stem}_particlepositions.txt"

            rows.append({
                "split": split,
                "class_name": class_name,
                "image_filename": img_path.name,
                "image_stem": stem,
                "image_path": str(img_path.relative_to(RAW_ROOT)),
                "tag_path": str(tag_path.relative_to(RAW_ROOT)) if tag_path.exists() else None,
                "particle_path": str(particle_path.relative_to(RAW_ROOT)) if particle_path.exists() else None,
                "tag_exists": tag_path.exists(),
                "particle_exists": particle_path.exists(),
                "image_size_bytes": img_path.stat().st_size
            })

df = pd.DataFrame(rows)

print(f"\nTotal gambar RAW: {len(df)}")
print(f"Jumlah kelas RAW: {df['class_name'].nunique() if len(df) > 0 else 0}")

metadata_summary = (
    df.groupby(["split"])
      .agg(
          image_count=("image_path", "count"),
          class_count=("class_name", "nunique"),
          tag_available=("tag_exists", "sum"),
          particle_available=("particle_exists", "sum")
      )
      .reset_index()
)

print("\nKelengkapan metadata per split:")
print(metadata_summary.to_string(index=False))

missing_tag = df[df["tag_exists"] == False]
missing_particle = df[df["particle_exists"] == False]

print(f"\nFile tanpa tags: {len(missing_tag)}")
print(f"File tanpa particle_positions: {len(missing_particle)}")

if len(missing_tag) > 0:
    print("\nContoh file tanpa tags:")
    print(missing_tag.head(10)[["split", "class_name", "image_filename"]].to_string(index=False))

if len(missing_particle) > 0:
    print("\nContoh file tanpa particle_positions:")
    print(missing_particle.head(10)[["split", "class_name", "image_filename"]].to_string(index=False))

# Preview 1 file tags dengan delimiter ;
print("\nContoh isi file tags yang berhasil dibaca:")

sample_tag_row = df[df["tag_exists"] == True].head(1)

tag_preview_records = []

if len(sample_tag_row) > 0:
    sample_tag_rel = sample_tag_row.iloc[0]["tag_path"]
    sample_tag_path = RAW_ROOT / sample_tag_rel

    print(f"File contoh: {sample_tag_rel}")

    try:
        tag_df = pd.read_csv(
            sample_tag_path,
            sep=";",
            header=None,
            names=["field", "value"],
            engine="python"
        )

        print(tag_df.head(15).to_string(index=False))
        tag_preview_records = tag_df.head(15).astype(str).to_dict(orient="records")

    except Exception as e:
        print(f"Tags masih gagal dibaca: {repr(e)}")
else:
    print("Tidak ada file tags yang tersedia untuk preview.")

# Simpan output
inventory_path = OUTPUT_DIR / "raw_metadata_inventory.csv"
summary_path = OUTPUT_DIR / "raw_metadata_summary.csv"
json_path = OUTPUT_DIR / "raw_metadata_check.json"

df.to_csv(inventory_path, index=False)
metadata_summary.to_csv(summary_path, index=False)

result = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "raw_root": str(RAW_ROOT),
    "total_raw_images": int(len(df)),
    "raw_class_count": int(df["class_name"].nunique()) if len(df) > 0 else 0,
    "metadata_summary": metadata_summary.to_dict(orient="records"),
    "missing_tag_count": int(len(missing_tag)),
    "missing_particle_position_count": int(len(missing_particle)),
    "sample_tag_preview": tag_preview_records,
    "saved_files": {
        "inventory": str(inventory_path),
        "summary": str(summary_path),
        "json": str(json_path)
    }
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

print("\nFile hasil verifikasi tersimpan di:")
print(f"- {inventory_path}")
print(f"- {summary_path}")
print(f"- {json_path}")

In [ ]:
# Sel 13 — Cek Identifier RAW dan Overlap antar Split

from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'
RAW_ROOT = DATASET_ROOT / "context_virus_RAW"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Verifikasi identifier dan overlap RAW")
print(f"Folder RAW: {RAW_ROOT}")
print(f"Folder tersedia: {RAW_ROOT.exists()}")

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"Folder RAW tidak ditemukan: {RAW_ROOT}")

official_splits = ["train", "validation", "test"]

def md5_hash_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def read_tag_file(tag_path):
    try:
        tag_df = pd.read_csv(
            tag_path,
            sep=";",
            header=None,
            names=["field", "value"],
            engine="python",
            dtype=str
        )
        tag_df["field"] = tag_df["field"].astype(str).str.strip()
        tag_df["value"] = tag_df["value"].astype(str).str.strip()
        return dict(zip(tag_df["field"], tag_df["value"]))
    except Exception:
        return {}

rows = []

for split in official_splits:
    split_dir = RAW_ROOT / split
    if not split_dir.exists():
        continue

    class_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()])

    for class_dir in class_dirs:
        class_name = class_dir.name
        image_files = sorted(class_dir.glob("*.tif"))

        for img_path in image_files:
            stem = img_path.stem
            tag_path = class_dir / "tags" / f"{img_path.name}_tags.csv"
            particle_path = class_dir / "particle_positions" / f"{stem}_particlepositions.txt"

            tags = read_tag_file(tag_path) if tag_path.exists() else {}

            rows.append({
                "split": split,
                "class_name": class_name,
                "image_filename": img_path.name,
                "image_stem": stem,
                "relative_image_path": str(img_path.relative_to(RAW_ROOT)),
                "md5": md5_hash_file(img_path),
                "Heigt": tags.get("Heigt"),
                "Width": tags.get("Width"),
                "GridposX": tags.get("GridposX"),
                "GridposY": tags.get("GridposY"),
                "GridposZ": tags.get("GridposZ"),
                "Xscale": tags.get("Xscale"),
                "Yscale": tags.get("Yscale"),
                "AccVoltage": tags.get("AccVoltage"),
                "Defocus": tags.get("Defocus"),
                "Date": tags.get("Date"),
                "Magnification": tags.get("Magnification"),
                "Photographer": tags.get("Photographer"),
                "CameraModel": tags.get("CameraModel"),
                "tag_exists": tag_path.exists(),
                "particle_exists": particle_path.exists()
            })

df = pd.DataFrame(rows)

print(f"\nTotal gambar RAW dicek: {len(df)}")
print(f"Jumlah kelas: {df['class_name'].nunique() if len(df) > 0 else 0}")
print(f"Jumlah hash unik gambar RAW: {df['md5'].nunique() if len(df) > 0 else 0}")

# 1. Exact duplicate berdasarkan isi gambar RAW antar split
hash_split = (
    df.groupby(["md5", "split"])
      .size()
      .reset_index(name="count")
)

hash_multi = (
    hash_split.groupby("md5")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("count", "sum")
      )
      .reset_index()
)

raw_duplicate_hash = hash_multi[hash_multi["split_count"] > 1].copy()

print(f"\nGambar RAW identik yang muncul di lebih dari satu split: {len(raw_duplicate_hash)}")

if len(raw_duplicate_hash) > 0:
    detail_dup_hash = df[df["md5"].isin(raw_duplicate_hash["md5"])].copy()
    print("\nContoh gambar RAW identik lintas split:")
    print(detail_dup_hash.head(20)[["split", "class_name", "image_filename", "md5"]].to_string(index=False))
else:
    detail_dup_hash = pd.DataFrame(columns=df.columns)
    print("Tidak ditemukan gambar RAW identik lintas split berdasarkan hash file.")

# 2. Overlap nama file RAW dalam kelas yang sama antar split
df["class_filename_key"] = df["class_name"] + "__" + df["image_filename"]

filename_split = (
    df.groupby(["class_filename_key", "split"])
      .size()
      .reset_index(name="count")
)

filename_multi = (
    filename_split.groupby("class_filename_key")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("count", "sum")
      )
      .reset_index()
)

raw_filename_overlap = filename_multi[filename_multi["split_count"] > 1].copy()

print(f"\nNama file RAW yang sama dalam kelas yang sama dan muncul lintas split: {len(raw_filename_overlap)}")

if len(raw_filename_overlap) > 0:
    detail_filename = df[df["class_filename_key"].isin(raw_filename_overlap["class_filename_key"])].copy()
    print("\nContoh overlap nama file RAW:")
    print(detail_filename.head(20)[["split", "class_name", "image_filename"]].to_string(index=False))
else:
    detail_filename = pd.DataFrame(columns=df.columns)
    print("Tidak ditemukan overlap nama file RAW dalam kelas yang sama antar split.")

# 3. Overlap metadata signature
signature_cols = [
    "class_name",
    "Heigt",
    "Width",
    "GridposX",
    "GridposY",
    "GridposZ",
    "Xscale",
    "Yscale",
    "Date",
    "Magnification",
    "Photographer",
    "CameraModel"
]

df["metadata_signature"] = df[signature_cols].fillna("").astype(str).agg("__".join, axis=1)

sig_split = (
    df.groupby(["metadata_signature", "split"])
      .size()
      .reset_index(name="count")
)

sig_multi = (
    sig_split.groupby("metadata_signature")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("count", "sum")
      )
      .reset_index()
)

metadata_overlap = sig_multi[sig_multi["split_count"] > 1].copy()

print(f"\nMetadata signature yang muncul di lebih dari satu split: {len(metadata_overlap)}")

if len(metadata_overlap) > 0:
    detail_metadata = df[df["metadata_signature"].isin(metadata_overlap["metadata_signature"])].copy()
    print("\nContoh metadata signature overlap:")
    print(detail_metadata.head(20)[["split", "class_name", "image_filename", "Date", "GridposX", "GridposY", "GridposZ"]].to_string(index=False))
else:
    detail_metadata = pd.DataFrame(columns=df.columns)
    print("Tidak ditemukan metadata signature yang sama lintas split.")

# Ringkasan per split
split_summary = (
    df.groupby("split")
      .agg(
          image_count=("relative_image_path", "count"),
          class_count=("class_name", "nunique"),
          unique_hash=("md5", "nunique"),
          unique_metadata_signature=("metadata_signature", "nunique")
      )
      .reset_index()
)

print("\nRingkasan per split:")
print(split_summary.to_string(index=False))

# Simpan output
inventory_path = OUTPUT_DIR / "raw_identifier_inventory.csv"
split_summary_path = OUTPUT_DIR / "raw_identifier_split_summary.csv"
raw_duplicate_hash_path = OUTPUT_DIR / "raw_duplicate_hash_across_splits.csv"
raw_filename_overlap_path = OUTPUT_DIR / "raw_filename_overlap_across_splits.csv"
metadata_overlap_path = OUTPUT_DIR / "raw_metadata_signature_overlap_across_splits.csv"
json_path = OUTPUT_DIR / "raw_identifier_overlap_check.json"

df.to_csv(inventory_path, index=False)
split_summary.to_csv(split_summary_path, index=False)
raw_duplicate_hash.to_csv(raw_duplicate_hash_path, index=False)
raw_filename_overlap.to_csv(raw_filename_overlap_path, index=False)
metadata_overlap.to_csv(metadata_overlap_path, index=False)

result = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "raw_root": str(RAW_ROOT),
    "total_raw_images_checked": int(len(df)),
    "raw_class_count": int(df["class_name"].nunique()) if len(df) > 0 else 0,
    "unique_raw_image_hash_count": int(df["md5"].nunique()) if len(df) > 0 else 0,
    "raw_duplicate_hash_across_split_count": int(len(raw_duplicate_hash)),
    "raw_filename_overlap_across_split_count": int(len(raw_filename_overlap)),
    "metadata_signature_overlap_across_split_count": int(len(metadata_overlap)),
    "saved_files": {
        "inventory": str(inventory_path),
        "split_summary": str(split_summary_path),
        "raw_duplicate_hash": str(raw_duplicate_hash_path),
        "raw_filename_overlap": str(raw_filename_overlap_path),
        "metadata_overlap": str(metadata_overlap_path),
        "json": str(json_path)
    }
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

print("\nFile hasil verifikasi tersimpan di:")
print(f"- {inventory_path}")
print(f"- {split_summary_path}")
print(f"- {raw_duplicate_hash_path}")
print(f"- {raw_filename_overlap_path}")
print(f"- {metadata_overlap_path}")
print(f"- {json_path}")

In [ ]:
# Sel 14 — Cek Awal Near-Duplicate antar Split Resmi

from pathlib import Path
import pandas as pd
import numpy as np
import json
from PIL import Image
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'
DATA_1NM = DATASET_ROOT / "context_virus_1nm_256x256"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Verifikasi awal near-duplicate gambar 1nm_256x256")
print(f"Folder gambar: {DATA_1NM}")
print(f"Folder tersedia: {DATA_1NM.exists()}")

if not DATA_1NM.exists():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_1NM}")

official_splits = ["train", "validation", "test"]

def average_hash(image_path, hash_size=8):
    """
    Menghasilkan perceptual hash sederhana 64-bit.
    Gambar yang sangat mirip biasanya punya jarak Hamming kecil.
    """
    img = Image.open(image_path).convert("L").resize((hash_size, hash_size))
    pixels = np.asarray(img, dtype=np.float32)
    avg = pixels.mean()
    bits = pixels > avg
    return ''.join(['1' if b else '0' for b in bits.flatten()])

def hamming_distance(hash1, hash2):
    return sum(c1 != c2 for c1, c2 in zip(hash1, hash2))

rows = []

for p in sorted(DATA_1NM.rglob("*.tif")):
    rel = p.relative_to(DATA_1NM)
    parts = rel.parts

    split = parts[0] if len(parts) > 0 else None

    if split not in official_splits:
        continue

    if "crop_outlines" in parts:
        continue

    if len(parts) >= 2 and parts[1] == "_EXCLUDED":
        class_status = "excluded"
        class_name = parts[2] if len(parts) >= 3 else None
    else:
        class_status = "main"
        class_name = parts[1] if len(parts) >= 2 else None

    stem = p.stem
    base_id = stem.split("_")[0] if "_" in stem else stem

    try:
        ahash = average_hash(p)
        readable = True
        error = None
    except Exception as e:
        ahash = None
        readable = False
        error = repr(e)

    rows.append({
        "relative_path": str(rel),
        "split": split,
        "class_status": class_status,
        "class_name": class_name,
        "filename": p.name,
        "stem": stem,
        "base_id": base_id,
        "ahash": ahash,
        "readable": readable,
        "error": error
    })

df = pd.DataFrame(rows)

print(f"\nTotal file dicek: {len(df)}")
print(f"File berhasil dibaca: {df['readable'].sum()}")
print(f"File gagal dibaca: {(df['readable'] == False).sum()}")

valid_df = df[(df["readable"] == True) & (df["ahash"].notna())].copy()

# Cek hash identik lintas split
hash_split = (
    valid_df.groupby(["ahash", "split"])
      .size()
      .reset_index(name="count")
)

hash_multi = (
    hash_split.groupby("ahash")
      .agg(
          split_count=("split", "nunique"),
          splits=("split", lambda x: sorted(list(set(x)))),
          total_files=("count", "sum")
      )
      .reset_index()
)

same_ahash_across_split = hash_multi[hash_multi["split_count"] > 1].copy()

print(f"\nPerceptual hash identik yang muncul lintas split: {len(same_ahash_across_split)}")

# Cari kandidat near-duplicate antar split dalam kelas yang sama
threshold = 5
candidate_pairs = []

for (class_status, class_name), group in valid_df.groupby(["class_status", "class_name"]):
    group = group.reset_index(drop=True)

    split_groups = {
        split: group[group["split"] == split].reset_index(drop=True)
        for split in official_splits
    }

    split_pairs = [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test")
    ]

    for split_a, split_b in split_pairs:
        ga = split_groups[split_a]
        gb = split_groups[split_b]

        if len(ga) == 0 or len(gb) == 0:
            continue

        for _, row_a in ga.iterrows():
            hash_a = row_a["ahash"]

            for _, row_b in gb.iterrows():
                hash_b = row_b["ahash"]
                dist = hamming_distance(hash_a, hash_b)

                if dist <= threshold:
                    candidate_pairs.append({
                        "class_status": class_status,
                        "class_name": class_name,
                        "split_a": split_a,
                        "file_a": row_a["relative_path"],
                        "base_id_a": row_a["base_id"],
                        "split_b": split_b,
                        "file_b": row_b["relative_path"],
                        "base_id_b": row_b["base_id"],
                        "hamming_distance": dist,
                        "ahash_a": hash_a,
                        "ahash_b": hash_b
                    })

candidate_df = pd.DataFrame(candidate_pairs)

print(f"\nKandidat near-duplicate lintas split dengan jarak hash <= {threshold}: {len(candidate_df)}")

if len(candidate_df) > 0:
    summary_candidate = (
        candidate_df.groupby(["class_status", "class_name", "split_a", "split_b"])
        .size()
        .reset_index(name="candidate_pair_count")
        .sort_values("candidate_pair_count", ascending=False)
    )

    print("\nRingkasan kandidat near-duplicate terbanyak:")
    print(summary_candidate.head(30).to_string(index=False))

    print("\nContoh pasangan kandidat:")
    print(candidate_df.head(20)[[
        "class_status", "class_name",
        "split_a", "file_a",
        "split_b", "file_b",
        "hamming_distance"
    ]].to_string(index=False))
else:
    summary_candidate = pd.DataFrame(columns=[
        "class_status", "class_name", "split_a", "split_b", "candidate_pair_count"
    ])
    print("Tidak ada kandidat near-duplicate berdasarkan threshold awal ini.")

# Simpan output
hash_inventory_path = OUTPUT_DIR / "near_duplicate_ahash_inventory.csv"
same_ahash_path = OUTPUT_DIR / "near_duplicate_same_ahash_across_splits.csv"
candidate_path = OUTPUT_DIR / "near_duplicate_candidate_pairs.csv"
candidate_summary_path = OUTPUT_DIR / "near_duplicate_candidate_summary.csv"
json_path = OUTPUT_DIR / "near_duplicate_check.json"

df.to_csv(hash_inventory_path, index=False)
same_ahash_across_split.to_csv(same_ahash_path, index=False)
candidate_df.to_csv(candidate_path, index=False)
summary_candidate.to_csv(candidate_summary_path, index=False)

result = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "data_root": str(DATA_1NM),
    "checked_splits": official_splits,
    "excluded_crop_outlines": True,
    "included_class_status": ["main", "excluded"],
    "hash_method": "average_hash_8x8",
    "near_duplicate_hamming_threshold": threshold,
    "total_files_checked": int(len(df)),
    "readable_files": int(df["readable"].sum()),
    "unreadable_files": int((df["readable"] == False).sum()),
    "same_ahash_across_split_count": int(len(same_ahash_across_split)),
    "near_duplicate_candidate_pair_count": int(len(candidate_df)),
    "saved_files": {
        "hash_inventory": str(hash_inventory_path),
        "same_ahash_across_splits": str(same_ahash_path),
        "candidate_pairs": str(candidate_path),
        "candidate_summary": str(candidate_summary_path),
        "json": str(json_path)
    }
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

print("\nFile hasil verifikasi tersimpan di:")
print(f"- {hash_inventory_path}")
print(f"- {same_ahash_path}")
print(f"- {candidate_path}")
print(f"- {candidate_summary_path}")
print(f"- {json_path}")

In [ ]:
# Sel 15 — Validasi Visual Kandidat Near-Duplicate Terkuat

from pathlib import Path
import pandas as pd
import numpy as np
import json
from PIL import Image, ImageDraw, ImageFont
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

EXTRACT_DIR = Path('/content/tem_virus_dataset_extracted')
DATASET_ROOT = EXTRACT_DIR / 'TEM virus dataset'
DATA_1NM = DATASET_ROOT / "context_virus_1nm_256x256"

candidate_path = OUTPUT_DIR / "near_duplicate_candidate_pairs.csv"

print("Validasi visual kandidat near-duplicate")
print(f"File kandidat: {candidate_path}")
print(f"File tersedia: {candidate_path.exists()}")

if not candidate_path.exists():
    raise FileNotFoundError(f"File kandidat tidak ditemukan: {candidate_path}")

df = pd.read_csv(candidate_path)

print(f"\nTotal kandidat awal: {len(df)}")

# Fokus ke kelas utama saja, bukan _EXCLUDED
main_df = df[df["class_status"] == "main"].copy()

# Kandidat paling kuat: hamming distance kecil
strong_df = main_df[main_df["hamming_distance"] <= 2].copy()
exact_ahash_df = main_df[main_df["hamming_distance"] == 0].copy()

print(f"Kandidat kelas utama: {len(main_df)}")
print(f"Kandidat kuat kelas utama, jarak hash <= 2: {len(strong_df)}")
print(f"Kandidat hash identik kelas utama, jarak hash = 0: {len(exact_ahash_df)}")

def load_gray_resized(rel_path, size=(128, 128)):
    img = Image.open(DATA_1NM / rel_path).convert("L").resize(size)
    arr = np.asarray(img, dtype=np.float32)
    return img, arr

def mse(a, b):
    return float(np.mean((a - b) ** 2))

def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def corrcoef(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    if np.std(a_flat) == 0 or np.std(b_flat) == 0:
        return None
    return float(np.corrcoef(a_flat, b_flat)[0, 1])

# Hitung metrik tambahan untuk kandidat kuat
metric_rows = []

for _, row in strong_df.iterrows():
    try:
        img_a, arr_a = load_gray_resized(row["file_a"])
        img_b, arr_b = load_gray_resized(row["file_b"])

        metric_rows.append({
            **row.to_dict(),
            "mse_128": mse(arr_a, arr_b),
            "mae_128": mae(arr_a, arr_b),
            "corr_128": corrcoef(arr_a, arr_b),
            "readable": True,
            "error": None
        })
    except Exception as e:
        metric_rows.append({
            **row.to_dict(),
            "mse_128": None,
            "mae_128": None,
            "corr_128": None,
            "readable": False,
            "error": repr(e)
        })

metric_df = pd.DataFrame(metric_rows)

# Urutkan kandidat paling mirip menurut hamming, MSE, dan korelasi
if len(metric_df) > 0:
    metric_df = metric_df.sort_values(
        by=["hamming_distance", "mse_128", "mae_128"],
        ascending=[True, True, True]
    )

print("\nRingkasan kandidat kuat per kelas:")
if len(metric_df) > 0:
    summary = (
        metric_df.groupby(["class_name", "split_a", "split_b"])
        .size()
        .reset_index(name="pair_count")
        .sort_values("pair_count", ascending=False)
    )
    print(summary.head(30).to_string(index=False))
else:
    summary = pd.DataFrame(columns=["class_name", "split_a", "split_b", "pair_count"])
    print("Tidak ada kandidat kuat pada kelas utama.")

print("\nContoh kandidat paling kuat:")
if len(metric_df) > 0:
    print(metric_df.head(20)[[
        "class_name", "split_a", "file_a", "split_b", "file_b",
        "hamming_distance", "mse_128", "mae_128", "corr_128"
    ]].to_string(index=False))
else:
    print("Tidak ada data untuk ditampilkan.")

# Buat contact sheet visual
def make_contact_sheet(pair_df, output_path, max_pairs=24, thumb_size=(140, 140)):
    pair_df = pair_df.head(max_pairs).copy()

    if len(pair_df) == 0:
        return False

    row_height = thumb_size[1] + 70
    sheet_width = thumb_size[0] * 2 + 80
    sheet_height = row_height * len(pair_df)

    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
    draw = ImageDraw.Draw(sheet)

    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    for idx, (_, row) in enumerate(pair_df.iterrows()):
        y = idx * row_height

        img_a = Image.open(DATA_1NM / row["file_a"]).convert("RGB").resize(thumb_size)
        img_b = Image.open(DATA_1NM / row["file_b"]).convert("RGB").resize(thumb_size)

        sheet.paste(img_a, (10, y + 25))
        sheet.paste(img_b, (thumb_size[0] + 40, y + 25))

        title = (
            f"{row['class_name']} | {row['split_a']} vs {row['split_b']} | "
            f"H={row['hamming_distance']} | MSE={row.get('mse_128', 0):.1f}"
        )
        draw.text((10, y + 5), title, fill="black", font=font)

        draw.text((10, y + thumb_size[1] + 30), Path(row["file_a"]).name, fill="black", font=font)
        draw.text((thumb_size[0] + 40, y + thumb_size[1] + 30), Path(row["file_b"]).name, fill="black", font=font)

    sheet.save(output_path)
    return True

contact_sheet_strong = OUTPUT_DIR / "near_duplicate_strong_pairs_contact_sheet.png"
contact_sheet_exact = OUTPUT_DIR / "near_duplicate_exact_ahash_pairs_contact_sheet.png"

created_strong = make_contact_sheet(metric_df, contact_sheet_strong, max_pairs=24)

if len(metric_df) > 0:
    exact_metric_df = metric_df[metric_df["hamming_distance"] == 0].copy()
else:
    exact_metric_df = pd.DataFrame()

created_exact = make_contact_sheet(exact_metric_df, contact_sheet_exact, max_pairs=24)

# Simpan output
strong_candidate_path = OUTPUT_DIR / "near_duplicate_strong_main_candidates.csv"
summary_path = OUTPUT_DIR / "near_duplicate_strong_main_summary.csv"
json_path = OUTPUT_DIR / "near_duplicate_visual_validation.json"

metric_df.to_csv(strong_candidate_path, index=False)
summary.to_csv(summary_path, index=False)

result = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "candidate_source": str(candidate_path),
    "focus": "main classes only, official train/validation/test, no crop_outlines",
    "total_initial_candidates": int(len(df)),
    "total_main_candidates": int(len(main_df)),
    "strong_main_candidates_hamming_le_2": int(len(strong_df)),
    "exact_ahash_main_candidates_hamming_0": int(len(exact_ahash_df)),
    "metric_rows": int(len(metric_df)),
    "contact_sheet_strong_created": bool(created_strong),
    "contact_sheet_exact_created": bool(created_exact),
    "saved_files": {
        "strong_candidates": str(strong_candidate_path),
        "summary": str(summary_path),
        "contact_sheet_strong": str(contact_sheet_strong) if created_strong else None,
        "contact_sheet_exact": str(contact_sheet_exact) if created_exact else None,
        "json": str(json_path)
    }
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4, ensure_ascii=False)

print("\nFile hasil verifikasi tersimpan di:")
print(f"- {strong_candidate_path}")
print(f"- {summary_path}")
if created_strong:
    print(f"- {contact_sheet_strong}")
if created_exact:
    print(f"- {contact_sheet_exact}")
print(f"- {json_path}")

In [ ]:
# Sel 16 — Kompilasi Ringkasan Verifikasi Dataset

from pathlib import Path
import json
import pandas as pd
from datetime import datetime

PROJECT_DIR = Path('/content/drive/MyDrive/Riset_TEM_Virus')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Menyusun ringkasan akhir verifikasi dataset")
print(f"Folder output: {OUTPUT_DIR}")

# Helper aman untuk membaca JSON
def load_json_safe(path):
    path = Path(path)
    if path.exists():
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            return {"error": repr(e), "path": str(path)}
    return None

# Baca beberapa ringkasan utama jika tersedia
sel1 = load_json_safe(OUTPUT_DIR / "sel1_zip_check.json")
sel2 = load_json_safe(OUTPUT_DIR / "sel2_zip_inventory_summary.json")
sel3 = load_json_safe(OUTPUT_DIR / "sel3_extract_status.json")
sel4 = load_json_safe(OUTPUT_DIR / "sel4_folder_structure_summary.json")
sel5 = load_json_safe(OUTPUT_DIR / "sel5_tif_count_summary.json")
sel8 = load_json_safe(OUTPUT_DIR / "sel8_precise_grouping_summary.json")
sel9 = load_json_safe(OUTPUT_DIR / "sel9_exact_duplicate_hash_summary.json")
sel10 = load_json_safe(OUTPUT_DIR / "sel10_dataset_code_check_summary.json")
sel11 = load_json_safe(OUTPUT_DIR / "sel11_matlab_code_analysis_summary.json")
raw_metadata = load_json_safe(OUTPUT_DIR / "raw_metadata_check.json")
raw_identifier = load_json_safe(OUTPUT_DIR / "raw_identifier_overlap_check.json")
near_dup = load_json_safe(OUTPUT_DIR / "near_duplicate_check.json")
near_dup_visual = load_json_safe(OUTPUT_DIR / "near_duplicate_visual_validation.json")

# Manifest semua file output yang sudah dibuat
output_files = sorted([p for p in OUTPUT_DIR.rglob("*") if p.is_file()])

manifest_rows = []
for p in output_files:
    manifest_rows.append({
        "filename": p.name,
        "relative_path": str(p.relative_to(OUTPUT_DIR)),
        "full_path": str(p),
        "suffix": p.suffix.lower() if p.suffix else "",
        "size_kb": round(p.stat().st_size / 1024, 2),
        "modified_time": datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
    })

manifest_df = pd.DataFrame(manifest_rows)

# Ringkasan manual berdasarkan hasil verifikasi
final_summary = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "project_dir": str(PROJECT_DIR),
    "output_dir": str(OUTPUT_DIR),

    "dataset_file": {
        "zip_path": sel1.get("zip_path") if sel1 else None,
        "zip_exists": sel1.get("zip_exists") if sel1 else None,
        "zip_size_mb": sel1.get("zip_size_mb") if sel1 else None,
        "zip_size_gb": sel1.get("zip_size_gb") if sel1 else None,
        "zip_readable": sel2.get("zip_readable") if sel2 else None
    },

    "zip_inventory": {
        "total_items": sel2.get("total_items") if sel2 else None,
        "total_files": sel2.get("total_files") if sel2 else None,
        "total_dirs": sel2.get("total_dirs") if sel2 else None,
        "extension_counter": sel2.get("extension_counter") if sel2 else None
    },

    "extraction": {
        "extract_dir": sel3.get("extract_dir") if sel3 else None,
        "total_files_extracted": sel3.get("total_files_extracted") if sel3 else None,
        "total_dirs_extracted": sel3.get("total_dirs_extracted") if sel3 else None
    },

    "folder_structure": {
        "dataset_root": sel4.get("dataset_root") if sel4 else None,
        "immediate_children": sel4.get("immediate_children") if sel4 else None,
        "detected_split_dirs": sel4.get("detected_split_dirs") if sel4 else None
    },

    "classes_and_images": {
        "total_tif_files": sel5.get("total_tif_files") if sel5 else None,
        "combined_classes_by_version": sel5.get("combined_classes_by_version") if sel5 else None
    },

    "official_splits": {
        "official_split_no_crop_files": sel8.get("total_official_no_crop_files") if sel8 else None,
        "official_summary": sel8.get("official_summary") if sel8 else None,
        "group_overlap_official_split_count": sel8.get("potential_group_overlap_official_split_count") if sel8 else None,
        "exact_filename_overlap_official_split_count": sel8.get("exact_filename_overlap_official_split_count") if sel8 else None
    },

    "exact_duplicate_check": {
        "total_files_checked": sel9.get("total_files_checked") if sel9 else None,
        "unique_hash_count": sel9.get("unique_hash_count") if sel9 else None,
        "duplicate_hashes_across_official_splits": sel9.get("duplicate_hashes_across_official_splits") if sel9 else None
    },

    "dataset_code": {
        "code_zip_exists": sel10.get("code_zip_exists") if sel10 else None,
        "total_files_in_code_zip": sel10.get("total_files_in_code_zip") if sel10 else None,
        "m_files": sel11.get("m_files") if sel11 else None,
        "virus_count_from_getAllVirusData": sel11.get("virus_count_from_getAllVirusData") if sel11 else None,
        "virus_list": sel11.get("virus_list") if sel11 else None
    },

    "raw_metadata": {
        "total_raw_images": raw_metadata.get("total_raw_images") if raw_metadata else None,
        "raw_class_count": raw_metadata.get("raw_class_count") if raw_metadata else None,
        "missing_tag_count": raw_metadata.get("missing_tag_count") if raw_metadata else None,
        "missing_particle_position_count": raw_metadata.get("missing_particle_position_count") if raw_metadata else None
    },

    "raw_identifier_overlap": {
        "total_raw_images_checked": raw_identifier.get("total_raw_images_checked") if raw_identifier else None,
        "raw_duplicate_hash_across_split_count": raw_identifier.get("raw_duplicate_hash_across_split_count") if raw_identifier else None,
        "raw_filename_overlap_across_split_count": raw_identifier.get("raw_filename_overlap_across_split_count") if raw_identifier else None,
        "metadata_signature_overlap_across_split_count": raw_identifier.get("metadata_signature_overlap_across_split_count") if raw_identifier else None
    },

    "near_duplicate_check": {
        "hash_method": near_dup.get("hash_method") if near_dup else None,
        "total_files_checked": near_dup.get("total_files_checked") if near_dup else None,
        "same_ahash_across_split_count": near_dup.get("same_ahash_across_split_count") if near_dup else None,
        "near_duplicate_candidate_pair_count": near_dup.get("near_duplicate_candidate_pair_count") if near_dup else None,
        "strong_main_candidates_hamming_le_2": near_dup_visual.get("strong_main_candidates_hamming_le_2") if near_dup_visual else None,
        "exact_ahash_main_candidates_hamming_0": near_dup_visual.get("exact_ahash_main_candidates_hamming_0") if near_dup_visual else None
    },

    "verification_outputs_count": len(output_files)
}

# Simpan ringkasan akhir
master_json_path = OUTPUT_DIR / "dataset_verification_master_summary.json"
manifest_path = OUTPUT_DIR / "dataset_verification_outputs_manifest.csv"

with open(master_json_path, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4, ensure_ascii=False)

manifest_df.to_csv(manifest_path, index=False)

print("\nRingkasan utama")
print(f"ZIP ditemukan: {final_summary['dataset_file']['zip_exists']}")
print(f"ZIP bisa dibaca: {final_summary['dataset_file']['zip_readable']}")
print(f"Total file .tif: {final_summary['classes_and_images']['total_tif_files']}")
print(f"Jumlah kelas dari kode asli: {final_summary['dataset_code']['virus_count_from_getAllVirusData']}")
print(f"Jumlah gambar RAW: {final_summary['raw_metadata']['total_raw_images']}")
print(f"Metadata RAW hilang: tags={final_summary['raw_metadata']['missing_tag_count']}, particle_positions={final_summary['raw_metadata']['missing_particle_position_count']}")
print(f"Exact duplicate antar split resmi: {final_summary['exact_duplicate_check']['duplicate_hashes_across_official_splits']}")
print(f"Kandidat near-duplicate awal: {final_summary['near_duplicate_check']['near_duplicate_candidate_pair_count']}")
print(f"Kandidat near-duplicate kuat kelas utama: {final_summary['near_duplicate_check']['strong_main_candidates_hamming_le_2']}")
print(f"Jumlah file output verifikasi: {final_summary['verification_outputs_count']}")

print("\nFile ringkasan akhir tersimpan di:")
print(f"- {master_json_path}")
print(f"- {manifest_path}")

# Bagian 2 — Validasi Leakage dan Split


In [ ]:
# Sel 1 — Cek Runtime dan Path Verifikasi Sebelumnya

import os
from pathlib import Path
import pandas as pd

# Path utama
drive_root = Path("/content/drive/MyDrive")
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Path folder penting dataset
crop_root = dataset_root / "context_virus_1nm_256x256"
raw_root = dataset_root / "context_virus_RAW"

# File hasil verifikasi sebelumnya
near_dup_csv = outputs_dir / "near_duplicate_strong_main_candidates.csv"
contact_sheet_strong = outputs_dir / "near_duplicate_strong_pairs_contact_sheet.png"
contact_sheet_exact = outputs_dir / "near_duplicate_exact_ahash_pairs_contact_sheet.png"

# Cek status
status = {
    "Google Drive mounted (/content/drive/MyDrive)": drive_root.exists(),
    "Dataset root tersedia": dataset_root.exists(),
    "Folder crop 1nm_256x256 tersedia": crop_root.exists(),
    "Folder RAW tersedia": raw_root.exists(),
    "Folder outputs tersedia": outputs_dir.exists(),
    "File near_duplicate_strong_main_candidates.csv tersedia": near_dup_csv.exists(),
    "Contact sheet strong pairs tersedia": contact_sheet_strong.exists(),
    "Contact sheet exact ahash pairs tersedia": contact_sheet_exact.exists(),
}

print("=== Status Path Awal ===")
for k, v in status.items():
    print(f"{k}: {v}")

print("\n=== Detail Path ===")
print("Dataset root:", dataset_root)
print("Crop root   :", crop_root)
print("RAW root    :", raw_root)
print("Outputs dir :", outputs_dir)
print("Near-dup CSV:", near_dup_csv)
print("Strong sheet:", contact_sheet_strong)
print("Exact sheet :", contact_sheet_exact)

# Ringkasan tambahan jika file CSV tersedia
if near_dup_csv.exists():
    try:
        df = pd.read_csv(near_dup_csv)
        print("\n=== Ringkasan near_duplicate_strong_main_candidates.csv ===")
        print("Shape:", df.shape)
        print("Kolom:", list(df.columns))
        display(df.head())
    except Exception as e:
        print("\nCSV ditemukan, tetapi gagal dibaca.")
        print("Error:", repr(e))
else:
    print("\nPERINGATAN: File CSV kandidat near-duplicate belum ditemukan di path yang diharapkan.")

# Ringkasan folder output
if outputs_dir.exists():
    print("\n=== Beberapa file di folder outputs ===")
    files = sorted(outputs_dir.iterdir())
    for f in files[:30]:
        size_kb = f.stat().st_size / 1024 if f.is_file() else 0
        print(f"- {f.name} ({size_kb:.1f} KB)")
    if len(files) > 30:
        print(f"... dan {len(files) - 30} file lainnya.")

In [ ]:
# Sel 2 — Load dan Audit Ringkas Kandidat Near-Duplicate

import os
from pathlib import Path
import pandas as pd
import numpy as np

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
near_dup_csv = outputs_dir / "near_duplicate_strong_main_candidates.csv"

# Load kandidat near-duplicate kuat
df = pd.read_csv(near_dup_csv)

print("=== Informasi Dasar Kandidat Near-Duplicate Kuat ===")
print("Shape:", df.shape)
print("Kolom:", list(df.columns))

# Tambahkan kolom bantu
df["split_pair"] = df["split_a"].astype(str) + " -> " + df["split_b"].astype(str)

# Cek apakah file gambar benar-benar ada di folder crop
df["path_a"] = df["file_a"].apply(lambda x: str(crop_root / x))
df["path_b"] = df["file_b"].apply(lambda x: str(crop_root / x))
df["exists_a"] = df["path_a"].apply(lambda x: Path(x).exists())
df["exists_b"] = df["path_b"].apply(lambda x: Path(x).exists())

print("\n=== Cek Ketersediaan File Gambar ===")
print("File A tersedia:", df["exists_a"].sum(), "/", len(df))
print("File B tersedia:", df["exists_b"].sum(), "/", len(df))
print("Pasangan lengkap tersedia:", ((df["exists_a"]) & (df["exists_b"])).sum(), "/", len(df))

if not df["exists_a"].all() or not df["exists_b"].all():
    print("\nContoh file yang tidak ditemukan:")
    display(df.loc[(~df["exists_a"]) | (~df["exists_b"]),
                   ["class_name", "split_a", "file_a", "exists_a", "split_b", "file_b", "exists_b"]].head(10))

print("\n=== Distribusi Hamming Distance ===")
display(df["hamming_distance"].value_counts().sort_index().reset_index().rename(
    columns={"index": "hamming_distance", "hamming_distance": "count"}
))

print("\n=== Distribusi Pasangan Split ===")
display(df["split_pair"].value_counts().reset_index().rename(
    columns={"index": "split_pair", "split_pair": "count"}
))

print("\n=== Distribusi Kandidat per Kelas ===")
display(df["class_name"].value_counts().reset_index().rename(
    columns={"index": "class_name", "class_name": "count"}
))

print("\n=== Statistik Metrik Kemiripan ===")
metric_cols = ["hamming_distance", "mse_128", "mae_128", "corr_128"]
display(df[metric_cols].describe())

print("\n=== Contoh 10 Kandidat dengan Hamming Distance Terendah ===")
display(df.sort_values(["hamming_distance", "mse_128", "mae_128"]).head(10)[
    ["class_name", "split_a", "file_a", "split_b", "file_b",
     "hamming_distance", "mse_128", "mae_128", "corr_128"]
])

print("\n=== Contoh 10 Kandidat Acak ===")
display(df.sample(min(10, len(df)), random_state=42)[
    ["class_name", "split_a", "file_a", "split_b", "file_b",
     "hamming_distance", "mse_128", "mae_128", "corr_128"]
])

In [ ]:
# Sel 3 — Buat Sampel Kandidat Near-Duplicate untuk Validasi Visual

import os
from pathlib import Path
import pandas as pd
import numpy as np

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
near_dup_csv = outputs_dir / "near_duplicate_strong_main_candidates.csv"

# Load ulang kandidat
df = pd.read_csv(near_dup_csv)

# Tambahkan path lengkap
df["left_image"] = df["file_a"].apply(lambda x: str(crop_root / x))
df["right_image"] = df["file_b"].apply(lambda x: str(crop_root / x))
df["exists_left"] = df["left_image"].apply(lambda x: Path(x).exists())
df["exists_right"] = df["right_image"].apply(lambda x: Path(x).exists())

# Ambil hanya pasangan yang lengkap
df_ok = df[(df["exists_left"]) & (df["exists_right"])].copy()

print("Total kandidat awal:", len(df))
print("Total kandidat dengan file lengkap:", len(df_ok))

# Target sampel kandidat: 40 pasangan
# Dibuat agak seimbang berdasarkan hamming distance:
# - distance 0: maksimal 12
# - distance 1: maksimal 14
# - distance 2: sisanya sampai total 40
sample_plan = {
    0: 12,
    1: 14,
    2: 14
}

sample_parts = []

for hd, n_target in sample_plan.items():
    sub = df_ok[df_ok["hamming_distance"] == hd].copy()
    n_take = min(n_target, len(sub))
    if n_take > 0:
        # Sampling acak tapi reproducible
        sample_parts.append(sub.sample(n=n_take, random_state=100 + hd))

candidate_sample = pd.concat(sample_parts, ignore_index=True)

# Jika karena alasan tertentu belum mencapai 40, tambahkan dari sisa kandidat
target_total = 40
if len(candidate_sample) < target_total:
    used_index = set(candidate_sample.index)
    # Gunakan pembanding berbasis file agar aman, bukan index setelah concat
    used_pairs = set(zip(candidate_sample["file_a"], candidate_sample["file_b"]))
    remaining = df_ok[~df_ok.apply(lambda r: (r["file_a"], r["file_b"]) in used_pairs, axis=1)].copy()
    extra_needed = target_total - len(candidate_sample)
    extra = remaining.sample(n=min(extra_needed, len(remaining)), random_state=999)
    candidate_sample = pd.concat([candidate_sample, extra], ignore_index=True)

# Acak urutan kandidat
candidate_sample = candidate_sample.sample(frac=1, random_state=123).reset_index(drop=True)

# Buat ID sementara khusus kandidat
candidate_sample.insert(0, "pair_id", [f"CAND_{i+1:03d}" for i in range(len(candidate_sample))])

# Tambahkan kategori internal
candidate_sample["original_category"] = "candidate"

# Kolom untuk validasi manual nanti
candidate_sample["human_label"] = ""
candidate_sample["note"] = ""

# Rapikan kolom utama
candidate_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "left_image",
    "split_b",
    "file_b",
    "right_image",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "original_category",
    "human_label",
    "note"
]

candidate_sample_out = candidate_sample[candidate_cols].copy()

# Simpan output kandidat
candidate_sample_path = outputs_dir / "v1_candidate_near_duplicate_sample_40.csv"
candidate_sample_out.to_csv(candidate_sample_path, index=False)

print("\n=== Sampel Kandidat Near-Duplicate Dibuat ===")
print("Jumlah sampel kandidat:", len(candidate_sample_out))
print("File disimpan ke:")
print(candidate_sample_path)

print("\n=== Distribusi Hamming Distance dalam Sampel ===")
display(candidate_sample_out["hamming_distance"].value_counts().sort_index().reset_index().rename(
    columns={"index": "hamming_distance", "hamming_distance": "count"}
))

print("\n=== Distribusi Kelas dalam Sampel ===")
display(candidate_sample_out["class_name"].value_counts().reset_index().rename(
    columns={"index": "class_name", "class_name": "count"}
))

print("\n=== Distribusi Split Pair dalam Sampel ===")
tmp = candidate_sample_out.copy()
tmp["split_pair"] = tmp["split_a"].astype(str) + " -> " + tmp["split_b"].astype(str)
display(tmp["split_pair"].value_counts().reset_index().rename(
    columns={"index": "split_pair", "split_pair": "count"}
))

print("\n=== Preview Sampel Kandidat ===")
display(candidate_sample_out.head(10))

In [ ]:
# Sel 4 — Buat Sampel Kontrol Negatif Random Non-Kandidat

import os
from pathlib import Path
import pandas as pd
import numpy as np
import random

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

strong_csv = outputs_dir / "near_duplicate_strong_main_candidates.csv"
broad_candidate_csv = outputs_dir / "near_duplicate_candidate_pairs.csv"
candidate_sample_csv = outputs_dir / "v1_candidate_near_duplicate_sample_40.csv"

# Load kandidat kuat
strong_df = pd.read_csv(strong_csv)

# Fungsi untuk membuat pasangan file menjadi key yang urut,
# agar A-B dan B-A dianggap pasangan yang sama
def pair_key(a, b):
    return tuple(sorted([str(a), str(b)]))

# Buat exclusion set dari kandidat kuat
exclude_pairs = set()
for _, r in strong_df.iterrows():
    exclude_pairs.add(pair_key(r["file_a"], r["file_b"]))

# Jika file kandidat near-duplicate awal tersedia, pakai juga untuk exclusion
# agar kontrol benar-benar bukan kandidat dari daftar yang lebih luas
if broad_candidate_csv.exists():
    broad_df = pd.read_csv(broad_candidate_csv)
    print("File kandidat luas ditemukan:", broad_candidate_csv)
    print("Shape kandidat luas:", broad_df.shape)

    # Coba deteksi nama kolom file pasangan
    possible_a = [c for c in broad_df.columns if c.lower() in ["file_a", "path_a", "image_a", "left_image"]]
    possible_b = [c for c in broad_df.columns if c.lower() in ["file_b", "path_b", "image_b", "right_image"]]

    if len(possible_a) > 0 and len(possible_b) > 0:
        col_a = possible_a[0]
        col_b = possible_b[0]
        for _, r in broad_df.iterrows():
            exclude_pairs.add(pair_key(r[col_a], r[col_b]))
        print(f"Exclusion juga memakai kandidat luas dengan kolom: {col_a}, {col_b}")
    else:
        print("Kolom pasangan pada kandidat luas tidak terdeteksi otomatis.")
        print("Kolom tersedia:", list(broad_df.columns))
else:
    print("File kandidat luas tidak ditemukan. Exclusion hanya memakai kandidat kuat.")

print("Total pasangan dalam exclusion set:", len(exclude_pairs))

# Buat inventory semua gambar crop pada split resmi
records = []
for split in ["train", "validation", "test"]:
    split_dir = crop_root / split
    if not split_dir.exists():
        continue

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        for img_path in sorted(class_dir.glob("*.tif")):
            rel_path = str(img_path.relative_to(crop_root))
            records.append({
                "split": split,
                "class_name": class_dir.name,
                "file": rel_path,
                "image_path": str(img_path)
            })

inv = pd.DataFrame(records)

print("\n=== Inventory Gambar Crop ===")
print("Total gambar:", len(inv))
print("Jumlah kelas:", inv["class_name"].nunique())
display(inv.groupby(["split"])["file"].count().reset_index(name="n_images"))
display(inv.groupby(["class_name", "split"])["file"].count().reset_index(name="n_images").head(20))

# Buat kontrol negatif:
# - lintas split
# - kelas sama
# - bukan pasangan kandidat
# - file tersedia
random.seed(404)
np.random.seed(404)

split_pairs = [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]

target_controls = 15
controls = []
used_controls = set()

max_attempts = 20000
attempts = 0

classes = sorted(inv["class_name"].unique())

while len(controls) < target_controls and attempts < max_attempts:
    attempts += 1

    cls = random.choice(classes)
    sp_a, sp_b = random.choice(split_pairs)

    left_pool = inv[(inv["class_name"] == cls) & (inv["split"] == sp_a)]
    right_pool = inv[(inv["class_name"] == cls) & (inv["split"] == sp_b)]

    if len(left_pool) == 0 or len(right_pool) == 0:
        continue

    left = left_pool.sample(1, random_state=random.randint(0, 10**9)).iloc[0]
    right = right_pool.sample(1, random_state=random.randint(0, 10**9)).iloc[0]

    key = pair_key(left["file"], right["file"])

    if key in exclude_pairs:
        continue
    if key in used_controls:
        continue

    used_controls.add(key)

    controls.append({
        "pair_id": f"CTRL_{len(controls)+1:03d}",
        "class_name": cls,
        "split_a": sp_a,
        "file_a": left["file"],
        "left_image": left["image_path"],
        "split_b": sp_b,
        "file_b": right["file"],
        "right_image": right["image_path"],
        "hamming_distance": "",
        "mse_128": "",
        "mae_128": "",
        "corr_128": "",
        "original_category": "control",
        "human_label": "",
        "note": ""
    })

control_df = pd.DataFrame(controls)

control_csv = outputs_dir / "v1_control_negative_random_sample_15.csv"
control_df.to_csv(control_csv, index=False)

print("\n=== Sampel Kontrol Negatif Dibuat ===")
print("Jumlah kontrol:", len(control_df))
print("Jumlah attempts:", attempts)
print("File disimpan ke:")
print(control_csv)

print("\n=== Distribusi Kelas Kontrol ===")
display(control_df["class_name"].value_counts().reset_index().rename(
    columns={"index": "class_name", "class_name": "count"}
))

print("\n=== Distribusi Split Pair Kontrol ===")
tmp = control_df.copy()
tmp["split_pair"] = tmp["split_a"].astype(str) + " -> " + tmp["split_b"].astype(str)
display(tmp["split_pair"].value_counts().reset_index().rename(
    columns={"index": "split_pair", "split_pair": "count"}
))

print("\n=== Preview Kontrol Negatif ===")
display(control_df)

In [ ]:
# Sel 5 — Gabungkan Kandidat + Kontrol, Acak, dan Siapkan File Validasi Manual

import pandas as pd
from pathlib import Path

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

candidate_csv = outputs_dir / "v1_candidate_near_duplicate_sample_40.csv"
control_csv = outputs_dir / "v1_control_negative_random_sample_15.csv"

# Load
cand = pd.read_csv(candidate_csv)
ctrl = pd.read_csv(control_csv)

print("Jumlah kandidat:", len(cand))
print("Jumlah kontrol:", len(ctrl))

# Gabungkan
df_all = pd.concat([cand, ctrl], ignore_index=True)

# Simpan kolom asli (untuk ground truth nanti, tidak ditampilkan)
df_all["original_category_hidden"] = df_all["original_category"]

# Acak urutan
df_all = df_all.sample(frac=1, random_state=777).reset_index(drop=True)

# Buat ID baru supaya tidak ketahuan urutan awal
df_all["pair_id"] = [f"PAIR_{i+1:03d}" for i in range(len(df_all))]

# Versi untuk VALIDASI MANUAL (tanpa info kandidat/kontrol)
df_manual = df_all[[
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "left_image",
    "split_b",
    "file_b",
    "right_image",
    "human_label",
    "note"
]].copy()

# Versi FULL (untuk kita, nanti evaluasi)
df_full = df_all.copy()

# Simpan
manual_csv = outputs_dir / "v1_visual_validation_blind.csv"
full_csv = outputs_dir / "v1_visual_validation_with_groundtruth.csv"

df_manual.to_csv(manual_csv, index=False)
df_full.to_csv(full_csv, index=False)

print("\n=== File Validasi Visual Dibuat ===")
print("Blind (untuk anotasi manual):")
print(manual_csv)

print("\nFull (dengan ground truth tersembunyi):")
print(full_csv)

print("\nJumlah total pasangan:", len(df_all))
print("\nPreview data blind:")
display(df_manual.head(10))

In [ ]:
# Sel 6 — Generate Contact Sheet Campuran untuk Validasi Visual Blind

from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont
from IPython.display import display, Image as IPyImage
import math
import textwrap

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

blind_csv = outputs_dir / "v1_visual_validation_blind.csv"
contact_sheet_path = outputs_dir / "v1_visual_validation_blind_contact_sheet.png"

df = pd.read_csv(blind_csv)

print("Jumlah pasangan untuk contact sheet:", len(df))

# Parameter tampilan
thumb_size = 160
gap = 10
text_h = 72
tile_w = thumb_size * 2 + gap
tile_h = thumb_size + text_h + gap

n_cols = 5
n_rows = math.ceil(len(df) / n_cols)

sheet_w = n_cols * tile_w + (n_cols + 1) * gap
sheet_h = n_rows * tile_h + (n_rows + 1) * gap

sheet = Image.new("RGB", (sheet_w, sheet_h), "white")
draw = ImageDraw.Draw(sheet)

# Font default aman di Colab
try:
    font = ImageFont.truetype("DejaVuSans.ttf", 14)
    font_small = ImageFont.truetype("DejaVuSans.ttf", 11)
except:
    font = ImageFont.load_default()
    font_small = ImageFont.load_default()

def load_tem_image(path, size=160):
    """Load tif, convert ke grayscale/RGB, autocontrast, resize."""
    img = Image.open(path)
    img = img.convert("L")
    img = ImageOps.autocontrast(img)
    img = img.resize((size, size))
    img = img.convert("RGB")
    return img

def short_text(s, max_len=34):
    s = str(s)
    if len(s) <= max_len:
        return s
    return s[:max_len-3] + "..."

missing = []

for idx, row in df.iterrows():
    r = idx // n_cols
    c = idx % n_cols

    x0 = gap + c * tile_w + c * gap
    y0 = gap + r * tile_h + r * gap

    pair_id = row["pair_id"]
    cls = row["class_name"]
    split_info = f'{row["split_a"]} → {row["split_b"]}'

    left_path = Path(row["left_image"])
    right_path = Path(row["right_image"])

    # Text header
    draw.rectangle([x0, y0, x0 + tile_w, y0 + text_h], fill=(245, 245, 245), outline=(180, 180, 180))
    draw.text((x0 + 5, y0 + 5), f"{pair_id} | {cls}", fill="black", font=font)
    draw.text((x0 + 5, y0 + 25), f"{split_info}", fill="black", font=font_small)
    draw.text((x0 + 5, y0 + 42), f"L: {short_text(row['file_a'])}", fill="black", font=font_small)
    draw.text((x0 + 5, y0 + 56), f"R: {short_text(row['file_b'])}", fill="black", font=font_small)

    # Area gambar
    img_y = y0 + text_h

    if left_path.exists() and right_path.exists():
        left_img = load_tem_image(left_path, thumb_size)
        right_img = load_tem_image(right_path, thumb_size)

        sheet.paste(left_img, (x0, img_y))
        sheet.paste(right_img, (x0 + thumb_size + gap, img_y))

        # Border kiri dan kanan
        draw.rectangle([x0, img_y, x0 + thumb_size, img_y + thumb_size], outline=(0, 0, 0), width=1)
        draw.rectangle([x0 + thumb_size + gap, img_y, x0 + thumb_size + gap + thumb_size, img_y + thumb_size], outline=(0, 0, 0), width=1)

        # Label L/R kecil
        draw.text((x0 + 4, img_y + 4), "L", fill="black", font=font)
        draw.text((x0 + thumb_size + gap + 4, img_y + 4), "R", fill="black", font=font)
    else:
        missing.append(pair_id)
        draw.text((x0 + 5, img_y + 40), "IMAGE NOT FOUND", fill="red", font=font)

# Simpan
sheet.save(contact_sheet_path)

print("\n=== Contact Sheet Berhasil Dibuat ===")
print("Path:")
print(contact_sheet_path)
print("Ukuran sheet:", sheet.size)
print("Jumlah missing image:", len(missing))

if missing:
    print("PAIR_ID missing:", missing)

print("\nPreview contact sheet:")
display(IPyImage(filename=str(contact_sheet_path)))

In [ ]:
# Sel 7 — Siapkan File Anotasi Manual dan Panduan Label

from pathlib import Path
import pandas as pd

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

blind_csv = outputs_dir / "v1_visual_validation_blind.csv"
annotation_csv = outputs_dir / "v1_visual_validation_manual_annotation.csv"
guide_txt = outputs_dir / "v1_visual_validation_labeling_guide.txt"

# Load file blind
df = pd.read_csv(blind_csv)

# Pastikan kolom human_label dan note kosong, bukan NaN
df["human_label"] = ""
df["note"] = ""

# Tambahkan kolom bantuan, tetapi tetap tidak membuka candidate/control
df["allowed_labels"] = "duplicate / similar / not duplicate / uncertain"

# Urutan kolom yang enak untuk anotasi
cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "human_label",
    "note",
    "allowed_labels",
    "left_image",
    "right_image"
]

df_annot = df[cols].copy()

# Simpan CSV anotasi manual
df_annot.to_csv(annotation_csv, index=False)

# Buat panduan label
guide = """PANDUAN VALIDASI VISUAL NEAR-DUPLICATE TEM VIRUS

File contact sheet:
v1_visual_validation_blind_contact_sheet.png

File anotasi:
v1_visual_validation_manual_annotation.csv

Isi kolom human_label dengan salah satu dari:
1. duplicate
   Jika pasangan gambar tampak hampir sama atau sangat mungkin berasal dari area/source yang sama.
   Contoh indikator: struktur utama sama, posisi objek mirip, tekstur latar sangat mirip, hanya beda crop/kontras/rotasi kecil.

2. similar
   Jika pasangan gambar tampak mirip secara visual, tetapi tidak cukup kuat disebut duplicate.
   Contoh: kelas sama dan bentuk virus mirip, tetapi posisi/area/tekstur tidak jelas sama.

3. not duplicate
   Jika pasangan gambar jelas berbeda.
   Contoh: objek berbeda, jumlah partikel berbeda jauh, area visual berbeda, pola latar berbeda.

4. uncertain
   Jika ragu antara duplicate/similar/not duplicate.

Catatan penting:
- Jangan melihat file v1_visual_validation_with_groundtruth.csv saat anotasi.
- Gunakan contact sheet blind agar tidak bias.
- Kolom note boleh diisi alasan singkat, misalnya:
  'tekstur latar sama', 'objek sama tapi crop berbeda', 'mirip kelas saja', 'ragu karena blur'.
- Tidak perlu sempurna, yang penting konsisten.
"""

guide_txt.write_text(guide, encoding="utf-8")

print("=== File Anotasi Manual Siap ===")
print("CSV anotasi manual:")
print(annotation_csv)

print("\nPanduan label:")
print(guide_txt)

print("\nJumlah pasangan untuk dianotasi:", len(df_annot))

print("\nCek kolom:")
print(list(df_annot.columns))

print("\nPreview 10 baris pertama:")
display(df_annot.head(10))

In [ ]:
# Sel 8 — Cek Hasil Anotasi Manual dan Bandingkan dengan Ground Truth Tersembunyi

from pathlib import Path
import pandas as pd
import json

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

annotation_csv = outputs_dir / "v1_visual_validation_manual_annotation.csv"
groundtruth_csv = outputs_dir / "v1_visual_validation_with_groundtruth.csv"

merged_out_csv = outputs_dir / "v1_visual_validation_manual_results_merged.csv"
summary_out_csv = outputs_dir / "v1_visual_validation_manual_results_summary.csv"
summary_out_json = outputs_dir / "v1_visual_validation_manual_results_summary.json"

# Load file
annot = pd.read_csv(annotation_csv)
gt = pd.read_csv(groundtruth_csv)

print("=== File Dibaca ===")
print("Annotation shape:", annot.shape)
print("Groundtruth shape:", gt.shape)

# Normalisasi label
allowed_labels = ["duplicate", "similar", "not duplicate", "uncertain"]

annot["human_label_raw"] = annot["human_label"]
annot["human_label"] = (
    annot["human_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Cek kosong dan invalid
missing_label = annot[annot["human_label"] == ""].copy()
invalid_label = annot[
    (annot["human_label"] != "") & (~annot["human_label"].isin(allowed_labels))
].copy()

print("\n=== Cek Kelengkapan Label ===")
print("Total pasangan:", len(annot))
print("Label terisi:", (annot["human_label"] != "").sum())
print("Label kosong:", len(missing_label))
print("Label salah/invalid:", len(invalid_label))

if len(missing_label) > 0:
    print("\nPAIR_ID yang masih kosong:")
    display(missing_label[["pair_id", "class_name", "split_a", "file_a", "split_b", "file_b", "human_label_raw"]])

if len(invalid_label) > 0:
    print("\nPAIR_ID dengan label invalid:")
    display(invalid_label[["pair_id", "class_name", "human_label_raw", "human_label"]])
    print("\nLabel yang diperbolehkan:", allowed_labels)

# Siapkan ground truth category
if "original_category_hidden" in gt.columns:
    gt_category_col = "original_category_hidden"
elif "original_category" in gt.columns:
    gt_category_col = "original_category"
else:
    raise ValueError("Kolom original_category/original_category_hidden tidak ditemukan di groundtruth CSV.")

gt_small = gt[["pair_id", gt_category_col]].copy()
gt_small = gt_small.rename(columns={gt_category_col: "original_category_hidden"})

# Merge
merged = annot.merge(gt_small, on="pair_id", how="left")

# Simpan merged, walaupun masih ada label kosong/invalid
merged.to_csv(merged_out_csv, index=False)

print("\n=== Cek Merge Ground Truth ===")
print("Merged shape:", merged.shape)
print("Ground truth missing:", merged["original_category_hidden"].isna().sum())

# Tampilkan ringkasan hanya kalau semua label valid dan lengkap
if len(missing_label) == 0 and len(invalid_label) == 0:
    print("\n=== Distribusi Label Manual Keseluruhan ===")
    label_counts = merged["human_label"].value_counts().reindex(allowed_labels, fill_value=0).reset_index()
    label_counts.columns = ["human_label", "count"]
    display(label_counts)

    print("\n=== Distribusi Label Manual per Kategori Tersembunyi ===")
    category_label_table = pd.crosstab(
        merged["original_category_hidden"],
        merged["human_label"]
    ).reindex(columns=allowed_labels, fill_value=0)
    display(category_label_table)

    print("\n=== Ringkasan Candidate vs Control ===")
    summary_rows = []
    for cat in ["candidate", "control"]:
        sub = merged[merged["original_category_hidden"] == cat]
        row = {
            "original_category": cat,
            "n_pairs": len(sub),
            "duplicate": (sub["human_label"] == "duplicate").sum(),
            "similar": (sub["human_label"] == "similar").sum(),
            "not_duplicate": (sub["human_label"] == "not duplicate").sum(),
            "uncertain": (sub["human_label"] == "uncertain").sum(),
            "duplicate_or_similar": sub["human_label"].isin(["duplicate", "similar"]).sum()
        }
        row["duplicate_or_similar_rate"] = row["duplicate_or_similar"] / row["n_pairs"] if row["n_pairs"] > 0 else 0
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)

    # Simpan summary
    summary_df.to_csv(summary_out_csv, index=False)

    summary_json = {
        "total_pairs": int(len(merged)),
        "label_counts": label_counts.to_dict(orient="records"),
        "category_summary": summary_df.to_dict(orient="records"),
        "outputs": {
            "merged_csv": str(merged_out_csv),
            "summary_csv": str(summary_out_csv),
            "summary_json": str(summary_out_json)
        }
    }

    summary_out_json.write_text(json.dumps(summary_json, indent=2), encoding="utf-8")

    print("\n=== Output Hasil Validasi Visual Disimpan ===")
    print("Merged CSV :", merged_out_csv)
    print("Summary CSV:", summary_out_csv)
    print("Summary JSON:", summary_out_json)

else:
    print("\nAnotasi belum lengkap/valid. Perbaiki dulu label kosong atau invalid di CSV anotasi, lalu jalankan ulang Sel 8.")
    print("Merged sementara tetap disimpan ke:")
    print(merged_out_csv)

In [ ]:
# Sel 8A — Cek File Anotasi Manual Versi Filled

from pathlib import Path
import pandas as pd

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

filled_csv = outputs_dir / "v1_visual_validation_manual_annotation_filled.csv"

print("File filled tersedia:", filled_csv.exists())
print("Path:", filled_csv)

if filled_csv.exists():
    df = pd.read_csv(filled_csv)

    print("\nShape:", df.shape)
    print("Kolom:", list(df.columns))

    df["human_label_check"] = (
        df["human_label"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    print("\nLabel terisi:", (df["human_label_check"] != "").sum())
    print("Label kosong:", (df["human_label_check"] == "").sum())

    print("\nDistribusi label:")
    display(df["human_label_check"].value_counts(dropna=False).reset_index().rename(
        columns={"index": "human_label", "human_label_check": "count"}
    ))

    print("\nPreview:")
    display(df[["pair_id", "class_name", "human_label", "note"]].head(10))
else:
    print("\nUpload dulu file CSV yang sudah kamu isi ke folder outputs dengan nama:")
    print("v1_visual_validation_manual_annotation_filled.csv")

In [ ]:
# Sel 9 — Evaluasi Hasil Anotasi Filled terhadap Candidate/Control

from pathlib import Path
import pandas as pd
import json

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

filled_csv = outputs_dir / "v1_visual_validation_manual_annotation_filled.csv"
groundtruth_csv = outputs_dir / "v1_visual_validation_with_groundtruth.csv"

merged_out_csv = outputs_dir / "v1_visual_validation_manual_results_merged_filled.csv"
summary_out_csv = outputs_dir / "v1_visual_validation_manual_results_summary_filled.csv"
summary_out_json = outputs_dir / "v1_visual_validation_manual_results_summary_filled.json"

# Load
annot = pd.read_csv(filled_csv)
gt = pd.read_csv(groundtruth_csv)

# Normalisasi label
allowed_labels = ["duplicate", "similar", "not duplicate", "uncertain"]

annot["human_label_raw"] = annot["human_label"]
annot["human_label"] = (
    annot["human_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Cek label
missing_label = annot[annot["human_label"] == ""].copy()
invalid_label = annot[
    (annot["human_label"] != "") & (~annot["human_label"].isin(allowed_labels))
].copy()

print("=== Cek Label Filled ===")
print("Total pasangan:", len(annot))
print("Label terisi:", (annot["human_label"] != "").sum())
print("Label kosong:", len(missing_label))
print("Label invalid:", len(invalid_label))

if len(invalid_label) > 0:
    print("\nLabel invalid ditemukan:")
    display(invalid_label[["pair_id", "human_label_raw", "human_label"]])

# Ambil kategori tersembunyi
if "original_category_hidden" in gt.columns:
    gt_category_col = "original_category_hidden"
elif "original_category" in gt.columns:
    gt_category_col = "original_category"
else:
    raise ValueError("Kolom original_category/original_category_hidden tidak ditemukan.")

gt_small = gt[["pair_id", gt_category_col]].copy()
gt_small = gt_small.rename(columns={gt_category_col: "original_category_hidden"})

# Ambil metrik tambahan dari groundtruth jika ada
extra_cols = [
    "pair_id",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128"
]
available_extra_cols = [c for c in extra_cols if c in gt.columns]
gt_extra = gt[available_extra_cols].copy()

# Merge
merged = annot.merge(gt_small, on="pair_id", how="left")
merged = merged.merge(gt_extra, on="pair_id", how="left")

print("\n=== Cek Merge ===")
print("Merged shape:", merged.shape)
print("Ground truth missing:", merged["original_category_hidden"].isna().sum())

# Simpan hasil merge
merged.to_csv(merged_out_csv, index=False)

if len(missing_label) == 0 and len(invalid_label) == 0:
    print("\n=== Distribusi Label Manual Keseluruhan ===")
    label_counts = (
        merged["human_label"]
        .value_counts()
        .reindex(allowed_labels, fill_value=0)
        .reset_index()
    )
    label_counts.columns = ["human_label", "count"]
    display(label_counts)

    print("\n=== Crosstab Candidate/Control vs Label Manual ===")
    crosstab = pd.crosstab(
        merged["original_category_hidden"],
        merged["human_label"]
    ).reindex(columns=allowed_labels, fill_value=0)
    display(crosstab)

    print("\n=== Ringkasan Candidate vs Control ===")
    summary_rows = []
    for cat in ["candidate", "control"]:
        sub = merged[merged["original_category_hidden"] == cat].copy()

        duplicate_n = (sub["human_label"] == "duplicate").sum()
        similar_n = (sub["human_label"] == "similar").sum()
        notdup_n = (sub["human_label"] == "not duplicate").sum()
        uncertain_n = (sub["human_label"] == "uncertain").sum()
        dup_or_sim_n = sub["human_label"].isin(["duplicate", "similar"]).sum()

        row = {
            "original_category": cat,
            "n_pairs": len(sub),
            "duplicate": int(duplicate_n),
            "similar": int(similar_n),
            "not_duplicate": int(notdup_n),
            "uncertain": int(uncertain_n),
            "duplicate_or_similar": int(dup_or_sim_n),
            "duplicate_rate": duplicate_n / len(sub) if len(sub) > 0 else 0,
            "duplicate_or_similar_rate": dup_or_sim_n / len(sub) if len(sub) > 0 else 0,
            "not_duplicate_rate": notdup_n / len(sub) if len(sub) > 0 else 0
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)

    print("\n=== Detail Pasangan yang Dinilai Duplicate ===")
    display(merged[merged["human_label"] == "duplicate"][
        ["pair_id", "original_category_hidden", "class_name", "split_a", "file_a",
         "split_b", "file_b", "human_label", "note",
         "hamming_distance", "mse_128", "mae_128", "corr_128"]
    ])

    print("\n=== Detail Kontrol yang Dinilai Similar/Duplicate ===")
    display(merged[
        (merged["original_category_hidden"] == "control") &
        (merged["human_label"].isin(["duplicate", "similar"]))
    ][
        ["pair_id", "class_name", "split_a", "file_a",
         "split_b", "file_b", "human_label", "note"]
    ])

    # Simpan summary
    summary_df.to_csv(summary_out_csv, index=False)

    summary_json = {
        "total_pairs": int(len(merged)),
        "label_counts": label_counts.to_dict(orient="records"),
        "category_summary": summary_df.to_dict(orient="records"),
        "outputs": {
            "merged_csv": str(merged_out_csv),
            "summary_csv": str(summary_out_csv),
            "summary_json": str(summary_out_json)
        }
    }

    summary_out_json.write_text(json.dumps(summary_json, indent=2), encoding="utf-8")

    print("\n=== Output Disimpan ===")
    print("Merged CSV :", merged_out_csv)
    print("Summary CSV:", summary_out_csv)
    print("Summary JSON:", summary_out_json)

else:
    print("\nMasih ada label kosong atau invalid. Perbaiki file filled lalu jalankan ulang Sel 9.")

In [ ]:
# Sel 10 — Cek dan Ekstrak dataset code.zip untuk Analisis Mapping Crop ke RAW

from pathlib import Path
import zipfile
import pandas as pd
import json
import os

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Target zip kode dataset
dataset_code_zip = crop_root / "dataset code.zip"

# Folder ekstraksi aman di outputs, bukan di folder dataset asli
extract_dir = outputs_dir / "v2_dataset_code_extracted"
extract_dir.mkdir(parents=True, exist_ok=True)

print("=== Cek dataset code.zip ===")
print("Crop root:", crop_root)
print("dataset code.zip path:", dataset_code_zip)
print("File tersedia:", dataset_code_zip.exists())

# Cari zip lain jika path utama tidak ditemukan
if not dataset_code_zip.exists():
    print("\nFile dataset code.zip tidak ditemukan di path utama.")
    print("Mencari file .zip lain di dataset_root...")
    zip_candidates = list(dataset_root.rglob("*.zip"))
    for z in zip_candidates:
        print("-", z)
else:
    print("Ukuran zip MB:", dataset_code_zip.stat().st_size / (1024 * 1024))

# Ekstrak jika tersedia
manifest_records = []

if dataset_code_zip.exists():
    with zipfile.ZipFile(dataset_code_zip, "r") as zf:
        zip_infos = zf.infolist()

        print("\n=== Isi ZIP ===")
        print("Jumlah item dalam zip:", len(zip_infos))

        # Ekstrak semua ke folder outputs/v2_dataset_code_extracted
        zf.extractall(extract_dir)

        for info in zip_infos:
            manifest_records.append({
                "filename_in_zip": info.filename,
                "is_dir": info.is_dir(),
                "file_size": info.file_size,
                "compressed_size": info.compress_size,
                "extracted_path": str(extract_dir / info.filename)
            })

    manifest_df = pd.DataFrame(manifest_records)

    # Simpan manifest
    manifest_csv = outputs_dir / "v2_dataset_code_manifest.csv"
    manifest_df.to_csv(manifest_csv, index=False)

    # Cari file kode penting
    code_exts = [".m", ".py", ".txt", ".md", ".csv"]
    code_files = []

    for p in extract_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in code_exts:
            code_files.append(p)

    code_df = pd.DataFrame({
        "file_name": [p.name for p in code_files],
        "suffix": [p.suffix for p in code_files],
        "relative_path": [str(p.relative_to(extract_dir)) for p in code_files],
        "full_path": [str(p) for p in code_files],
        "size_kb": [p.stat().st_size / 1024 for p in code_files]
    }).sort_values(["suffix", "file_name"])

    code_manifest_csv = outputs_dir / "v2_dataset_code_files_manifest.csv"
    code_df.to_csv(code_manifest_csv, index=False)

    # Ringkasan JSON
    summary = {
        "dataset_code_zip": str(dataset_code_zip),
        "zip_exists": True,
        "extract_dir": str(extract_dir),
        "n_items_in_zip": len(zip_infos),
        "n_code_like_files": len(code_files),
        "manifest_csv": str(manifest_csv),
        "code_manifest_csv": str(code_manifest_csv)
    }

    summary_json = outputs_dir / "v2_dataset_code_extraction_summary.json"
    summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\n=== Ekstraksi Selesai ===")
    print("Folder ekstraksi:")
    print(extract_dir)

    print("\nManifest ZIP:")
    print(manifest_csv)

    print("\nManifest file kode:")
    print(code_manifest_csv)

    print("\nSummary JSON:")
    print(summary_json)

    print("\n=== File Kode yang Ditemukan ===")
    display(code_df)

    # Highlight script penting
    important_keywords = [
        "getAllVirusData",
        "augmentContextVirusImages",
        "getImageTagInfo",
        "Virus",
        "Context",
        "crop",
        "particle"
    ]

    important = code_df[
        code_df["file_name"].apply(
            lambda x: any(k.lower() in x.lower() for k in important_keywords)
        )
    ].copy()

    print("\n=== Kandidat Script Penting ===")
    if len(important) > 0:
        display(important)
    else:
        print("Belum ada nama file yang cocok dengan keyword penting.")

else:
    summary = {
        "dataset_code_zip": str(dataset_code_zip),
        "zip_exists": False,
        "extract_dir": str(extract_dir)
    }

    summary_json = outputs_dir / "v2_dataset_code_extraction_summary.json"
    summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nTidak bisa ekstrak karena dataset code.zip tidak tersedia.")
    print("Summary JSON tetap disimpan:")
    print(summary_json)

In [ ]:
# Sel 11 — Analisis Isi Script MATLAB untuk Mapping Crop ke RAW

from pathlib import Path
import pandas as pd
import re
import json

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
extract_dir = outputs_dir / "v2_dataset_code_extracted"

# File output
overview_csv = outputs_dir / "v2_matlab_script_overview.csv"
keyword_hits_csv = outputs_dir / "v2_matlab_keyword_hits.csv"
full_text_txt = outputs_dir / "v2_matlab_code_full_text_with_line_numbers.txt"
summary_json = outputs_dir / "v2_matlab_code_analysis_summary.json"

matlab_files = sorted(extract_dir.rglob("*.m"))

print("=== File MATLAB Ditemukan ===")
print("Jumlah file .m:", len(matlab_files))
for p in matlab_files:
    print("-", p.name)

def read_text_safe(path):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    return path.read_text(errors="ignore")

# Keyword penting untuk mapping crop ke RAW
keywords = [
    "imread",
    "imwrite",
    "imcrop",
    "crop",
    "particle",
    "position",
    "positions",
    "context",
    "raw",
    "tag",
    "filename",
    "fileName",
    "imageName",
    "folder",
    "train",
    "validation",
    "test",
    "rand",
    "split",
    "mkdir",
    "csv",
    "txt",
    "tif",
    "resize",
    "augment",
    "select"
]

overview_records = []
hit_records = []
full_text_parts = []

for path in matlab_files:
    text = read_text_safe(path)
    lines = text.splitlines()

    overview_records.append({
        "file_name": path.name,
        "relative_path": str(path.relative_to(extract_dir)),
        "n_lines": len(lines),
        "size_kb": path.stat().st_size / 1024
    })

    full_text_parts.append("=" * 90)
    full_text_parts.append(f"FILE: {path.name}")
    full_text_parts.append("=" * 90)

    for i, line in enumerate(lines, start=1):
        numbered = f"{i:04d}: {line}"
        full_text_parts.append(numbered)

        low = line.lower()
        matched = [kw for kw in keywords if kw.lower() in low]
        if matched:
            hit_records.append({
                "file_name": path.name,
                "line_no": i,
                "matched_keywords": ", ".join(matched),
                "line_text": line.strip()
            })

# Simpan full text dengan line number
full_text_txt.write_text("\n".join(full_text_parts), encoding="utf-8")

overview_df = pd.DataFrame(overview_records)
hits_df = pd.DataFrame(hit_records)

overview_df.to_csv(overview_csv, index=False)
hits_df.to_csv(keyword_hits_csv, index=False)

summary = {
    "n_matlab_files": len(matlab_files),
    "matlab_files": [p.name for p in matlab_files],
    "overview_csv": str(overview_csv),
    "keyword_hits_csv": str(keyword_hits_csv),
    "full_text_txt": str(full_text_txt),
    "n_keyword_hits": int(len(hits_df))
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\n=== Overview Script ===")
display(overview_df)

print("\n=== Jumlah Keyword Hit per File ===")
if len(hits_df) > 0:
    display(hits_df.groupby("file_name").size().reset_index(name="n_hits").sort_values("n_hits", ascending=False))
else:
    print("Tidak ada keyword hit.")

print("\n=== Baris Penting Terkait imread/imwrite/crop/particle/position ===")
important_pattern = r"imread|imwrite|imcrop|crop|particle|position|positions|raw|filename|fileName|imageName|tif"
important_hits = hits_df[
    hits_df["line_text"].str.contains(important_pattern, case=False, regex=True, na=False)
].copy()

display(important_hits)

print("\n=== Baris Penting Terkait Split Train/Validation/Test ===")
split_pattern = r"train|validation|test|split|rand|select"
split_hits = hits_df[
    hits_df["line_text"].str.contains(split_pattern, case=False, regex=True, na=False)
].copy()

display(split_hits)

print("\n=== Preview Excerpt per File: Baris dengan Keyword Penting ±2 Baris ===")

for path in matlab_files:
    text = read_text_safe(path)
    lines = text.splitlines()

    hit_line_nos = hits_df.loc[hits_df["file_name"] == path.name, "line_no"].tolist()
    important_line_nos = []

    for ln in hit_line_nos:
        line_text = lines[ln - 1] if 0 <= ln - 1 < len(lines) else ""
        if re.search(important_pattern + "|" + split_pattern, line_text, flags=re.IGNORECASE):
            important_line_nos.append(ln)

    if not important_line_nos:
        continue

    print("\n" + "=" * 80)
    print("FILE:", path.name)
    print("=" * 80)

    shown = set()
    for ln in important_line_nos:
        start = max(1, ln - 2)
        end = min(len(lines), ln + 2)

        block_key = (start, end)
        if block_key in shown:
            continue
        shown.add(block_key)

        print(f"\n--- sekitar baris {ln} ---")
        for j in range(start, end + 1):
            prefix = ">>" if j == ln else "  "
            print(f"{prefix} {j:04d}: {lines[j-1]}")

print("\n=== Output Disimpan ===")
print("Overview CSV     :", overview_csv)
print("Keyword hits CSV :", keyword_hits_csv)
print("Full text TXT    :", full_text_txt)
print("Summary JSON     :", summary_json)

In [ ]:
# Sel 12 — Uji Mapping Awal Crop 1nm_256x256 ke RAW Berdasarkan iImg_iPar

from pathlib import Path
import pandas as pd
import re
import json
from collections import defaultdict

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"
raw_root = dataset_root / "context_virus_RAW"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

mapping_csv = outputs_dir / "v2_crop_to_raw_initial_mapping.csv"
mapping_summary_csv = outputs_dir / "v2_crop_to_raw_initial_mapping_summary.csv"
mapping_summary_json = outputs_dir / "v2_crop_to_raw_initial_mapping_summary.json"

splits = ["train", "validation", "test"]

def count_particle_positions(txt_path):
    """Hitung jumlah particleposition dalam file particle_positions."""
    if not txt_path.exists():
        return None
    try:
        text = txt_path.read_text(encoding="utf-8", errors="ignore")
        return text.count("particleposition")
    except Exception:
        return None

records = []
summary_records = []

for split in splits:
    crop_split_dir = crop_root / split
    raw_split_dir = raw_root / split

    if not crop_split_dir.exists():
        print("Folder crop split tidak ditemukan:", crop_split_dir)
        continue

    for class_dir in sorted(crop_split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name
        raw_class_dir = raw_split_dir / class_name

        crop_files = sorted(class_dir.glob("*.tif"))
        raw_files = sorted(raw_class_dir.glob("*.tif")) if raw_class_dir.exists() else []

        # Mapping iImg -> raw file berdasarkan urutan sorted raw_files
        raw_index_map = {i + 1: p for i, p in enumerate(raw_files)}

        # Hitung crop per base iImg
        crop_base_counts = defaultdict(int)

        for crop_path in crop_files:
            m = re.match(r"^(\d+)_(\d+)\.tif$", crop_path.name)

            if m:
                iImg = int(m.group(1))
                iPar = int(m.group(2))
            else:
                iImg = None
                iPar = None

            raw_path = raw_index_map.get(iImg, None) if iImg is not None else None

            if raw_path is not None:
                particle_txt = raw_class_dir / "particle_positions" / f"{raw_path.stem}_particlepositions.txt"
                tags_csv = raw_class_dir / "tags" / f"{raw_path.name}_tags.csv"
                particle_count = count_particle_positions(particle_txt)
            else:
                particle_txt = None
                tags_csv = None
                particle_count = None

            if iImg is not None:
                crop_base_counts[iImg] += 1

            records.append({
                "split": split,
                "class_name": class_name,
                "crop_file": str(crop_path.relative_to(crop_root)),
                "crop_filename": crop_path.name,
                "iImg_from_crop": iImg,
                "iPar_from_crop": iPar,
                "raw_file_mapped": str(raw_path.relative_to(raw_root)) if raw_path is not None else "",
                "raw_filename_mapped": raw_path.name if raw_path is not None else "",
                "particle_positions_file": str(particle_txt.relative_to(raw_root)) if particle_txt is not None else "",
                "particle_positions_exists": particle_txt.exists() if particle_txt is not None else False,
                "tags_file": str(tags_csv.relative_to(raw_root)) if tags_csv is not None else "",
                "tags_exists": tags_csv.exists() if tags_csv is not None else False,
                "raw_particle_count": particle_count
            })

        # Ringkasan per split/kelas
        valid_iImg = [k for k in crop_base_counts.keys() if k is not None]
        mapped_iImg = [k for k in valid_iImg if k in raw_index_map]
        unmapped_iImg = [k for k in valid_iImg if k not in raw_index_map]

        # Cek kecocokan crop count vs jumlah particleposition
        per_img_checks = []
        for iImg, n_crops in crop_base_counts.items():
            raw_path = raw_index_map.get(iImg, None)
            if raw_path is None:
                per_img_checks.append(False)
                continue

            particle_txt = raw_class_dir / "particle_positions" / f"{raw_path.stem}_particlepositions.txt"
            particle_count = count_particle_positions(particle_txt)

            if particle_count is None:
                per_img_checks.append(False)
            else:
                per_img_checks.append(n_crops == particle_count)

        n_match = sum(per_img_checks)
        n_checked = len(per_img_checks)

        summary_records.append({
            "split": split,
            "class_name": class_name,
            "n_crop_files": len(crop_files),
            "n_raw_files": len(raw_files),
            "n_unique_iImg_in_crop": len(valid_iImg),
            "min_iImg": min(valid_iImg) if valid_iImg else None,
            "max_iImg": max(valid_iImg) if valid_iImg else None,
            "n_iImg_mapped_to_raw": len(mapped_iImg),
            "n_iImg_unmapped": len(unmapped_iImg),
            "n_iImg_particle_count_checked": n_checked,
            "n_iImg_crop_count_matches_particle_count": n_match,
            "particle_count_match_rate": n_match / n_checked if n_checked > 0 else None
        })

mapping_df = pd.DataFrame(records)
summary_df = pd.DataFrame(summary_records)

mapping_df.to_csv(mapping_csv, index=False)
summary_df.to_csv(mapping_summary_csv, index=False)

summary_json = {
    "n_mapping_rows": int(len(mapping_df)),
    "n_summary_rows": int(len(summary_df)),
    "mapping_csv": str(mapping_csv),
    "summary_csv": str(mapping_summary_csv),
    "overall": {
        "n_crop_files": int(len(mapping_df)),
        "n_with_iImg": int(mapping_df["iImg_from_crop"].notna().sum()),
        "n_with_raw_mapping": int((mapping_df["raw_file_mapped"] != "").sum()),
        "n_particle_file_exists": int(mapping_df["particle_positions_exists"].sum()),
        "n_tags_file_exists": int(mapping_df["tags_exists"].sum())
    }
}

mapping_summary_json.write_text(json.dumps(summary_json, indent=2), encoding="utf-8")

print("=== Mapping Awal Crop ke RAW Selesai ===")
print("Total baris mapping crop:", len(mapping_df))
print("Crop dengan iImg terbaca:", mapping_df["iImg_from_crop"].notna().sum())
print("Crop berhasil mendapat raw mapping:", (mapping_df["raw_file_mapped"] != "").sum())
print("Particle positions ditemukan:", mapping_df["particle_positions_exists"].sum())
print("Tags ditemukan:", mapping_df["tags_exists"].sum())

print("\nFile mapping disimpan:")
print(mapping_csv)
print(mapping_summary_csv)
print(mapping_summary_json)

print("\n=== Ringkasan Mapping per Split/Kelas ===")
display(summary_df)

print("\n=== Kelas/Split dengan Masalah Mapping ===")
problem_df = summary_df[
    (summary_df["n_iImg_unmapped"] > 0) |
    (summary_df["particle_count_match_rate"].fillna(0) < 1.0)
].copy()

if len(problem_df) > 0:
    display(problem_df)
else:
    print("Tidak ada masalah besar: semua iImg terpetakan dan crop count cocok dengan particle count.")

print("\n=== Preview Mapping 20 Baris ===")
display(mapping_df.head(20))

In [ ]:
# Sel 13 — Validasi Mapping dengan Logika Particle Center ala prepareContextVirusImages.m

from pathlib import Path
import pandas as pd
import json
from collections import defaultdict

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
raw_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset/context_virus_RAW")

mapping_csv = outputs_dir / "v2_crop_to_raw_initial_mapping.csv"

enhanced_csv = outputs_dir / "v2_crop_to_raw_mapping_enhanced_validation.csv"
enhanced_summary_csv = outputs_dir / "v2_crop_to_raw_mapping_enhanced_summary.csv"
enhanced_summary_json = outputs_dir / "v2_crop_to_raw_mapping_enhanced_summary.json"

mapping_df = pd.read_csv(mapping_csv)

def count_expected_centers_like_matlab(particle_txt_path):
    """
    Menghitung jumlah crop expected mengikuti logika prepareContextVirusImages.m:
    - Tiap blok 'particleposition' dibaca sebagai daftar koordinat.
    - Jika jumlah titik koordinat = 2, dibuat 1 center dari mean dua titik.
    - Jika jumlah titik koordinat selain 2, script memakai semua titik sebagai center.
    """
    path = Path(particle_txt_path)
    if not path.exists():
        return None

    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    expected_centers = 0
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if line == "pixelsize":
            # MATLAB skip one line setelah pixelsize
            i += 2
            continue

        if line == "particleposition":
            i += 1

            # Kadang ada header x;y
            if i < len(lines) and lines[i].strip().lower() == "x;y":
                i += 1

            coords = []
            while i < len(lines):
                cur = lines[i].strip()
                if cur == "particleposition":
                    break
                if cur == "" or cur == "pixelsize":
                    i += 1
                    continue

                # Koordinat biasanya format x;y
                if ";" in cur:
                    parts = cur.split(";")
                    if len(parts) >= 2:
                        try:
                            x = float(parts[0])
                            y = float(parts[1])
                            coords.append((x, y))
                        except:
                            pass
                i += 1

            if len(coords) == 2:
                expected_centers += 1
            else:
                expected_centers += len(coords)

            # jangan i += 1 di sini kalau berhenti di particleposition,
            # supaya loop berikutnya memproses blok berikutnya
            continue

        i += 1

    return expected_centers

# Buat tabel per RAW image: berapa crop aktual dan expected centers
group_cols = [
    "split",
    "class_name",
    "raw_file_mapped",
    "raw_filename_mapped",
    "particle_positions_file"
]

raw_group = (
    mapping_df
    .groupby(group_cols, dropna=False)
    .agg(
        n_crop_actual=("crop_file", "count"),
        min_iPar=("iPar_from_crop", "min"),
        max_iPar=("iPar_from_crop", "max"),
        n_unique_iPar=("iPar_from_crop", "nunique")
    )
    .reset_index()
)

expected_list = []
particle_file_exists_list = []

for _, row in raw_group.iterrows():
    rel_particle = row["particle_positions_file"]
    if isinstance(rel_particle, str) and rel_particle != "":
        particle_path = raw_root / rel_particle
    else:
        particle_path = Path("")

    particle_file_exists_list.append(particle_path.exists())
    expected_list.append(count_expected_centers_like_matlab(particle_path) if particle_path.exists() else None)

raw_group["particle_positions_exists"] = particle_file_exists_list
raw_group["n_expected_centers_matlab_logic"] = expected_list
raw_group["actual_equals_expected"] = raw_group["n_crop_actual"] == raw_group["n_expected_centers_matlab_logic"]
raw_group["diff_actual_minus_expected"] = raw_group["n_crop_actual"] - raw_group["n_expected_centers_matlab_logic"]

# Ringkasan per split/kelas
summary = (
    raw_group
    .groupby(["split", "class_name"], dropna=False)
    .agg(
        n_raw_images=("raw_file_mapped", "count"),
        n_raw_images_particle_file_exists=("particle_positions_exists", "sum"),
        n_crop_actual_total=("n_crop_actual", "sum"),
        n_expected_centers_total=("n_expected_centers_matlab_logic", "sum"),
        n_raw_images_match=("actual_equals_expected", "sum"),
        n_raw_images_mismatch=("actual_equals_expected", lambda s: (~s).sum())
    )
    .reset_index()
)

summary["match_rate"] = summary["n_raw_images_match"] / summary["n_raw_images"]

# Simpan
raw_group.to_csv(enhanced_csv, index=False)
summary.to_csv(enhanced_summary_csv, index=False)

summary_json = {
    "n_raw_image_rows": int(len(raw_group)),
    "n_raw_images_match": int(raw_group["actual_equals_expected"].sum()),
    "n_raw_images_mismatch": int((~raw_group["actual_equals_expected"]).sum()),
    "overall_match_rate": float(raw_group["actual_equals_expected"].mean()),
    "n_crop_actual_total": int(raw_group["n_crop_actual"].sum()),
    "n_expected_centers_total": int(raw_group["n_expected_centers_matlab_logic"].sum()),
    "enhanced_csv": str(enhanced_csv),
    "enhanced_summary_csv": str(enhanced_summary_csv),
    "enhanced_summary_json": str(enhanced_summary_json)
}

enhanced_summary_json.write_text(json.dumps(summary_json, indent=2), encoding="utf-8")

print("=== Validasi Enhanced Mapping Selesai ===")
print("Jumlah RAW image dicek:", len(raw_group))
print("RAW image match:", raw_group["actual_equals_expected"].sum())
print("RAW image mismatch:", (~raw_group["actual_equals_expected"]).sum())
print("Overall match rate:", raw_group["actual_equals_expected"].mean())
print("Total crop aktual:", raw_group["n_crop_actual"].sum())
print("Total expected centers:", raw_group["n_expected_centers_matlab_logic"].sum())

print("\nOutput disimpan:")
print(enhanced_csv)
print(enhanced_summary_csv)
print(enhanced_summary_json)

print("\n=== Ringkasan per Split/Kelas ===")
display(summary)

print("\n=== RAW Image yang Masih Mismatch ===")
mismatch = raw_group[~raw_group["actual_equals_expected"]].copy()
if len(mismatch) > 0:
    display(mismatch.head(50))
else:
    print("Tidak ada mismatch. Mapping crop ke RAW tervalidasi sangat kuat.")

print("\n=== Preview RAW Group Mapping ===")
display(raw_group.head(20))

In [ ]:
# Sel 14 — Load Mapping Crop ke RAW dan Inspeksi Format Metadata Tags RAW

from pathlib import Path
import pandas as pd
import json

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
raw_root = dataset_root / "context_virus_RAW"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# File mapping hasil V2
mapping_csv = outputs_dir / "v2_crop_to_raw_initial_mapping.csv"

# Output V3 awal
v3_preview_csv = outputs_dir / "v3_raw_tags_preview_sample.csv"
v3_tag_structure_json = outputs_dir / "v3_raw_tags_structure_check.json"

print("=== Cek File Mapping V2 ===")
print("Mapping CSV:", mapping_csv)
print("Mapping tersedia:", mapping_csv.exists())

# Load mapping
mapping_df = pd.read_csv(mapping_csv)

print("\n=== Ringkasan Mapping Crop ke RAW ===")
print("Shape mapping:", mapping_df.shape)
print("Jumlah crop:", len(mapping_df))
print("Jumlah RAW unik:", mapping_df["raw_file_mapped"].nunique())
print("Jumlah tags_file unik:", mapping_df["tags_file"].nunique())

print("\nKolom mapping:")
print(list(mapping_df.columns))

# Ambil RAW unik
raw_unique = (
    mapping_df[
        ["split", "class_name", "raw_file_mapped", "raw_filename_mapped", "tags_file", "particle_positions_file"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\n=== RAW Unique Preview ===")
display(raw_unique.head(10))

# Fungsi baca tags CSV
def read_tag_csv(tag_rel_path):
    tag_path = raw_root / tag_rel_path
    if not tag_path.exists():
        return None, tag_path, "missing"

    try:
        df = pd.read_csv(tag_path, sep=";", engine="python")
        return df, tag_path, "ok"
    except Exception as e:
        return None, tag_path, repr(e)

# Baca beberapa sample tags
sample_n = min(10, len(raw_unique))
sample_rows = raw_unique.sample(sample_n, random_state=14).reset_index(drop=True)

preview_records = []
structure_records = []

print("\n=== Preview Beberapa File Tags RAW ===")

for idx, row in sample_rows.iterrows():
    tag_rel = row["tags_file"]
    tag_df, tag_path, status = read_tag_csv(tag_rel)

    print("\n" + "="*80)
    print("Sample:", idx + 1)
    print("Class:", row["class_name"])
    print("Split:", row["split"])
    print("RAW:", row["raw_file_mapped"])
    print("Tags:", tag_rel)
    print("Status:", status)

    if tag_df is not None:
        print("Shape tags:", tag_df.shape)
        print("Columns:", list(tag_df.columns))
        display(tag_df.head(8))

        structure_records.append({
            "sample_idx": idx + 1,
            "split": row["split"],
            "class_name": row["class_name"],
            "raw_file_mapped": row["raw_file_mapped"],
            "tags_file": tag_rel,
            "status": status,
            "n_rows": len(tag_df),
            "n_cols": len(tag_df.columns),
            "columns": list(tag_df.columns)
        })

        # Simpan preview baris awal
        tmp = tag_df.head(20).copy()
        tmp.insert(0, "sample_idx", idx + 1)
        tmp.insert(1, "split", row["split"])
        tmp.insert(2, "class_name", row["class_name"])
        tmp.insert(3, "raw_file_mapped", row["raw_file_mapped"])
        tmp.insert(4, "tags_file", tag_rel)
        preview_records.append(tmp)
    else:
        structure_records.append({
            "sample_idx": idx + 1,
            "split": row["split"],
            "class_name": row["class_name"],
            "raw_file_mapped": row["raw_file_mapped"],
            "tags_file": tag_rel,
            "status": status,
            "n_rows": None,
            "n_cols": None,
            "columns": None
        })

# Gabungkan preview
if preview_records:
    preview_df = pd.concat(preview_records, ignore_index=True)
    preview_df.to_csv(v3_preview_csv, index=False)
else:
    preview_df = pd.DataFrame()

# Simpan struktur
structure_summary = {
    "mapping_csv": str(mapping_csv),
    "raw_root": str(raw_root),
    "n_crop_rows": int(len(mapping_df)),
    "n_unique_raw": int(raw_unique["raw_file_mapped"].nunique()),
    "n_unique_tags_file": int(raw_unique["tags_file"].nunique()),
    "sample_n": int(sample_n),
    "structure_records": structure_records,
    "preview_csv": str(v3_preview_csv),
    "structure_json": str(v3_tag_structure_json)
}

v3_tag_structure_json.write_text(json.dumps(structure_summary, indent=2, default=str), encoding="utf-8")

print("\n=== Output Sel 14 Disimpan ===")
print("Preview sample tags CSV:")
print(v3_preview_csv)
print("Structure check JSON:")
print(v3_tag_structure_json)

In [ ]:
# Sel 15 — Ekstrak Metadata Tags RAW Menjadi Tabel per RAW Image

from pathlib import Path
import pandas as pd
import json
import numpy as np

# Path utama
dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
raw_root = dataset_root / "context_virus_RAW"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input dari V2
mapping_csv = outputs_dir / "v2_crop_to_raw_initial_mapping.csv"

# Output V3
raw_metadata_csv = outputs_dir / "v3_raw_metadata_from_tags.csv"
raw_metadata_missing_csv = outputs_dir / "v3_raw_metadata_missingness_summary.csv"
raw_metadata_unique_csv = outputs_dir / "v3_raw_metadata_unique_summary.csv"
raw_metadata_json = outputs_dir / "v3_raw_metadata_from_tags_summary.json"

# Load mapping crop -> RAW
mapping_df = pd.read_csv(mapping_csv)

# RAW unik + jumlah crop per RAW
raw_unique = (
    mapping_df
    .groupby(["split", "class_name", "raw_file_mapped", "raw_filename_mapped", "tags_file", "particle_positions_file"], dropna=False)
    .agg(
        n_crops_from_raw=("crop_file", "count"),
        min_iPar=("iPar_from_crop", "min"),
        max_iPar=("iPar_from_crop", "max")
    )
    .reset_index()
)

print("=== Input RAW Unique ===")
print("Jumlah RAW unik:", len(raw_unique))
print("Total crop dari mapping:", raw_unique["n_crops_from_raw"].sum())

def read_tag_key_value(tag_rel_path):
    """
    Membaca file tags CSV format key-value:
    baris contoh:
    Heigt;2048
    Width;2048
    GridposX;0
    ...
    """
    tag_path = raw_root / tag_rel_path

    if not tag_path.exists():
        return {}, False, "missing"

    try:
        df = pd.read_csv(tag_path, sep=";", header=None, dtype=str, engine="python")
        df = df.dropna(how="all")

        tag_dict = {}
        for _, row in df.iterrows():
            if len(row) < 2:
                continue

            key = str(row.iloc[0]).strip()
            value = str(row.iloc[1]).strip()

            if key == "" or key.lower() == "nan":
                continue

            tag_dict[key] = value

        return tag_dict, True, "ok"

    except Exception as e:
        return {}, False, repr(e)

def normalize_decimal_value(x):
    """
    Ubah angka Eropa '0,373' menjadi '0.373' jika memungkinkan.
    Kalau bukan angka, biarkan sebagai string.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    s2 = s.replace(",", ".")
    try:
        return float(s2)
    except:
        return s

records = []
all_keys = set()
status_records = []

for _, row in raw_unique.iterrows():
    tag_rel = row["tags_file"]
    tag_dict, exists, status = read_tag_key_value(tag_rel)
    all_keys.update(tag_dict.keys())

    rec = {
        "split": row["split"],
        "class_name": row["class_name"],
        "raw_file_mapped": row["raw_file_mapped"],
        "raw_filename_mapped": row["raw_filename_mapped"],
        "tags_file": tag_rel,
        "particle_positions_file": row["particle_positions_file"],
        "n_crops_from_raw": row["n_crops_from_raw"],
        "min_iPar": row["min_iPar"],
        "max_iPar": row["max_iPar"],
        "tags_exists": exists,
        "tags_read_status": status
    }

    # Masukkan semua key tag
    for k, v in tag_dict.items():
        rec[k] = v

    records.append(rec)

    status_records.append({
        "tags_file": tag_rel,
        "tags_exists": exists,
        "tags_read_status": status,
        "n_keys": len(tag_dict),
        "keys": list(tag_dict.keys())
    })

raw_meta = pd.DataFrame(records)

print("\n=== Metadata Tags Berhasil Diekstrak ===")
print("Shape raw_meta:", raw_meta.shape)
print("Jumlah key metadata unik:", len(all_keys))
print("Daftar key metadata unik:")
print(sorted(all_keys))

# Normalisasi nama kolom typo umum
# Jangan hapus kolom asli; buat versi standar jika perlu.
if "Heigt" in raw_meta.columns and "Height" not in raw_meta.columns:
    raw_meta["Height"] = raw_meta["Heigt"]

# Field target V3
target_fields = [
    "Date",
    "Photographer",
    "CameraModel",
    "Magnification",
    "GridposX",
    "GridposY",
    "GridposZ"
]

# Buat versi numeric untuk field yang biasanya angka
numeric_candidates = [
    "Height", "Width", "GridposX", "GridposY", "GridposZ",
    "Xscale", "Yscale", "AccVoltage", "Defocus", "Magnification"
]

for col in numeric_candidates:
    if col in raw_meta.columns:
        raw_meta[col + "_num"] = raw_meta[col].apply(normalize_decimal_value)

# Missingness target fields
missing_records = []
for col in target_fields:
    if col in raw_meta.columns:
        missing_records.append({
            "field": col,
            "exists_as_column": True,
            "missing_count": int(raw_meta[col].isna().sum()),
            "non_missing_count": int(raw_meta[col].notna().sum()),
            "missing_rate": float(raw_meta[col].isna().mean()),
            "n_unique": int(raw_meta[col].nunique(dropna=True))
        })
    else:
        missing_records.append({
            "field": col,
            "exists_as_column": False,
            "missing_count": len(raw_meta),
            "non_missing_count": 0,
            "missing_rate": 1.0,
            "n_unique": 0
        })

missing_df = pd.DataFrame(missing_records)

# Unique summary untuk semua key penting
unique_records = []
for col in target_fields:
    if col in raw_meta.columns:
        vc = raw_meta[col].value_counts(dropna=False).head(20)
        unique_records.append({
            "field": col,
            "n_unique": int(raw_meta[col].nunique(dropna=True)),
            "top_values": vc.to_dict()
        })
    else:
        unique_records.append({
            "field": col,
            "n_unique": 0,
            "top_values": {}
        })

unique_df = pd.DataFrame(unique_records)

# Simpan
raw_meta.to_csv(raw_metadata_csv, index=False)
missing_df.to_csv(raw_metadata_missing_csv, index=False)
unique_df.to_csv(raw_metadata_unique_csv, index=False)

summary = {
    "n_raw_unique": int(len(raw_unique)),
    "n_crop_total": int(raw_unique["n_crops_from_raw"].sum()),
    "n_metadata_rows": int(len(raw_meta)),
    "n_metadata_keys_unique": int(len(all_keys)),
    "metadata_keys_unique": sorted(list(all_keys)),
    "target_fields": target_fields,
    "raw_metadata_csv": str(raw_metadata_csv),
    "missing_summary_csv": str(raw_metadata_missing_csv),
    "unique_summary_csv": str(raw_metadata_unique_csv)
}

raw_metadata_json.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

print("\n=== Missingness Target Fields ===")
display(missing_df)

print("\n=== Unique Summary Target Fields ===")
display(unique_df)

print("\n=== Preview Metadata RAW ===")
preview_cols = [
    "split", "class_name", "raw_filename_mapped", "n_crops_from_raw",
    "Date", "Photographer", "CameraModel", "Magnification",
    "GridposX", "GridposY", "GridposZ"
]
preview_cols = [c for c in preview_cols if c in raw_meta.columns]
display(raw_meta[preview_cols].head(20))

print("\n=== Output Sel 15 Disimpan ===")
print("Raw metadata CSV:")
print(raw_metadata_csv)
print("Missingness summary CSV:")
print(raw_metadata_missing_csv)
print("Unique summary CSV:")
print(raw_metadata_unique_csv)
print("Summary JSON:")
print(raw_metadata_json)

In [ ]:
# Sel 16 — Gabungkan Metadata RAW ke Level Crop dan Buat Kandidat Group Identifier

from pathlib import Path
import pandas as pd
import numpy as np
import json

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input
mapping_csv = outputs_dir / "v2_crop_to_raw_initial_mapping.csv"
raw_metadata_csv = outputs_dir / "v3_raw_metadata_from_tags.csv"

# Output
crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"
group_columns_json = outputs_dir / "v3_group_candidate_columns_summary.json"

# Load
mapping_df = pd.read_csv(mapping_csv)
raw_meta = pd.read_csv(raw_metadata_csv)

print("=== Input ===")
print("Mapping crop shape:", mapping_df.shape)
print("Raw metadata shape:", raw_meta.shape)
print("Jumlah crop:", len(mapping_df))
print("Jumlah RAW unik di mapping:", mapping_df["raw_file_mapped"].nunique())
print("Jumlah RAW unik di metadata:", raw_meta["raw_file_mapped"].nunique())

# Kolom metadata yang dibutuhkan
meta_cols = [
    "split",
    "class_name",
    "raw_file_mapped",
    "raw_filename_mapped",
    "n_crops_from_raw",
    "Date",
    "Photographer",
    "CameraModel",
    "Magnification",
    "GridposX",
    "GridposY",
    "GridposZ"
]

meta_cols = [c for c in meta_cols if c in raw_meta.columns]

# Ambil metadata RAW unik
raw_meta_small = raw_meta[meta_cols].drop_duplicates(subset=["raw_file_mapped"]).copy()

# Merge metadata ke level crop
crop_meta = mapping_df.merge(
    raw_meta_small.drop(columns=["split", "class_name", "raw_filename_mapped"], errors="ignore"),
    on="raw_file_mapped",
    how="left"
)

print("\n=== Setelah Merge ===")
print("Crop metadata shape:", crop_meta.shape)
print("Metadata Date missing:", crop_meta["Date"].isna().sum() if "Date" in crop_meta.columns else "Date column missing")
print("Metadata Magnification missing:", crop_meta["Magnification"].isna().sum() if "Magnification" in crop_meta.columns else "Magnification column missing")

# Normalisasi string untuk group
def clean_group_value(x):
    if pd.isna(x):
        return "NA"
    s = str(x).strip()
    if s == "" or s.lower() in ["nan", "none", "null"]:
        return "NA"
    return s

group_base_cols = [
    "Date",
    "Photographer",
    "CameraModel",
    "Magnification",
    "GridposX",
    "GridposY",
    "GridposZ"
]

for col in group_base_cols:
    if col in crop_meta.columns:
        crop_meta[col + "_clean"] = crop_meta[col].apply(clean_group_value)
    else:
        crop_meta[col + "_clean"] = "NA"

# Fungsi gabung group id
def make_group_id(df, cols):
    clean_cols = [c + "_clean" for c in cols]
    return df[clean_cols].astype(str).agg(" | ".join, axis=1)

# Kandidat group identifier sesuai instruksi V3
group_specs = {
    "G01_Date": ["Date"],
    "G02_Photographer": ["Photographer"],
    "G03_CameraModel": ["CameraModel"],
    "G04_Magnification": ["Magnification"],
    "G05_GridposX": ["GridposX"],
    "G06_GridposY": ["GridposY"],
    "G07_GridposZ": ["GridposZ"],
    "G08_Date_Photographer": ["Date", "Photographer"],
    "G09_Date_Magnification": ["Date", "Magnification"],
    "G10_Date_Photographer_CameraModel": ["Date", "Photographer", "CameraModel"],
    "G11_Date_Photographer_Magnification": ["Date", "Photographer", "Magnification"],
    "G12_Date_Photographer_CameraModel_Magnification": ["Date", "Photographer", "CameraModel", "Magnification"],
    "G13_Date_Photographer_CameraModel_GridposXYZ_Magnification": [
        "Date", "Photographer", "CameraModel",
        "GridposX", "GridposY", "GridposZ", "Magnification"
    ],
    # Tambahan teknis untuk pembanding, bukan kandidat final
    "G14_RAWSource": ["raw_file_mapped"] if "raw_file_mapped" in crop_meta.columns else ["Date"]
}

# Buat clean raw_file_mapped juga untuk G14
crop_meta["raw_file_mapped_clean"] = crop_meta["raw_file_mapped"].apply(clean_group_value)

# Buat group id
created_group_cols = []

for group_name, cols in group_specs.items():
    clean_cols = []
    for c in cols:
        if c == "raw_file_mapped":
            clean_cols.append("raw_file_mapped_clean")
        else:
            clean_cols.append(c + "_clean")

    crop_meta[group_name] = crop_meta[clean_cols].astype(str).agg(" | ".join, axis=1)
    created_group_cols.append(group_name)

# Simpan crop-level metadata
crop_meta.to_csv(crop_meta_csv, index=False)

# Ringkasan awal kolom group
summary = {
    "n_crop_rows": int(len(crop_meta)),
    "n_raw_unique": int(crop_meta["raw_file_mapped"].nunique()),
    "group_specs": group_specs,
    "created_group_cols": created_group_cols,
    "crop_meta_csv": str(crop_meta_csv)
}

group_columns_json.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

print("\n=== Group Candidate Columns Dibuat ===")
print("Jumlah group candidate columns:", len(created_group_cols))
print(created_group_cols)

print("\nOutput crop-level metadata:")
print(crop_meta_csv)
print("Summary JSON:")
print(group_columns_json)

print("\n=== Preview Crop Metadata + Group Candidate ===")
preview_cols = [
    "split", "class_name", "crop_file", "raw_file_mapped",
    "Date", "Photographer", "CameraModel", "Magnification",
    "GridposX", "GridposY", "GridposZ",
    "G01_Date",
    "G08_Date_Photographer",
    "G13_Date_Photographer_CameraModel_GridposXYZ_Magnification",
    "G14_RAWSource"
]
preview_cols = [c for c in preview_cols if c in crop_meta.columns]
display(crop_meta[preview_cols].head(20))

print("\n=== Cek Jumlah Unique Awal per Group Candidate ===")
unique_preview = []
for g in created_group_cols:
    unique_preview.append({
        "group_candidate": g,
        "n_unique_groups": int(crop_meta[g].nunique()),
        "mean_crop_per_group": float(crop_meta.groupby(g)["crop_file"].count().mean()),
        "median_crop_per_group": float(crop_meta.groupby(g)["crop_file"].count().median()),
        "min_crop_per_group": int(crop_meta.groupby(g)["crop_file"].count().min()),
        "max_crop_per_group": int(crop_meta.groupby(g)["crop_file"].count().max())
    })

unique_preview_df = pd.DataFrame(unique_preview)
display(unique_preview_df)

In [ ]:
# Sel 17 — Evaluasi Kandidat Group Identifier untuk Leakage-Aware Split (Revisi Final)

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input dari Sel 16
crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"

# Output Sel 17
group_overall_csv = outputs_dir / "v3_group_identifier_overall_summary.csv"
group_class_csv = outputs_dir / "v3_group_identifier_per_class_summary.csv"
group_split_overlap_csv = outputs_dir / "v3_group_identifier_split_overlap_summary.csv"
group_top_groups_csv = outputs_dir / "v3_group_identifier_top_largest_groups.csv"
group_eval_json = outputs_dir / "v3_group_identifier_evaluation_summary.json"

# Load crop-level metadata
# keep_default_na=False penting supaya string "NA" tidak otomatis dibaca sebagai NaN
df = pd.read_csv(crop_meta_csv, keep_default_na=False)

print("=== Input Crop-Level Metadata ===")
print("Shape:", df.shape)
print("Jumlah crop:", len(df))
print("Jumlah RAW unik:", df["raw_file_mapped"].nunique())
print("Jumlah kelas:", df["class_name"].nunique())
print("Split:", df["split"].unique())

# Deteksi kolom kandidat group secara ketat
group_cols = [
    c for c in df.columns
    if re.match(r"^G\d+_", c)
]

group_cols = sorted(group_cols, key=lambda x: int(re.match(r"^G(\d+)_", x).group(1)))

# Pastikan semua group candidate tidak kosong/NaN
for g in group_cols:
    df[g] = df[g].fillna("NA").astype(str)
    df.loc[df[g].str.strip() == "", g] = "NA"

print("\n=== Kandidat Group Identifier ===")
print("Jumlah kandidat:", len(group_cols))
for g in group_cols:
    print("-", g)

total_crops = len(df)
total_raw = df["raw_file_mapped"].nunique()
total_classes = df["class_name"].nunique()

def classify_group_candidate(n_groups, mean_crop, median_crop, max_crop, n_cross_split_groups, max_group_rate):
    """
    Heuristik sementara untuk membantu interpretasi.
    Bukan keputusan final.
    """
    if n_groups <= total_classes or max_group_rate >= 0.25 or mean_crop >= 500:
        return "terlalu longgar"
    elif n_groups >= 0.85 * total_raw or median_crop <= 6:
        return "cenderung terlalu ketat"
    else:
        return "moderat / perlu dievaluasi"

def safe_int(x, default=0):
    if pd.isna(x):
        return default
    return int(x)

def safe_float(x, default=0.0):
    if pd.isna(x):
        return default
    return float(x)

overall_records = []
class_records = []
split_overlap_records = []
top_group_records = []

for g in group_cols:
    # Ukuran group crop-level
    group_sizes = df.groupby(g, dropna=False).agg(
        n_crops=("crop_file", "count"),
        n_raw=("raw_file_mapped", "nunique"),
        n_classes=("class_name", "nunique"),
        n_splits=("split", "nunique")
    ).reset_index().rename(columns={g: "group_id"})

    n_groups = len(group_sizes)
    mean_crop = group_sizes["n_crops"].mean()
    median_crop = group_sizes["n_crops"].median()
    min_crop = group_sizes["n_crops"].min()
    max_crop = group_sizes["n_crops"].max()
    max_group_rate = max_crop / total_crops

    mean_raw = group_sizes["n_raw"].mean()
    median_raw = group_sizes["n_raw"].median()
    max_raw = group_sizes["n_raw"].max()

    # Group yang muncul lintas official split
    cross_split = group_sizes[group_sizes["n_splits"] > 1]
    n_cross_split_groups = len(cross_split)
    crop_in_cross_split_groups = df[df[g].isin(cross_split["group_id"])]["crop_file"].count()
    raw_in_cross_split_groups = df[df[g].isin(cross_split["group_id"])]["raw_file_mapped"].nunique()

    # Group yang berisi lebih dari satu kelas
    multi_class = group_sizes[group_sizes["n_classes"] > 1]
    n_multi_class_groups = len(multi_class)
    crop_in_multi_class_groups = df[df[g].isin(multi_class["group_id"])]["crop_file"].count()

    preliminary_flag = classify_group_candidate(
        n_groups=n_groups,
        mean_crop=mean_crop,
        median_crop=median_crop,
        max_crop=max_crop,
        n_cross_split_groups=n_cross_split_groups,
        max_group_rate=max_group_rate
    )

    overall_records.append({
        "group_candidate": g,
        "n_unique_groups": int(n_groups),
        "mean_crop_per_group": safe_float(mean_crop),
        "median_crop_per_group": safe_float(median_crop),
        "min_crop_per_group": safe_int(min_crop),
        "max_crop_per_group": safe_int(max_crop),
        "max_group_crop_rate": safe_float(max_group_rate),
        "mean_raw_per_group": safe_float(mean_raw),
        "median_raw_per_group": safe_float(median_raw),
        "max_raw_per_group": safe_int(max_raw),
        "n_cross_split_groups": int(n_cross_split_groups),
        "crop_in_cross_split_groups": int(crop_in_cross_split_groups),
        "raw_in_cross_split_groups": int(raw_in_cross_split_groups),
        "cross_split_group_rate": safe_float(n_cross_split_groups / n_groups if n_groups > 0 else 0),
        "crop_cross_split_rate": safe_float(crop_in_cross_split_groups / total_crops if total_crops > 0 else 0),
        "n_multi_class_groups": int(n_multi_class_groups),
        "crop_in_multi_class_groups": int(crop_in_multi_class_groups),
        "multi_class_group_rate": safe_float(n_multi_class_groups / n_groups if n_groups > 0 else 0),
        "preliminary_interpretation": preliminary_flag
    })

    # Ringkasan per kelas
    for cls, sub in df.groupby("class_name", dropna=False):
        cls_group_sizes = sub.groupby(g, dropna=False).size()

        if len(cls_group_sizes) == 0:
            n_groups_in_class = 0
            mean_cls = 0
            median_cls = 0
            min_cls = 0
            max_cls = 0
        else:
            n_groups_in_class = cls_group_sizes.shape[0]
            mean_cls = cls_group_sizes.mean()
            median_cls = cls_group_sizes.median()
            min_cls = cls_group_sizes.min()
            max_cls = cls_group_sizes.max()

        class_records.append({
            "group_candidate": g,
            "class_name": cls,
            "n_crops_class": int(len(sub)),
            "n_raw_class": int(sub["raw_file_mapped"].nunique()),
            "n_groups_in_class": int(n_groups_in_class),
            "mean_crop_per_group_in_class": safe_float(mean_cls),
            "median_crop_per_group_in_class": safe_float(median_cls),
            "min_crop_per_group_in_class": safe_int(min_cls),
            "max_crop_per_group_in_class": safe_int(max_cls)
        })

    # Detail split overlap per candidate
    if len(cross_split) > 0:
        for _, row in cross_split.iterrows():
            gid = row["group_id"]
            sub = df[df[g] == gid]
            split_overlap_records.append({
                "group_candidate": g,
                "group_id": gid,
                "n_crops": int(row["n_crops"]),
                "n_raw": int(row["n_raw"]),
                "n_classes": int(row["n_classes"]),
                "n_splits": int(row["n_splits"]),
                "split_counts": json.dumps(sub["split"].value_counts().to_dict(), ensure_ascii=False),
                "class_counts": json.dumps(sub["class_name"].value_counts().to_dict(), ensure_ascii=False)
            })

    # Top largest groups
    top_largest = group_sizes.sort_values("n_crops", ascending=False).head(10)
    for rank, (_, row) in enumerate(top_largest.iterrows(), start=1):
        gid = row["group_id"]
        sub = df[df[g] == gid]
        top_group_records.append({
            "group_candidate": g,
            "rank": rank,
            "group_id": gid,
            "n_crops": int(row["n_crops"]),
            "n_raw": int(row["n_raw"]),
            "n_classes": int(row["n_classes"]),
            "n_splits": int(row["n_splits"]),
            "split_counts": json.dumps(sub["split"].value_counts().to_dict(), ensure_ascii=False),
            "class_counts": json.dumps(sub["class_name"].value_counts().to_dict(), ensure_ascii=False)
        })

overall_df = pd.DataFrame(overall_records)
class_df = pd.DataFrame(class_records)
split_overlap_df = pd.DataFrame(split_overlap_records)
top_groups_df = pd.DataFrame(top_group_records)

# Simpan output
overall_df.to_csv(group_overall_csv, index=False)
class_df.to_csv(group_class_csv, index=False)
split_overlap_df.to_csv(group_split_overlap_csv, index=False)
top_groups_df.to_csv(group_top_groups_csv, index=False)

summary_json = {
    "input_csv": str(crop_meta_csv),
    "n_crops": int(total_crops),
    "n_raw_unique": int(total_raw),
    "n_classes": int(total_classes),
    "group_candidates": group_cols,
    "outputs": {
        "overall_summary_csv": str(group_overall_csv),
        "per_class_summary_csv": str(group_class_csv),
        "split_overlap_summary_csv": str(group_split_overlap_csv),
        "top_largest_groups_csv": str(group_top_groups_csv),
        "summary_json": str(group_eval_json)
    }
}

group_eval_json.write_text(json.dumps(summary_json, indent=2, default=str), encoding="utf-8")

print("\n=== Overall Group Identifier Summary ===")
display(overall_df)

print("\n=== Per-Class Summary Preview ===")
display(class_df.head(40))

print("\n=== Split-Overlap Summary Preview ===")
if len(split_overlap_df) > 0:
    display(split_overlap_df.head(40))
else:
    print("Tidak ada group yang muncul lintas split pada semua kandidat.")

print("\n=== Top Largest Groups Preview ===")
display(top_groups_df.head(40))

print("\n=== Output Sel 17 Disimpan ===")
print("Overall summary CSV:")
print(group_overall_csv)
print("Per-class summary CSV:")
print(group_class_csv)
print("Split-overlap summary CSV:")
print(group_split_overlap_csv)
print("Top largest groups CSV:")
print(group_top_groups_csv)
print("Summary JSON:")
print(group_eval_json)

In [ ]:
# Sel 18 — Buat Ringkasan Interpretasi Kandidat Group Identifier untuk MASTER CHAT

from pathlib import Path
import pandas as pd
import json

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input dari Sel 17
overall_csv = outputs_dir / "v3_group_identifier_overall_summary.csv"
per_class_csv = outputs_dir / "v3_group_identifier_per_class_summary.csv"

# Output Sel 18
decision_table_csv = outputs_dir / "v3_group_identifier_decision_support_table.csv"
decision_table_json = outputs_dir / "v3_group_identifier_decision_support_summary.json"

overall = pd.read_csv(overall_csv)
per_class = pd.read_csv(per_class_csv)

print("=== Input Sel 18 ===")
print("Overall summary:", overall.shape)
print("Per-class summary:", per_class.shape)

def interpret_row(row):
    g = row["group_candidate"]
    n_groups = row["n_unique_groups"]
    mean_crop = row["mean_crop_per_group"]
    median_crop = row["median_crop_per_group"]
    max_crop = row["max_crop_per_group"]
    crop_cross = row["crop_cross_split_rate"]
    multi_class = row["multi_class_group_rate"]
    prelim = row["preliminary_interpretation"]

    notes = []

    if prelim == "terlalu longgar":
        notes.append("terlalu longgar untuk split berbasis group")
    elif prelim == "cenderung terlalu ketat":
        notes.append("cenderung terlalu granular/ketat")
    else:
        notes.append("ukuran group relatif moderat")

    if crop_cross >= 0.90:
        notes.append("sangat banyak group tersebar lintas official split")
    elif crop_cross >= 0.50:
        notes.append("cukup banyak group tersebar lintas official split")
    elif crop_cross > 0:
        notes.append("ada sebagian group lintas official split")
    else:
        notes.append("tidak ada overlap lintas official split pada definisi group ini")

    if multi_class > 0:
        notes.append("sebagian group berisi lebih dari satu kelas")
    else:
        notes.append("tidak mencampur kelas")

    if max_crop >= 1000:
        notes.append("ada group sangat besar yang berpotensi mendominasi split")
    elif max_crop >= 500:
        notes.append("ada group besar, perlu perhatian saat splitting")

    # Status rekomendasi diskusi, bukan final
    if g in [
        "G09_Date_Magnification",
        "G11_Date_Photographer_Magnification",
        "G12_Date_Photographer_CameraModel_Magnification"
    ]:
        discussion_status = "layak dibahas serius di MASTER CHAT"
    elif g in [
        "G01_Date",
        "G08_Date_Photographer",
        "G10_Date_Photographer_CameraModel"
    ]:
        discussion_status = "layak dibahas, tetapi official split overlap sangat tinggi"
    elif g in [
        "G13_Date_Photographer_CameraModel_GridposXYZ_Magnification",
        "G14_RAWSource"
    ]:
        discussion_status = "layak sebagai pembanding konservatif, tetapi cenderung terlalu ketat"
    else:
        discussion_status = "kurang ideal sebagai kandidat utama"

    return discussion_status, "; ".join(notes)

decision_rows = []

for _, row in overall.iterrows():
    discussion_status, notes = interpret_row(row)

    decision_rows.append({
        "group_candidate": row["group_candidate"],
        "n_unique_groups": int(row["n_unique_groups"]),
        "mean_crop_per_group": row["mean_crop_per_group"],
        "median_crop_per_group": row["median_crop_per_group"],
        "min_crop_per_group": int(row["min_crop_per_group"]),
        "max_crop_per_group": int(row["max_crop_per_group"]),
        "mean_raw_per_group": row["mean_raw_per_group"],
        "median_raw_per_group": row["median_raw_per_group"],
        "n_cross_split_groups": int(row["n_cross_split_groups"]),
        "crop_cross_split_rate": row["crop_cross_split_rate"],
        "n_multi_class_groups": int(row["n_multi_class_groups"]),
        "multi_class_group_rate": row["multi_class_group_rate"],
        "preliminary_interpretation": row["preliminary_interpretation"],
        "discussion_status_for_master": discussion_status,
        "interpretation_notes": notes
    })

decision_df = pd.DataFrame(decision_rows)

# Tambahkan ranking diskusi manual berbasis status
priority_map = {
    "layak dibahas serius di MASTER CHAT": 1,
    "layak dibahas, tetapi official split overlap sangat tinggi": 2,
    "layak sebagai pembanding konservatif, tetapi cenderung terlalu ketat": 3,
    "kurang ideal sebagai kandidat utama": 4
}

decision_df["discussion_priority"] = decision_df["discussion_status_for_master"].map(priority_map)
decision_df = decision_df.sort_values(["discussion_priority", "group_candidate"]).reset_index(drop=True)

# Simpan
decision_df.to_csv(decision_table_csv, index=False)

summary = {
    "input_overall_csv": str(overall_csv),
    "input_per_class_csv": str(per_class_csv),
    "decision_table_csv": str(decision_table_csv),
    "n_candidates": int(len(decision_df)),
    "priority_counts": decision_df["discussion_status_for_master"].value_counts().to_dict()
}

decision_table_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n=== Decision Support Table untuk MASTER CHAT ===")
display(decision_df)

print("\n=== Ringkasan Prioritas Diskusi ===")
display(decision_df["discussion_status_for_master"].value_counts().reset_index().rename(
    columns={"index": "discussion_status_for_master", "discussion_status_for_master": "count"}
))

print("\n=== Output Sel 18 Disimpan ===")
print("Decision support CSV:")
print(decision_table_csv)
print("Decision support JSON:")
print(decision_table_json)

In [ ]:
# Sel 19 — V4 Simulasi Split Group-Aware Seed 42

from pathlib import Path
import pandas as pd
import numpy as np
import json
import random

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input dari V3
crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"

# Output V4 sesuai arahan MASTER
summary_csv = outputs_dir / "v4_group_split_simulation_summary.csv"
class_dist_csv = outputs_dir / "v4_group_split_class_distribution.csv"
leakage_csv = outputs_dir / "v4_group_split_leakage_check.csv"
recommendation_json = outputs_dir / "v4_group_split_recommendation.json"

# Kandidat group sesuai keputusan MASTER
candidate_groups = [
    "G09_Date_Magnification",
    "G13_Date_Photographer_CameraModel_GridposXYZ_Magnification",
    "G14_RAWSource",
    "G12_Date_Photographer_CameraModel_Magnification"  # opsional
]

# Mulai dari seed 42 dulu
seeds = [42]

# Threshold awal feasibility, belum final
MIN_VAL_PER_CLASS = 5
MIN_TEST_PER_CLASS = 5
MIN_TRAIN_PER_CLASS = 10
MIN_TRAIN_RATIO = 0.50
MAX_MEAN_ABS_PCT_POINT = 12.0
MAX_ABS_PCT_POINT = 35.0

# Load data
df = pd.read_csv(crop_meta_csv, keep_default_na=False)

print("=== Input V4 ===")
print("Shape:", df.shape)
print("Jumlah crop:", len(df))
print("Jumlah RAW unik:", df["raw_file_mapped"].nunique())
print("Jumlah kelas:", df["class_name"].nunique())
print("Official split:", df["split"].unique())

# Validasi kandidat group
missing_candidates = [g for g in candidate_groups if g not in df.columns]
if missing_candidates:
    raise ValueError(f"Kandidat group tidak ditemukan di df: {missing_candidates}")

# Pastikan group id aman sebagai string
for g in candidate_groups:
    df[g] = df[g].fillna("NA").astype(str)
    df.loc[df[g].str.strip() == "", g] = "NA"

splits = ["train", "validation", "test"]
classes = sorted(df["class_name"].unique())

# Official distribution sebagai target
official_class_counts = (
    df.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=classes, columns=splits, fill_value=0)
)

official_total_counts = df["split"].value_counts().reindex(splits, fill_value=0)
official_ratios = official_total_counts / len(df)

print("\n=== Official Split Counts ===")
display(official_total_counts.reset_index().rename(columns={"index": "split", "split": "n_crops"}))

print("\n=== Official Split Ratios ===")
display((official_ratios * 100).round(2).reset_index().rename(columns={"index": "split", "split": "percent"}))

print("\n=== Official Class Counts Preview ===")
display(official_class_counts)

def assign_groups_for_one_class(class_group_sizes, target_counts, seed):
    """
    Assign group dalam satu kelas ke train/validation/test.
    Tujuan: mendekati target official class distribution tanpa memecah group.
    """
    rng = np.random.default_rng(seed)

    groups = class_group_sizes.copy()
    groups["rand"] = rng.random(len(groups))

    # Urutkan group besar dulu, random sebagai tie-breaker
    groups = groups.sort_values(["n_crops", "rand"], ascending=[False, True]).reset_index(drop=True)

    current = {s: 0 for s in splits}
    assignment = {}

    # Hindari target nol dengan epsilon kecil
    target = {s: max(float(target_counts.get(s, 0)), 1.0) for s in splits}

    for _, row in groups.iterrows():
        gid = row["group_id"]
        size = int(row["n_crops"])

        best_split = None
        best_score = None

        for s in splits:
            temp = current.copy()
            temp[s] += size

            # Skor: normalized squared error terhadap target class count
            score = sum(((temp[k] - target[k]) / target[k]) ** 2 for k in splits)

            # Penalti ringan kalau split masih kosong dan ada kesempatan mengisi
            # supaya validation/test tidak mudah kosong
            empty_after = sum(1 for k in splits if temp[k] == 0)
            score += empty_after * 0.01

            if best_score is None or score < best_score:
                best_score = score
                best_split = s

        assignment[gid] = best_split
        current[best_split] += size

    return assignment

def simulate_one_candidate(df, group_col, seed):
    """
    Simulasi split untuk satu kandidat group dan satu seed.
    Karena kandidat yang dipilih tidak mencampur kelas, assignment dilakukan per kelas.
    """
    random.seed(seed)
    np.random.seed(seed)

    assign_records = []

    # Cek apakah group bercampur kelas
    group_class_check = df.groupby(group_col)["class_name"].nunique()
    n_multi_class_groups = int((group_class_check > 1).sum())

    if n_multi_class_groups > 0:
        print(f"PERINGATAN: {group_col} punya {n_multi_class_groups} multi-class groups.")

    for cls in classes:
        sub = df[df["class_name"] == cls].copy()

        class_group_sizes = (
            sub.groupby(group_col)
            .agg(n_crops=("crop_file", "count"))
            .reset_index()
            .rename(columns={group_col: "group_id"})
        )

        target_counts = official_class_counts.loc[cls].to_dict()

        # Seed dibuat berbeda per kelas agar reproducible tapi tidak sama persis
        class_seed = seed + abs(hash(cls)) % 100000

        assignment = assign_groups_for_one_class(class_group_sizes, target_counts, class_seed)

        for gid, sim_split in assignment.items():
            assign_records.append({
                "class_name": cls,
                "group_id": gid,
                "simulated_split": sim_split
            })

    assign_df = pd.DataFrame(assign_records)

    sim = df.merge(
        assign_df,
        left_on=["class_name", group_col],
        right_on=["class_name", "group_id"],
        how="left"
    )

    if sim["simulated_split"].isna().sum() > 0:
        raise ValueError(f"Ada crop yang tidak mendapat simulated_split pada {group_col}, seed {seed}")

    return sim, assign_df

summary_records = []
class_dist_records = []
leakage_records = []

for group_col in candidate_groups:
    for seed in seeds:
        print(f"\n=== Simulasi: {group_col} | seed={seed} ===")

        sim, assign_df = simulate_one_candidate(df, group_col, seed)

        # 1-2. jumlah crop dan persentase split
        split_counts = sim["simulated_split"].value_counts().reindex(splits, fill_value=0)
        split_pcts = split_counts / len(sim)

        # 3. jumlah kelas per split
        n_classes_by_split = (
            sim.groupby("simulated_split")["class_name"]
            .nunique()
            .reindex(splits, fill_value=0)
        )

        # 4-6. distribusi crop per kelas pada tiap split
        sim_class_counts = (
            sim.groupby(["class_name", "simulated_split"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=classes, columns=splits, fill_value=0)
        )

        missing_train = sim_class_counts.index[sim_class_counts["train"] == 0].tolist()
        missing_val = sim_class_counts.index[sim_class_counts["validation"] == 0].tolist()
        missing_test = sim_class_counts.index[sim_class_counts["test"] == 0].tolist()

        min_train_class = int(sim_class_counts["train"].min())
        min_val_class = int(sim_class_counts["validation"].min())
        min_test_class = int(sim_class_counts["test"].min())

        # 7. jumlah group unik per split
        groups_by_split = (
            sim.groupby("simulated_split")[group_col]
            .nunique()
            .reindex(splits, fill_value=0)
        )

        # 8-9. leakage check group muncul di lebih dari satu split
        group_split_counts = sim.groupby(group_col)["simulated_split"].nunique()
        leaked_groups = group_split_counts[group_split_counts > 1]
        n_total_groups = int(group_split_counts.shape[0])
        n_leaked_groups = int(len(leaked_groups))
        group_leakage_rate = n_leaked_groups / n_total_groups if n_total_groups > 0 else 0

        leaked_crop_count = int(sim[sim[group_col].isin(leaked_groups.index)]["crop_file"].count()) if n_leaked_groups > 0 else 0
        leaked_crop_rate = leaked_crop_count / len(sim) if len(sim) > 0 else 0

        # 10. distribusi per kelas dibanding official split
        abs_pct_points = []
        for cls in classes:
            class_total = sim_class_counts.loc[cls].sum()
            official_class_total = official_class_counts.loc[cls].sum()

            for sp in splits:
                sim_n = int(sim_class_counts.loc[cls, sp])
                off_n = int(official_class_counts.loc[cls, sp])

                sim_pct = sim_n / class_total if class_total > 0 else 0
                off_pct = off_n / official_class_total if official_class_total > 0 else 0
                diff_pp = (sim_pct - off_pct) * 100

                abs_pct_points.append(abs(diff_pp))

                class_dist_records.append({
                    "group_candidate": group_col,
                    "seed": seed,
                    "class_name": cls,
                    "split": sp,
                    "simulated_n_crops": sim_n,
                    "simulated_pct_within_class": sim_pct,
                    "official_n_crops": off_n,
                    "official_pct_within_class": off_pct,
                    "diff_pct_point_vs_official": diff_pp,
                    "abs_diff_pct_point_vs_official": abs(diff_pp)
                })

        mean_abs_pp = float(np.mean(abs_pct_points)) if abs_pct_points else 0.0
        max_abs_pp = float(np.max(abs_pct_points)) if abs_pct_points else 0.0

        # 11. feasibility
        all_classes_present_all_splits = (
            len(missing_train) == 0 and len(missing_val) == 0 and len(missing_test) == 0
        )
        no_group_leakage = n_leaked_groups == 0
        sufficient_min_per_class = (
            min_train_class >= MIN_TRAIN_PER_CLASS and
            min_val_class >= MIN_VAL_PER_CLASS and
            min_test_class >= MIN_TEST_PER_CLASS
        )
        train_not_too_small = split_pcts["train"] >= MIN_TRAIN_RATIO
        distribution_close_enough = (
            mean_abs_pp <= MAX_MEAN_ABS_PCT_POINT and
            max_abs_pp <= MAX_ABS_PCT_POINT
        )

        feasible = (
            all_classes_present_all_splits and
            no_group_leakage and
            sufficient_min_per_class and
            train_not_too_small and
            distribution_close_enough
        )

        # Simpan summary
        summary_records.append({
            "group_candidate": group_col,
            "seed": seed,
            "n_crops_total": int(len(sim)),
            "n_groups_total": n_total_groups,
            "train_crops": int(split_counts["train"]),
            "validation_crops": int(split_counts["validation"]),
            "test_crops": int(split_counts["test"]),
            "train_pct": float(split_pcts["train"]),
            "validation_pct": float(split_pcts["validation"]),
            "test_pct": float(split_pcts["test"]),
            "train_n_classes": int(n_classes_by_split["train"]),
            "validation_n_classes": int(n_classes_by_split["validation"]),
            "test_n_classes": int(n_classes_by_split["test"]),
            "missing_train_classes": json.dumps(missing_train, ensure_ascii=False),
            "missing_validation_classes": json.dumps(missing_val, ensure_ascii=False),
            "missing_test_classes": json.dumps(missing_test, ensure_ascii=False),
            "min_crop_per_class_train": min_train_class,
            "min_crop_per_class_validation": min_val_class,
            "min_crop_per_class_test": min_test_class,
            "train_n_groups": int(groups_by_split["train"]),
            "validation_n_groups": int(groups_by_split["validation"]),
            "test_n_groups": int(groups_by_split["test"]),
            "n_leaked_groups": n_leaked_groups,
            "group_leakage_rate": float(group_leakage_rate),
            "leaked_crop_count": leaked_crop_count,
            "leaked_crop_rate": float(leaked_crop_rate),
            "mean_abs_pct_point_vs_official": mean_abs_pp,
            "max_abs_pct_point_vs_official": max_abs_pp,
            "all_classes_present_all_splits": bool(all_classes_present_all_splits),
            "no_group_leakage": bool(no_group_leakage),
            "sufficient_min_per_class": bool(sufficient_min_per_class),
            "train_not_too_small": bool(train_not_too_small),
            "distribution_close_enough": bool(distribution_close_enough),
            "feasible_preliminary": bool(feasible)
        })

        leakage_records.append({
            "group_candidate": group_col,
            "seed": seed,
            "n_total_groups": n_total_groups,
            "n_leaked_groups": n_leaked_groups,
            "group_leakage_rate": float(group_leakage_rate),
            "leaked_crop_count": leaked_crop_count,
            "leaked_crop_rate": float(leaked_crop_rate),
            "leaked_group_ids_preview": json.dumps(list(leaked_groups.index[:20]), ensure_ascii=False)
        })

        print("Split counts:")
        display(split_counts.reset_index().rename(columns={"index": "split", "simulated_split": "n_crops"}))

        print("Class count by split:")
        display(n_classes_by_split.reset_index().rename(columns={"index": "split", "class_name": "n_classes"}))

        print("Missing validation classes:", missing_val)
        print("Missing test classes:", missing_test)
        print("Group leakage:", n_leaked_groups, "/", n_total_groups)
        print("Mean abs pct-point vs official:", round(mean_abs_pp, 3))
        print("Max abs pct-point vs official:", round(max_abs_pp, 3))
        print("Feasible preliminary:", feasible)

# Buat dataframe output
summary_df = pd.DataFrame(summary_records)
class_dist_df = pd.DataFrame(class_dist_records)
leakage_df = pd.DataFrame(leakage_records)

# Simpan output utama V4
summary_df.to_csv(summary_csv, index=False)
class_dist_df.to_csv(class_dist_csv, index=False)
leakage_df.to_csv(leakage_csv, index=False)

# Recommendation sementara berdasarkan seed 42
recommendation = {
    "stage": "V4_seed_42_format_check",
    "note": "Ini hasil awal seed 42. Belum memilih group final. Multi-seed masih perlu dijalankan setelah format disetujui.",
    "seeds": seeds,
    "candidate_groups": candidate_groups,
    "feasibility_thresholds": {
        "MIN_VAL_PER_CLASS": MIN_VAL_PER_CLASS,
        "MIN_TEST_PER_CLASS": MIN_TEST_PER_CLASS,
        "MIN_TRAIN_PER_CLASS": MIN_TRAIN_PER_CLASS,
        "MIN_TRAIN_RATIO": MIN_TRAIN_RATIO,
        "MAX_MEAN_ABS_PCT_POINT": MAX_MEAN_ABS_PCT_POINT,
        "MAX_ABS_PCT_POINT": MAX_ABS_PCT_POINT
    },
    "summary_csv": str(summary_csv),
    "class_distribution_csv": str(class_dist_csv),
    "leakage_check_csv": str(leakage_csv),
    "summary_records": summary_df.to_dict(orient="records")
}

recommendation_json.write_text(json.dumps(recommendation, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n=== V4 Seed 42 Summary ===")
display(summary_df)

print("\n=== V4 Leakage Check ===")
display(leakage_df)

print("\n=== V4 Class Distribution Preview ===")
display(class_dist_df.head(60))

print("\n=== Output V4 Seed 42 Disimpan ===")
print("Summary CSV:")
print(summary_csv)
print("Class distribution CSV:")
print(class_dist_csv)
print("Leakage check CSV:")
print(leakage_csv)
print("Recommendation JSON:")
print(recommendation_json)

In [ ]:
# Sel 20 — V4 Multi-Seed Group-Aware Split Simulation

from pathlib import Path
import pandas as pd
import numpy as np
import json
import random

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input dari V3
crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"

# Output V4 final multi-seed
summary_csv = outputs_dir / "v4_group_split_simulation_summary.csv"
class_dist_csv = outputs_dir / "v4_group_split_class_distribution.csv"
leakage_csv = outputs_dir / "v4_group_split_leakage_check.csv"
recommendation_json = outputs_dir / "v4_group_split_recommendation.json"

candidate_groups = [
    "G09_Date_Magnification",
    "G13_Date_Photographer_CameraModel_GridposXYZ_Magnification",
    "G14_RAWSource",
    "G12_Date_Photographer_CameraModel_Magnification"
]

seeds = [42, 123, 456, 789, 1024]

# Threshold awal feasibility
MIN_VAL_PER_CLASS = 5
MIN_TEST_PER_CLASS = 5
MIN_TRAIN_PER_CLASS = 10
MIN_TRAIN_RATIO = 0.50
MAX_MEAN_ABS_PCT_POINT = 12.0
MAX_ABS_PCT_POINT = 35.0

df = pd.read_csv(crop_meta_csv, keep_default_na=False)

print("=== Input V4 Multi-Seed ===")
print("Shape:", df.shape)
print("Jumlah crop:", len(df))
print("Jumlah RAW unik:", df["raw_file_mapped"].nunique())
print("Jumlah kelas:", df["class_name"].nunique())
print("Seeds:", seeds)

missing_candidates = [g for g in candidate_groups if g not in df.columns]
if missing_candidates:
    raise ValueError(f"Kandidat group tidak ditemukan: {missing_candidates}")

for g in candidate_groups:
    df[g] = df[g].fillna("NA").astype(str)
    df.loc[df[g].str.strip() == "", g] = "NA"

splits = ["train", "validation", "test"]
classes = sorted(df["class_name"].unique())

official_class_counts = (
    df.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=classes, columns=splits, fill_value=0)
)

def stable_class_seed(base_seed, cls):
    """
    Hindari hash() Python yang bisa berubah antar runtime.
    """
    return base_seed + sum(ord(ch) for ch in str(cls)) * 17

def assign_groups_for_one_class(class_group_sizes, target_counts, seed):
    rng = np.random.default_rng(seed)

    groups = class_group_sizes.copy()
    groups["rand"] = rng.random(len(groups))
    groups = groups.sort_values(["n_crops", "rand"], ascending=[False, True]).reset_index(drop=True)

    current = {s: 0 for s in splits}
    assignment = {}
    target = {s: max(float(target_counts.get(s, 0)), 1.0) for s in splits}

    for _, row in groups.iterrows():
        gid = row["group_id"]
        size = int(row["n_crops"])

        best_split = None
        best_score = None

        for s in splits:
            temp = current.copy()
            temp[s] += size

            score = sum(((temp[k] - target[k]) / target[k]) ** 2 for k in splits)

            empty_after = sum(1 for k in splits if temp[k] == 0)
            score += empty_after * 0.01

            if best_score is None or score < best_score:
                best_score = score
                best_split = s

        assignment[gid] = best_split
        current[best_split] += size

    return assignment

def simulate_one_candidate(df, group_col, seed):
    assign_records = []

    group_class_check = df.groupby(group_col)["class_name"].nunique()
    n_multi_class_groups = int((group_class_check > 1).sum())

    if n_multi_class_groups > 0:
        print(f"PERINGATAN: {group_col} punya {n_multi_class_groups} multi-class groups.")

    for cls in classes:
        sub = df[df["class_name"] == cls].copy()

        class_group_sizes = (
            sub.groupby(group_col)
            .agg(n_crops=("crop_file", "count"))
            .reset_index()
            .rename(columns={group_col: "group_id"})
        )

        target_counts = official_class_counts.loc[cls].to_dict()
        class_seed = stable_class_seed(seed, cls)

        assignment = assign_groups_for_one_class(class_group_sizes, target_counts, class_seed)

        for gid, sim_split in assignment.items():
            assign_records.append({
                "class_name": cls,
                "group_id": gid,
                "simulated_split": sim_split
            })

    assign_df = pd.DataFrame(assign_records)

    sim = df.merge(
        assign_df,
        left_on=["class_name", group_col],
        right_on=["class_name", "group_id"],
        how="left"
    )

    if sim["simulated_split"].isna().sum() > 0:
        raise ValueError(f"Ada crop yang tidak mendapat simulated_split pada {group_col}, seed {seed}")

    return sim, assign_df

summary_records = []
class_dist_records = []
leakage_records = []

for group_col in candidate_groups:
    for seed in seeds:
        print(f"\n=== Simulasi: {group_col} | seed={seed} ===")

        sim, assign_df = simulate_one_candidate(df, group_col, seed)

        split_counts = sim["simulated_split"].value_counts().reindex(splits, fill_value=0)
        split_pcts = split_counts / len(sim)

        n_classes_by_split = (
            sim.groupby("simulated_split")["class_name"]
            .nunique()
            .reindex(splits, fill_value=0)
        )

        sim_class_counts = (
            sim.groupby(["class_name", "simulated_split"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=classes, columns=splits, fill_value=0)
        )

        missing_train = sim_class_counts.index[sim_class_counts["train"] == 0].tolist()
        missing_val = sim_class_counts.index[sim_class_counts["validation"] == 0].tolist()
        missing_test = sim_class_counts.index[sim_class_counts["test"] == 0].tolist()

        min_train_class = int(sim_class_counts["train"].min())
        min_val_class = int(sim_class_counts["validation"].min())
        min_test_class = int(sim_class_counts["test"].min())

        groups_by_split = (
            sim.groupby("simulated_split")[group_col]
            .nunique()
            .reindex(splits, fill_value=0)
        )

        group_split_counts = sim.groupby(group_col)["simulated_split"].nunique()
        leaked_groups = group_split_counts[group_split_counts > 1]

        n_total_groups = int(group_split_counts.shape[0])
        n_leaked_groups = int(len(leaked_groups))
        group_leakage_rate = n_leaked_groups / n_total_groups if n_total_groups > 0 else 0

        leaked_crop_count = int(sim[sim[group_col].isin(leaked_groups.index)]["crop_file"].count()) if n_leaked_groups > 0 else 0
        leaked_crop_rate = leaked_crop_count / len(sim) if len(sim) > 0 else 0

        abs_pct_points = []

        for cls in classes:
            class_total = sim_class_counts.loc[cls].sum()
            official_class_total = official_class_counts.loc[cls].sum()

            for sp in splits:
                sim_n = int(sim_class_counts.loc[cls, sp])
                off_n = int(official_class_counts.loc[cls, sp])

                sim_pct = sim_n / class_total if class_total > 0 else 0
                off_pct = off_n / official_class_total if official_class_total > 0 else 0
                diff_pp = (sim_pct - off_pct) * 100

                abs_pct_points.append(abs(diff_pp))

                class_dist_records.append({
                    "group_candidate": group_col,
                    "seed": seed,
                    "class_name": cls,
                    "split": sp,
                    "simulated_n_crops": sim_n,
                    "simulated_pct_within_class": sim_pct,
                    "official_n_crops": off_n,
                    "official_pct_within_class": off_pct,
                    "diff_pct_point_vs_official": diff_pp,
                    "abs_diff_pct_point_vs_official": abs(diff_pp)
                })

        mean_abs_pp = float(np.mean(abs_pct_points)) if abs_pct_points else 0.0
        max_abs_pp = float(np.max(abs_pct_points)) if abs_pct_points else 0.0

        all_classes_present_all_splits = (
            len(missing_train) == 0 and len(missing_val) == 0 and len(missing_test) == 0
        )
        no_group_leakage = n_leaked_groups == 0
        sufficient_min_per_class = (
            min_train_class >= MIN_TRAIN_PER_CLASS and
            min_val_class >= MIN_VAL_PER_CLASS and
            min_test_class >= MIN_TEST_PER_CLASS
        )
        train_not_too_small = split_pcts["train"] >= MIN_TRAIN_RATIO
        distribution_close_enough = (
            mean_abs_pp <= MAX_MEAN_ABS_PCT_POINT and
            max_abs_pp <= MAX_ABS_PCT_POINT
        )

        feasible = (
            all_classes_present_all_splits and
            no_group_leakage and
            sufficient_min_per_class and
            train_not_too_small and
            distribution_close_enough
        )

        summary_records.append({
            "group_candidate": group_col,
            "seed": seed,
            "n_crops_total": int(len(sim)),
            "n_groups_total": n_total_groups,
            "train_crops": int(split_counts["train"]),
            "validation_crops": int(split_counts["validation"]),
            "test_crops": int(split_counts["test"]),
            "train_pct": float(split_pcts["train"]),
            "validation_pct": float(split_pcts["validation"]),
            "test_pct": float(split_pcts["test"]),
            "train_n_classes": int(n_classes_by_split["train"]),
            "validation_n_classes": int(n_classes_by_split["validation"]),
            "test_n_classes": int(n_classes_by_split["test"]),
            "missing_train_classes": json.dumps(missing_train, ensure_ascii=False),
            "missing_validation_classes": json.dumps(missing_val, ensure_ascii=False),
            "missing_test_classes": json.dumps(missing_test, ensure_ascii=False),
            "min_crop_per_class_train": min_train_class,
            "min_crop_per_class_validation": min_val_class,
            "min_crop_per_class_test": min_test_class,
            "train_n_groups": int(groups_by_split["train"]),
            "validation_n_groups": int(groups_by_split["validation"]),
            "test_n_groups": int(groups_by_split["test"]),
            "n_leaked_groups": n_leaked_groups,
            "group_leakage_rate": float(group_leakage_rate),
            "leaked_crop_count": leaked_crop_count,
            "leaked_crop_rate": float(leaked_crop_rate),
            "mean_abs_pct_point_vs_official": mean_abs_pp,
            "max_abs_pct_point_vs_official": max_abs_pp,
            "all_classes_present_all_splits": bool(all_classes_present_all_splits),
            "no_group_leakage": bool(no_group_leakage),
            "sufficient_min_per_class": bool(sufficient_min_per_class),
            "train_not_too_small": bool(train_not_too_small),
            "distribution_close_enough": bool(distribution_close_enough),
            "feasible_preliminary": bool(feasible)
        })

        leakage_records.append({
            "group_candidate": group_col,
            "seed": seed,
            "n_total_groups": n_total_groups,
            "n_leaked_groups": n_leaked_groups,
            "group_leakage_rate": float(group_leakage_rate),
            "leaked_crop_count": leaked_crop_count,
            "leaked_crop_rate": float(leaked_crop_rate),
            "leaked_group_ids_preview": json.dumps(list(leaked_groups.index[:20]), ensure_ascii=False)
        })

        print(
            f"train/val/test = "
            f"{int(split_counts['train'])}/{int(split_counts['validation'])}/{int(split_counts['test'])} | "
            f"leak={n_leaked_groups} | "
            f"mean_abs_pp={mean_abs_pp:.3f} | "
            f"max_abs_pp={max_abs_pp:.3f} | "
            f"feasible={feasible}"
        )

summary_df = pd.DataFrame(summary_records)
class_dist_df = pd.DataFrame(class_dist_records)
leakage_df = pd.DataFrame(leakage_records)

summary_df.to_csv(summary_csv, index=False)
class_dist_df.to_csv(class_dist_csv, index=False)
leakage_df.to_csv(leakage_csv, index=False)

# Agregasi stabilitas per kandidat
agg_df = (
    summary_df
    .groupby("group_candidate")
    .agg(
        n_seeds=("seed", "nunique"),
        feasible_all_seeds=("feasible_preliminary", "all"),
        feasible_count=("feasible_preliminary", "sum"),
        train_crops_mean=("train_crops", "mean"),
        train_crops_std=("train_crops", "std"),
        validation_crops_mean=("validation_crops", "mean"),
        validation_crops_std=("validation_crops", "std"),
        test_crops_mean=("test_crops", "mean"),
        test_crops_std=("test_crops", "std"),
        min_val_class_min=("min_crop_per_class_validation", "min"),
        min_test_class_min=("min_crop_per_class_test", "min"),
        leaked_groups_max=("n_leaked_groups", "max"),
        mean_abs_pp_mean=("mean_abs_pct_point_vs_official", "mean"),
        mean_abs_pp_std=("mean_abs_pct_point_vs_official", "std"),
        max_abs_pp_max=("max_abs_pct_point_vs_official", "max")
    )
    .reset_index()
)

# Ranking sementara:
# 1 feasible semua seed
# 2 leakage max rendah
# 3 mean_abs_pp rendah
# 4 max_abs_pp rendah
agg_df["rank_score"] = (
    (~agg_df["feasible_all_seeds"]).astype(int) * 1000
    + agg_df["leaked_groups_max"] * 100
    + agg_df["mean_abs_pp_mean"]
    + agg_df["max_abs_pp_max"] / 100
)

agg_df = agg_df.sort_values("rank_score").reset_index(drop=True)

recommendation = {
    "stage": "V4_multi_seed_simulation",
    "note": "Belum memilih group final; ini ringkasan feasibility multi-seed untuk dibawa ke MASTER CHAT.",
    "seeds": seeds,
    "candidate_groups": candidate_groups,
    "feasibility_thresholds": {
        "MIN_VAL_PER_CLASS": MIN_VAL_PER_CLASS,
        "MIN_TEST_PER_CLASS": MIN_TEST_PER_CLASS,
        "MIN_TRAIN_PER_CLASS": MIN_TRAIN_PER_CLASS,
        "MIN_TRAIN_RATIO": MIN_TRAIN_RATIO,
        "MAX_MEAN_ABS_PCT_POINT": MAX_MEAN_ABS_PCT_POINT,
        "MAX_ABS_PCT_POINT": MAX_ABS_PCT_POINT
    },
    "summary_csv": str(summary_csv),
    "class_distribution_csv": str(class_dist_csv),
    "leakage_check_csv": str(leakage_csv),
    "aggregate_summary": agg_df.to_dict(orient="records")
}

recommendation_json.write_text(json.dumps(recommendation, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n=== V4 Multi-Seed Summary ===")
display(summary_df)

print("\n=== V4 Multi-Seed Aggregate by Group Candidate ===")
display(agg_df)

print("\n=== V4 Multi-Seed Leakage Check ===")
display(leakage_df)

print("\n=== Output V4 Multi-Seed Disimpan ===")
print("Summary CSV:")
print(summary_csv)
print("Class distribution CSV:")
print(class_dist_csv)
print("Leakage check CSV:")
print(leakage_csv)
print("Recommendation JSON:")
print(recommendation_json)

In [ ]:
# Sel 21 — Finalisasi Ringkasan dan Recommendation JSON V4

from pathlib import Path
import pandas as pd
import json

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

summary_csv = outputs_dir / "v4_group_split_simulation_summary.csv"
class_dist_csv = outputs_dir / "v4_group_split_class_distribution.csv"
leakage_csv = outputs_dir / "v4_group_split_leakage_check.csv"
recommendation_json = outputs_dir / "v4_group_split_recommendation.json"

summary_df = pd.read_csv(summary_csv)
class_dist_df = pd.read_csv(class_dist_csv)
leakage_df = pd.read_csv(leakage_csv)

print("=== Input V4 Finalization ===")
print("Summary shape:", summary_df.shape)
print("Class distribution shape:", class_dist_df.shape)
print("Leakage shape:", leakage_df.shape)

# Agregasi final per kandidat
agg = (
    summary_df
    .groupby("group_candidate")
    .agg(
        n_seeds=("seed", "nunique"),
        feasible_all_seeds=("feasible_preliminary", "all"),
        feasible_count=("feasible_preliminary", "sum"),
        train_crops_mean=("train_crops", "mean"),
        validation_crops_mean=("validation_crops", "mean"),
        test_crops_mean=("test_crops", "mean"),
        train_crops_std=("train_crops", "std"),
        validation_crops_std=("validation_crops", "std"),
        test_crops_std=("test_crops", "std"),
        min_val_class_min=("min_crop_per_class_validation", "min"),
        min_test_class_min=("min_crop_per_class_test", "min"),
        leaked_groups_max=("n_leaked_groups", "max"),
        leaked_crop_rate_max=("leaked_crop_rate", "max"),
        mean_abs_pp_mean=("mean_abs_pct_point_vs_official", "mean"),
        max_abs_pp_max=("max_abs_pct_point_vs_official", "max")
    )
    .reset_index()
)

# Tambahkan kategori interpretasi metodologis
def methodological_role(group):
    if group == "G14_RAWSource":
        return "pembanding konservatif terbaik; paling aman dari sisi RAW leakage, tetapi paling granular"
    if group.startswith("G13_"):
        return "cadangan konservatif; dekat dengan official split, tetapi masih cenderung granular"
    if group == "G09_Date_Magnification":
        return "kandidat utama metodologis; lebih moderat dan memakai metadata acquisition yang masuk akal"
    if group == "G12_Date_Photographer_CameraModel_Magnification":
        return "opsional; hasil identik dengan G09 sehingga tidak perlu diprioritaskan"
    return "tidak diprioritaskan"

agg["methodological_role"] = agg["group_candidate"].apply(methodological_role)

# Ranking teknis berdasarkan kedekatan distribusi + leakage + feasibility
agg["technical_rank_score"] = (
    (~agg["feasible_all_seeds"]).astype(int) * 1000
    + agg["leaked_groups_max"] * 100
    + agg["mean_abs_pp_mean"]
    + agg["max_abs_pp_max"] / 100
)

agg = agg.sort_values("technical_rank_score").reset_index(drop=True)

# Buat recommendation final
recommendation = {
    "stage": "V4_final_group_split_simulation",
    "status": "completed",
    "important_note": (
        "V4 belum menetapkan group final. Hasil ini adalah bahan keputusan untuk MASTER CHAT. "
        "Tidak ada training model dilakukan."
    ),
    "seeds": sorted(summary_df["seed"].unique().tolist()),
    "candidate_groups_tested": sorted(summary_df["group_candidate"].unique().tolist()),
    "overall_findings": {
        "all_candidates_feasible_all_seeds": bool(agg["feasible_all_seeds"].all()),
        "max_leaked_groups_all_candidates": int(agg["leaked_groups_max"].max()),
        "all_splits_have_14_classes": bool(
            (summary_df["train_n_classes"].eq(14).all()) and
            (summary_df["validation_n_classes"].eq(14).all()) and
            (summary_df["test_n_classes"].eq(14).all())
        ),
        "seed_variation_observed": bool(
            (agg["train_crops_std"].fillna(0).gt(0).any()) or
            (agg["validation_crops_std"].fillna(0).gt(0).any()) or
            (agg["test_crops_std"].fillna(0).gt(0).any())
        )
    },
    "technical_best_by_distribution": {
        "best": "G14_RAWSource",
        "reason": (
            "Paling dekat dengan official split: mean_abs_pct_point_vs_official paling kecil "
            "dan max_abs_pct_point_vs_official paling kecil, serta leakage 0."
        )
    },
    "methodological_primary_candidate": {
        "candidate": "G09_Date_Magnification",
        "reason": (
            "Lebih moderat dibanding RAWSource/G13 karena jumlah group 352, "
            "menggunakan kombinasi metadata acquisition yang masuk akal, tidak mencampur kelas, "
            "dan tetap feasible tanpa leakage pada semua seed."
        )
    },
    "backup_candidate": {
        "candidate": "G13_Date_Photographer_CameraModel_GridposXYZ_Magnification",
        "reason": (
            "Distribusi sangat dekat dengan official split dan leakage 0, "
            "tetapi lebih granular dibanding G09."
        )
    },
    "conservative_baseline": {
        "candidate": "G14_RAWSource",
        "reason": (
            "Paling konservatif karena memastikan semua crop dari RAW yang sama berada pada split yang sama. "
            "Cocok sebagai pembanding atau skenario paling aman."
        )
    },
    "not_prioritized": {
        "candidate": "G12_Date_Photographer_CameraModel_Magnification",
        "reason": (
            "Hasil simulasi identik dengan G09, sehingga tidak memberi manfaat tambahan untuk diprioritaskan."
        )
    },
    "scenario_b_feasibility": {
        "leakage_aware_split_is_feasible": True,
        "reason": (
            "Semua kandidat utama menghasilkan split train/validation/test dengan 14 kelas, "
            "tidak ada group leakage, validation/test memiliki jumlah sampel per kelas yang cukup, "
            "dan train set tidak terlalu kecil."
        )
    },
    "aggregate_summary": agg.to_dict(orient="records"),
    "output_files": {
        "summary_csv": str(summary_csv),
        "class_distribution_csv": str(class_dist_csv),
        "leakage_check_csv": str(leakage_csv),
        "recommendation_json": str(recommendation_json)
    }
}

recommendation_json.write_text(
    json.dumps(recommendation, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== V4 Final Aggregate Summary ===")
display(agg)

print("\n=== V4 Final Recommendation ===")
print("Technical best by distribution: G14_RAWSource")
print("Methodological primary candidate: G09_Date_Magnification")
print("Backup candidate: G13_Date_Photographer_CameraModel_GridposXYZ_Magnification")
print("Conservative baseline: G14_RAWSource")
print("Scenario B leakage-aware split feasible: True")

print("\nRecommendation JSON updated:")
print(recommendation_json)

In [ ]:
# Sel 22 — V5 Membuat File Split Manifest Final TEM Virus

from pathlib import Path
import pandas as pd
import numpy as np
import json

# ============================================================
# PATH
# ============================================================

dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"
outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"

# Output V5
manifest_A_csv = outputs_dir / "v5_split_manifest_A_official.csv"
manifest_B_csv = outputs_dir / "v5_split_manifest_B_G09_Date_Magnification.csv"
manifest_B_cons_csv = outputs_dir / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
class_dist_summary_csv = outputs_dir / "v5_split_class_distribution_summary.csv"
leakage_summary_csv = outputs_dir / "v5_split_leakage_check_summary.csv"
decision_support_json = outputs_dir / "v5_split_final_decision_support.json"

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(crop_meta_csv, keep_default_na=False)

print("=== Input V5 ===")
print("Crop-level metadata:", crop_meta_csv)
print("Shape:", df.shape)
print("Jumlah crop:", len(df))
print("Jumlah kelas:", df["class_name"].nunique())
print("Official split:", sorted(df["split"].unique()))

required_cols = [
    "split",
    "class_name",
    "crop_file",
    "crop_filename",
    "raw_file_mapped",
    "G09_Date_Magnification",
    "G14_RAWSource"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

# Pastikan hanya official train/validation/test, bukan augmented_train
official_allowed_splits = ["train", "validation", "test"]
df = df[df["split"].isin(official_allowed_splits)].copy()

# Pastikan path file crop ada
df["filepath"] = df["crop_file"].apply(lambda x: str(crop_root / x))
df["file_exists"] = df["filepath"].apply(lambda x: Path(x).exists())

print("\n=== Cek File Crop ===")
print("File exists:", int(df["file_exists"].sum()), "/", len(df))
if not df["file_exists"].all():
    display(df.loc[~df["file_exists"], ["crop_file", "filepath"]].head(20))
    raise ValueError("Ada file crop yang tidak ditemukan. Stop dulu.")

# Label map deterministik
classes = sorted(df["class_name"].unique())
label_map = {cls: i for i, cls in enumerate(classes)}
df["label_id"] = df["class_name"].map(label_map)

print("\n=== Label Map ===")
for cls, lid in label_map.items():
    print(f"{lid}: {cls}")

# ============================================================
# FUNGSI SIMULASI SPLIT GROUP-AWARE
# ============================================================

splits = ["train", "validation", "test"]

official_class_counts = (
    df.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=classes, columns=splits, fill_value=0)
)

def stable_class_seed(base_seed, cls):
    return base_seed + sum(ord(ch) for ch in str(cls)) * 17

def assign_groups_for_one_class(class_group_sizes, target_counts, seed):
    """
    Greedy group assignment per class.
    Tujuan: mendekati official class distribution tanpa memecah group.
    """
    rng = np.random.default_rng(seed)

    groups = class_group_sizes.copy()
    groups["rand"] = rng.random(len(groups))
    groups = groups.sort_values(["n_crops", "rand"], ascending=[False, True]).reset_index(drop=True)

    current = {s: 0 for s in splits}
    assignment = {}
    target = {s: max(float(target_counts.get(s, 0)), 1.0) for s in splits}

    for _, row in groups.iterrows():
        gid = row["group_id"]
        size = int(row["n_crops"])

        best_split = None
        best_score = None

        for s in splits:
            temp = current.copy()
            temp[s] += size

            score = sum(((temp[k] - target[k]) / target[k]) ** 2 for k in splits)
            empty_after = sum(1 for k in splits if temp[k] == 0)
            score += empty_after * 0.01

            if best_score is None or score < best_score:
                best_score = score
                best_split = s

        assignment[gid] = best_split
        current[best_split] += size

    return assignment

def make_group_aware_split(data, group_col, seed=42):
    """
    Membuat split train/validation/test berdasarkan group_col.
    Group tidak boleh pecah lintas split.
    Assignment dilakukan per kelas.
    """
    assign_records = []

    # Safety check: group multi-class
    group_class_check = data.groupby(group_col)["class_name"].nunique()
    n_multi_class_groups = int((group_class_check > 1).sum())
    if n_multi_class_groups > 0:
        print(f"PERINGATAN: {group_col} memiliki {n_multi_class_groups} group multi-class.")

    for cls in classes:
        sub = data[data["class_name"] == cls].copy()

        class_group_sizes = (
            sub.groupby(group_col)
            .agg(n_crops=("crop_file", "count"))
            .reset_index()
            .rename(columns={group_col: "group_id"})
        )

        target_counts = official_class_counts.loc[cls].to_dict()
        class_seed = stable_class_seed(seed, cls)

        assignment = assign_groups_for_one_class(class_group_sizes, target_counts, class_seed)

        for gid, split_name in assignment.items():
            assign_records.append({
                "class_name": cls,
                "group_id": gid,
                "split_final": split_name
            })

    assign_df = pd.DataFrame(assign_records)

    out = data.merge(
        assign_df,
        left_on=["class_name", group_col],
        right_on=["class_name", "group_id"],
        how="left"
    )

    if out["split_final"].isna().sum() > 0:
        raise ValueError(f"Ada baris tanpa split_final untuk {group_col}")

    return out

# ============================================================
# BUAT SKENARIO SPLIT
# ============================================================

# Skenario A — official split
A = df.copy()
A["split_final"] = A["split"]
A["raw_source_id"] = A["raw_file_mapped"]

# Skenario B — G09_Date_Magnification
B = make_group_aware_split(df, "G09_Date_Magnification", seed=42)
B["raw_source_id"] = B["raw_file_mapped"]

# Skenario B-conservative — G14_RAWSource
B_cons = make_group_aware_split(df, "G14_RAWSource", seed=42)
B_cons["raw_source_id"] = B_cons["raw_file_mapped"]

# ============================================================
# MANIFEST OUTPUT
# ============================================================

manifest_A = A[[
    "filepath",
    "crop_filename",
    "class_name",
    "label_id",
    "split_final",
    "raw_source_id",
    "G09_Date_Magnification",
    "G14_RAWSource"
]].copy()

manifest_A = manifest_A.rename(columns={
    "crop_filename": "filename",
    "split_final": "split"
})

manifest_B = B[[
    "filepath",
    "crop_filename",
    "class_name",
    "label_id",
    "split_final",
    "raw_source_id",
    "G09_Date_Magnification"
]].copy()

manifest_B = manifest_B.rename(columns={
    "crop_filename": "filename",
    "split_final": "split",
    "G09_Date_Magnification": "group_id"
})

manifest_B_cons = B_cons[[
    "filepath",
    "crop_filename",
    "class_name",
    "label_id",
    "split_final",
    "raw_source_id",
    "G14_RAWSource"
]].copy()

manifest_B_cons = manifest_B_cons.rename(columns={
    "crop_filename": "filename",
    "split_final": "split",
    "G14_RAWSource": "group_id"
})

# Simpan manifest
manifest_A.to_csv(manifest_A_csv, index=False)
manifest_B.to_csv(manifest_B_csv, index=False)
manifest_B_cons.to_csv(manifest_B_cons_csv, index=False)

print("\n=== Manifest Disimpan ===")
print("A official:", manifest_A_csv)
print("B G09:", manifest_B_csv)
print("B conservative G14:", manifest_B_cons_csv)

# ============================================================
# DISTRIBUSI KELAS
# ============================================================

scenario_frames = {
    "A_official": A,
    "B_G09_Date_Magnification": B,
    "B_conservative_G14_RAWSource": B_cons
}

class_dist_records = []

for scenario_name, data in scenario_frames.items():
    class_counts = (
        data.groupby(["class_name", "split_final"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=classes, columns=splits, fill_value=0)
    )

    split_totals = data["split_final"].value_counts().reindex(splits, fill_value=0)
    split_n_classes = data.groupby("split_final")["class_name"].nunique().reindex(splits, fill_value=0)

    min_per_split = {
        s: int(class_counts[s].min()) for s in splits
    }

    for cls in classes:
        for s in splits:
            class_dist_records.append({
                "scenario": scenario_name,
                "class_name": cls,
                "split": s,
                "n_crops": int(class_counts.loc[cls, s]),
                "split_total_crops": int(split_totals[s]),
                "split_n_classes": int(split_n_classes[s]),
                "min_crop_per_class_in_split": int(min_per_split[s])
            })

class_dist_df = pd.DataFrame(class_dist_records)
class_dist_df.to_csv(class_dist_summary_csv, index=False)

print("\n=== Distribusi Kelas Disimpan ===")
print(class_dist_summary_csv)

# ============================================================
# LEAKAGE CHECK
# ============================================================

def leakage_check(data, scenario_name, group_col, group_label):
    group_split_counts = data.groupby(group_col)["split_final"].nunique()
    leaked_groups = group_split_counts[group_split_counts > 1]

    n_total_groups = int(group_split_counts.shape[0])
    n_leaked_groups = int(len(leaked_groups))

    leaked_crop_count = int(data[data[group_col].isin(leaked_groups.index)].shape[0]) if n_leaked_groups > 0 else 0
    leaked_crop_rate = leaked_crop_count / len(data) if len(data) > 0 else 0.0

    return {
        "scenario": scenario_name,
        "group_basis": group_label,
        "n_total_groups": n_total_groups,
        "n_leaked_groups": n_leaked_groups,
        "group_leakage_rate": n_leaked_groups / n_total_groups if n_total_groups > 0 else 0.0,
        "leaked_crop_count": leaked_crop_count,
        "leaked_crop_rate": leaked_crop_rate,
        "status": "PASS" if n_leaked_groups == 0 else "FAIL",
        "leaked_group_preview": json.dumps(list(leaked_groups.index[:20]), ensure_ascii=False)
    }

leakage_records = []

# Skenario A: cek terhadap G09 dan G14 supaya terlihat official split bocor/tidak terhadap group metadata
leakage_records.append(leakage_check(A, "A_official", "G09_Date_Magnification", "G09_Date_Magnification"))
leakage_records.append(leakage_check(A, "A_official", "G14_RAWSource", "G14_RAWSource"))

# Skenario B: basis utamanya G09; cek juga G14 sebagai sanity check
leakage_records.append(leakage_check(B, "B_G09_Date_Magnification", "G09_Date_Magnification", "G09_Date_Magnification"))
leakage_records.append(leakage_check(B, "B_G09_Date_Magnification", "G14_RAWSource", "G14_RAWSource"))

# Skenario B-conservative: basis utamanya G14; cek juga G09 sebagai informasi tambahan
leakage_records.append(leakage_check(B_cons, "B_conservative_G14_RAWSource", "G14_RAWSource", "G14_RAWSource"))
leakage_records.append(leakage_check(B_cons, "B_conservative_G14_RAWSource", "G09_Date_Magnification", "G09_Date_Magnification"))

leakage_df = pd.DataFrame(leakage_records)
leakage_df.to_csv(leakage_summary_csv, index=False)

print("\n=== Leakage Check Disimpan ===")
print(leakage_summary_csv)

# ============================================================
# DECISION SUPPORT JSON
# ============================================================

def scenario_summary(data, scenario_name):
    class_counts = (
        data.groupby(["class_name", "split_final"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=classes, columns=splits, fill_value=0)
    )

    split_totals = data["split_final"].value_counts().reindex(splits, fill_value=0)
    split_n_classes = data.groupby("split_final")["class_name"].nunique().reindex(splits, fill_value=0)

    return {
        "scenario": scenario_name,
        "n_total_crops": int(len(data)),
        "split_totals": {s: int(split_totals[s]) for s in splits},
        "split_n_classes": {s: int(split_n_classes[s]) for s in splits},
        "min_crop_per_class": {s: int(class_counts[s].min()) for s in splits},
        "max_crop_per_class": {s: int(class_counts[s].max()) for s in splits}
    }

decision_support = {
    "stage": "V5_final_split_manifest_creation",
    "status": "completed",
    "important_note": (
        "Tidak ada training model, tidak membuat model, tidak memakai augmented_train, "
        "tidak menghapus near-duplicate, dan tidak mengubah data asli."
    ),
    "label_map": label_map,
    "scenarios": {
        "A_official": {
            "description": "Official dataset split untuk literature-comparable evaluation.",
            "group_policy": "Tidak group-aware; memakai folder train/validation/test resmi.",
            "manifest_csv": str(manifest_A_csv)
        },
        "B_G09_Date_Magnification": {
            "description": "Leakage-aware split utama berbasis Date + Magnification.",
            "group_policy": "Group G09 tidak boleh muncul lintas train/validation/test.",
            "manifest_csv": str(manifest_B_csv)
        },
        "B_conservative_G14_RAWSource": {
            "description": "Split konservatif berbasis RAWSource.",
            "group_policy": "Crop dari RAW image yang sama tidak boleh muncul lintas split.",
            "manifest_csv": str(manifest_B_cons_csv)
        }
    },
    "scenario_summaries": [
        scenario_summary(A, "A_official"),
        scenario_summary(B, "B_G09_Date_Magnification"),
        scenario_summary(B_cons, "B_conservative_G14_RAWSource")
    ],
    "leakage_summary": leakage_df.to_dict(orient="records"),
    "output_files": {
        "v5_split_manifest_A_official": str(manifest_A_csv),
        "v5_split_manifest_B_G09_Date_Magnification": str(manifest_B_csv),
        "v5_split_manifest_B_conservative_G14_RAWSource": str(manifest_B_cons_csv),
        "v5_split_class_distribution_summary": str(class_dist_summary_csv),
        "v5_split_leakage_check_summary": str(leakage_summary_csv),
        "v5_split_final_decision_support": str(decision_support_json)
    }
}

decision_support_json.write_text(
    json.dumps(decision_support, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== Decision Support JSON Disimpan ===")
print(decision_support_json)

# ============================================================
# PREVIEW OUTPUT
# ============================================================

print("\n=== Ringkasan Scenario ===")
scenario_summary_df = pd.DataFrame([
    scenario_summary(A, "A_official"),
    scenario_summary(B, "B_G09_Date_Magnification"),
    scenario_summary(B_cons, "B_conservative_G14_RAWSource")
])
display(scenario_summary_df)

print("\n=== Leakage Check Summary ===")
display(leakage_df)

print("\n=== Class Distribution Preview ===")
display(class_dist_df.head(60))

print("\n=== Manifest A Preview ===")
display(manifest_A.head())

print("\n=== Manifest B G09 Preview ===")
display(manifest_B.head())

print("\n=== Manifest B Conservative G14 Preview ===")
display(manifest_B_cons.head())

print("\n=== V5 SELESAI ===")
print("1.", manifest_A_csv)
print("2.", manifest_B_csv)
print("3.", manifest_B_cons_csv)
print("4.", class_dist_summary_csv)
print("5.", leakage_summary_csv)
print("6.", decision_support_json)

In [ ]:
# Sel 23 — V6 Input Check: Near-Duplicate Strong vs Metadata Crop/G14

from pathlib import Path
import pandas as pd
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

# Input utama V6
near_dup_csv = outputs_dir / "near_duplicate_strong_main_candidates.csv"
crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"

# Manifest V5, hanya untuk referensi/sanity check
manifest_A_csv = outputs_dir / "v5_split_manifest_A_official.csv"
manifest_B_g09_csv = outputs_dir / "v5_split_manifest_B_G09_Date_Magnification.csv"
manifest_B_g14_csv = outputs_dir / "v5_split_manifest_B_conservative_G14_RAWSource.csv"

print("=== V6 Input File Check ===")
input_files = {
    "near_duplicate_strong_main_candidates": near_dup_csv,
    "v3_crop_level_metadata_with_group_candidates": crop_meta_csv,
    "v5_split_manifest_A_official": manifest_A_csv,
    "v5_split_manifest_B_G09_Date_Magnification": manifest_B_g09_csv,
    "v5_split_manifest_B_conservative_G14_RAWSource": manifest_B_g14_csv
}

for name, path in input_files.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists())

# Stop kalau input wajib tidak ada
if not near_dup_csv.exists():
    raise FileNotFoundError(f"File near-duplicate kuat tidak ditemukan: {near_dup_csv}")

if not crop_meta_csv.exists():
    raise FileNotFoundError(f"File crop metadata V3 tidak ditemukan: {crop_meta_csv}")

# ============================================================
# LOAD DATA
# ============================================================

near_df = pd.read_csv(near_dup_csv)
crop_meta = pd.read_csv(crop_meta_csv, keep_default_na=False)

print("\n=== Loaded Data ===")
print("Near-duplicate shape:", near_df.shape)
print("Crop metadata shape:", crop_meta.shape)

print("\nKolom near-duplicate:")
print(list(near_df.columns))

print("\nKolom crop_meta penting yang tersedia:")
important_cols = [
    "split",
    "class_name",
    "crop_file",
    "crop_filename",
    "raw_file_mapped",
    "G09_Date_Magnification",
    "G14_RAWSource"
]
for c in important_cols:
    print(f"{c}: {c in crop_meta.columns}")

missing_important = [c for c in important_cols if c not in crop_meta.columns]
if missing_important:
    raise ValueError(f"Kolom penting tidak ditemukan di crop_meta: {missing_important}")

# ============================================================
# FILTER NEAR-DUPLICATE KUAT HASH DISTANCE <= 2
# ============================================================

if "hamming_distance" in near_df.columns:
    near_strong = near_df[near_df["hamming_distance"] <= 2].copy()
else:
    print("\nPERINGATAN: Kolom hamming_distance tidak ditemukan. Semua pasangan dipakai.")
    near_strong = near_df.copy()

print("\n=== Near-Duplicate Strong Filter ===")
print("Total pasangan near-duplicate kandidat awal:", len(near_df))
print("Total pasangan hamming_distance <= 2:", len(near_strong))

if "hamming_distance" in near_strong.columns:
    print("\nDistribusi hamming_distance:")
    display(
        near_strong["hamming_distance"]
        .value_counts()
        .sort_index()
        .reset_index()
        .rename(columns={"index": "hamming_distance", "hamming_distance": "count"})
    )

# ============================================================
# CEK KOLOM PASANGAN FILE
# ============================================================

required_pair_cols = ["file_a", "file_b"]

missing_pair_cols = [c for c in required_pair_cols if c not in near_strong.columns]
if missing_pair_cols:
    raise ValueError(f"Kolom pasangan file tidak ditemukan di near-duplicate CSV: {missing_pair_cols}")

# Pastikan format file_a/file_b sama dengan crop_meta["crop_file"]
near_strong["file_a"] = near_strong["file_a"].astype(str)
near_strong["file_b"] = near_strong["file_b"].astype(str)
crop_meta["crop_file"] = crop_meta["crop_file"].astype(str)

all_crop_files = set(crop_meta["crop_file"])

near_strong["file_a_found_in_crop_meta"] = near_strong["file_a"].isin(all_crop_files)
near_strong["file_b_found_in_crop_meta"] = near_strong["file_b"].isin(all_crop_files)

missing_a = near_strong[~near_strong["file_a_found_in_crop_meta"]].copy()
missing_b = near_strong[~near_strong["file_b_found_in_crop_meta"]].copy()

print("\n=== Pair File Coverage in Crop Metadata ===")
print("file_a found:", int(near_strong["file_a_found_in_crop_meta"].sum()), "/", len(near_strong))
print("file_b found:", int(near_strong["file_b_found_in_crop_meta"].sum()), "/", len(near_strong))
print("missing file_a:", len(missing_a))
print("missing file_b:", len(missing_b))

if len(missing_a) > 0:
    print("\nPreview missing file_a:")
    display(missing_a[["file_a", "file_b"]].head(20))

if len(missing_b) > 0:
    print("\nPreview missing file_b:")
    display(missing_b[["file_a", "file_b"]].head(20))

# ============================================================
# PREVIEW DATA
# ============================================================

print("\n=== Preview Near-Duplicate Strong ===")
preview_cols = [
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128"
]
preview_cols = [c for c in preview_cols if c in near_strong.columns]
display(near_strong[preview_cols].head(20))

print("\n=== Preview Crop Metadata Lookup ===")
display(crop_meta[important_cols].head(20))

print("\n=== Sel 23 selesai ===")
print("Jika file_a dan file_b found semuanya 100%, lanjut Sel 24 untuk merge pasangan dengan G14/G09.")

In [ ]:
# Sel 24 — V6 Cross-check Near-Duplicate vs G14 RAWSource

from pathlib import Path
import pandas as pd
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

near_dup_csv = outputs_dir / "near_duplicate_strong_main_candidates.csv"
crop_meta_csv = outputs_dir / "v3_crop_level_metadata_with_group_candidates.csv"

# Output V6
v6_detail_csv = outputs_dir / "v6_near_duplicate_vs_G14_check.csv"
v6_summary_json = outputs_dir / "v6_near_duplicate_vs_G14_summary.json"
v6_class_summary_csv = outputs_dir / "v6_near_duplicate_vs_G14_class_summary.csv"

# ============================================================
# LOAD DATA
# ============================================================

near_df = pd.read_csv(near_dup_csv)
crop_meta = pd.read_csv(crop_meta_csv, keep_default_na=False)

# Filter kandidat kuat hash distance <= 2
if "hamming_distance" in near_df.columns:
    near_strong = near_df[near_df["hamming_distance"] <= 2].copy()
else:
    near_strong = near_df.copy()

near_strong["file_a"] = near_strong["file_a"].astype(str)
near_strong["file_b"] = near_strong["file_b"].astype(str)
crop_meta["crop_file"] = crop_meta["crop_file"].astype(str)

print("=== Input V6 Sel 24 ===")
print("Near-duplicate strong pairs:", len(near_strong))
print("Crop metadata rows:", len(crop_meta))

# ============================================================
# PREPARE LOOKUP METADATA
# ============================================================

lookup_cols = [
    "crop_file",
    "split",
    "class_name",
    "raw_file_mapped",
    "G14_RAWSource",
    "G09_Date_Magnification"
]

missing_cols = [c for c in lookup_cols if c not in crop_meta.columns]
if missing_cols:
    raise ValueError(f"Kolom lookup tidak ditemukan di crop_meta: {missing_cols}")

lookup = crop_meta[lookup_cols].drop_duplicates(subset=["crop_file"]).copy()

# Side A lookup
lookup_a = lookup.rename(columns={
    "crop_file": "file_a",
    "split": "split_a_meta",
    "class_name": "class_name_a_meta",
    "raw_file_mapped": "raw_source_id_a",
    "G14_RAWSource": "G14_RAWSource_a",
    "G09_Date_Magnification": "G09_Date_Magnification_a"
})

# Side B lookup
lookup_b = lookup.rename(columns={
    "crop_file": "file_b",
    "split": "split_b_meta",
    "class_name": "class_name_b_meta",
    "raw_file_mapped": "raw_source_id_b",
    "G14_RAWSource": "G14_RAWSource_b",
    "G09_Date_Magnification": "G09_Date_Magnification_b"
})

# ============================================================
# MERGE PAIRS
# ============================================================

merged = near_strong.merge(lookup_a, on="file_a", how="left")
merged = merged.merge(lookup_b, on="file_b", how="left")

# Check merge completeness
missing_a = merged["G14_RAWSource_a"].isna().sum()
missing_b = merged["G14_RAWSource_b"].isna().sum()

print("\n=== Merge Check ===")
print("Missing metadata side A:", missing_a)
print("Missing metadata side B:", missing_b)

if missing_a > 0 or missing_b > 0:
    print("\nPreview missing metadata:")
    display(merged[
        merged["G14_RAWSource_a"].isna() | merged["G14_RAWSource_b"].isna()
    ].head(20))
    raise ValueError("Ada pasangan yang gagal merge metadata.")

# ============================================================
# COMPUTE SAME/DIFFERENT GROUP
# ============================================================

merged["same_G14_RAWSource"] = merged["G14_RAWSource_a"] == merged["G14_RAWSource_b"]
merged["same_raw_source_id"] = merged["raw_source_id_a"] == merged["raw_source_id_b"]
merged["same_G09_Date_Magnification"] = (
    merged["G09_Date_Magnification_a"] == merged["G09_Date_Magnification_b"]
)

merged["same_class_meta"] = merged["class_name_a_meta"] == merged["class_name_b_meta"]
merged["same_official_split_meta"] = merged["split_a_meta"] == merged["split_b_meta"]
merged["cross_official_split_meta"] = merged["split_a_meta"] != merged["split_b_meta"]

# Tambahan: raw filename tanpa prefix split untuk inspeksi tambahan
# Ini bukan pengganti G14, hanya bantuan diagnostik.
def remove_split_prefix(x):
    s = str(x)
    parts = s.split("/", 1)
    if len(parts) == 2 and parts[0] in ["train", "validation", "test"]:
        return parts[1]
    return s

merged["raw_source_no_split_a"] = merged["raw_source_id_a"].apply(remove_split_prefix)
merged["raw_source_no_split_b"] = merged["raw_source_id_b"].apply(remove_split_prefix)
merged["same_raw_source_no_split"] = (
    merged["raw_source_no_split_a"] == merged["raw_source_no_split_b"]
)

# ============================================================
# OVERALL SUMMARY
# ============================================================

total_pairs = len(merged)

n_same_g14 = int(merged["same_G14_RAWSource"].sum())
n_diff_g14 = int((~merged["same_G14_RAWSource"]).sum())

n_same_g09 = int(merged["same_G09_Date_Magnification"].sum())
n_diff_g09 = int((~merged["same_G09_Date_Magnification"]).sum())

n_same_raw_id = int(merged["same_raw_source_id"].sum())
n_same_raw_no_split = int(merged["same_raw_source_no_split"].sum())

g14_coverage_rate = n_same_g14 / total_pairs if total_pairs > 0 else 0
g09_same_rate = n_same_g09 / total_pairs if total_pairs > 0 else 0
raw_no_split_same_rate = n_same_raw_no_split / total_pairs if total_pairs > 0 else 0

# Hamming-level summary
hamming_summary = []
if "hamming_distance" in merged.columns:
    for hd, sub in merged.groupby("hamming_distance"):
        hamming_summary.append({
            "hamming_distance": int(hd),
            "n_pairs": int(len(sub)),
            "same_G14_RAWSource": int(sub["same_G14_RAWSource"].sum()),
            "different_G14_RAWSource": int((~sub["same_G14_RAWSource"]).sum()),
            "same_G14_rate": float(sub["same_G14_RAWSource"].mean()),
            "same_G09_Date_Magnification": int(sub["same_G09_Date_Magnification"].sum()),
            "same_G09_rate": float(sub["same_G09_Date_Magnification"].mean())
        })

# ============================================================
# CLASS SUMMARY
# ============================================================

class_col = "class_name"
if class_col not in merged.columns:
    # fallback kalau kolom asal tidak ada
    merged["class_name"] = merged["class_name_a_meta"]

class_summary = (
    merged
    .groupby("class_name")
    .agg(
        n_pairs=("file_a", "count"),
        same_G14_RAWSource=("same_G14_RAWSource", "sum"),
        same_G09_Date_Magnification=("same_G09_Date_Magnification", "sum"),
        same_raw_source_no_split=("same_raw_source_no_split", "sum"),
        mean_hamming_distance=("hamming_distance", "mean") if "hamming_distance" in merged.columns else ("file_a", "count")
    )
    .reset_index()
)

class_summary["different_G14_RAWSource"] = class_summary["n_pairs"] - class_summary["same_G14_RAWSource"]
class_summary["same_G14_rate"] = class_summary["same_G14_RAWSource"] / class_summary["n_pairs"]
class_summary["same_G09_rate"] = class_summary["same_G09_Date_Magnification"] / class_summary["n_pairs"]
class_summary["same_raw_no_split_rate"] = class_summary["same_raw_source_no_split"] / class_summary["n_pairs"]

class_summary = class_summary.sort_values(
    ["n_pairs", "same_G14_rate"],
    ascending=[False, False]
).reset_index(drop=True)

# ============================================================
# SAVE DETAIL OUTPUT
# ============================================================

detail_cols = [
    "class_name",
    "split_a",
    "file_a",
    "split_a_meta",
    "class_name_a_meta",
    "raw_source_id_a",
    "G14_RAWSource_a",
    "G09_Date_Magnification_a",
    "split_b",
    "file_b",
    "split_b_meta",
    "class_name_b_meta",
    "raw_source_id_b",
    "G14_RAWSource_b",
    "G09_Date_Magnification_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "same_G14_RAWSource",
    "same_raw_source_id",
    "same_G09_Date_Magnification",
    "same_raw_source_no_split",
    "same_class_meta",
    "cross_official_split_meta"
]

detail_cols = [c for c in detail_cols if c in merged.columns]
merged[detail_cols].to_csv(v6_detail_csv, index=False)

class_summary.to_csv(v6_class_summary_csv, index=False)

# Interpretasi awal otomatis
if g14_coverage_rate >= 0.80:
    g14_interpretation = (
        "Mayoritas besar near-duplicate tercakup oleh G14_RAWSource. "
        "B-G14 kuat sebagai leakage-aware split utama."
    )
elif g14_coverage_rate >= 0.50:
    g14_interpretation = (
        "Sebagian besar near-duplicate tercakup oleh G14_RAWSource, "
        "tetapi masih ada porsi berbeda G14 yang perlu diperhatikan."
    )
else:
    g14_interpretation = (
        "Banyak near-duplicate berada pada G14_RAWSource berbeda. "
        "A-clean mungkin perlu dipertimbangkan setelah MASTER CHAT meninjau hasil."
    )

if n_diff_g14 > 0:
    aclean_note = (
        "A-clean belum langsung dibuat, tetapi perlu dipertimbangkan/didiskusikan "
        "karena ada pasangan near-duplicate dengan G14 berbeda."
    )
else:
    aclean_note = (
        "A-clean belum mendesak dari sisi G14 karena semua near-duplicate kuat berada dalam G14 yang sama."
    )

summary = {
    "stage": "V6_near_duplicate_vs_G14_RAWSource_check",
    "status": "completed",
    "input_files": {
        "near_duplicate_strong_main_candidates": str(near_dup_csv),
        "crop_level_metadata": str(crop_meta_csv)
    },
    "total_near_duplicate_pairs": int(total_pairs),
    "hamming_distance_filter": "<= 2",
    "overall_G14": {
        "same_G14_RAWSource": n_same_g14,
        "different_G14_RAWSource": n_diff_g14,
        "same_G14_rate": g14_coverage_rate,
        "different_G14_rate": n_diff_g14 / total_pairs if total_pairs > 0 else 0
    },
    "overall_G09": {
        "same_G09_Date_Magnification": n_same_g09,
        "different_G09_Date_Magnification": n_diff_g09,
        "same_G09_rate": g09_same_rate,
        "different_G09_rate": n_diff_g09 / total_pairs if total_pairs > 0 else 0
    },
    "additional_raw_no_split_diagnostic": {
        "same_raw_source_no_split": n_same_raw_no_split,
        "same_raw_source_no_split_rate": raw_no_split_same_rate
    },
    "hamming_summary": hamming_summary,
    "interpretation": {
        "g14_interpretation": g14_interpretation,
        "a_clean_note": aclean_note,
        "b_g14_main_split_note": (
            "B-G14 layak sebagai RAW source-aware leakage-aware split utama jika MASTER CHAT menerima "
            "bahwa mitigasi utama harus berbasis RAW source."
        ),
        "g09_sensitivity_note": (
            "G09 tetap berguna sebagai acquisition-aware sensitivity analysis karena merepresentasikan Date + Magnification."
        )
    },
    "outputs": {
        "detail_csv": str(v6_detail_csv),
        "class_summary_csv": str(v6_class_summary_csv),
        "summary_json": str(v6_summary_json)
    }
}

v6_summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n=== V6 Overall Summary ===")
print("Total near-duplicate strong pairs:", total_pairs)
print("Same G14_RAWSource:", n_same_g14)
print("Different G14_RAWSource:", n_diff_g14)
print("Same G14 rate:", round(g14_coverage_rate * 100, 2), "%")
print("Same G09_Date_Magnification:", n_same_g09)
print("Different G09_Date_Magnification:", n_diff_g09)
print("Same G09 rate:", round(g09_same_rate * 100, 2), "%")
print("Same raw_source_no_split:", n_same_raw_no_split)
print("Same raw_source_no_split rate:", round(raw_no_split_same_rate * 100, 2), "%")

print("\n=== Hamming Distance Summary ===")
if hamming_summary:
    display(pd.DataFrame(hamming_summary))
else:
    print("Tidak ada hamming_distance summary.")

print("\n=== Class Summary ===")
display(class_summary)

print("\n=== Detail Preview ===")
display(merged[detail_cols].head(30))

print("\n=== Output V6 Disimpan ===")
print("1.", v6_detail_csv)
print("2.", v6_summary_json)
print("3.", v6_class_summary_csv)

# Bagian V7 — Validasi Visual Hamming Distance 0


In [ ]:
# Sel V7-1 — Load V6 Result dan Filter Pasangan Hamming Distance 0

from pathlib import Path
import pandas as pd
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"

v6_detail_csv = outputs_dir / "v6_near_duplicate_vs_G14_check.csv"

# Output sementara V7 untuk memudahkan Sel V7-2
v7_hamming0_pairs_csv = outputs_dir / "v7_hamming0_pairs_for_visual_check.csv"

print("=== Sel V7-1: Input Check ===")
print("V6 detail CSV:", v6_detail_csv)
print("V6 detail exists:", v6_detail_csv.exists())
print("Crop root:", crop_root)
print("Crop root exists:", crop_root.exists())

if not v6_detail_csv.exists():
    raise FileNotFoundError(f"File V6 tidak ditemukan: {v6_detail_csv}")

if not crop_root.exists():
    raise FileNotFoundError(f"Folder crop tidak ditemukan: {crop_root}")

# ============================================================
# LOAD V6 DETAIL
# ============================================================

df = pd.read_csv(v6_detail_csv, keep_default_na=False)

print("\n=== Loaded V6 Detail ===")
print("Shape:", df.shape)
print("Kolom:")
print(list(df.columns))

required_cols = [
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "hamming_distance"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

# ============================================================
# FILTER HAMMING DISTANCE 0
# ============================================================

df["hamming_distance"] = pd.to_numeric(df["hamming_distance"], errors="coerce")

h0 = df[df["hamming_distance"] == 0].copy()
h0 = h0.reset_index(drop=True)

h0["pair_id"] = [f"H0_PAIR_{i+1:03d}" for i in range(len(h0))]

print("\n=== Filter Hamming Distance 0 ===")
print("Total pasangan V6:", len(df))
print("Total pasangan hamming_distance == 0:", len(h0))

if len(h0) == 0:
    raise ValueError("Tidak ada pasangan hamming_distance == 0. Stop dulu.")

# ============================================================
# CEK PATH GAMBAR
# ============================================================

h0["left_image"] = h0["file_a"].apply(lambda x: str(crop_root / x))
h0["right_image"] = h0["file_b"].apply(lambda x: str(crop_root / x))

h0["left_exists"] = h0["left_image"].apply(lambda x: Path(x).exists())
h0["right_exists"] = h0["right_image"].apply(lambda x: Path(x).exists())

print("\n=== Cek File Gambar ===")
print("Left image exists:", int(h0["left_exists"].sum()), "/", len(h0))
print("Right image exists:", int(h0["right_exists"].sum()), "/", len(h0))
print("Missing left:", int((~h0["left_exists"]).sum()))
print("Missing right:", int((~h0["right_exists"]).sum()))

if (~h0["left_exists"]).any() or (~h0["right_exists"]).any():
    print("\nPreview missing image:")
    display(h0.loc[
        (~h0["left_exists"]) | (~h0["right_exists"]),
        ["pair_id", "class_name", "file_a", "left_image", "left_exists", "file_b", "right_image", "right_exists"]
    ])
    raise ValueError("Ada image yang tidak ditemukan. Stop dulu.")

# ============================================================
# SIMPAN FILTERED PAIRS SEMENTARA
# ============================================================

cols_to_save = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "same_G14_RAWSource",
    "same_G09_Date_Magnification",
    "left_image",
    "right_image"
]

cols_to_save = [c for c in cols_to_save if c in h0.columns]

h0[cols_to_save].to_csv(v7_hamming0_pairs_csv, index=False)

print("\n=== Output Sementara V7 Disimpan ===")
print(v7_hamming0_pairs_csv)

print("\n=== Preview Pasangan Hamming Distance 0 ===")
display(h0[cols_to_save].head(32))

print("\n=== Sel V7-1 selesai ===")
print("Jika jumlah pasangan sekitar 32 dan semua image exists, lanjut ke Sel V7-2 untuk membuat contact sheet.")

In [ ]:
# Sel V7-2 — Buat Contact Sheet dan File Anotasi Manual Hamming Distance 0

from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont
import textwrap
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

v7_pairs_csv = outputs_dir / "v7_hamming0_pairs_for_visual_check.csv"

contact_sheet_path = outputs_dir / "v7_hamming0_visual_contact_sheet.png"
annotation_csv = outputs_dir / "v7_hamming0_manual_annotation.csv"

print("=== Sel V7-2: Load Hamming 0 Pairs ===")
print("Input pairs CSV:", v7_pairs_csv)
print("Exists:", v7_pairs_csv.exists())

if not v7_pairs_csv.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {v7_pairs_csv}")

df = pd.read_csv(v7_pairs_csv, keep_default_na=False)

print("Jumlah pasangan:", len(df))
print("Kolom:", list(df.columns))

# ============================================================
# CEK KOLOM WAJIB
# ============================================================

required_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "left_image",
    "right_image"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

df["left_exists"] = df["left_image"].apply(lambda x: Path(x).exists())
df["right_exists"] = df["right_image"].apply(lambda x: Path(x).exists())

print("\n=== Cek Image Exists ===")
print("Left exists:", int(df["left_exists"].sum()), "/", len(df))
print("Right exists:", int(df["right_exists"].sum()), "/", len(df))

if (~df["left_exists"]).any() or (~df["right_exists"]).any():
    display(df.loc[
        (~df["left_exists"]) | (~df["right_exists"]),
        ["pair_id", "left_image", "left_exists", "right_image", "right_exists"]
    ])
    raise ValueError("Ada image yang tidak ditemukan. Stop dulu.")

# ============================================================
# BUAT FILE ANOTASI MANUAL
# ============================================================

annotation_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b"
]

annot = df[annotation_cols].copy()

annot = annot.rename(columns={
    "G14_RAWSource_a": "G14_a",
    "G14_RAWSource_b": "G14_b",
    "G09_Date_Magnification_a": "G09_a",
    "G09_Date_Magnification_b": "G09_b"
})

annot["human_label"] = ""
annot["note"] = ""

annot.to_csv(annotation_csv, index=False)

# ============================================================
# BUAT CONTACT SHEET
# ============================================================

# Ukuran visual
thumb_size = 180
gap = 18
text_height = 105
row_height = thumb_size + text_height + 18
sheet_width = 2 * thumb_size + 3 * gap
sheet_height = len(df) * row_height + gap

sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
draw = ImageDraw.Draw(sheet)

try:
    font_title = ImageFont.truetype("DejaVuSans-Bold.ttf", 13)
    font_small = ImageFont.truetype("DejaVuSans.ttf", 10)
    font_label = ImageFont.truetype("DejaVuSans-Bold.ttf", 11)
except:
    font_title = ImageFont.load_default()
    font_small = ImageFont.load_default()
    font_label = ImageFont.load_default()

def load_thumb(path, size=180):
    img = Image.open(path).convert("RGB")
    img = ImageOps.contain(img, (size, size))
    canvas = Image.new("RGB", (size, size), "white")
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

def draw_wrapped_text(draw, text, xy, font, max_width_chars=48, line_spacing=12, fill="black"):
    x, y = xy
    lines = []
    for part in str(text).split("\n"):
        lines.extend(textwrap.wrap(part, width=max_width_chars) if part else [""])
    for line in lines:
        draw.text((x, y), line, font=font, fill=fill)
        y += line_spacing
    return y

for idx, row in df.iterrows():
    y0 = gap + idx * row_height
    x_left = gap
    x_right = gap * 2 + thumb_size

    # Header info
    title = f"{row['pair_id']} | {row['class_name']} | {row['split_a']} vs {row['split_b']}"
    draw.text((gap, y0), title, font=font_title, fill="black")

    # L/R labels
    draw.text((x_left, y0 + 22), "LEFT / A", font=font_label, fill="black")
    draw.text((x_right, y0 + 22), "RIGHT / B", font=font_label, fill="black")

    # Images
    img_left = load_thumb(row["left_image"], thumb_size)
    img_right = load_thumb(row["right_image"], thumb_size)

    img_y = y0 + 40
    sheet.paste(img_left, (x_left, img_y))
    sheet.paste(img_right, (x_right, img_y))

    # Border
    draw.rectangle([x_left, img_y, x_left + thumb_size, img_y + thumb_size], outline="black", width=1)
    draw.rectangle([x_right, img_y, x_right + thumb_size, img_y + thumb_size], outline="black", width=1)

    # Text below
    text_y = img_y + thumb_size + 6

    left_short = str(row["file_a"])
    right_short = str(row["file_b"])

    draw_wrapped_text(
        draw,
        f"A: {left_short}",
        (x_left, text_y),
        font_small,
        max_width_chars=30,
        line_spacing=11
    )

    draw_wrapped_text(
        draw,
        f"B: {right_short}",
        (x_right, text_y),
        font_small,
        max_width_chars=30,
        line_spacing=11
    )

# Simpan contact sheet
sheet.save(contact_sheet_path)

# Summary
summary = {
    "stage": "V7_hamming0_contact_sheet_and_annotation_creation",
    "n_pairs": int(len(df)),
    "contact_sheet_path": str(contact_sheet_path),
    "annotation_csv": str(annotation_csv),
    "allowed_labels": ["duplicate", "similar", "not duplicate", "uncertain"],
    "note": "Isi kolom human_label secara manual. Jangan mengubah kolom metadata lain."
}

summary_path = outputs_dir / "v7_hamming0_contact_sheet_creation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n=== Output Sel V7-2 Disimpan ===")
print("Contact sheet:")
print(contact_sheet_path)
print("Annotation CSV:")
print(annotation_csv)
print("Creation summary:")
print(summary_path)

print("\nUkuran contact sheet:", sheet.size)
print("Jumlah pasangan:", len(df))

print("\n=== Preview Annotation CSV ===")
display(annot.head(32))

print("\n=== Sel V7-2 selesai ===")
print("Buka contact sheet, lalu isi human_label di CSV anotasi.")
print("Allowed labels: duplicate / similar / not duplicate / uncertain")

In [ ]:
# Sel V7-3 — Cek Anotasi Manual Hamming 0 dan Buat Summary V7

from pathlib import Path
import pandas as pd
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

filled_csv = outputs_dir / "v7_hamming0_manual_annotation_filled.csv"
pairs_csv = outputs_dir / "v7_hamming0_pairs_for_visual_check.csv"

# Output V7
summary_json = outputs_dir / "v7_hamming0_validation_summary.json"
class_summary_csv = outputs_dir / "v7_hamming0_validation_class_summary.csv"

print("=== Sel V7-3: Input Check ===")
print("Filled annotation CSV:", filled_csv)
print("Exists:", filled_csv.exists())
print("Pairs CSV:", pairs_csv)
print("Exists:", pairs_csv.exists())

if not filled_csv.exists():
    print("\nFile filled belum ditemukan dengan nama:")
    print(filled_csv)
    print("\nCek file CSV V7 yang ada di outputs:")
    for p in outputs_dir.glob("*v7*hamming0*annotation*.csv"):
        print("-", p)
    raise FileNotFoundError("Upload dulu file filled dengan nama v7_hamming0_manual_annotation_filled.csv")

if not pairs_csv.exists():
    raise FileNotFoundError(f"Pairs CSV tidak ditemukan: {pairs_csv}")

# ============================================================
# LOAD DATA
# ============================================================

annot = pd.read_csv(filled_csv, keep_default_na=False)
pairs = pd.read_csv(pairs_csv, keep_default_na=False)

print("\n=== Loaded Data ===")
print("Annotation shape:", annot.shape)
print("Pairs shape:", pairs.shape)

print("\nKolom annotation:")
print(list(annot.columns))

# ============================================================
# VALIDASI KOLOM DAN LABEL
# ============================================================

required_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_a",
    "G14_b",
    "G09_a",
    "G09_b",
    "human_label",
    "note"
]

missing_cols = [c for c in required_cols if c not in annot.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan di annotation filled: {missing_cols}")

allowed_labels = ["duplicate", "similar", "not duplicate", "uncertain"]

annot["human_label_raw"] = annot["human_label"]
annot["human_label"] = (
    annot["human_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

missing_label = annot[annot["human_label"] == ""].copy()
invalid_label = annot[
    (annot["human_label"] != "") &
    (~annot["human_label"].isin(allowed_labels))
].copy()

print("\n=== Cek Label Manual ===")
print("Total pasangan:", len(annot))
print("Label terisi:", int((annot["human_label"] != "").sum()))
print("Label kosong:", len(missing_label))
print("Label invalid:", len(invalid_label))

if len(missing_label) > 0:
    print("\nPair yang masih kosong:")
    display(missing_label[["pair_id", "class_name", "file_a", "file_b", "human_label_raw"]])

if len(invalid_label) > 0:
    print("\nPair dengan label invalid:")
    display(invalid_label[["pair_id", "class_name", "human_label_raw", "human_label"]])
    print("Label valid:", allowed_labels)

if len(missing_label) > 0 or len(invalid_label) > 0:
    raise ValueError("Masih ada label kosong atau invalid. Perbaiki CSV filled lalu jalankan ulang Sel V7-3.")

# ============================================================
# HITUNG SUMMARY
# ============================================================

total_pairs = len(annot)

label_counts = (
    annot["human_label"]
    .value_counts()
    .reindex(allowed_labels, fill_value=0)
    .reset_index()
)
label_counts.columns = ["human_label", "count"]

n_duplicate = int((annot["human_label"] == "duplicate").sum())
n_similar = int((annot["human_label"] == "similar").sum())
n_not_duplicate = int((annot["human_label"] == "not duplicate").sum())
n_uncertain = int((annot["human_label"] == "uncertain").sum())

duplicate_rate = n_duplicate / total_pairs if total_pairs > 0 else 0
duplicate_or_similar = int(annot["human_label"].isin(["duplicate", "similar"]).sum())
duplicate_or_similar_rate = duplicate_or_similar / total_pairs if total_pairs > 0 else 0

# Distribusi per kelas
class_summary = (
    annot
    .groupby("class_name")
    .agg(
        n_pairs=("pair_id", "count"),
        duplicate=("human_label", lambda s: int((s == "duplicate").sum())),
        similar=("human_label", lambda s: int((s == "similar").sum())),
        not_duplicate=("human_label", lambda s: int((s == "not duplicate").sum())),
        uncertain=("human_label", lambda s: int((s == "uncertain").sum()))
    )
    .reset_index()
)

class_summary["duplicate_or_similar"] = class_summary["duplicate"] + class_summary["similar"]
class_summary["duplicate_rate"] = class_summary["duplicate"] / class_summary["n_pairs"]
class_summary["duplicate_or_similar_rate"] = class_summary["duplicate_or_similar"] / class_summary["n_pairs"]

class_summary = class_summary.sort_values(
    ["n_pairs", "duplicate_or_similar_rate"],
    ascending=[False, False]
).reset_index(drop=True)

class_summary.to_csv(class_summary_csv, index=False)

# Distribusi pasangan split
annot["split_pair"] = annot["split_a"].astype(str) + " -> " + annot["split_b"].astype(str)

split_pair_summary = (
    annot
    .groupby("split_pair")
    .agg(
        n_pairs=("pair_id", "count"),
        duplicate=("human_label", lambda s: int((s == "duplicate").sum())),
        similar=("human_label", lambda s: int((s == "similar").sum())),
        not_duplicate=("human_label", lambda s: int((s == "not duplicate").sum())),
        uncertain=("human_label", lambda s: int((s == "uncertain").sum()))
    )
    .reset_index()
)

split_pair_summary["duplicate_or_similar"] = split_pair_summary["duplicate"] + split_pair_summary["similar"]
split_pair_summary["duplicate_rate"] = split_pair_summary["duplicate"] / split_pair_summary["n_pairs"]
split_pair_summary["duplicate_or_similar_rate"] = split_pair_summary["duplicate_or_similar"] / split_pair_summary["n_pairs"]

# ============================================================
# INTERPRETASI AWAL OTOMATIS
# ============================================================

if duplicate_rate >= 0.50:
    duplicate_interpretation = "Mayoritas pasangan hamming 0 adalah duplicate ketat."
elif duplicate_or_similar_rate >= 0.50:
    duplicate_interpretation = "Mayoritas pasangan hamming 0 minimal similar, tetapi tidak semuanya duplicate ketat."
else:
    duplicate_interpretation = "Mayoritas pasangan hamming 0 bukan duplicate/similar kuat secara visual."

if duplicate_rate >= 0.30:
    aclean_note = "A-clean perlu dipertimbangkan kuat sebelum baseline official."
elif duplicate_or_similar_rate >= 0.50:
    aclean_note = "A-clean perlu didiskusikan, tetapi keputusan tergantung apakah similar dianggap cukup berisiko."
else:
    aclean_note = "A-clean belum mendesak; near-duplicate dapat dilaporkan sebagai risiko metodologis."

summary = {
    "stage": "V7_hamming0_visual_validation",
    "status": "completed",
    "input_annotation_filled": str(filled_csv),
    "total_pairs": int(total_pairs),
    "label_counts": label_counts.to_dict(orient="records"),
    "duplicate": n_duplicate,
    "similar": n_similar,
    "not_duplicate": n_not_duplicate,
    "uncertain": n_uncertain,
    "duplicate_rate": duplicate_rate,
    "duplicate_or_similar": duplicate_or_similar,
    "duplicate_or_similar_rate": duplicate_or_similar_rate,
    "class_summary_csv": str(class_summary_csv),
    "split_pair_summary": split_pair_summary.to_dict(orient="records"),
    "interpretation": {
        "duplicate_interpretation": duplicate_interpretation,
        "a_clean_note": aclean_note,
        "baseline_note": "Keputusan baseline DenseNet201 tetap menunggu MASTER CHAT setelah membaca V7.",
        "near_duplicate_reporting_note": "Near-duplicate hamming 0 dapat dilaporkan sebagai risiko metodologis, terutama jika duplicate/similar cukup tinggi."
    },
    "outputs": {
        "summary_json": str(summary_json),
        "class_summary_csv": str(class_summary_csv)
    }
}

summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ============================================================
# DISPLAY
# ============================================================

print("\n=== Label Counts ===")
display(label_counts)

print("\n=== Overall Summary ===")
print("Total pairs:", total_pairs)
print("Duplicate:", n_duplicate)
print("Similar:", n_similar)
print("Not duplicate:", n_not_duplicate)
print("Uncertain:", n_uncertain)
print("Duplicate rate:", round(duplicate_rate * 100, 2), "%")
print("Duplicate or similar:", duplicate_or_similar)
print("Duplicate or similar rate:", round(duplicate_or_similar_rate * 100, 2), "%")

print("\n=== Class Summary ===")
display(class_summary)

print("\n=== Split Pair Summary ===")
display(split_pair_summary)

print("\n=== Interpretasi Awal ===")
print(duplicate_interpretation)
print(aclean_note)

print("\n=== Output V7 Disimpan ===")
print("Summary JSON:")
print(summary_json)
print("Class summary CSV:")
print(class_summary_csv)

print("\n=== Sel V7-3 selesai ===")

In [ ]:
# Sel V8-1 — Load V6, Filter G14 Berbeda, dan Pilih Sampel V8

from pathlib import Path
import pandas as pd
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

dataset_root = Path("/content/tem_virus_dataset_extracted/TEM virus dataset")
crop_root = dataset_root / "context_virus_1nm_256x256"

v6_detail_csv = outputs_dir / "v6_near_duplicate_vs_G14_check.csv"

# Output sementara untuk Sel V8-2
v8_selected_pairs_csv = outputs_dir / "v8_g14_different_nearduplicate_selected_pairs.csv"

print("=== Sel V8-1: Input Check ===")
print("V6 detail CSV:", v6_detail_csv)
print("V6 detail exists:", v6_detail_csv.exists())
print("Crop root:", crop_root)
print("Crop root exists:", crop_root.exists())

if not v6_detail_csv.exists():
    raise FileNotFoundError(f"File V6 tidak ditemukan: {v6_detail_csv}")

if not crop_root.exists():
    raise FileNotFoundError(f"Folder crop tidak ditemukan: {crop_root}")

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(v6_detail_csv, keep_default_na=False)

print("\n=== Loaded V6 Detail ===")
print("Shape:", df.shape)
print("Kolom:")
print(list(df.columns))

required_cols = [
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "same_G14_RAWSource"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

# ============================================================
# FILTER: hamming_distance <= 2 dan G14 berbeda
# ============================================================

df["hamming_distance"] = pd.to_numeric(df["hamming_distance"], errors="coerce")
df["mse_128"] = pd.to_numeric(df["mse_128"], errors="coerce")
df["mae_128"] = pd.to_numeric(df["mae_128"], errors="coerce")
df["corr_128"] = pd.to_numeric(df["corr_128"], errors="coerce")

# Normalisasi boolean, untuk jaga-jaga jika terbaca sebagai string
if df["same_G14_RAWSource"].dtype == object:
    df["same_G14_RAWSource_bool"] = (
        df["same_G14_RAWSource"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
    )
else:
    df["same_G14_RAWSource_bool"] = df["same_G14_RAWSource"].astype(bool)

filtered = df[
    (df["hamming_distance"] <= 2) &
    (df["same_G14_RAWSource_bool"] == False)
].copy()

filtered = filtered.reset_index(drop=True)

print("\n=== Filter V8 ===")
print("Total pasangan V6:", len(df))
print("Pasangan hamming_distance <= 2 dan G14 berbeda:", len(filtered))

print("\nDistribusi hamming_distance setelah filter:")
display(
    filtered["hamming_distance"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "hamming_distance", "hamming_distance": "count"})
)

print("\nDistribusi kelas setelah filter:")
display(
    filtered["class_name"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "class_name", "class_name": "count"})
)

# ============================================================
# PILIH 20 PASANGAN
# Prioritas:
# 1. hamming_distance paling kecil
# 2. corr_128 paling tinggi
# 3. mse_128 paling rendah
# ============================================================

filtered_sorted = filtered.sort_values(
    by=["hamming_distance", "corr_128", "mse_128"],
    ascending=[True, False, True]
).reset_index(drop=True)

selected = filtered_sorted.head(20).copy()
selected["pair_id"] = [f"V8_PAIR_{i+1:03d}" for i in range(len(selected))]

print("\n=== Selected Pairs for V8 ===")
print("Jumlah pasangan terpilih:", len(selected))

print("\nDistribusi hamming_distance selected:")
display(
    selected["hamming_distance"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "hamming_distance", "hamming_distance": "count"})
)

print("\nDistribusi kelas selected:")
display(
    selected["class_name"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "class_name", "class_name": "count"})
)

# ============================================================
# CEK FILE GAMBAR
# ============================================================

selected["left_image"] = selected["file_a"].apply(lambda x: str(crop_root / x))
selected["right_image"] = selected["file_b"].apply(lambda x: str(crop_root / x))

selected["left_exists"] = selected["left_image"].apply(lambda x: Path(x).exists())
selected["right_exists"] = selected["right_image"].apply(lambda x: Path(x).exists())

print("\n=== Cek File Gambar Selected ===")
print("Left image exists:", int(selected["left_exists"].sum()), "/", len(selected))
print("Right image exists:", int(selected["right_exists"].sum()), "/", len(selected))
print("Missing left:", int((~selected["left_exists"]).sum()))
print("Missing right:", int((~selected["right_exists"]).sum()))

if (~selected["left_exists"]).any() or (~selected["right_exists"]).any():
    print("\nPreview missing image:")
    display(selected.loc[
        (~selected["left_exists"]) | (~selected["right_exists"]),
        ["pair_id", "class_name", "file_a", "left_image", "left_exists", "file_b", "right_image", "right_exists"]
    ])
    raise ValueError("Ada image yang tidak ditemukan. Stop dulu.")

# ============================================================
# SIMPAN SELECTED PAIRS
# ============================================================

cols_to_save = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "same_G14_RAWSource",
    "same_G09_Date_Magnification",
    "left_image",
    "right_image"
]

cols_to_save = [c for c in cols_to_save if c in selected.columns]

selected[cols_to_save].to_csv(v8_selected_pairs_csv, index=False)

# Summary sementara
v8_selection_summary = {
    "stage": "V8_selection_g14_different_nearduplicate",
    "input_v6_detail_csv": str(v6_detail_csv),
    "total_v6_pairs": int(len(df)),
    "filtered_hamming_leq_2_g14_different": int(len(filtered)),
    "selected_pairs": int(len(selected)),
    "selection_rule": "sort by hamming_distance asc, corr_128 desc, mse_128 asc; take top 20",
    "selected_pairs_csv": str(v8_selected_pairs_csv),
    "selected_hamming_distribution": selected["hamming_distance"].value_counts().sort_index().to_dict(),
    "selected_class_distribution": selected["class_name"].value_counts().to_dict()
}

v8_selection_summary_json = outputs_dir / "v8_g14_different_nearduplicate_selection_summary.json"
v8_selection_summary_json.write_text(
    json.dumps(v8_selection_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== Output Sementara V8 Disimpan ===")
print("Selected pairs CSV:")
print(v8_selected_pairs_csv)
print("Selection summary JSON:")
print(v8_selection_summary_json)

print("\n=== Preview Selected Pairs ===")
display(selected[cols_to_save])

print("\n=== Sel V8-1 selesai ===")
print("Jika selected pairs = 10–20 dan semua image exists, lanjut Sel V8-2 untuk membuat contact sheet dan file anotasi manual.")

In [ ]:
# Sel V8-2 — Buat Contact Sheet dan File Anotasi Manual V8

from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont
import textwrap
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

v8_pairs_csv = outputs_dir / "v8_g14_different_nearduplicate_selected_pairs.csv"

contact_sheet_path = outputs_dir / "v8_g14_different_nearduplicate_contact_sheet.png"
annotation_csv = outputs_dir / "v8_g14_different_nearduplicate_manual_annotation.csv"
creation_summary_json = outputs_dir / "v8_g14_different_nearduplicate_contact_sheet_summary.json"

print("=== Sel V8-2: Load Selected Pairs ===")
print("Input selected pairs CSV:", v8_pairs_csv)
print("Exists:", v8_pairs_csv.exists())

if not v8_pairs_csv.exists():
    raise FileNotFoundError(f"File selected pairs tidak ditemukan: {v8_pairs_csv}")

df = pd.read_csv(v8_pairs_csv, keep_default_na=False)

print("Jumlah pasangan selected:", len(df))
print("Kolom:", list(df.columns))

# ============================================================
# CEK KOLOM WAJIB
# ============================================================

required_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "left_image",
    "right_image"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

# ============================================================
# CEK IMAGE
# ============================================================

df["left_exists"] = df["left_image"].apply(lambda x: Path(x).exists())
df["right_exists"] = df["right_image"].apply(lambda x: Path(x).exists())

print("\n=== Cek Image Exists ===")
print("Left exists:", int(df["left_exists"].sum()), "/", len(df))
print("Right exists:", int(df["right_exists"].sum()), "/", len(df))

if (~df["left_exists"]).any() or (~df["right_exists"]).any():
    display(df.loc[
        (~df["left_exists"]) | (~df["right_exists"]),
        ["pair_id", "left_image", "left_exists", "right_image", "right_exists"]
    ])
    raise ValueError("Ada image yang tidak ditemukan. Stop dulu.")

# ============================================================
# BUAT FILE ANOTASI MANUAL
# ============================================================

annotation_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_RAWSource_a",
    "G14_RAWSource_b",
    "G09_Date_Magnification_a",
    "G09_Date_Magnification_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128"
]

annot = df[annotation_cols].copy()

annot = annot.rename(columns={
    "G14_RAWSource_a": "G14_a",
    "G14_RAWSource_b": "G14_b",
    "G09_Date_Magnification_a": "G09_a",
    "G09_Date_Magnification_b": "G09_b"
})

annot["visual_cause_label"] = ""
annot["note"] = ""

annot.to_csv(annotation_csv, index=False)

# ============================================================
# BUAT CONTACT SHEET
# ============================================================

thumb_size = 190
gap = 18
text_height = 135
row_height = thumb_size + text_height + 22
sheet_width = 2 * thumb_size + 3 * gap
sheet_height = len(df) * row_height + gap

sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
draw = ImageDraw.Draw(sheet)

try:
    font_title = ImageFont.truetype("DejaVuSans-Bold.ttf", 13)
    font_small = ImageFont.truetype("DejaVuSans.ttf", 10)
    font_label = ImageFont.truetype("DejaVuSans-Bold.ttf", 11)
except:
    font_title = ImageFont.load_default()
    font_small = ImageFont.load_default()
    font_label = ImageFont.load_default()

def load_thumb(path, size=190):
    img = Image.open(path).convert("RGB")
    img = ImageOps.contain(img, (size, size))
    canvas = Image.new("RGB", (size, size), "white")
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

def draw_wrapped_text(draw, text, xy, font, max_width_chars=34, line_spacing=11, fill="black"):
    x, y = xy
    lines = []
    for part in str(text).split("\n"):
        if part:
            lines.extend(textwrap.wrap(part, width=max_width_chars))
        else:
            lines.append("")
    for line in lines:
        draw.text((x, y), line, font=font, fill=fill)
        y += line_spacing
    return y

for idx, row in df.iterrows():
    y0 = gap + idx * row_height
    x_left = gap
    x_right = gap * 2 + thumb_size

    title = (
        f"{row['pair_id']} | {row['class_name']} | "
        f"{row['split_a']} vs {row['split_b']} | "
        f"H={row['hamming_distance']} | corr={float(row['corr_128']):.3f}"
    )
    draw.text((gap, y0), title, font=font_title, fill="black")

    draw.text((x_left, y0 + 24), "LEFT / A", font=font_label, fill="black")
    draw.text((x_right, y0 + 24), "RIGHT / B", font=font_label, fill="black")

    img_left = load_thumb(row["left_image"], thumb_size)
    img_right = load_thumb(row["right_image"], thumb_size)

    img_y = y0 + 43
    sheet.paste(img_left, (x_left, img_y))
    sheet.paste(img_right, (x_right, img_y))

    draw.rectangle([x_left, img_y, x_left + thumb_size, img_y + thumb_size], outline="black", width=1)
    draw.rectangle([x_right, img_y, x_right + thumb_size, img_y + thumb_size], outline="black", width=1)

    text_y = img_y + thumb_size + 7

    left_text = f"A: {row['file_a']}\nRAW: {row['raw_source_id_a']}"
    right_text = f"B: {row['file_b']}\nRAW: {row['raw_source_id_b']}"

    draw_wrapped_text(draw, left_text, (x_left, text_y), font_small, max_width_chars=31)
    draw_wrapped_text(draw, right_text, (x_right, text_y), font_small, max_width_chars=31)

sheet.save(contact_sheet_path)

# ============================================================
# SUMMARY JSON
# ============================================================

allowed_labels = [
    "background_dominated",
    "possible_same_biological_sample_or_session",
    "hash_false_positive_or_class_similarity",
    "true_duplicate_unclear_source",
    "uncertain"
]

summary = {
    "stage": "V8_contact_sheet_and_manual_annotation_creation",
    "n_pairs": int(len(df)),
    "contact_sheet_path": str(contact_sheet_path),
    "annotation_csv": str(annotation_csv),
    "allowed_labels": allowed_labels,
    "instruction": "Isi kolom visual_cause_label menggunakan salah satu label valid. Jangan ubah kolom metadata lain.",
    "note": "V8 hanya investigasi visual penyebab kemiripan, bukan cleaning."
}

creation_summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== Output Sel V8-2 Disimpan ===")
print("Contact sheet:")
print(contact_sheet_path)
print("Annotation CSV:")
print(annotation_csv)
print("Creation summary JSON:")
print(creation_summary_json)

print("\nUkuran contact sheet:", sheet.size)
print("Jumlah pasangan:", len(df))

print("\n=== Preview Annotation CSV ===")
display(annot)

print("\n=== Sel V8-2 selesai ===")
print("Buka contact sheet, lalu isi kolom visual_cause_label di CSV anotasi.")
print("Allowed labels:")
for label in allowed_labels:
    print("-", label)

In [ ]:
# Sel V8-3 — Cek Anotasi Manual V8 dan Buat Summary

from pathlib import Path
import pandas as pd
import json

# ============================================================
# PATH
# ============================================================

outputs_dir = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

filled_csv = outputs_dir / "v8_g14_different_nearduplicate_manual_annotation_filled.csv"
selected_pairs_csv = outputs_dir / "v8_g14_different_nearduplicate_selected_pairs.csv"

# Output V8
summary_json = outputs_dir / "v8_g14_different_nearduplicate_summary.json"
class_summary_csv = outputs_dir / "v8_g14_different_nearduplicate_class_summary.csv"

print("=== Sel V8-3: Input Check ===")
print("Filled annotation CSV:", filled_csv)
print("Exists:", filled_csv.exists())
print("Selected pairs CSV:", selected_pairs_csv)
print("Exists:", selected_pairs_csv.exists())

if not filled_csv.exists():
    print("\nFile filled belum ditemukan dengan nama:")
    print(filled_csv)
    print("\nCek file V8 annotation yang ada di outputs:")
    for p in outputs_dir.glob("*v8*g14*different*annotation*.csv"):
        print("-", p)
    raise FileNotFoundError("Upload dulu file filled dengan nama v8_g14_different_nearduplicate_manual_annotation_filled.csv")

if not selected_pairs_csv.exists():
    raise FileNotFoundError(f"Selected pairs CSV tidak ditemukan: {selected_pairs_csv}")

# ============================================================
# LOAD DATA
# ============================================================

annot = pd.read_csv(filled_csv, keep_default_na=False)
pairs = pd.read_csv(selected_pairs_csv, keep_default_na=False)

print("\n=== Loaded Data ===")
print("Annotation shape:", annot.shape)
print("Selected pairs shape:", pairs.shape)

print("\nKolom annotation:")
print(list(annot.columns))

# ============================================================
# VALIDASI KOLOM DAN LABEL
# ============================================================

required_cols = [
    "pair_id",
    "class_name",
    "split_a",
    "file_a",
    "split_b",
    "file_b",
    "raw_source_id_a",
    "raw_source_id_b",
    "G14_a",
    "G14_b",
    "G09_a",
    "G09_b",
    "hamming_distance",
    "mse_128",
    "mae_128",
    "corr_128",
    "visual_cause_label",
    "note"
]

missing_cols = [c for c in required_cols if c not in annot.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan di annotation filled: {missing_cols}")

allowed_labels = [
    "background_dominated",
    "possible_same_biological_sample_or_session",
    "hash_false_positive_or_class_similarity",
    "true_duplicate_unclear_source",
    "uncertain"
]

annot["visual_cause_label_raw"] = annot["visual_cause_label"]
annot["visual_cause_label"] = (
    annot["visual_cause_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

missing_label = annot[annot["visual_cause_label"] == ""].copy()
invalid_label = annot[
    (annot["visual_cause_label"] != "") &
    (~annot["visual_cause_label"].isin(allowed_labels))
].copy()

print("\n=== Cek Label Manual V8 ===")
print("Total pasangan:", len(annot))
print("Label terisi:", int((annot["visual_cause_label"] != "").sum()))
print("Label kosong:", len(missing_label))
print("Label invalid:", len(invalid_label))

if len(missing_label) > 0:
    print("\nPair yang masih kosong:")
    display(missing_label[["pair_id", "class_name", "file_a", "file_b", "visual_cause_label_raw"]])

if len(invalid_label) > 0:
    print("\nPair dengan label invalid:")
    display(invalid_label[["pair_id", "class_name", "visual_cause_label_raw", "visual_cause_label"]])
    print("Label valid:")
    for label in allowed_labels:
        print("-", label)

if len(missing_label) > 0 or len(invalid_label) > 0:
    raise ValueError("Masih ada label kosong atau invalid. Perbaiki CSV filled lalu jalankan ulang Sel V8-3.")

# ============================================================
# SUMMARY OVERALL
# ============================================================

total_pairs = len(annot)

label_counts = (
    annot["visual_cause_label"]
    .value_counts()
    .reindex(allowed_labels, fill_value=0)
    .reset_index()
)
label_counts.columns = ["visual_cause_label", "count"]
label_counts["rate"] = label_counts["count"] / total_pairs

# Distribusi per kelas
class_summary = (
    annot
    .groupby("class_name")
    .agg(
        n_pairs=("pair_id", "count"),
        background_dominated=("visual_cause_label", lambda s: int((s == "background_dominated").sum())),
        possible_same_biological_sample_or_session=("visual_cause_label", lambda s: int((s == "possible_same_biological_sample_or_session").sum())),
        hash_false_positive_or_class_similarity=("visual_cause_label", lambda s: int((s == "hash_false_positive_or_class_similarity").sum())),
        true_duplicate_unclear_source=("visual_cause_label", lambda s: int((s == "true_duplicate_unclear_source").sum())),
        uncertain=("visual_cause_label", lambda s: int((s == "uncertain").sum())),
        mean_hamming_distance=("hamming_distance", "mean"),
        mean_corr_128=("corr_128", "mean"),
        mean_mse_128=("mse_128", "mean")
    )
    .reset_index()
)

class_summary["dominant_label"] = class_summary[
    [
        "background_dominated",
        "possible_same_biological_sample_or_session",
        "hash_false_positive_or_class_similarity",
        "true_duplicate_unclear_source",
        "uncertain"
    ]
].idxmax(axis=1)

class_summary = class_summary.sort_values(
    ["n_pairs", "true_duplicate_unclear_source", "possible_same_biological_sample_or_session"],
    ascending=[False, False, False]
).reset_index(drop=True)

class_summary.to_csv(class_summary_csv, index=False)

# Distribusi per split pair
annot["split_pair"] = annot["split_a"].astype(str) + " -> " + annot["split_b"].astype(str)

split_pair_summary = (
    annot
    .groupby("split_pair")
    .agg(
        n_pairs=("pair_id", "count"),
        background_dominated=("visual_cause_label", lambda s: int((s == "background_dominated").sum())),
        possible_same_biological_sample_or_session=("visual_cause_label", lambda s: int((s == "possible_same_biological_sample_or_session").sum())),
        hash_false_positive_or_class_similarity=("visual_cause_label", lambda s: int((s == "hash_false_positive_or_class_similarity").sum())),
        true_duplicate_unclear_source=("visual_cause_label", lambda s: int((s == "true_duplicate_unclear_source").sum())),
        uncertain=("visual_cause_label", lambda s: int((s == "uncertain").sum()))
    )
    .reset_index()
)

# Distribusi per hamming
hamming_summary = (
    annot
    .groupby("hamming_distance")
    .agg(
        n_pairs=("pair_id", "count"),
        background_dominated=("visual_cause_label", lambda s: int((s == "background_dominated").sum())),
        possible_same_biological_sample_or_session=("visual_cause_label", lambda s: int((s == "possible_same_biological_sample_or_session").sum())),
        hash_false_positive_or_class_similarity=("visual_cause_label", lambda s: int((s == "hash_false_positive_or_class_similarity").sum())),
        true_duplicate_unclear_source=("visual_cause_label", lambda s: int((s == "true_duplicate_unclear_source").sum())),
        uncertain=("visual_cause_label", lambda s: int((s == "uncertain").sum())),
        mean_corr_128=("corr_128", "mean"),
        mean_mse_128=("mse_128", "mean")
    )
    .reset_index()
)

# ============================================================
# INTERPRETASI AWAL OTOMATIS
# ============================================================

dominant_label = label_counts.sort_values("count", ascending=False).iloc[0]["visual_cause_label"]
dominant_count = int(label_counts.sort_values("count", ascending=False).iloc[0]["count"])
dominant_rate = dominant_count / total_pairs if total_pairs > 0 else 0

true_dup_count = int((annot["visual_cause_label"] == "true_duplicate_unclear_source").sum())
possible_session_count = int((annot["visual_cause_label"] == "possible_same_biological_sample_or_session").sum())
background_count = int((annot["visual_cause_label"] == "background_dominated").sum())
hash_false_count = int((annot["visual_cause_label"] == "hash_false_positive_or_class_similarity").sum())
uncertain_count = int((annot["visual_cause_label"] == "uncertain").sum())

if true_dup_count / total_pairs >= 0.30:
    cleaning_note = "A-clean perlu dipertimbangkan kuat karena true_duplicate_unclear_source cukup tinggi."
elif (true_dup_count + possible_session_count) / total_pairs >= 0.50:
    cleaning_note = "A-clean perlu didiskusikan, tetapi lebih aman sebagai sensitivity analysis setelah baseline awal."
elif hash_false_count / total_pairs >= 0.50 or background_count / total_pairs >= 0.50:
    cleaning_note = "A-clean belum perlu dibuat sebelum baseline; near-duplicate lebih cocok dilaporkan sebagai risiko/metodologi."
else:
    cleaning_note = "A-clean belum otomatis perlu; keputusan tetap dibawa ke MASTER CHAT."

if true_dup_count == 0:
    baseline_note = "Baseline DenseNet201 boleh dimulai tanpa cleaning, dengan catatan near-duplicate dilaporkan sebagai risiko metodologis."
else:
    baseline_note = "Baseline masih bisa dimulai tanpa cleaning untuk comparability, tetapi perlu catatan eksplisit dan opsi A-clean sensitivity."

summary = {
    "stage": "V8_g14_different_nearduplicate_visual_cause_validation",
    "status": "completed",
    "input_filled_annotation": str(filled_csv),
    "total_pairs": int(total_pairs),
    "label_counts": label_counts.to_dict(orient="records"),
    "dominant_label": dominant_label,
    "dominant_count": dominant_count,
    "dominant_rate": dominant_rate,
    "counts": {
        "background_dominated": background_count,
        "possible_same_biological_sample_or_session": possible_session_count,
        "hash_false_positive_or_class_similarity": hash_false_count,
        "true_duplicate_unclear_source": true_dup_count,
        "uncertain": uncertain_count
    },
    "split_pair_summary": split_pair_summary.to_dict(orient="records"),
    "hamming_summary": hamming_summary.to_dict(orient="records"),
    "interpretation": {
        "majority_cause": dominant_label,
        "cleaning_note": cleaning_note,
        "baseline_note": baseline_note,
        "a_clean_sensitivity_note": "A-clean lebih aman dijadikan sensitivity analysis setelah baseline awal, kecuali MASTER CHAT memutuskan true duplicate harus dibersihkan sebelum baseline."
    },
    "outputs": {
        "summary_json": str(summary_json),
        "class_summary_csv": str(class_summary_csv)
    }
}

summary_json.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ============================================================
# DISPLAY
# ============================================================

print("\n=== Label Counts V8 ===")
display(label_counts)

print("\n=== Overall Summary V8 ===")
print("Total pairs:", total_pairs)
print("Dominant label:", dominant_label)
print("Dominant count:", dominant_count)
print("Dominant rate:", round(dominant_rate * 100, 2), "%")
print("background_dominated:", background_count)
print("possible_same_biological_sample_or_session:", possible_session_count)
print("hash_false_positive_or_class_similarity:", hash_false_count)
print("true_duplicate_unclear_source:", true_dup_count)
print("uncertain:", uncertain_count)

print("\n=== Class Summary V8 ===")
display(class_summary)

print("\n=== Split Pair Summary V8 ===")
display(split_pair_summary)

print("\n=== Hamming Summary V8 ===")
display(hamming_summary)

print("\n=== Interpretasi Awal V8 ===")
print(cleaning_note)
print(baseline_note)

print("\n=== Output V8 Disimpan ===")
print("Summary JSON:")
print(summary_json)
print("Class summary CSV:")
print(class_summary_csv)

print("\n=== Sel V8-3 selesai ===")

# Bagian Baseline — DenseNet201 TEM Virus


In [ ]:
# Sel B1 — Cek Runtime, GPU, Drive, Manifest, Dataset, Outputs, dan Checkpoints

import os
import glob
import json
import torch
from pathlib import Path

print("=== Sel B1: Runtime, GPU, Drive, dan Path Check ===")

# 1. Cek GPU
print("\n=== GPU Check ===")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
else:
    print("GPU tidak terdeteksi. Pastikan Runtime Colab memakai GPU: Runtime > Change runtime type > GPU")

# 2. Mount Google Drive
print("\n=== Google Drive Mount ===")
drive_root = Path("/content/drive")

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive folder sudah ada:", drive_root)

# Jika belum benar-benar mounted, coba mount
mydrive = Path("/content/drive/MyDrive")
if not mydrive.exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("MyDrive tersedia:", mydrive)

# 3. Path utama project
PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"

# 4. Buat folder output dan checkpoints jika belum ada
print("\n=== Output Folder Check ===")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Project dir exists:", PROJECT_DIR.exists())
print("Output dir:", OUTPUT_DIR)
print("Output dir exists:", OUTPUT_DIR.exists())
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Checkpoint dir exists:", CHECKPOINT_DIR.exists())

# 5. Cek manifest Skenario A official
print("\n=== Manifest A Official Check ===")
print("Manifest A path:", MANIFEST_A_PATH)
print("Manifest A exists:", MANIFEST_A_PATH.exists())

# 6. Cek folder dataset ekstraksi
print("\n=== Dataset Folder Check ===")

dataset_candidates = [
    Path("/content/tem_virus_dataset_extracted/TEM virus dataset/context_virus_1nm_256x256"),
    Path("/content/tem_virus_dataset_extracted/context_virus_1nm_256x256"),
    PROJECT_DIR / "TEM virus dataset" / "context_virus_1nm_256x256",
    PROJECT_DIR / "data" / "context_virus_1nm_256x256",
    PROJECT_DIR / "context_virus_1nm_256x256",
]

found_dataset_dirs = []

for p in dataset_candidates:
    if p.exists():
        found_dataset_dirs.append(p)

# Pencarian ringan jika kandidat standar tidak ketemu
if len(found_dataset_dirs) == 0:
    search_roots = [
        Path("/content"),
        PROJECT_DIR
    ]
    for root in search_roots:
        if root.exists():
            matches = glob.glob(str(root / "**" / "context_virus_1nm_256x256"), recursive=True)
            for m in matches:
                mp = Path(m)
                if mp.exists() and mp not in found_dataset_dirs:
                    found_dataset_dirs.append(mp)

print("Jumlah kandidat folder dataset ditemukan:", len(found_dataset_dirs))
for i, p in enumerate(found_dataset_dirs, start=1):
    print(f"{i}. {p}")

if len(found_dataset_dirs) > 0:
    DATASET_DIR = found_dataset_dirs[0]
    print("\nDataset dir terpilih sementara:", DATASET_DIR)
else:
    DATASET_DIR = None
    print("\nFolder dataset context_virus_1nm_256x256 belum ditemukan.")
    print("Nanti jika memang hilang karena runtime reset, kita recovery dari ZIP Drive pada langkah berikutnya.")

# 7. Ringkasan status
print("\n=== Ringkasan Status Sel B1 ===")
status = {
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "project_dir_exists": PROJECT_DIR.exists(),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "checkpoint_dir_exists": CHECKPOINT_DIR.exists(),
    "manifest_A_exists": MANIFEST_A_PATH.exists(),
    "dataset_dir_found": DATASET_DIR is not None,
    "dataset_dir": str(DATASET_DIR) if DATASET_DIR is not None else None,
}

for k, v in status.items():
    print(f"{k}: {v}")

# 8. Simpan status awal B1 ke JSON
b1_status_path = OUTPUT_DIR / "b1_runtime_path_status.json"
with open(b1_status_path, "w") as f:
    json.dump(status, f, indent=2)

print("\nStatus Sel B1 disimpan ke:")
print(b1_status_path)

In [ ]:
# Sel B2 — Cari ZIP Dataset TEM Virus dan Cek Isi untuk Recovery

import os
import glob
import json
import zipfile
from pathlib import Path

print("=== Sel B2: Cari ZIP Dataset dan Cek Isi ===")

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FOLDER_NAME = "context_virus_1nm_256x256"

# 1. Cari file ZIP kandidat di folder project terlebih dahulu
print("\n=== Scan ZIP di folder project ===")
project_zip_candidates = list(PROJECT_DIR.rglob("*.zip")) if PROJECT_DIR.exists() else []

print("Jumlah ZIP di PROJECT_DIR:", len(project_zip_candidates))
for i, p in enumerate(project_zip_candidates[:30], start=1):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"{i}. {p} | {size_mb:.2f} MB")

# 2. Jika belum ketemu, cari ringan di MyDrive dengan kata kunci yang mungkin
print("\n=== Scan ZIP kandidat di MyDrive dengan keyword TEM/virus ===")
MYDRIVE = Path("/content/drive/MyDrive")

all_candidate_patterns = [
    str(MYDRIVE / "**" / "*TEM*.zip"),
    str(MYDRIVE / "**" / "*tem*.zip"),
    str(MYDRIVE / "**" / "*Virus*.zip"),
    str(MYDRIVE / "**" / "*virus*.zip"),
]

keyword_zip_candidates = []
for pattern in all_candidate_patterns:
    matches = glob.glob(pattern, recursive=True)
    for m in matches:
        mp = Path(m)
        if mp.exists() and mp not in keyword_zip_candidates:
            keyword_zip_candidates.append(mp)

print("Jumlah ZIP kandidat keyword:", len(keyword_zip_candidates))
for i, p in enumerate(keyword_zip_candidates[:50], start=1):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"{i}. {p} | {size_mb:.2f} MB")

# 3. Gabungkan kandidat, prioritaskan yang ada di folder project
zip_candidates = []
for p in project_zip_candidates + keyword_zip_candidates:
    if p not in zip_candidates:
        zip_candidates.append(p)

print("\n=== Total ZIP Kandidat Unik ===")
print("Total:", len(zip_candidates))

# 4. Cek isi ZIP: cari folder context_virus_1nm_256x256
zip_scan_results = []

for idx, zip_path in enumerate(zip_candidates, start=1):
    result = {
        "zip_path": str(zip_path),
        "size_mb": zip_path.stat().st_size / (1024 * 1024),
        "is_readable_zip": False,
        "contains_target_folder": False,
        "target_entries_count": 0,
        "sample_target_entries": [],
        "error": None
    }

    print(f"\n--- Cek ZIP {idx}/{len(zip_candidates)} ---")
    print("ZIP:", zip_path)
    print(f"Size: {result['size_mb']:.2f} MB")

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            result["is_readable_zip"] = True
            names = zf.namelist()

            target_entries = [n for n in names if TARGET_FOLDER_NAME in n]
            result["target_entries_count"] = len(target_entries)
            result["contains_target_folder"] = len(target_entries) > 0
            result["sample_target_entries"] = target_entries[:10]

            print("Readable ZIP:", True)
            print(f"Jumlah entry mengandung '{TARGET_FOLDER_NAME}':", len(target_entries))

            if len(target_entries) > 0:
                print("Contoh entry target:")
                for s in target_entries[:10]:
                    print(" ", s)

            # Hitung kasar file gambar dalam target folder
            image_entries = [
                n for n in target_entries
                if n.lower().endswith((".tif", ".tiff", ".png", ".jpg", ".jpeg"))
            ]
            result["target_image_entries_count"] = len(image_entries)
            print("Jumlah image entry dalam target folder:", len(image_entries))

    except Exception as e:
        result["error"] = str(e)
        print("Readable ZIP:", False)
        print("Error:", e)

    zip_scan_results.append(result)

# 5. Pilih ZIP terbaik sementara
valid_target_zips = [
    r for r in zip_scan_results
    if r["is_readable_zip"] and r["contains_target_folder"]
]

print("\n=== Rekomendasi Recovery ===")
if len(valid_target_zips) > 0:
    # Pilih yang target image entry paling banyak
    best_zip = sorted(
        valid_target_zips,
        key=lambda x: x.get("target_image_entries_count", 0),
        reverse=True
    )[0]

    print("ZIP dataset yang direkomendasikan:")
    print(best_zip["zip_path"])
    print(f"Size: {best_zip['size_mb']:.2f} MB")
    print("Jumlah image entry target:", best_zip.get("target_image_entries_count", None))
else:
    best_zip = None
    print("Belum ditemukan ZIP yang berisi folder context_virus_1nm_256x256.")
    print("Nanti kita perlu cek lokasi/nama ZIP dataset di Drive.")

# 6. Simpan hasil scan
b2_scan_path = OUTPUT_DIR / "b2_zip_recovery_scan.json"
with open(b2_scan_path, "w") as f:
    json.dump({
        "target_folder_name": TARGET_FOLDER_NAME,
        "total_zip_candidates": len(zip_candidates),
        "zip_scan_results": zip_scan_results,
        "recommended_zip": best_zip
    }, f, indent=2)

print("\nHasil scan Sel B2 disimpan ke:")
print(b2_scan_path)

print("\n=== Ringkasan Sel B2 ===")
print("target_folder_name:", TARGET_FOLDER_NAME)
print("total_zip_candidates:", len(zip_candidates))
print("valid_target_zips:", len(valid_target_zips))
print("recommended_zip:", best_zip["zip_path"] if best_zip else None)

In [ ]:
# Sel B3 — Recovery Ekstraksi Dataset context_virus_1nm_256x256 ke /content

import os
import json
import zipfile
from pathlib import Path
from time import time

print("=== Sel B3: Recovery Ekstraksi Dataset ===")

ZIP_PATH = Path("/content/drive/MyDrive/Riset_TEM_Virus/data_raw/TEM virus dataset.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
TARGET_FOLDER_NAME = "context_virus_1nm_256x256"
OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / TARGET_FOLDER_NAME

print("\n=== Path Check ===")
print("ZIP_PATH:", ZIP_PATH)
print("ZIP exists:", ZIP_PATH.exists())
print("EXTRACT_ROOT:", EXTRACT_ROOT)
print("EXPECTED_DATASET_DIR:", EXPECTED_DATASET_DIR)
print("Dataset already exists:", EXPECTED_DATASET_DIR.exists())

# 1. Jika sudah ada, tidak perlu ekstrak ulang
if EXPECTED_DATASET_DIR.exists():
    print("\nDataset sudah tersedia. Ekstraksi dilewati.")
else:
    print("\nDataset belum tersedia. Mulai ekstraksi folder target dari ZIP...")
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    start_time = time()

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        names = zf.namelist()

        # Ambil hanya entry yang berada dalam context_virus_1nm_256x256
        target_entries = [n for n in names if TARGET_FOLDER_NAME in n]

        print("Total entry target ditemukan:", len(target_entries))

        # Ekstrak semua entry target
        for i, member in enumerate(target_entries, start=1):
            zf.extract(member, EXTRACT_ROOT)

            if i % 2000 == 0 or i == len(target_entries):
                elapsed_min = (time() - start_time) / 60
                print(f"Ekstraksi progress: {i}/{len(target_entries)} entry | elapsed: {elapsed_min:.2f} menit")

    print("\nEkstraksi selesai.")

# 2. Verifikasi folder hasil ekstraksi
print("\n=== Verifikasi Dataset Folder ===")
print("EXPECTED_DATASET_DIR exists:", EXPECTED_DATASET_DIR.exists())

# 3. Hitung file gambar per split standar
image_exts = (".tif", ".tiff", ".png", ".jpg", ".jpeg")
split_counts = {}

if EXPECTED_DATASET_DIR.exists():
    for split in ["train", "validation", "test", "augmented_train"]:
        split_dir = EXPECTED_DATASET_DIR / split
        if split_dir.exists():
            image_files = [
                p for p in split_dir.rglob("*")
                if p.is_file() and p.suffix.lower() in image_exts
            ]
            class_dirs = [p.name for p in split_dir.iterdir() if p.is_dir()]
            split_counts[split] = {
                "exists": True,
                "num_images": len(image_files),
                "num_classes": len(class_dirs),
                "classes": sorted(class_dirs)
            }
        else:
            split_counts[split] = {
                "exists": False,
                "num_images": 0,
                "num_classes": 0,
                "classes": []
            }

print("\n=== Split Counts di Folder Dataset ===")
for split, info in split_counts.items():
    print(f"\nSplit: {split}")
    print("exists:", info["exists"])
    print("num_images:", info["num_images"])
    print("num_classes:", info["num_classes"])
    if info["exists"]:
        print("classes:", info["classes"])

# 4. Simpan status B3
b3_status = {
    "zip_path": str(ZIP_PATH),
    "zip_exists": ZIP_PATH.exists(),
    "extract_root": str(EXTRACT_ROOT),
    "dataset_dir": str(EXPECTED_DATASET_DIR),
    "dataset_dir_exists": EXPECTED_DATASET_DIR.exists(),
    "split_counts": split_counts
}

b3_status_path = OUTPUT_DIR / "b3_dataset_extraction_status.json"
with open(b3_status_path, "w") as f:
    json.dump(b3_status, f, indent=2)

print("\nStatus Sel B3 disimpan ke:")
print(b3_status_path)

print("\n=== Ringkasan Sel B3 ===")
print("dataset_dir_exists:", EXPECTED_DATASET_DIR.exists())
print("dataset_dir:", EXPECTED_DATASET_DIR)

In [ ]:
# Sel B4 — Load Manifest A Official dan Cek Split, Kelas, serta Kolom

import os
import json
import pandas as pd
from pathlib import Path

print("=== Sel B4: Load Manifest A Official ===")

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"

DATASET_DIR = Path("/content/tem_virus_dataset_extracted/TEM virus dataset/context_virus_1nm_256x256")

print("\n=== Path Check ===")
print("Manifest A:", MANIFEST_A_PATH)
print("Manifest exists:", MANIFEST_A_PATH.exists())
print("Dataset dir:", DATASET_DIR)
print("Dataset dir exists:", DATASET_DIR.exists())

# 1. Load manifest
df = pd.read_csv(MANIFEST_A_PATH)

print("\n=== Manifest Shape ===")
print("Shape:", df.shape)

print("\n=== Kolom Manifest ===")
print(list(df.columns))

print("\n=== Preview Manifest ===")
display(df.head())

# 2. Cek kemungkinan nama kolom penting
possible_split_cols = ["split", "official_split", "subset", "data_split"]
possible_class_cols = ["class_name", "label_name", "class", "category"]
possible_path_cols = ["filepath", "file_path", "path", "image_path", "relative_path", "rel_path"]

split_col = next((c for c in possible_split_cols if c in df.columns), None)
class_col = next((c for c in possible_class_cols if c in df.columns), None)
path_col = next((c for c in possible_path_cols if c in df.columns), None)

print("\n=== Deteksi Kolom Penting ===")
print("split_col:", split_col)
print("class_col:", class_col)
print("path_col:", path_col)

if split_col is None:
    raise ValueError("Kolom split tidak ditemukan. Cek nama kolom manifest.")
if class_col is None:
    raise ValueError("Kolom class/label tidak ditemukan. Cek nama kolom manifest.")
if path_col is None:
    print("PERINGATAN: Kolom path belum terdeteksi otomatis. Nanti kita cek manual dari daftar kolom.")

# 3. Cek jumlah data per split
print("\n=== Jumlah Data per Split ===")
split_counts = df[split_col].value_counts(dropna=False).sort_index()
display(split_counts.rename("count").reset_index().rename(columns={"index": split_col}))

# 4. Cek kelas
print("\n=== Kelas Unik ===")
classes = sorted(df[class_col].dropna().unique().tolist())
print("Jumlah kelas:", len(classes))
print(classes)

print("\n=== Jumlah Data per Kelas ===")
class_counts = df[class_col].value_counts().sort_index()
display(class_counts.rename("count").reset_index().rename(columns={"index": class_col}))

# 5. Cek jumlah data per split dan kelas
print("\n=== Crosstab Split x Class ===")
split_class_table = pd.crosstab(df[class_col], df[split_col])
display(split_class_table)

# 6. Cek apakah _EXCLUDED masuk manifest
print("\n=== Cek _EXCLUDED dan augmented_train ===")
contains_excluded = df[class_col].astype(str).str.contains("_EXCLUDED", case=False, regex=False).any()
print("Ada _EXCLUDED di manifest:", contains_excluded)

contains_augmented = df[split_col].astype(str).str.contains("augmented", case=False, regex=False).any()
print("Ada augmented_train/augmented di manifest:", contains_augmented)

# 7. Cek split yang tersedia
unique_splits = sorted(df[split_col].dropna().astype(str).unique().tolist())
print("Split unik:", unique_splits)

# 8. Simpan ringkasan B4
b4_summary = {
    "manifest_path": str(MANIFEST_A_PATH),
    "dataset_dir": str(DATASET_DIR),
    "shape": list(df.shape),
    "columns": list(df.columns),
    "split_col": split_col,
    "class_col": class_col,
    "path_col": path_col,
    "split_counts": split_counts.to_dict(),
    "num_classes": len(classes),
    "classes": classes,
    "contains_excluded": bool(contains_excluded),
    "contains_augmented": bool(contains_augmented),
    "unique_splits": unique_splits
}

b4_summary_path = OUTPUT_DIR / "b4_manifest_A_official_check.json"
with open(b4_summary_path, "w") as f:
    json.dump(b4_summary, f, indent=2)

print("\nRingkasan Sel B4 disimpan ke:")
print(b4_summary_path)

print("\n=== Ringkasan Sel B4 ===")
print("Manifest loaded:", True)
print("Jumlah data:", len(df))
print("Jumlah kelas:", len(classes))
print("Ada _EXCLUDED:", contains_excluded)
print("Ada augmented:", contains_augmented)

In [ ]:
# Sel B5 — Cek Label Mapping, Filepath, dan Validitas Gambar Manifest A

import os
import json
import random
import pandas as pd
from pathlib import Path
from PIL import Image

print("=== Sel B5: Cek Label Mapping, Filepath, dan Validitas Gambar ===")

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"

df = pd.read_csv(MANIFEST_A_PATH)

print("\n=== Manifest Loaded ===")
print("Shape:", df.shape)

# 1. Cek label mapping
print("\n=== Label Mapping Check ===")
label_map_df = (
    df[["class_name", "label_id"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

display(label_map_df)

num_classes = label_map_df["class_name"].nunique()
num_label_ids = label_map_df["label_id"].nunique()
label_ids_sorted = sorted(label_map_df["label_id"].unique().tolist())

print("Jumlah class_name unik:", num_classes)
print("Jumlah label_id unik:", num_label_ids)
print("Label ID:", label_ids_sorted)
print("Label ID sesuai 0 sampai 13:", label_ids_sorted == list(range(14)))

# Cek apakah satu class hanya punya satu label_id
class_to_label_count = df.groupby("class_name")["label_id"].nunique()
bad_class_label = class_to_label_count[class_to_label_count != 1]

# Cek apakah satu label_id hanya punya satu class
label_to_class_count = df.groupby("label_id")["class_name"].nunique()
bad_label_class = label_to_class_count[label_to_class_count != 1]

print("\nClass dengan label_id lebih dari satu:", len(bad_class_label))
if len(bad_class_label) > 0:
    display(bad_class_label)

print("Label_id dengan class lebih dari satu:", len(bad_label_class))
if len(bad_label_class) > 0:
    display(bad_label_class)

# 2. Cek filepath tersedia
print("\n=== Filepath Availability Check ===")
df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())

missing_df = df[~df["filepath_exists"]].copy()
print("Total filepath:", len(df))
print("Filepath exists:", int(df["filepath_exists"].sum()))
print("Filepath missing:", len(missing_df))

if len(missing_df) > 0:
    print("\nContoh filepath missing:")
    display(missing_df.head(20)[["filepath", "filename", "class_name", "split"]])

# 3. Cek ekstensi file
print("\n=== File Extension Check ===")
df["extension"] = df["filepath"].apply(lambda x: Path(str(x)).suffix.lower())
ext_counts = df["extension"].value_counts().sort_index()
display(ext_counts.rename("count").reset_index().rename(columns={"index": "extension"}))

# 4. Cek beberapa gambar bisa dibuka
print("\n=== Image Open Sanity Check ===")

sample_rows = []
for split in ["train", "validation", "test"]:
    split_df = df[(df["split"] == split) & (df["filepath_exists"])]
    sample_n = min(10, len(split_df))
    sample_rows.append(split_df.sample(sample_n, random_state=42))

sample_df = pd.concat(sample_rows, ignore_index=True)

image_check_records = []
failed_images = []

for idx, row in sample_df.iterrows():
    fp = Path(row["filepath"])
    record = {
        "split": row["split"],
        "class_name": row["class_name"],
        "label_id": int(row["label_id"]),
        "filepath": str(fp),
        "filename": row["filename"],
        "can_open": False,
        "mode": None,
        "size": None,
        "error": None
    }

    try:
        with Image.open(fp) as img:
            record["can_open"] = True
            record["mode"] = img.mode
            record["size"] = list(img.size)
    except Exception as e:
        record["error"] = str(e)
        failed_images.append(record)

    image_check_records.append(record)

image_check_df = pd.DataFrame(image_check_records)

print("Jumlah sample image dicek:", len(image_check_df))
print("Berhasil dibuka:", int(image_check_df["can_open"].sum()))
print("Gagal dibuka:", int((~image_check_df["can_open"]).sum()))

display(image_check_df[["split", "class_name", "filename", "can_open", "mode", "size", "error"]])

# 5. Ringkasan per split setelah filepath check
print("\n=== Split Counts dari Filepath yang Ada ===")
available_split_counts = (
    df[df["filepath_exists"]]
    .groupby("split")
    .size()
    .rename("available_count")
    .reset_index()
)

display(available_split_counts)

# 6. Simpan output penting
label_mapping_path = OUTPUT_DIR / "b5_label_mapping_A_official.json"
label_mapping = {
    str(int(row["label_id"])): row["class_name"]
    for _, row in label_map_df.iterrows()
}

with open(label_mapping_path, "w") as f:
    json.dump(label_mapping, f, indent=2)

b5_summary = {
    "manifest_path": str(MANIFEST_A_PATH),
    "manifest_shape": list(df.shape),
    "num_classes": int(num_classes),
    "num_label_ids": int(num_label_ids),
    "label_ids_sorted": label_ids_sorted,
    "label_id_expected_0_to_13": label_ids_sorted == list(range(14)),
    "bad_class_label_count": int(len(bad_class_label)),
    "bad_label_class_count": int(len(bad_label_class)),
    "total_filepaths": int(len(df)),
    "filepath_exists": int(df["filepath_exists"].sum()),
    "filepath_missing": int(len(missing_df)),
    "extension_counts": ext_counts.to_dict(),
    "image_sample_checked": int(len(image_check_df)),
    "image_sample_open_success": int(image_check_df["can_open"].sum()),
    "image_sample_open_failed": int((~image_check_df["can_open"]).sum()),
    "label_mapping_path": str(label_mapping_path)
}

b5_summary_path = OUTPUT_DIR / "b5_manifest_integrity_check.json"
with open(b5_summary_path, "w") as f:
    json.dump(b5_summary, f, indent=2)

image_check_path = OUTPUT_DIR / "b5_image_open_sample_check.csv"
image_check_df.to_csv(image_check_path, index=False)

print("\n=== File Tersimpan ===")
print("Label mapping:", label_mapping_path)
print("B5 summary:", b5_summary_path)
print("Image open sample check:", image_check_path)

print("\n=== Ringkasan Sel B5 ===")
for k, v in b5_summary.items():
    print(f"{k}: {v}")

In [ ]:
# Sel B6 — Simpan Pre-Registered Experiment Plan Baseline DenseNet201 Skenario A Seed 42

import json
from pathlib import Path
from datetime import datetime

print("=== Sel B6: Pre-Registered Experiment Plan ===")

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
DATASET_DIR = Path("/content/tem_virus_dataset_extracted/TEM virus dataset/context_virus_1nm_256x256")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

experiment_plan = {
    "created_at": datetime.now().isoformat(),
    "leaf_name": "LEAF Eksperimen & Coding — Baseline DenseNet201 TEM Virus",
    "project_scope": "TEM Virus Image Classification",
    "compute": "Google Colab Free",
    "gpu_expected": "Tesla T4 or available Colab GPU",
    "dataset": "TEM Virus subset 14 kelas",
    "dataset_folder": str(DATASET_DIR),
    "manifest_used": str(MANIFEST_A_PATH),
    "scenario_first": "A official split",
    "evaluation_name": "literature-comparable evaluation",
    "baseline_model": "DenseNet201 pretrained ImageNet",
    "first_seed": 42,
    "final_primary_seeds_later": [42, 123, 456, 789, 1024],
    "input_size": 224,
    "channel_handling": "grayscale copied/converted to 3-channel RGB",
    "num_classes": 14,
    "augmented_train": "not used",
    "A_clean_before_baseline": "not used",
    "scenario_B": "not used in first baseline",
    "scenario_C": "not used in first baseline",
    "train_augmentation": [
        "light rotation only",
        "horizontal flip",
        "vertical flip",
        "light brightness/contrast adjustment if safe"
    ],
    "validation_augmentation": "none",
    "test_augmentation": "none",
    "test_set_usage": "test set evaluated only after model selection from validation",
    "model_selection_metric": "validation_macro_f1",
    "tie_breaker_metric": "validation_accuracy",
    "single_seed_result_interpretation": "pipeline validation only, not final claim",
    "failed_runs": "must be logged",
    "stop_gate": {
        "condition": "if DenseNet201 A seed 42 test accuracy < 0.90",
        "action": "stop and investigate preprocessing, split, transform, input, and training setup before trying modern models"
    },
    "benchmark_main": {
        "paper": "Matuszewski & Sintorn 2021",
        "model": "DenseNet201 pretrained ImageNet + fine-tuning",
        "accuracy": 0.931,
        "f1_score": 0.921
    },
    "fairness_constraints": [
        "no data leakage",
        "no test-set tuning",
        "no cherry-picking split",
        "no augmented_train",
        "no A-clean before baseline",
        "no scenario B/C before scenario A seed 42 is completed",
        "no final claim before MASTER CHAT review"
    ],
    "planned_outputs": {
        "best_checkpoint": str(CHECKPOINT_DIR),
        "training_history": str(OUTPUT_DIR),
        "classification_report": str(OUTPUT_DIR),
        "confusion_matrix": str(OUTPUT_DIR),
        "metrics_json_csv": str(OUTPUT_DIR),
        "failed_run_log": str(OUTPUT_DIR / "failed_runs_log.csv")
    }
}

# Simpan JSON
json_path = OUTPUT_DIR / "b6_preregistered_experiment_plan_DenseNet201_A_seed42.json"
with open(json_path, "w") as f:
    json.dump(experiment_plan, f, indent=2)

# Simpan TXT yang mudah dibaca
txt_path = OUTPUT_DIR / "b6_preregistered_experiment_plan_DenseNet201_A_seed42.txt"
with open(txt_path, "w") as f:
    f.write("# Pre-Registered Experiment Plan — DenseNet201 Scenario A Seed 42\n\n")
    for key, value in experiment_plan.items():
        f.write(f"{key}: {value}\n")

print("\n=== Experiment Plan Tersimpan ===")
print("JSON:", json_path)
print("TXT :", txt_path)

print("\n=== Ringkasan Keputusan Penting ===")
print("Scenario:", experiment_plan["scenario_first"])
print("Evaluation name:", experiment_plan["evaluation_name"])
print("Model:", experiment_plan["baseline_model"])
print("Seed:", experiment_plan["first_seed"])
print("Input size:", experiment_plan["input_size"])
print("Model selection metric:", experiment_plan["model_selection_metric"])
print("Tie breaker metric:", experiment_plan["tie_breaker_metric"])
print("Stop-gate:", experiment_plan["stop_gate"])
print("Augmented train:", experiment_plan["augmented_train"])
print("A-clean before baseline:", experiment_plan["A_clean_before_baseline"])

In [ ]:
# Sel B7 — Setup Seed, Transform, Dataset, dan DataLoader

import os
import random
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("=== Sel B7: Setup Seed, Transform, Dataset, dan DataLoader ===")

# 1. Konfigurasi dasar
SEED = 42
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16   # aman untuk DenseNet201 di Tesla T4 Colab Free
NUM_WORKERS = 2

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Set seed agar eksperimen lebih reproducible
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Untuk reproducibility. Bisa sedikit lebih lambat, tapi aman untuk baseline.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Runtime Check ===")
print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# 3. Load manifest
df = pd.read_csv(MANIFEST_A_PATH)

# Pastikan hanya split yang dipakai
allowed_splits = ["train", "validation", "test"]
df = df[df["split"].isin(allowed_splits)].copy()

# Pastikan tidak ada _EXCLUDED dan tidak ada augmented
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["split"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

# 4. Pisahkan split
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\n=== Split Size ===")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# 5. Cek label
label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)
id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_to_id = {v: k for k, v in id_to_class.items()}

print("\n=== Label Mapping ===")
display(label_mapping)
print("NUM_CLASSES valid:", len(id_to_class) == NUM_CLASSES)

# 6. Transform
# Gambar asli grayscale mode L, di Dataset akan dikonversi ke RGB 3-channel.
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print("\n=== Transform Setup ===")
print("Train transform:", train_transform)
print("Validation/test transform:", eval_transform)

# 7. Dataset class
class TEMVirusManifestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = row["filepath"]
        label = int(row["label_id"])

        # Convert grayscale ke RGB 3-channel untuk DenseNet pretrained ImageNet
        image = Image.open(filepath).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "filepath": filepath,
            "class_name": row["class_name"],
            "filename": row["filename"]
        }

# 8. Buat dataset
train_dataset = TEMVirusManifestDataset(train_df, transform=train_transform)
val_dataset = TEMVirusManifestDataset(val_df, transform=eval_transform)
test_dataset = TEMVirusManifestDataset(test_df, transform=eval_transform)

print("\n=== Dataset Size ===")
print("train_dataset:", len(train_dataset))
print("val_dataset:", len(val_dataset))
print("test_dataset:", len(test_dataset))

# 9. Buat DataLoader
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("\n=== DataLoader Setup ===")
print("Batch size:", BATCH_SIZE)
print("Num workers:", NUM_WORKERS)
print("Pin memory:", pin_memory)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# 10. Ambil satu batch untuk sanity check
print("\n=== One Batch Sanity Check ===")
batch = next(iter(train_loader))

images = batch["image"]
labels = batch["label"]

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Image min:", float(images.min()))
print("Image max:", float(images.max()))
print("Label sample:", labels[:16].tolist())
print("Class sample:", batch["class_name"][:5])

# 11. Simpan ringkasan setup DataLoader
b7_summary = {
    "seed": SEED,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": pin_memory,
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "test_size": len(test_dataset),
    "train_batches": len(train_loader),
    "val_batches": len(val_loader),
    "test_batches": len(test_loader),
    "image_batch_shape": list(images.shape),
    "label_batch_shape": list(labels.shape),
    "id_to_class": {str(k): v for k, v in id_to_class.items()},
    "train_transform": str(train_transform),
    "eval_transform": str(eval_transform)
}

b7_summary_path = OUTPUT_DIR / "b7_dataloader_setup_summary.json"
with open(b7_summary_path, "w") as f:
    json.dump(b7_summary, f, indent=2)

print("\nRingkasan Sel B7 disimpan ke:")
print(b7_summary_path)

print("\n=== Ringkasan Sel B7 ===")
for k, v in b7_summary.items():
    if k not in ["id_to_class", "train_transform", "eval_transform"]:
        print(f"{k}: {v}")

In [ ]:
# Sel B8 — Tampilkan Contoh Batch Gambar dari DataLoader

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("=== Sel B8: Tampilkan Contoh Batch Gambar ===")

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ambil satu batch baru dari train_loader
sample_batch = next(iter(train_loader))

images = sample_batch["image"].cpu()
labels = sample_batch["label"].cpu().numpy()
class_names = sample_batch["class_name"]
filenames = sample_batch["filename"]

print("\n=== Batch Info ===")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Jumlah gambar ditampilkan:", min(16, len(images)))

# Denormalisasi ImageNet agar gambar bisa dilihat normal
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(img_tensor):
    img = img_tensor * imagenet_std + imagenet_mean
    img = torch.clamp(img, 0, 1)
    return img

# Plot 16 gambar
n_show = min(16, len(images))
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

for i in range(n_show):
    img = denormalize(images[i])
    img_np = img.permute(1, 2, 0).numpy()

    axes[i].imshow(img_np)
    axes[i].set_title(f"{class_names[i]}\nlabel={labels[i]}\n{filenames[i]}", fontsize=8)
    axes[i].axis("off")

# Matikan axis kosong kalau ada
for j in range(n_show, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# Simpan gambar contoh batch
sample_batch_path = OUTPUT_DIR / "b8_sample_train_batch_augmented.png"
plt.savefig(sample_batch_path, dpi=200, bbox_inches="tight")
plt.show()

print("\nContoh batch gambar disimpan ke:")
print(sample_batch_path)

# Simpan metadata contoh batch
b8_records = []
for i in range(n_show):
    b8_records.append({
        "index_in_batch": i,
        "label_id": int(labels[i]),
        "class_name": str(class_names[i]),
        "filename": str(filenames[i])
    })

b8_summary = {
    "sample_type": "train batch after train_transform augmentation",
    "image_batch_shape": list(images.shape),
    "num_images_shown": int(n_show),
    "saved_figure": str(sample_batch_path),
    "records": b8_records
}

b8_summary_path = OUTPUT_DIR / "b8_sample_train_batch_summary.json"
with open(b8_summary_path, "w") as f:
    json.dump(b8_summary, f, indent=2)

print("Ringkasan Sel B8 disimpan ke:")
print(b8_summary_path)

print("\n=== Ringkasan Sel B8 ===")
print("Visualisasi batch train selesai.")
print("Catatan: gambar ini sudah melewati augmentasi train ringan.")

In [ ]:
# Sel B9 — Siapkan DenseNet201 Pretrained ImageNet dan Ganti Classifier 14 Kelas

import json
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.models as models

print("=== Sel B9: Setup DenseNet201 Pretrained ImageNet ===")

# 1. Konfigurasi
NUM_CLASSES = 14
PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n=== Device Check ===")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA memory allocated before model:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved before model:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

# 2. Load DenseNet201 pretrained ImageNet
print("\n=== Loading DenseNet201 Pretrained ImageNet ===")
print("Catatan: jika weights belum ada di Colab, cell ini mungkin download pretrained weights dulu.")

try:
    weights = models.DenseNet201_Weights.IMAGENET1K_V1
    model = models.densenet201(weights=weights)
    weights_name = "DenseNet201_Weights.IMAGENET1K_V1"
except Exception as e:
    print("Gagal load dengan weights API baru. Error:", e)
    print("Mencoba fallback pretrained=True...")
    model = models.densenet201(pretrained=True)
    weights_name = "pretrained=True fallback"

print("Weights:", weights_name)

# 3. Cek classifier lama
old_classifier = model.classifier
old_in_features = model.classifier.in_features

print("\n=== Original Classifier ===")
print(old_classifier)
print("in_features:", old_in_features)

# 4. Ganti classifier menjadi 14 kelas
model.classifier = nn.Linear(old_in_features, NUM_CLASSES)

print("\n=== New Classifier ===")
print(model.classifier)

# 5. Kirim model ke device
model = model.to(device)

# 6. Hitung parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n=== Parameter Count ===")
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

# 7. Test forward pass dengan 1 batch dari train_loader
print("\n=== Forward Pass Sanity Check ===")
model.eval()

batch = next(iter(train_loader))
images = batch["image"].to(device, non_blocking=True)
labels = batch["label"].to(device, non_blocking=True)

with torch.no_grad():
    outputs = model(images)

print("Input image shape:", tuple(images.shape))
print("Output logits shape:", tuple(outputs.shape))
print("Label shape:", tuple(labels.shape))
print("Output dtype:", outputs.dtype)
print("Output min:", float(outputs.min()))
print("Output max:", float(outputs.max()))

assert outputs.shape[0] == images.shape[0], "Batch output tidak sesuai batch input."
assert outputs.shape[1] == NUM_CLASSES, "Jumlah output class tidak sesuai NUM_CLASSES=14."

print("Forward pass OK: output shape sesuai [batch_size, 14].")

# 8. Cek memory GPU setelah model
if torch.cuda.is_available():
    print("\n=== CUDA Memory After Model ===")
    print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

# 9. Simpan ringkasan model setup
b9_summary = {
    "model_name": "DenseNet201",
    "weights": weights_name,
    "num_classes": NUM_CLASSES,
    "old_classifier": str(old_classifier),
    "old_in_features": int(old_in_features),
    "new_classifier": str(model.classifier),
    "total_params": int(total_params),
    "trainable_params": int(trainable_params),
    "device": str(device),
    "forward_pass_input_shape": list(images.shape),
    "forward_pass_output_shape": list(outputs.shape),
    "forward_pass_ok": True
}

b9_summary_path = OUTPUT_DIR / "b9_densenet201_model_setup_summary.json"
with open(b9_summary_path, "w") as f:
    json.dump(b9_summary, f, indent=2)

print("\nRingkasan Sel B9 disimpan ke:")
print(b9_summary_path)

print("\n=== Ringkasan Sel B9 ===")
for k, v in b9_summary.items():
    print(f"{k}: {v}")

In [ ]:
# Sel B10 — Setup Fungsi Training, Evaluasi, Metrik, Loss, Optimizer, Scheduler, dan Mixed Precision

import os
import json
import time
import copy
import math
import traceback
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("=== Sel B10: Setup Training Utilities ===")

# 1. Konfigurasi training baseline
SEED = 42
NUM_CLASSES = 14
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "DenseNet201_A_official_seed42"
BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"
HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
FAILED_RUN_LOG_PATH = OUTPUT_DIR / "failed_runs_log.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n=== Training Config ===")
print("Run name:", RUN_NAME)
print("Device:", device)
print("Max epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Model selection metric:", MODEL_SELECTION_METRIC)
print("Tie breaker metric:", TIE_BREAKER_METRIC)
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)

# 2. Loss function
# Baseline awal: CrossEntropyLoss standar, tanpa class weight.
# Jika hasil buruk, baru investigasi class imbalance/weighted loss pada tahap berikutnya.
criterion = nn.CrossEntropyLoss()

# 3. Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# 4. Scheduler sederhana berdasarkan validation macro F1
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# 5. Mixed precision
use_amp = torch.cuda.is_available()
try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
    amp_api = "torch.amp"
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)
    amp_api = "torch.cuda.amp"

print("\n=== AMP Setup ===")
print("Use AMP:", use_amp)
print("AMP API:", amp_api)

# 6. Fungsi metrik
def compute_classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

# 7. Fungsi train 1 epoch
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    start_time = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp and device.type == "cuda":
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        preds = torch.argmax(outputs.detach(), dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

        # Progress ringan setiap 50 batch
        if batch_idx % 50 == 0 or batch_idx == len(loader):
            elapsed = time.time() - start_time
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={elapsed/60:.2f} min"
            )

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_classification_metrics(all_labels, all_preds)
    metrics["loss"] = float(epoch_loss)

    return metrics

# 8. Fungsi evaluasi
@torch.no_grad()
def evaluate_model(model, loader, criterion, device, split_name="validation"):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp and device.type == "cuda":
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_classification_metrics(all_labels, all_preds)
    metrics["loss"] = float(epoch_loss)

    # Prefix nama split
    prefixed = {}
    for k, v in metrics.items():
        prefixed[f"{split_name}_{k}"] = v

    return prefixed

# 9. Fungsi logging failed run
def log_failed_run(run_name, error_message, traceback_text):
    record = {
        "timestamp": datetime.now().isoformat(),
        "run_name": run_name,
        "error_message": str(error_message),
        "traceback": str(traceback_text)
    }

    if FAILED_RUN_LOG_PATH.exists():
        old_df = pd.read_csv(FAILED_RUN_LOG_PATH)
        new_df = pd.concat([old_df, pd.DataFrame([record])], ignore_index=True)
    else:
        new_df = pd.DataFrame([record])

    new_df.to_csv(FAILED_RUN_LOG_PATH, index=False)
    print("Failed run logged to:", FAILED_RUN_LOG_PATH)

# 10. Simpan setup B10
b10_summary = {
    "run_name": RUN_NAME,
    "seed": SEED,
    "device": str(device),
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "criterion": str(criterion),
    "optimizer": str(optimizer),
    "scheduler": str(scheduler),
    "use_amp": bool(use_amp),
    "amp_api": amp_api,
    "model_selection_metric": MODEL_SELECTION_METRIC,
    "tie_breaker_metric": TIE_BREAKER_METRIC,
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "last_checkpoint_path": str(LAST_CKPT_PATH),
    "history_csv_path": str(HISTORY_CSV_PATH),
    "history_json_path": str(HISTORY_JSON_PATH),
    "failed_run_log_path": str(FAILED_RUN_LOG_PATH)
}

b10_summary_path = OUTPUT_DIR / f"{RUN_NAME}_b10_training_setup_summary.json"
with open(b10_summary_path, "w") as f:
    json.dump(b10_summary, f, indent=2)

print("\nRingkasan Sel B10 disimpan ke:")
print(b10_summary_path)

print("\n=== Ringkasan Sel B10 ===")
print("Training utilities siap.")
print("Belum ada training dijalankan di sel ini.")

In [ ]:
# Sel B11 — Training Baseline DenseNet201 Skenario A Official Seed 42

import os
import json
import time
import copy
import traceback
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch

print("=== Sel B11: Training Baseline DenseNet201 A Official Seed 42 ===")

# Pastikan variabel dari Sel B10 tersedia
required_vars = [
    "model", "train_loader", "val_loader", "criterion", "optimizer", "scheduler",
    "scaler", "device", "train_one_epoch", "evaluate_model",
    "RUN_NAME", "BEST_CKPT_PATH", "LAST_CKPT_PATH",
    "HISTORY_CSV_PATH", "HISTORY_JSON_PATH",
    "MAX_EPOCHS", "PATIENCE", "MODEL_SELECTION_METRIC", "TIE_BREAKER_METRIC",
    "use_amp", "id_to_class"
]

missing_vars = [v for v in required_vars if v not in globals()]
if len(missing_vars) > 0:
    raise RuntimeError(f"Variabel belum tersedia dari sel sebelumnya: {missing_vars}")

print("\n=== Training Start Info ===")
print("Run name:", RUN_NAME)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Max epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)
print("Selection metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Best checkpoint path:", BEST_CKPT_PATH)
print("Last checkpoint path:", LAST_CKPT_PATH)

if torch.cuda.is_available():
    print("CUDA memory allocated before training:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved before training:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

# 1. Resume logic jika runtime sempat putus dan last checkpoint sudah ada
RESUME_IF_LAST_EXISTS = True

start_epoch = 1
history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

if RESUME_IF_LAST_EXISTS and Path(LAST_CKPT_PATH).exists():
    print("\n=== Resume Checkpoint Ditemukan ===")
    print("Loading last checkpoint:", LAST_CKPT_PATH)

    checkpoint = torch.load(LAST_CKPT_PATH, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if "scheduler_state_dict" in checkpoint and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint["epoch"]) + 1
    history = checkpoint.get("history", [])
    best_metric = float(checkpoint.get("best_metric", -1.0))
    best_tie_metric = float(checkpoint.get("best_tie_metric", -1.0))
    best_epoch = checkpoint.get("best_epoch", None)
    epochs_without_improvement = int(checkpoint.get("epochs_without_improvement", 0))

    print("Resume dari epoch:", start_epoch)
    print("Best metric sebelumnya:", best_metric)
    print("Best tie metric sebelumnya:", best_tie_metric)
    print("Best epoch sebelumnya:", best_epoch)
    print("Epoch tanpa improvement sebelumnya:", epochs_without_improvement)
else:
    print("\nTidak ada last checkpoint. Training mulai dari awal.")

# 2. Training loop
training_start_time = time.time()

try:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start_time = time.time()

        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS}")
        print(f"Current LR: {current_lr:.8f}")
        print("=" * 80)

        # Train
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            use_amp=use_amp
        )

        # Validation
        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            split_name="val"
        )

        # Scheduler pakai val_macro_f1
        scheduler.step(val_metrics["val_macro_f1"])

        epoch_time_min = (time.time() - epoch_start_time) / 60

        # Gabungkan record epoch
        epoch_record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": float(epoch_time_min),
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(epoch_record)

        # Ambil metric utama dan tie breaker
        current_metric = float(epoch_record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(epoch_record[TIE_BREAKER_METRIC])

        # Improvement rule:
        # utama: val_macro_f1 naik
        # tie: jika sama sangat dekat, pilih val_accuracy lebih tinggi
        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat()
            }

            torch.save(best_checkpoint, BEST_CKPT_PATH)
            print(f"\n-> Best model updated at epoch {epoch}")
            print(f"   {MODEL_SELECTION_METRIC}: {best_metric:.4f}")
            print(f"   {TIE_BREAKER_METRIC}: {best_tie_metric:.4f}")
            print(f"   Saved to: {BEST_CKPT_PATH}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> No improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   Current {MODEL_SELECTION_METRIC}: {current_metric:.4f}")
            print(f"   Best {MODEL_SELECTION_METRIC}: {best_metric:.4f} at epoch {best_epoch}")

        # Simpan last checkpoint setiap epoch
        last_checkpoint = {
            "run_name": RUN_NAME,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat()
        }

        torch.save(last_checkpoint, LAST_CKPT_PATH)

        # Simpan history ke CSV dan JSON setiap epoch
        history_df = pd.DataFrame(history)
        history_df.to_csv(HISTORY_CSV_PATH, index=False)

        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        # Print ringkasan epoch
        print("\n=== Epoch Summary ===")
        print(f"Epoch time: {epoch_time_min:.2f} min")
        print(f"Train loss: {epoch_record['train_loss']:.4f}")
        print(f"Train acc : {epoch_record['train_accuracy']:.4f}")
        print(f"Train macro F1: {epoch_record['train_macro_f1']:.4f}")
        print(f"Val loss  : {epoch_record['val_loss']:.4f}")
        print(f"Val acc   : {epoch_record['val_accuracy']:.4f}")
        print(f"Val macro precision: {epoch_record['val_macro_precision']:.4f}")
        print(f"Val macro recall   : {epoch_record['val_macro_recall']:.4f}")
        print(f"Val macro F1       : {epoch_record['val_macro_f1']:.4f}")
        print(f"Val weighted F1    : {epoch_record['val_weighted_f1']:.4f}")
        print(f"Best epoch so far  : {best_epoch}")
        print(f"History saved to   : {HISTORY_CSV_PATH}")
        print(f"Last checkpoint    : {LAST_CKPT_PATH}")

        if torch.cuda.is_available():
            print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
            print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

        # Early stopping
        if epochs_without_improvement >= PATIENCE:
            print("\n=== Early Stopping Triggered ===")
            print(f"Tidak ada improvement selama {PATIENCE} epoch.")
            print(f"Best epoch: {best_epoch}")
            print(f"Best {MODEL_SELECTION_METRIC}: {best_metric:.4f}")
            break

    total_training_time_min = (time.time() - training_start_time) / 60

    # Simpan final training summary
    training_summary = {
        "run_name": RUN_NAME,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": float(total_training_time_min),
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat()
    }

    training_summary_path = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
    with open(training_summary_path, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n" + "=" * 80)
    print("=== Training Completed ===")
    print("=" * 80)
    print("Status: completed")
    print("Epochs completed:", len(history))
    print("Best epoch:", best_epoch)
    print(f"Best {MODEL_SELECTION_METRIC}: {best_metric:.4f}")
    print(f"Best {TIE_BREAKER_METRIC}: {best_tie_metric:.4f}")
    print(f"Total training time: {total_training_time_min:.2f} min")
    print("Training summary:", training_summary_path)
    print("Best checkpoint:", BEST_CKPT_PATH)
    print("Last checkpoint:", LAST_CKPT_PATH)
    print("History CSV:", HISTORY_CSV_PATH)

except Exception as e:
    tb = traceback.format_exc()
    print("\n=== TRAINING FAILED ===")
    print("Error:", e)
    print(tb)
    log_failed_run(RUN_NAME, str(e), tb)
    raise

In [ ]:
# Sel B12 — Evaluasi Best Checkpoint pada Validation dan Test Set

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B12: Evaluasi Best Checkpoint pada Validation dan Test ===")

# 1. Path dan konfigurasi
PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

RUN_NAME = "DenseNet201_A_official_seed42"
BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n=== Path Check ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Best checkpoint exists:", BEST_CKPT_PATH.exists())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if not BEST_CKPT_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint tidak ditemukan: {BEST_CKPT_PATH}")

# 2. Load best checkpoint
print("\n=== Load Best Checkpoint ===")
try:
    checkpoint = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(BEST_CKPT_PATH, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

best_epoch = checkpoint.get("best_epoch", checkpoint.get("epoch", None))
best_metric_name = checkpoint.get("best_metric_name", "val_macro_f1")
best_metric = checkpoint.get("best_metric", None)
best_tie_metric_name = checkpoint.get("best_tie_metric_name", "val_accuracy")
best_tie_metric = checkpoint.get("best_tie_metric", None)

print("Loaded checkpoint epoch:", checkpoint.get("epoch", None))
print("Best epoch:", best_epoch)
print("Best metric:", best_metric_name, best_metric)
print("Best tie metric:", best_tie_metric_name, best_tie_metric)

# 3. Ambil id_to_class
if "id_to_class" in checkpoint:
    id_to_class_ckpt = {int(k): v for k, v in checkpoint["id_to_class"].items()}
else:
    id_to_class_ckpt = id_to_class

class_names_ordered = [id_to_class_ckpt[i] for i in range(len(id_to_class_ckpt))]
print("\n=== Class Names Ordered ===")
for i, name in enumerate(class_names_ordered):
    print(f"{i}: {name}")

# 4. Fungsi prediksi lengkap
@torch.no_grad()
def predict_loader(model, loader, device, split_name):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []
    all_filepaths = []
    all_filenames = []
    all_class_names_true = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())
        all_filepaths.extend(batch["filepath"])
        all_filenames.extend(batch["filename"])
        all_class_names_true.extend(batch["class_name"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    avg_loss = running_loss / len(loader.dataset)

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": all_filepaths,
        "filename": all_filenames,
        "true_label": all_labels,
        "pred_label": all_preds,
        "true_class": [id_to_class_ckpt[int(x)] for x in all_labels],
        "pred_class": [id_to_class_ckpt[int(x)] for x in all_preds],
        "correct": [int(t == p) for t, p in zip(all_labels, all_preds)]
    })

    # Tambahkan probabilitas tiap kelas
    probs_arr = np.array(all_probs)
    for class_id, class_name in id_to_class_ckpt.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": float(avg_loss),
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds),
        "probs": probs_arr,
        "result_df": result_df
    }

# 5. Fungsi metrik lengkap
def compute_full_metrics(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names_ordered))),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names_ordered))),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(class_names_ordered)))
    )

    summary = {
        "split": split_name,
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1)
    }

    return summary, report_dict, report_text, cm

# 6. Evaluasi validation
print("\n=== Evaluasi Validation Set ===")
val_pred = predict_loader(model, val_loader, device, "validation")
val_summary, val_report_dict, val_report_text, val_cm = compute_full_metrics(
    val_pred["y_true"],
    val_pred["y_pred"],
    "validation"
)
val_summary["loss"] = val_pred["loss"]

print("\n=== Validation Metrics ===")
for k, v in val_summary.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

print("\n=== Validation Classification Report ===")
print(val_report_text)

# 7. Evaluasi test
print("\n=== Evaluasi Test Set ===")
test_pred = predict_loader(model, test_loader, device, "test")
test_summary, test_report_dict, test_report_text, test_cm = compute_full_metrics(
    test_pred["y_true"],
    test_pred["y_pred"],
    "test"
)
test_summary["loss"] = test_pred["loss"]

print("\n=== Test Metrics ===")
for k, v in test_summary.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

print("\n=== Test Classification Report ===")
print(test_report_text)

# 8. Simpan prediction CSV
val_pred_path = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
test_pred_path = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"

val_pred["result_df"].to_csv(val_pred_path, index=False)
test_pred["result_df"].to_csv(test_pred_path, index=False)

# 9. Simpan reports CSV/JSON/TXT
val_report_df = pd.DataFrame(val_report_dict).T
test_report_df = pd.DataFrame(test_report_dict).T

val_report_csv_path = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
test_report_csv_path = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"

val_report_txt_path = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
test_report_txt_path = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"

val_report_json_path = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
test_report_json_path = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"

val_report_df.to_csv(val_report_csv_path)
test_report_df.to_csv(test_report_csv_path)

with open(val_report_txt_path, "w") as f:
    f.write(val_report_text)

with open(test_report_txt_path, "w") as f:
    f.write(test_report_text)

with open(val_report_json_path, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(test_report_json_path, "w") as f:
    json.dump(test_report_dict, f, indent=2)

# 10. Simpan confusion matrix CSV
val_cm_df = pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered)
test_cm_df = pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered)

val_cm_csv_path = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
test_cm_csv_path = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

val_cm_df.to_csv(val_cm_csv_path)
test_cm_df.to_csv(test_cm_csv_path)

# 11. Plot confusion matrix
def save_confusion_matrix_plot(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))

    ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticklabels(class_names)

    # Tulis angka dalam sel
    max_val = cm.max() if cm.max() > 0 else 1
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm[i, j]
            text_color = "white" if value > max_val * 0.5 else "black"
            ax.text(j, i, str(value), ha="center", va="center", color=text_color, fontsize=8)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

val_cm_png_path = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.png"
test_cm_png_path = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.png"

save_confusion_matrix_plot(
    val_cm,
    class_names_ordered,
    "Validation Confusion Matrix — DenseNet201 A Official Seed 42",
    val_cm_png_path
)

save_confusion_matrix_plot(
    test_cm,
    class_names_ordered,
    "Test Confusion Matrix — DenseNet201 A Official Seed 42",
    test_cm_png_path
)

# 12. Simpan metrics summary
metrics_summary = {
    "run_name": RUN_NAME,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "best_validation_metric_name_from_training": best_metric_name,
    "best_validation_metric_from_training": float(best_metric) if best_metric is not None else None,
    "best_tie_metric_name_from_training": best_tie_metric_name,
    "best_tie_metric_from_training": float(best_tie_metric) if best_tie_metric is not None else None,
    "validation": val_summary,
    "test": test_summary,
    "benchmark_reference": {
        "accuracy": 0.931,
        "f1_score": 0.921
    },
    "test_accuracy_minus_benchmark_accuracy": float(test_summary["accuracy"] - 0.931),
    "test_macro_f1_minus_benchmark_f1": float(test_summary["macro_f1"] - 0.921),
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions_csv": str(val_pred_path),
        "test_predictions_csv": str(test_pred_path),
        "validation_classification_report_csv": str(val_report_csv_path),
        "test_classification_report_csv": str(test_report_csv_path),
        "validation_classification_report_txt": str(val_report_txt_path),
        "test_classification_report_txt": str(test_report_txt_path),
        "validation_classification_report_json": str(val_report_json_path),
        "test_classification_report_json": str(test_report_json_path),
        "validation_confusion_matrix_csv": str(val_cm_csv_path),
        "test_confusion_matrix_csv": str(test_cm_csv_path),
        "validation_confusion_matrix_png": str(val_cm_png_path),
        "test_confusion_matrix_png": str(test_cm_png_path)
    }
}

metrics_json_path = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
metrics_csv_path = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

with open(metrics_json_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)

summary_rows = []
for split_name, split_summary in [("validation", val_summary), ("test", test_summary)]:
    row = {"split": split_name}
    row.update(split_summary)
    summary_rows.append(row)

pd.DataFrame(summary_rows).to_csv(metrics_csv_path, index=False)

print("\n=== File Output Evaluasi Tersimpan ===")
print("Metrics JSON:", metrics_json_path)
print("Metrics CSV :", metrics_csv_path)
print("Validation predictions:", val_pred_path)
print("Test predictions      :", test_pred_path)
print("Validation report CSV :", val_report_csv_path)
print("Test report CSV       :", test_report_csv_path)
print("Validation CM CSV     :", val_cm_csv_path)
print("Test CM CSV           :", test_cm_csv_path)
print("Validation CM PNG     :", val_cm_png_path)
print("Test CM PNG           :", test_cm_png_path)

print("\n=== Ringkasan Sel B12 ===")
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)
print("Sel B12 selesai.")

In [ ]:
# Sel B13 — Load dan Cek Manifest Skenario B-G14 RAWSource Seed 42

import os
import json
import random
import pandas as pd
from pathlib import Path
from PIL import Image

import torch

print("=== Sel B13: Load Manifest B-G14 RAWSource ===")

# 1. Konfigurasi path
PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

MANIFEST_B_G14_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
DATASET_DIR = Path("/content/tem_virus_dataset_extracted/TEM virus dataset/context_virus_1nm_256x256")

RUN_NAME_B = "DenseNet201_B_G14_seed42"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("\n=== Runtime/GPU Status Ringan ===")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

print("\n=== Path Check ===")
print("Manifest B-G14:", MANIFEST_B_G14_PATH)
print("Manifest exists:", MANIFEST_B_G14_PATH.exists())
print("Dataset dir:", DATASET_DIR)
print("Dataset dir exists:", DATASET_DIR.exists())
print("Output dir:", OUTPUT_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Run name B:", RUN_NAME_B)

if not MANIFEST_B_G14_PATH.exists():
    raise FileNotFoundError(f"Manifest B-G14 tidak ditemukan: {MANIFEST_B_G14_PATH}")

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        "Folder dataset di /content tidak ditemukan. Runtime mungkin reset. "
        "Jangan lanjut training; perlu recovery dataset dari ZIP dulu."
    )

# 2. Load manifest B-G14
df_b = pd.read_csv(MANIFEST_B_G14_PATH)

print("\n=== Manifest B-G14 Shape ===")
print("Shape:", df_b.shape)

print("\n=== Kolom Manifest B-G14 ===")
print(list(df_b.columns))

print("\n=== Preview Manifest B-G14 ===")
display(df_b.head())

# 3. Deteksi kolom penting
required_cols = ["filepath", "filename", "class_name", "label_id", "split"]
missing_cols = [c for c in required_cols if c not in df_b.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Kolom wajib tidak ditemukan di manifest B-G14: {missing_cols}")

# 4. Pastikan tidak pakai augmented dan tidak ada _EXCLUDED
print("\n=== Cek Larangan Data ===")
contains_augmented_split = df_b["split"].astype(str).str.contains("augmented", case=False, regex=False).any()
contains_augmented_path = df_b["filepath"].astype(str).str.contains("augmented", case=False, regex=False).any()
contains_excluded = df_b["class_name"].astype(str).str.contains("_EXCLUDED", case=False, regex=False).any()

print("Ada augmented dari split:", contains_augmented_split)
print("Ada augmented dari filepath:", contains_augmented_path)
print("Ada _EXCLUDED:", contains_excluded)

if contains_augmented_split or contains_augmented_path:
    raise ValueError("Manifest B-G14 mengandung augmented_train/augmented. Stop sesuai instruksi MASTER CHAT.")

if contains_excluded:
    raise ValueError("Manifest B-G14 mengandung _EXCLUDED. Stop sesuai instruksi MASTER CHAT.")

# 5. Cek split
print("\n=== Jumlah Data per Split B-G14 ===")
split_counts_b = df_b["split"].value_counts(dropna=False).sort_index()
display(split_counts_b.rename("count").reset_index().rename(columns={"index": "split"}))

unique_splits_b = sorted(df_b["split"].dropna().astype(str).unique().tolist())
print("Split unik:", unique_splits_b)

expected_splits = ["test", "train", "validation"]
if sorted(unique_splits_b) != sorted(expected_splits):
    print("PERINGATAN: Split unik tidak persis train/validation/test. Cek output ini sebelum lanjut.")

# 6. Cek kelas dan label mapping
print("\n=== Label Mapping B-G14 ===")
label_map_b = (
    df_b[["class_name", "label_id"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

display(label_map_b)

classes_b = sorted(df_b["class_name"].dropna().unique().tolist())
label_ids_b = sorted(df_b["label_id"].dropna().astype(int).unique().tolist())

print("Jumlah kelas:", len(classes_b))
print("Classes:", classes_b)
print("Label IDs:", label_ids_b)
print("Label ID sesuai 0 sampai 13:", label_ids_b == list(range(14)))

if len(classes_b) != 14:
    raise ValueError(f"Jumlah kelas bukan 14. Jumlah kelas terdeteksi: {len(classes_b)}")

if label_ids_b != list(range(14)):
    raise ValueError(f"Label ID tidak sesuai 0 sampai 13: {label_ids_b}")

# 7. Crosstab split x class
print("\n=== Crosstab Split x Class B-G14 ===")
split_class_table_b = pd.crosstab(df_b["class_name"], df_b["split"])
display(split_class_table_b)

# 8. Cek filepath tersedia
print("\n=== Filepath Availability Check B-G14 ===")
df_b["filepath_exists"] = df_b["filepath"].apply(lambda x: Path(str(x)).exists())
missing_df_b = df_b[~df_b["filepath_exists"]].copy()

print("Total filepath:", len(df_b))
print("Filepath exists:", int(df_b["filepath_exists"].sum()))
print("Filepath missing:", len(missing_df_b))

if len(missing_df_b) > 0:
    print("\nContoh filepath missing:")
    display(missing_df_b.head(20)[["filepath", "filename", "class_name", "split"]])
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Stop dulu sebelum training.")

# 9. Cek sample gambar bisa dibuka
print("\n=== Image Open Sanity Check B-G14 ===")
sample_records = []

sample_df_list = []
for split in ["train", "validation", "test"]:
    part = df_b[df_b["split"] == split]
    n = min(5, len(part))
    if n > 0:
        sample_df_list.append(part.sample(n=n, random_state=42))

sample_df_b = pd.concat(sample_df_list, ignore_index=True)

for _, row in sample_df_b.iterrows():
    fp = Path(row["filepath"])
    rec = {
        "split": row["split"],
        "class_name": row["class_name"],
        "label_id": int(row["label_id"]),
        "filename": row["filename"],
        "filepath": str(fp),
        "can_open": False,
        "mode": None,
        "size": None,
        "error": None
    }

    try:
        with Image.open(fp) as img:
            rec["can_open"] = True
            rec["mode"] = img.mode
            rec["size"] = list(img.size)
    except Exception as e:
        rec["error"] = str(e)

    sample_records.append(rec)

sample_check_b = pd.DataFrame(sample_records)

print("Jumlah sample image dicek:", len(sample_check_b))
print("Berhasil dibuka:", int(sample_check_b["can_open"].sum()))
print("Gagal dibuka:", int((~sample_check_b["can_open"]).sum()))
display(sample_check_b[["split", "class_name", "filename", "can_open", "mode", "size", "error"]])

if int((~sample_check_b["can_open"]).sum()) > 0:
    raise RuntimeError("Ada sample gambar yang gagal dibuka. Stop dulu sebelum training.")

# 10. Simpan ringkasan B13
b13_summary = {
    "run_name": RUN_NAME_B,
    "scenario": "B-G14 RAWSource",
    "evaluation_name": "RAW source-aware leakage-controlled evaluation",
    "manifest_path": str(MANIFEST_B_G14_PATH),
    "dataset_dir": str(DATASET_DIR),
    "shape": list(df_b.shape),
    "columns": list(df_b.columns),
    "split_counts": split_counts_b.to_dict(),
    "unique_splits": unique_splits_b,
    "num_classes": len(classes_b),
    "classes": classes_b,
    "label_ids": label_ids_b,
    "label_id_expected_0_to_13": label_ids_b == list(range(14)),
    "contains_augmented_split": bool(contains_augmented_split),
    "contains_augmented_path": bool(contains_augmented_path),
    "contains_excluded": bool(contains_excluded),
    "total_filepaths": int(len(df_b)),
    "filepath_exists": int(df_b["filepath_exists"].sum()),
    "filepath_missing": int(len(missing_df_b)),
    "sample_image_checked": int(len(sample_check_b)),
    "sample_image_open_success": int(sample_check_b["can_open"].sum()),
    "sample_image_open_failed": int((~sample_check_b["can_open"]).sum())
}

b13_summary_path = OUTPUT_DIR / f"{RUN_NAME_B}_b13_manifest_check_summary.json"
with open(b13_summary_path, "w") as f:
    json.dump(b13_summary, f, indent=2)

sample_check_path = OUTPUT_DIR / f"{RUN_NAME_B}_b13_image_open_sample_check.csv"
sample_check_b.to_csv(sample_check_path, index=False)

print("\n=== File Tersimpan ===")
print("B13 summary:", b13_summary_path)
print("Sample image check:", sample_check_path)

print("\n=== Ringkasan Sel B13 ===")
print("Run name:", RUN_NAME_B)
print("Scenario: B-G14 RAWSource")
print("Jumlah data:", len(df_b))
print("Jumlah kelas:", len(classes_b))
print("Split counts:", split_counts_b.to_dict())
print("Filepath missing:", len(missing_df_b))
print("Sample image open failed:", int((~sample_check_b["can_open"]).sum()))
print("Sel B13 selesai.")

In [ ]:
# Sel B14 — Setup Dataset dan DataLoader Skenario B-G14 RAWSource Seed 42

import os
import gc
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("=== Sel B14: Setup Dataset dan DataLoader B-G14 RAWSource ===")

# 1. Bersihkan model/optimizer lama dari Skenario A agar tidak tercampur
print("\n=== Cleanup Model A dari Memory ===")
vars_to_delete = ["model", "optimizer", "scheduler", "scaler"]

for v in vars_to_delete:
    if v in globals():
        del globals()[v]
        print(f"Deleted variable: {v}")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")
    print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

# 2. Konfigurasi dasar B-G14
SEED_B = 42
NUM_CLASSES_B = 14
IMG_SIZE_B = 224
BATCH_SIZE_B = 16
NUM_WORKERS_B = 2

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

MANIFEST_B_G14_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
RUN_NAME_B = "DenseNet201_B_G14_seed42"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED_B)

print("\n=== Config B-G14 ===")
print("Run name:", RUN_NAME_B)
print("Seed:", SEED_B)
print("Num classes:", NUM_CLASSES_B)
print("Image size:", IMG_SIZE_B)
print("Batch size:", BATCH_SIZE_B)
print("Num workers:", NUM_WORKERS_B)
print("Manifest:", MANIFEST_B_G14_PATH)

# 3. Load manifest B-G14
df_b = pd.read_csv(MANIFEST_B_G14_PATH)

# Safety filter: hanya split utama, tanpa augmented dan tanpa _EXCLUDED
df_b = df_b[df_b["split"].isin(["train", "validation", "test"])].copy()
df_b = df_b[df_b["class_name"] != "_EXCLUDED"].copy()
df_b = df_b[~df_b["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

print("\n=== Manifest B-G14 Loaded ===")
print("Shape setelah safety filter:", df_b.shape)

# 4. Cek group_id overlap antar split
print("\n=== Group ID Overlap Check B-G14 ===")

if "group_id" not in df_b.columns:
    raise ValueError("Kolom group_id tidak ditemukan. Tidak aman lanjut untuk B-G14.")

group_sets = {
    split: set(df_b[df_b["split"] == split]["group_id"].astype(str).unique())
    for split in ["train", "validation", "test"]
}

overlap_train_val = group_sets["train"] & group_sets["validation"]
overlap_train_test = group_sets["train"] & group_sets["test"]
overlap_val_test = group_sets["validation"] & group_sets["test"]

print("Unique group_id train:", len(group_sets["train"]))
print("Unique group_id validation:", len(group_sets["validation"]))
print("Unique group_id test:", len(group_sets["test"]))
print("Overlap train-validation:", len(overlap_train_val))
print("Overlap train-test:", len(overlap_train_test))
print("Overlap validation-test:", len(overlap_val_test))

if len(overlap_train_val) > 0 or len(overlap_train_test) > 0 or len(overlap_val_test) > 0:
    print("\nContoh overlap group_id:")
    print("train-validation:", list(overlap_train_val)[:10])
    print("train-test:", list(overlap_train_test)[:10])
    print("validation-test:", list(overlap_val_test)[:10])
    raise ValueError("Ada overlap group_id antar split. Stop dulu sebelum training B-G14.")

# 5. Cek raw_source_id overlap juga, jika tersedia
print("\n=== RAW Source ID Overlap Check B-G14 ===")

if "raw_source_id" in df_b.columns:
    raw_sets = {
        split: set(df_b[df_b["split"] == split]["raw_source_id"].astype(str).unique())
        for split in ["train", "validation", "test"]
    }

    raw_overlap_train_val = raw_sets["train"] & raw_sets["validation"]
    raw_overlap_train_test = raw_sets["train"] & raw_sets["test"]
    raw_overlap_val_test = raw_sets["validation"] & raw_sets["test"]

    print("Unique raw_source_id train:", len(raw_sets["train"]))
    print("Unique raw_source_id validation:", len(raw_sets["validation"]))
    print("Unique raw_source_id test:", len(raw_sets["test"]))
    print("Overlap raw train-validation:", len(raw_overlap_train_val))
    print("Overlap raw train-test:", len(raw_overlap_train_test))
    print("Overlap raw validation-test:", len(raw_overlap_val_test))

    if len(raw_overlap_train_val) > 0 or len(raw_overlap_train_test) > 0 or len(raw_overlap_val_test) > 0:
        raise ValueError("Ada overlap raw_source_id antar split. Stop dulu sebelum training B-G14.")
else:
    print("Kolom raw_source_id tidak tersedia, skip raw_source_id overlap check.")

# 6. Pisahkan split
train_df_b = df_b[df_b["split"] == "train"].reset_index(drop=True)
val_df_b = df_b[df_b["split"] == "validation"].reset_index(drop=True)
test_df_b = df_b[df_b["split"] == "test"].reset_index(drop=True)

print("\n=== Split Size B-G14 ===")
print("Train:", train_df_b.shape)
print("Validation:", val_df_b.shape)
print("Test:", test_df_b.shape)

# 7. Label mapping B-G14
label_mapping_b = (
    df_b[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class_b = dict(zip(label_mapping_b["label_id"].astype(int), label_mapping_b["class_name"]))
class_to_id_b = {v: k for k, v in id_to_class_b.items()}

print("\n=== Label Mapping B-G14 ===")
display(label_mapping_b)
print("NUM_CLASSES valid:", len(id_to_class_b) == NUM_CLASSES_B)

# 8. Transform sama seperti Skenario A agar perbandingan fair
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform_b = transforms.Compose([
    transforms.Resize((IMG_SIZE_B, IMG_SIZE_B)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform_b = transforms.Compose([
    transforms.Resize((IMG_SIZE_B, IMG_SIZE_B)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print("\n=== Transform B-G14 ===")
print("Train transform:", train_transform_b)
print("Validation/test transform:", eval_transform_b)

# 9. Dataset class
class TEMVirusManifestDatasetB(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = row["filepath"]
        label = int(row["label_id"])

        # Gambar TEM asli grayscale, dikonversi ke RGB 3-channel untuk DenseNet ImageNet
        image = Image.open(filepath).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "filepath": filepath,
            "class_name": row["class_name"],
            "filename": row["filename"],
            "group_id": row["group_id"],
            "raw_source_id": row["raw_source_id"] if "raw_source_id" in row.index else None
        }

# 10. Buat dataset
train_dataset_b = TEMVirusManifestDatasetB(train_df_b, transform=train_transform_b)
val_dataset_b = TEMVirusManifestDatasetB(val_df_b, transform=eval_transform_b)
test_dataset_b = TEMVirusManifestDatasetB(test_df_b, transform=eval_transform_b)

print("\n=== Dataset Size B-G14 ===")
print("train_dataset_b:", len(train_dataset_b))
print("val_dataset_b:", len(val_dataset_b))
print("test_dataset_b:", len(test_dataset_b))

# 11. Buat DataLoader
pin_memory_b = torch.cuda.is_available()

train_loader_b = DataLoader(
    train_dataset_b,
    batch_size=BATCH_SIZE_B,
    shuffle=True,
    num_workers=NUM_WORKERS_B,
    pin_memory=pin_memory_b
)

val_loader_b = DataLoader(
    val_dataset_b,
    batch_size=BATCH_SIZE_B,
    shuffle=False,
    num_workers=NUM_WORKERS_B,
    pin_memory=pin_memory_b
)

test_loader_b = DataLoader(
    test_dataset_b,
    batch_size=BATCH_SIZE_B,
    shuffle=False,
    num_workers=NUM_WORKERS_B,
    pin_memory=pin_memory_b
)

print("\n=== DataLoader Setup B-G14 ===")
print("Batch size:", BATCH_SIZE_B)
print("Num workers:", NUM_WORKERS_B)
print("Pin memory:", pin_memory_b)
print("Train batches:", len(train_loader_b))
print("Val batches:", len(val_loader_b))
print("Test batches:", len(test_loader_b))

# 12. One batch sanity check
print("\n=== One Batch Sanity Check B-G14 ===")
batch_b = next(iter(train_loader_b))

images_b = batch_b["image"]
labels_b = batch_b["label"]

print("Image batch shape:", images_b.shape)
print("Label batch shape:", labels_b.shape)
print("Image dtype:", images_b.dtype)
print("Label dtype:", labels_b.dtype)
print("Image min:", float(images_b.min()))
print("Image max:", float(images_b.max()))
print("Label sample:", labels_b[:16].tolist())
print("Class sample:", batch_b["class_name"][:5])
print("Group ID sample:", batch_b["group_id"][:3])

# 13. Simpan ringkasan B14
b14_summary = {
    "run_name": RUN_NAME_B,
    "scenario": "B-G14 RAWSource",
    "evaluation_name": "RAW source-aware leakage-controlled evaluation",
    "seed": SEED_B,
    "num_classes": NUM_CLASSES_B,
    "img_size": IMG_SIZE_B,
    "batch_size": BATCH_SIZE_B,
    "num_workers": NUM_WORKERS_B,
    "pin_memory": pin_memory_b,
    "train_size": len(train_dataset_b),
    "val_size": len(val_dataset_b),
    "test_size": len(test_dataset_b),
    "train_batches": len(train_loader_b),
    "val_batches": len(val_loader_b),
    "test_batches": len(test_loader_b),
    "unique_group_id_train": len(group_sets["train"]),
    "unique_group_id_validation": len(group_sets["validation"]),
    "unique_group_id_test": len(group_sets["test"]),
    "overlap_group_train_validation": len(overlap_train_val),
    "overlap_group_train_test": len(overlap_train_test),
    "overlap_group_validation_test": len(overlap_val_test),
    "image_batch_shape": list(images_b.shape),
    "label_batch_shape": list(labels_b.shape),
    "id_to_class": {str(k): v for k, v in id_to_class_b.items()},
    "train_transform": str(train_transform_b),
    "eval_transform": str(eval_transform_b)
}

if "raw_source_id" in df_b.columns:
    b14_summary.update({
        "unique_raw_source_id_train": len(raw_sets["train"]),
        "unique_raw_source_id_validation": len(raw_sets["validation"]),
        "unique_raw_source_id_test": len(raw_sets["test"]),
        "overlap_raw_train_validation": len(raw_overlap_train_val),
        "overlap_raw_train_test": len(raw_overlap_train_test),
        "overlap_raw_validation_test": len(raw_overlap_val_test)
    })

b14_summary_path = OUTPUT_DIR / f"{RUN_NAME_B}_b14_dataloader_setup_summary.json"
with open(b14_summary_path, "w") as f:
    json.dump(b14_summary, f, indent=2)

print("\nRingkasan Sel B14 disimpan ke:")
print(b14_summary_path)

print("\n=== Ringkasan Sel B14 ===")
print("B-G14 DataLoader siap.")
print("Train/Val/Test:", len(train_dataset_b), len(val_dataset_b), len(test_dataset_b))
print("Group overlap train-val/test/val-test:", len(overlap_train_val), len(overlap_train_test), len(overlap_val_test))
print("Image batch shape:", list(images_b.shape))
print("Sel B14 selesai.")

In [ ]:
# Sel B15 — Setup DenseNet201 Baru untuk Skenario B-G14 Seed 42

import gc
import json
import time
import traceback
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("=== Sel B15: Setup DenseNet201 Baru untuk B-G14 RAWSource ===")

# 1. Bersihkan tensor/variabel besar yang tidak dibutuhkan
print("\n=== Cleanup Ringan Sebelum Model B-G14 ===")

cleanup_vars = [
    "batch", "images", "labels", "outputs",
    "batch_b", "images_b", "labels_b",
    "val_pred", "test_pred"
]

for v in cleanup_vars:
    if v in globals():
        del globals()[v]
        print(f"Deleted variable: {v}")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")
    print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

# 2. Konfigurasi B-G14
SEED_B = 42
NUM_CLASSES_B = 14
MAX_EPOCHS_B = 20
PATIENCE_B = 5
LEARNING_RATE_B = 1e-4
WEIGHT_DECAY_B = 1e-4

MODEL_SELECTION_METRIC_B = "val_macro_f1"
TIE_BREAKER_METRIC_B = "val_accuracy"

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

RUN_NAME_B = "DenseNet201_B_G14_seed42"

BEST_CKPT_PATH_B = CHECKPOINT_DIR / f"{RUN_NAME_B}_best.pt"
LAST_CKPT_PATH_B = CHECKPOINT_DIR / f"{RUN_NAME_B}_last.pt"
HISTORY_CSV_PATH_B = OUTPUT_DIR / f"{RUN_NAME_B}_training_history.csv"
HISTORY_JSON_PATH_B = OUTPUT_DIR / f"{RUN_NAME_B}_training_history.json"
FAILED_RUN_LOG_PATH = OUTPUT_DIR / "failed_runs_log.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device_b = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n=== Config Training B-G14 ===")
print("Run name:", RUN_NAME_B)
print("Device:", device_b)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Max epochs:", MAX_EPOCHS_B)
print("Patience:", PATIENCE_B)
print("Learning rate:", LEARNING_RATE_B)
print("Weight decay:", WEIGHT_DECAY_B)
print("Model selection metric:", MODEL_SELECTION_METRIC_B)
print("Tie breaker metric:", TIE_BREAKER_METRIC_B)
print("Best checkpoint:", BEST_CKPT_PATH_B)
print("Last checkpoint:", LAST_CKPT_PATH_B)

# 3. Pastikan DataLoader B-G14 tersedia
required_b14_vars = ["train_loader_b", "val_loader_b", "test_loader_b", "id_to_class_b"]
missing_b14_vars = [v for v in required_b14_vars if v not in globals()]
if len(missing_b14_vars) > 0:
    raise RuntimeError(f"Variabel dari Sel B14 belum tersedia: {missing_b14_vars}")

# 4. Load DenseNet201 pretrained ImageNet baru, bukan checkpoint A
print("\n=== Loading Fresh DenseNet201 ImageNet for B-G14 ===")
print("Catatan: ini model baru dari ImageNet, bukan model/checkpoint Skenario A.")

try:
    weights_b = models.DenseNet201_Weights.IMAGENET1K_V1
    model_b = models.densenet201(weights=weights_b)
    weights_name_b = "DenseNet201_Weights.IMAGENET1K_V1"
except Exception as e:
    print("Gagal load dengan weights API baru. Error:", e)
    print("Mencoba fallback pretrained=True...")
    model_b = models.densenet201(pretrained=True)
    weights_name_b = "pretrained=True fallback"

old_classifier_b = model_b.classifier
old_in_features_b = model_b.classifier.in_features

model_b.classifier = nn.Linear(old_in_features_b, NUM_CLASSES_B)
model_b = model_b.to(device_b)

print("\n=== Classifier Check B-G14 ===")
print("Weights:", weights_name_b)
print("Old classifier:", old_classifier_b)
print("New classifier:", model_b.classifier)

total_params_b = sum(p.numel() for p in model_b.parameters())
trainable_params_b = sum(p.numel() for p in model_b.parameters() if p.requires_grad)

print("\n=== Parameter Count B-G14 ===")
print("Total parameters:", f"{total_params_b:,}")
print("Trainable parameters:", f"{trainable_params_b:,}")

# 5. Setup loss, optimizer, scheduler
criterion_b = nn.CrossEntropyLoss()

optimizer_b = torch.optim.AdamW(
    model_b.parameters(),
    lr=LEARNING_RATE_B,
    weight_decay=WEIGHT_DECAY_B
)

scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b,
    mode="max",
    factor=0.5,
    patience=2
)

# 6. Mixed precision
use_amp_b = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler_b = GradScaler("cuda", enabled=use_amp_b)
    amp_api_b = "torch.amp"
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler_b = GradScaler(enabled=use_amp_b)
    amp_api_b = "torch.cuda.amp"

print("\n=== AMP Setup B-G14 ===")
print("Use AMP:", use_amp_b)
print("AMP API:", amp_api_b)

# 7. Fungsi metrik B-G14
def compute_classification_metrics_b(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

# 8. Fungsi train 1 epoch B-G14
def train_one_epoch_b(model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    start_time = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp and device.type == "cuda":
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        preds = torch.argmax(outputs.detach(), dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            elapsed = time.time() - start_time
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={elapsed/60:.2f} min"
            )

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_classification_metrics_b(all_labels, all_preds)
    metrics["loss"] = float(epoch_loss)

    return metrics

# 9. Fungsi evaluasi B-G14
@torch.no_grad()
def evaluate_model_b(model, loader, criterion, device, split_name="val"):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp_b and device.type == "cuda":
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_classification_metrics_b(all_labels, all_preds)
    metrics["loss"] = float(epoch_loss)

    prefixed = {}
    for k, v in metrics.items():
        prefixed[f"{split_name}_{k}"] = v

    return prefixed

# 10. Failed run logger B-G14
def log_failed_run_b(run_name, error_message, traceback_text):
    record = {
        "timestamp": datetime.now().isoformat(),
        "run_name": run_name,
        "error_message": str(error_message),
        "traceback": str(traceback_text)
    }

    if FAILED_RUN_LOG_PATH.exists():
        old_df = pd.read_csv(FAILED_RUN_LOG_PATH)
        new_df = pd.concat([old_df, pd.DataFrame([record])], ignore_index=True)
    else:
        new_df = pd.DataFrame([record])

    new_df.to_csv(FAILED_RUN_LOG_PATH, index=False)
    print("Failed run logged to:", FAILED_RUN_LOG_PATH)

# 11. Forward pass sanity check
print("\n=== Forward Pass Sanity Check B-G14 ===")
model_b.eval()

batch_check_b = next(iter(train_loader_b))
images_check_b = batch_check_b["image"].to(device_b, non_blocking=True)
labels_check_b = batch_check_b["label"].to(device_b, non_blocking=True)

with torch.no_grad():
    outputs_check_b = model_b(images_check_b)

print("Input image shape:", tuple(images_check_b.shape))
print("Output logits shape:", tuple(outputs_check_b.shape))
print("Label shape:", tuple(labels_check_b.shape))
print("Output dtype:", outputs_check_b.dtype)
print("Output min:", float(outputs_check_b.min()))
print("Output max:", float(outputs_check_b.max()))

assert outputs_check_b.shape[0] == images_check_b.shape[0], "Batch output tidak sesuai batch input."
assert outputs_check_b.shape[1] == NUM_CLASSES_B, "Jumlah output class tidak sesuai 14."

print("Forward pass OK: output shape sesuai [batch_size, 14].")

# 12. Simpan ringkasan B15
b15_summary = {
    "run_name": RUN_NAME_B,
    "scenario": "B-G14 RAWSource",
    "evaluation_name": "RAW source-aware leakage-controlled evaluation",
    "model_name": "DenseNet201",
    "weights": weights_name_b,
    "fresh_model_from_imagenet": True,
    "not_loaded_from_scenario_A": True,
    "num_classes": NUM_CLASSES_B,
    "old_classifier": str(old_classifier_b),
    "old_in_features": int(old_in_features_b),
    "new_classifier": str(model_b.classifier),
    "total_params": int(total_params_b),
    "trainable_params": int(trainable_params_b),
    "device": str(device_b),
    "max_epochs": MAX_EPOCHS_B,
    "patience": PATIENCE_B,
    "learning_rate": LEARNING_RATE_B,
    "weight_decay": WEIGHT_DECAY_B,
    "criterion": str(criterion_b),
    "optimizer": str(optimizer_b),
    "scheduler": str(scheduler_b),
    "use_amp": bool(use_amp_b),
    "amp_api": amp_api_b,
    "model_selection_metric": MODEL_SELECTION_METRIC_B,
    "tie_breaker_metric": TIE_BREAKER_METRIC_B,
    "best_checkpoint_path": str(BEST_CKPT_PATH_B),
    "last_checkpoint_path": str(LAST_CKPT_PATH_B),
    "history_csv_path": str(HISTORY_CSV_PATH_B),
    "history_json_path": str(HISTORY_JSON_PATH_B),
    "forward_pass_input_shape": list(images_check_b.shape),
    "forward_pass_output_shape": list(outputs_check_b.shape),
    "forward_pass_ok": True
}

b15_summary_path = OUTPUT_DIR / f"{RUN_NAME_B}_b15_model_training_setup_summary.json"
with open(b15_summary_path, "w") as f:
    json.dump(b15_summary, f, indent=2)

print("\nRingkasan Sel B15 disimpan ke:")
print(b15_summary_path)

if torch.cuda.is_available():
    print("\n=== CUDA Memory After B15 ===")
    print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

print("\n=== Ringkasan Sel B15 ===")
print("DenseNet201 B-G14 siap.")
print("Fresh ImageNet model:", True)
print("Forward pass OK:", True)
print("Belum ada training dijalankan di sel ini.")

In [ ]:
# Sel B16 — Training DenseNet201 Skenario B-G14 RAWSource Seed 42

import os
import json
import time
import traceback
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch

print("=== Sel B16: Training DenseNet201 B-G14 RAWSource Seed 42 ===")

# 1. Pastikan variabel dari Sel B14-B15 tersedia
required_vars_b = [
    "model_b", "train_loader_b", "val_loader_b",
    "criterion_b", "optimizer_b", "scheduler_b", "scaler_b",
    "device_b", "train_one_epoch_b", "evaluate_model_b",
    "RUN_NAME_B", "BEST_CKPT_PATH_B", "LAST_CKPT_PATH_B",
    "HISTORY_CSV_PATH_B", "HISTORY_JSON_PATH_B",
    "MAX_EPOCHS_B", "PATIENCE_B",
    "MODEL_SELECTION_METRIC_B", "TIE_BREAKER_METRIC_B",
    "use_amp_b", "id_to_class_b", "log_failed_run_b"
]

missing_vars_b = [v for v in required_vars_b if v not in globals()]
if len(missing_vars_b) > 0:
    raise RuntimeError(f"Variabel belum tersedia dari sel sebelumnya: {missing_vars_b}")

print("\n=== Training Start Info B-G14 ===")
print("Run name:", RUN_NAME_B)
print("Scenario: B-G14 RAWSource")
print("Evaluation name: RAW source-aware leakage-controlled evaluation")
print("Device:", device_b)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Max epochs:", MAX_EPOCHS_B)
print("Patience:", PATIENCE_B)
print("Selection metric:", MODEL_SELECTION_METRIC_B)
print("Tie breaker:", TIE_BREAKER_METRIC_B)
print("Best checkpoint path:", BEST_CKPT_PATH_B)
print("Last checkpoint path:", LAST_CKPT_PATH_B)

if torch.cuda.is_available():
    print("CUDA memory allocated before training:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
    print("CUDA memory reserved before training:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

# 2. Resume logic jika runtime sempat putus dan last checkpoint sudah ada
RESUME_IF_LAST_EXISTS_B = True

start_epoch_b = 1
history_b = []
best_metric_b = -1.0
best_tie_metric_b = -1.0
best_epoch_b = None
epochs_without_improvement_b = 0

if RESUME_IF_LAST_EXISTS_B and Path(LAST_CKPT_PATH_B).exists():
    print("\n=== Resume Checkpoint B-G14 Ditemukan ===")
    print("Loading last checkpoint:", LAST_CKPT_PATH_B)

    try:
        checkpoint_b = torch.load(LAST_CKPT_PATH_B, map_location=device_b, weights_only=False)
    except TypeError:
        checkpoint_b = torch.load(LAST_CKPT_PATH_B, map_location=device_b)

    model_b.load_state_dict(checkpoint_b["model_state_dict"])
    optimizer_b.load_state_dict(checkpoint_b["optimizer_state_dict"])

    if "scheduler_state_dict" in checkpoint_b and checkpoint_b["scheduler_state_dict"] is not None:
        scheduler_b.load_state_dict(checkpoint_b["scheduler_state_dict"])

    if "scaler_state_dict" in checkpoint_b and checkpoint_b["scaler_state_dict"] is not None:
        scaler_b.load_state_dict(checkpoint_b["scaler_state_dict"])

    start_epoch_b = int(checkpoint_b["epoch"]) + 1
    history_b = checkpoint_b.get("history", [])
    best_metric_b = float(checkpoint_b.get("best_metric", -1.0))
    best_tie_metric_b = float(checkpoint_b.get("best_tie_metric", -1.0))
    best_epoch_b = checkpoint_b.get("best_epoch", None)
    epochs_without_improvement_b = int(checkpoint_b.get("epochs_without_improvement", 0))

    print("Resume dari epoch:", start_epoch_b)
    print("Best metric sebelumnya:", best_metric_b)
    print("Best tie metric sebelumnya:", best_tie_metric_b)
    print("Best epoch sebelumnya:", best_epoch_b)
    print("Epoch tanpa improvement sebelumnya:", epochs_without_improvement_b)
else:
    print("\nTidak ada last checkpoint B-G14. Training mulai dari awal.")

# 3. Training loop
training_start_time_b = time.time()

try:
    for epoch_b in range(start_epoch_b, MAX_EPOCHS_B + 1):
        epoch_start_time_b = time.time()

        current_lr_b = optimizer_b.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch_b}/{MAX_EPOCHS_B} — B-G14")
        print(f"Current LR: {current_lr_b:.8f}")
        print("=" * 80)

        # Train
        train_metrics_b = train_one_epoch_b(
            model=model_b,
            loader=train_loader_b,
            criterion=criterion_b,
            optimizer=optimizer_b,
            scaler=scaler_b,
            device=device_b,
            use_amp=use_amp_b
        )

        # Validation
        val_metrics_b = evaluate_model_b(
            model=model_b,
            loader=val_loader_b,
            criterion=criterion_b,
            device=device_b,
            split_name="val"
        )

        # Scheduler berdasarkan validation macro F1
        scheduler_b.step(val_metrics_b["val_macro_f1"])

        epoch_time_min_b = (time.time() - epoch_start_time_b) / 60

        # Record epoch
        epoch_record_b = {
            "epoch": epoch_b,
            "lr": current_lr_b,
            "epoch_time_min": float(epoch_time_min_b),

            "train_loss": train_metrics_b["loss"],
            "train_accuracy": train_metrics_b["accuracy"],
            "train_macro_precision": train_metrics_b["macro_precision"],
            "train_macro_recall": train_metrics_b["macro_recall"],
            "train_macro_f1": train_metrics_b["macro_f1"],
            "train_weighted_f1": train_metrics_b["weighted_f1"],

            **val_metrics_b
        }

        history_b.append(epoch_record_b)

        current_metric_b = float(epoch_record_b[MODEL_SELECTION_METRIC_B])
        current_tie_metric_b = float(epoch_record_b[TIE_BREAKER_METRIC_B])

        improved_b = (
            (current_metric_b > best_metric_b + 1e-8) or
            (abs(current_metric_b - best_metric_b) <= 1e-8 and current_tie_metric_b > best_tie_metric_b)
        )

        if improved_b:
            best_metric_b = current_metric_b
            best_tie_metric_b = current_tie_metric_b
            best_epoch_b = epoch_b
            epochs_without_improvement_b = 0

            best_checkpoint_b = {
                "run_name": RUN_NAME_B,
                "scenario": "B-G14 RAWSource",
                "evaluation_name": "RAW source-aware leakage-controlled evaluation",
                "epoch": epoch_b,
                "best_epoch": best_epoch_b,
                "best_metric_name": MODEL_SELECTION_METRIC_B,
                "best_metric": best_metric_b,
                "best_tie_metric_name": TIE_BREAKER_METRIC_B,
                "best_tie_metric": best_tie_metric_b,
                "model_state_dict": model_b.state_dict(),
                "optimizer_state_dict": optimizer_b.state_dict(),
                "scheduler_state_dict": scheduler_b.state_dict() if scheduler_b is not None else None,
                "scaler_state_dict": scaler_b.state_dict() if scaler_b is not None else None,
                "history": history_b,
                "id_to_class": {str(k): v for k, v in id_to_class_b.items()},
                "saved_at": datetime.now().isoformat()
            }

            torch.save(best_checkpoint_b, BEST_CKPT_PATH_B)

            print(f"\n-> Best B-G14 model updated at epoch {epoch_b}")
            print(f"   {MODEL_SELECTION_METRIC_B}: {best_metric_b:.4f}")
            print(f"   {TIE_BREAKER_METRIC_B}: {best_tie_metric_b:.4f}")
            print(f"   Saved to: {BEST_CKPT_PATH_B}")
        else:
            epochs_without_improvement_b += 1
            print(f"\n-> No improvement ({epochs_without_improvement_b}/{PATIENCE_B})")
            print(f"   Current {MODEL_SELECTION_METRIC_B}: {current_metric_b:.4f}")
            print(f"   Best {MODEL_SELECTION_METRIC_B}: {best_metric_b:.4f} at epoch {best_epoch_b}")

        # Simpan last checkpoint tiap epoch
        last_checkpoint_b = {
            "run_name": RUN_NAME_B,
            "scenario": "B-G14 RAWSource",
            "evaluation_name": "RAW source-aware leakage-controlled evaluation",
            "epoch": epoch_b,
            "best_epoch": best_epoch_b,
            "best_metric_name": MODEL_SELECTION_METRIC_B,
            "best_metric": best_metric_b,
            "best_tie_metric_name": TIE_BREAKER_METRIC_B,
            "best_tie_metric": best_tie_metric_b,
            "epochs_without_improvement": epochs_without_improvement_b,
            "model_state_dict": model_b.state_dict(),
            "optimizer_state_dict": optimizer_b.state_dict(),
            "scheduler_state_dict": scheduler_b.state_dict() if scheduler_b is not None else None,
            "scaler_state_dict": scaler_b.state_dict() if scaler_b is not None else None,
            "history": history_b,
            "id_to_class": {str(k): v for k, v in id_to_class_b.items()},
            "saved_at": datetime.now().isoformat()
        }

        torch.save(last_checkpoint_b, LAST_CKPT_PATH_B)

        # Simpan history tiap epoch
        history_df_b = pd.DataFrame(history_b)
        history_df_b.to_csv(HISTORY_CSV_PATH_B, index=False)

        with open(HISTORY_JSON_PATH_B, "w") as f:
            json.dump(history_b, f, indent=2)

        # Ringkasan epoch
        print("\n=== Epoch Summary B-G14 ===")
        print(f"Epoch time: {epoch_time_min_b:.2f} min")
        print(f"Train loss: {epoch_record_b['train_loss']:.4f}")
        print(f"Train acc : {epoch_record_b['train_accuracy']:.4f}")
        print(f"Train macro F1: {epoch_record_b['train_macro_f1']:.4f}")
        print(f"Val loss  : {epoch_record_b['val_loss']:.4f}")
        print(f"Val acc   : {epoch_record_b['val_accuracy']:.4f}")
        print(f"Val macro precision: {epoch_record_b['val_macro_precision']:.4f}")
        print(f"Val macro recall   : {epoch_record_b['val_macro_recall']:.4f}")
        print(f"Val macro F1       : {epoch_record_b['val_macro_f1']:.4f}")
        print(f"Val weighted F1    : {epoch_record_b['val_weighted_f1']:.4f}")
        print(f"Best epoch so far  : {best_epoch_b}")
        print(f"History saved to   : {HISTORY_CSV_PATH_B}")
        print(f"Last checkpoint    : {LAST_CKPT_PATH_B}")

        if torch.cuda.is_available():
            print("CUDA memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**2, 2), "MB")
            print("CUDA memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**2, 2), "MB")

        # Early stopping
        if epochs_without_improvement_b >= PATIENCE_B:
            print("\n=== Early Stopping Triggered B-G14 ===")
            print(f"Tidak ada improvement selama {PATIENCE_B} epoch.")
            print(f"Best epoch: {best_epoch_b}")
            print(f"Best {MODEL_SELECTION_METRIC_B}: {best_metric_b:.4f}")
            break

    total_training_time_min_b = (time.time() - training_start_time_b) / 60

    # Simpan final training summary
    training_summary_b = {
        "run_name": RUN_NAME_B,
        "scenario": "B-G14 RAWSource",
        "evaluation_name": "RAW source-aware leakage-controlled evaluation",
        "status": "completed",
        "max_epochs": MAX_EPOCHS_B,
        "epochs_completed": len(history_b),
        "best_epoch": best_epoch_b,
        "best_metric_name": MODEL_SELECTION_METRIC_B,
        "best_metric": best_metric_b,
        "best_tie_metric_name": TIE_BREAKER_METRIC_B,
        "best_tie_metric": best_tie_metric_b,
        "total_training_time_min": float(total_training_time_min_b),
        "best_checkpoint_path": str(BEST_CKPT_PATH_B),
        "last_checkpoint_path": str(LAST_CKPT_PATH_B),
        "history_csv_path": str(HISTORY_CSV_PATH_B),
        "history_json_path": str(HISTORY_JSON_PATH_B),
        "completed_at": datetime.now().isoformat()
    }

    training_summary_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_training_summary.json"
    with open(training_summary_path_b, "w") as f:
        json.dump(training_summary_b, f, indent=2)

    print("\n" + "=" * 80)
    print("=== Training B-G14 Completed ===")
    print("=" * 80)
    print("Status: completed")
    print("Epochs completed:", len(history_b))
    print("Best epoch:", best_epoch_b)
    print(f"Best {MODEL_SELECTION_METRIC_B}: {best_metric_b:.4f}")
    print(f"Best {TIE_BREAKER_METRIC_B}: {best_tie_metric_b:.4f}")
    print(f"Total training time: {total_training_time_min_b:.2f} min")
    print("Training summary:", training_summary_path_b)
    print("Best checkpoint:", BEST_CKPT_PATH_B)
    print("Last checkpoint:", LAST_CKPT_PATH_B)
    print("History CSV:", HISTORY_CSV_PATH_B)

except Exception as e:
    tb = traceback.format_exc()
    print("\n=== TRAINING B-G14 FAILED ===")
    print("Error:", e)
    print(tb)
    log_failed_run_b(RUN_NAME_B, str(e), tb)
    raise

In [ ]:
# Sel B17 — Evaluasi Best Checkpoint DenseNet201 B-G14 pada Validation dan Test Set

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B17: Evaluasi Best Checkpoint DenseNet201 B-G14 ===")

# 1. Path dan konfigurasi
PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

RUN_NAME_B = "DenseNet201_B_G14_seed42"
BEST_CKPT_PATH_B = CHECKPOINT_DIR / f"{RUN_NAME_B}_best.pt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device_b = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n=== Path Check ===")
print("Best checkpoint B-G14:", BEST_CKPT_PATH_B)
print("Best checkpoint exists:", BEST_CKPT_PATH_B.exists())
print("Device:", device_b)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if not BEST_CKPT_PATH_B.exists():
    raise FileNotFoundError(f"Best checkpoint B-G14 tidak ditemukan: {BEST_CKPT_PATH_B}")

required_vars_b17 = ["model_b", "val_loader_b", "test_loader_b", "criterion_b"]
missing_vars_b17 = [v for v in required_vars_b17 if v not in globals()]
if len(missing_vars_b17) > 0:
    raise RuntimeError(
        f"Variabel dari sel sebelumnya belum tersedia: {missing_vars_b17}. "
        "Jangan lanjut. Jika runtime reset, kita perlu recovery dataset dan setup B-G14 lagi."
    )

# 2. Load best checkpoint B-G14
print("\n=== Load Best Checkpoint B-G14 ===")
try:
    checkpoint_b = torch.load(BEST_CKPT_PATH_B, map_location=device_b, weights_only=False)
except TypeError:
    checkpoint_b = torch.load(BEST_CKPT_PATH_B, map_location=device_b)

model_b.load_state_dict(checkpoint_b["model_state_dict"])
model_b = model_b.to(device_b)
model_b.eval()

best_epoch_b = checkpoint_b.get("best_epoch", checkpoint_b.get("epoch", None))
best_metric_name_b = checkpoint_b.get("best_metric_name", "val_macro_f1")
best_metric_b = checkpoint_b.get("best_metric", None)
best_tie_metric_name_b = checkpoint_b.get("best_tie_metric_name", "val_accuracy")
best_tie_metric_b = checkpoint_b.get("best_tie_metric", None)

print("Loaded checkpoint epoch:", checkpoint_b.get("epoch", None))
print("Best epoch:", best_epoch_b)
print("Best metric:", best_metric_name_b, best_metric_b)
print("Best tie metric:", best_tie_metric_name_b, best_tie_metric_b)

# 3. Class names
if "id_to_class" in checkpoint_b:
    id_to_class_eval_b = {int(k): v for k, v in checkpoint_b["id_to_class"].items()}
else:
    id_to_class_eval_b = id_to_class_b

class_names_ordered_b = [id_to_class_eval_b[i] for i in range(len(id_to_class_eval_b))]

print("\n=== Class Names Ordered B-G14 ===")
for i, name in enumerate(class_names_ordered_b):
    print(f"{i}: {name}")

# 4. Fungsi prediksi lengkap
@torch.no_grad()
def predict_loader_b17(model, loader, device, split_name):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []
    all_filepaths = []
    all_filenames = []
    all_group_ids = []
    all_raw_source_ids = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion_b(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())
        all_filepaths.extend(batch["filepath"])
        all_filenames.extend(batch["filename"])
        all_group_ids.extend(batch["group_id"])
        all_raw_source_ids.extend(batch["raw_source_id"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    avg_loss = running_loss / len(loader.dataset)

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": all_filepaths,
        "filename": all_filenames,
        "group_id": all_group_ids,
        "raw_source_id": all_raw_source_ids,
        "true_label": all_labels,
        "pred_label": all_preds,
        "true_class": [id_to_class_eval_b[int(x)] for x in all_labels],
        "pred_class": [id_to_class_eval_b[int(x)] for x in all_preds],
        "correct": [int(t == p) for t, p in zip(all_labels, all_preds)]
    })

    probs_arr = np.array(all_probs)
    for class_id, class_name in id_to_class_eval_b.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": float(avg_loss),
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds),
        "probs": probs_arr,
        "result_df": result_df
    }

# 5. Fungsi metrik lengkap
def compute_full_metrics_b17(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names_ordered_b))),
        target_names=class_names_ordered_b,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names_ordered_b))),
        target_names=class_names_ordered_b,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(class_names_ordered_b)))
    )

    summary = {
        "split": split_name,
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1)
    }

    return summary, report_dict, report_text, cm

# 6. Evaluasi validation B-G14
print("\n=== Evaluasi Validation Set B-G14 ===")
val_pred_b = predict_loader_b17(model_b, val_loader_b, device_b, "validation")
val_summary_b, val_report_dict_b, val_report_text_b, val_cm_b = compute_full_metrics_b17(
    val_pred_b["y_true"],
    val_pred_b["y_pred"],
    "validation"
)
val_summary_b["loss"] = val_pred_b["loss"]

print("\n=== Validation Metrics B-G14 ===")
for k, v in val_summary_b.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

print("\n=== Validation Classification Report B-G14 ===")
print(val_report_text_b)

# 7. Evaluasi test B-G14
print("\n=== Evaluasi Test Set B-G14 ===")
test_pred_b = predict_loader_b17(model_b, test_loader_b, device_b, "test")
test_summary_b, test_report_dict_b, test_report_text_b, test_cm_b = compute_full_metrics_b17(
    test_pred_b["y_true"],
    test_pred_b["y_pred"],
    "test"
)
test_summary_b["loss"] = test_pred_b["loss"]

print("\n=== Test Metrics B-G14 ===")
for k, v in test_summary_b.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

print("\n=== Test Classification Report B-G14 ===")
print(test_report_text_b)

# 8. Simpan prediction CSV
val_pred_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_validation_predictions.csv"
test_pred_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_test_predictions.csv"

val_pred_b["result_df"].to_csv(val_pred_path_b, index=False)
test_pred_b["result_df"].to_csv(test_pred_path_b, index=False)

# 9. Simpan classification report
val_report_df_b = pd.DataFrame(val_report_dict_b).T
test_report_df_b = pd.DataFrame(test_report_dict_b).T

val_report_csv_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_validation_classification_report.csv"
test_report_csv_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_test_classification_report.csv"

val_report_txt_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_validation_classification_report.txt"
test_report_txt_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_test_classification_report.txt"

val_report_json_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_validation_classification_report.json"
test_report_json_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_test_classification_report.json"

val_report_df_b.to_csv(val_report_csv_path_b)
test_report_df_b.to_csv(test_report_csv_path_b)

with open(val_report_txt_path_b, "w") as f:
    f.write(val_report_text_b)

with open(test_report_txt_path_b, "w") as f:
    f.write(test_report_text_b)

with open(val_report_json_path_b, "w") as f:
    json.dump(val_report_dict_b, f, indent=2)

with open(test_report_json_path_b, "w") as f:
    json.dump(test_report_dict_b, f, indent=2)

# 10. Simpan confusion matrix CSV
val_cm_df_b = pd.DataFrame(val_cm_b, index=class_names_ordered_b, columns=class_names_ordered_b)
test_cm_df_b = pd.DataFrame(test_cm_b, index=class_names_ordered_b, columns=class_names_ordered_b)

val_cm_csv_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_validation_confusion_matrix.csv"
test_cm_csv_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_test_confusion_matrix.csv"

val_cm_df_b.to_csv(val_cm_csv_path_b)
test_cm_df_b.to_csv(test_cm_csv_path_b)

# 11. Plot confusion matrix
def save_confusion_matrix_plot_b17(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))

    ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticklabels(class_names)

    max_val = cm.max() if cm.max() > 0 else 1
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm[i, j]
            text_color = "white" if value > max_val * 0.5 else "black"
            ax.text(j, i, str(value), ha="center", va="center", color=text_color, fontsize=8)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

val_cm_png_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_validation_confusion_matrix.png"
test_cm_png_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_test_confusion_matrix.png"

save_confusion_matrix_plot_b17(
    val_cm_b,
    class_names_ordered_b,
    "Validation Confusion Matrix — DenseNet201 B-G14 Seed 42",
    val_cm_png_path_b
)

save_confusion_matrix_plot_b17(
    test_cm_b,
    class_names_ordered_b,
    "Test Confusion Matrix — DenseNet201 B-G14 Seed 42",
    test_cm_png_path_b
)

# 12. Baca hasil A jika tersedia untuk pembanding internal
a_metrics_path = OUTPUT_DIR / "DenseNet201_A_official_seed42_evaluation_metrics_summary.json"
a_test_accuracy = None
a_test_macro_f1 = None
a_test_weighted_f1 = None

if a_metrics_path.exists():
    with open(a_metrics_path, "r") as f:
        a_metrics = json.load(f)
    a_test_accuracy = a_metrics.get("test", {}).get("accuracy", None)
    a_test_macro_f1 = a_metrics.get("test", {}).get("macro_f1", None)
    a_test_weighted_f1 = a_metrics.get("test", {}).get("weighted_f1", None)

# 13. Simpan metrics summary
metrics_summary_b = {
    "run_name": RUN_NAME_B,
    "scenario": "B-G14 RAWSource",
    "evaluation_name": "RAW source-aware leakage-controlled evaluation",
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH_B),
    "best_epoch_from_checkpoint": best_epoch_b,
    "best_validation_metric_name_from_training": best_metric_name_b,
    "best_validation_metric_from_training": float(best_metric_b) if best_metric_b is not None else None,
    "best_tie_metric_name_from_training": best_tie_metric_name_b,
    "best_tie_metric_from_training": float(best_tie_metric_b) if best_tie_metric_b is not None else None,
    "validation": val_summary_b,
    "test": test_summary_b,
    "benchmark_reference": {
        "scenario": "Matuszewski & Sintorn 2021 literature benchmark",
        "accuracy": 0.931,
        "f1_score": 0.921
    },
    "test_accuracy_minus_benchmark_accuracy": float(test_summary_b["accuracy"] - 0.931),
    "test_macro_f1_minus_benchmark_f1": float(test_summary_b["macro_f1"] - 0.921),
    "stop_gate_90_percent_passed": bool(test_summary_b["accuracy"] >= 0.90),
    "comparison_to_A_official_seed42_if_available": {
        "A_test_accuracy": a_test_accuracy,
        "A_test_macro_f1": a_test_macro_f1,
        "A_test_weighted_f1": a_test_weighted_f1,
        "B_minus_A_test_accuracy": float(test_summary_b["accuracy"] - a_test_accuracy) if a_test_accuracy is not None else None,
        "B_minus_A_test_macro_f1": float(test_summary_b["macro_f1"] - a_test_macro_f1) if a_test_macro_f1 is not None else None,
        "B_minus_A_test_weighted_f1": float(test_summary_b["weighted_f1"] - a_test_weighted_f1) if a_test_weighted_f1 is not None else None
    },
    "files_saved": {
        "validation_predictions_csv": str(val_pred_path_b),
        "test_predictions_csv": str(test_pred_path_b),
        "validation_classification_report_csv": str(val_report_csv_path_b),
        "test_classification_report_csv": str(test_report_csv_path_b),
        "validation_classification_report_txt": str(val_report_txt_path_b),
        "test_classification_report_txt": str(test_report_txt_path_b),
        "validation_classification_report_json": str(val_report_json_path_b),
        "test_classification_report_json": str(test_report_json_path_b),
        "validation_confusion_matrix_csv": str(val_cm_csv_path_b),
        "test_confusion_matrix_csv": str(test_cm_csv_path_b),
        "validation_confusion_matrix_png": str(val_cm_png_path_b),
        "test_confusion_matrix_png": str(test_cm_png_path_b)
    }
}

metrics_json_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_evaluation_metrics_summary.json"
metrics_csv_path_b = OUTPUT_DIR / f"{RUN_NAME_B}_evaluation_metrics_summary.csv"

with open(metrics_json_path_b, "w") as f:
    json.dump(metrics_summary_b, f, indent=2)

summary_rows_b = []
for split_name, split_summary in [("validation", val_summary_b), ("test", test_summary_b)]:
    row = {"split": split_name}
    row.update(split_summary)
    summary_rows_b.append(row)

pd.DataFrame(summary_rows_b).to_csv(metrics_csv_path_b, index=False)

print("\n=== File Output Evaluasi B-G14 Tersimpan ===")
print("Metrics JSON:", metrics_json_path_b)
print("Metrics CSV :", metrics_csv_path_b)
print("Validation predictions:", val_pred_path_b)
print("Test predictions      :", test_pred_path_b)
print("Validation report CSV :", val_report_csv_path_b)
print("Test report CSV       :", test_report_csv_path_b)
print("Validation CM CSV     :", val_cm_csv_path_b)
print("Test CM CSV           :", test_cm_csv_path_b)
print("Validation CM PNG     :", val_cm_png_path_b)
print("Test CM PNG           :", test_cm_png_path_b)

print("\n=== Ringkasan Sel B17 ===")
print("Best epoch B-G14:", best_epoch_b)
print(f"Validation accuracy B-G14: {val_summary_b['accuracy']:.4f}")
print(f"Validation macro F1 B-G14: {val_summary_b['macro_f1']:.4f}")
print(f"Test accuracy B-G14      : {test_summary_b['accuracy']:.4f}")
print(f"Test macro F1 B-G14      : {test_summary_b['macro_f1']:.4f}")
print(f"Test weighted F1 B-G14   : {test_summary_b['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary_b["accuracy"] >= 0.90)

if a_test_accuracy is not None:
    print("\n=== Pembanding Internal A vs B-G14 ===")
    print(f"A official test accuracy : {a_test_accuracy:.4f}")
    print(f"B-G14 test accuracy      : {test_summary_b['accuracy']:.4f}")
    print(f"B-G14 minus A accuracy   : {test_summary_b['accuracy'] - a_test_accuracy:.4f}")
    print(f"A official test macro F1 : {a_test_macro_f1:.4f}")
    print(f"B-G14 test macro F1      : {test_summary_b['macro_f1']:.4f}")
    print(f"B-G14 minus A macro F1   : {test_summary_b['macro_f1'] - a_test_macro_f1:.4f}")

print("Sel B17 selesai.")

In [ ]:
# Sel B18-0 — Mount Drive Setelah Pindah Runtime CPU

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
print("Output dir exists:", OUTPUT_DIR.exists())
print("Jumlah file di outputs:", len(list(OUTPUT_DIR.glob("*"))))

In [ ]:
# Sel B18 — Analisis Gap Validation vs Test

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

print("=== Sel B18: Analisis Gap Validation vs Test ===")
print("Mode kerja: analisis file hasil eksperimen, tanpa training model.")

# ============================================================
# 1. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MANIFEST_B_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"

SCENARIOS = {
    "A_official": {
        "scenario_name": "A official",
        "scenario_note": "literature-comparable evaluation",
        "manifest": MANIFEST_A_PATH,
        "validation_predictions": OUTPUT_DIR / "DenseNet201_A_official_seed42_validation_predictions.csv",
        "test_predictions": OUTPUT_DIR / "DenseNet201_A_official_seed42_test_predictions.csv",
        "validation_report": OUTPUT_DIR / "DenseNet201_A_official_seed42_validation_classification_report.csv",
        "test_report": OUTPUT_DIR / "DenseNet201_A_official_seed42_test_classification_report.csv",
        "validation_cm": OUTPUT_DIR / "DenseNet201_A_official_seed42_validation_confusion_matrix.csv",
        "test_cm": OUTPUT_DIR / "DenseNet201_A_official_seed42_test_confusion_matrix.csv",
    },
    "B_G14": {
        "scenario_name": "B-G14 RAWSource",
        "scenario_note": "RAW source-aware leakage-controlled evaluation",
        "manifest": MANIFEST_B_PATH,
        "validation_predictions": OUTPUT_DIR / "DenseNet201_B_G14_seed42_validation_predictions.csv",
        "test_predictions": OUTPUT_DIR / "DenseNet201_B_G14_seed42_test_predictions.csv",
        "validation_report": OUTPUT_DIR / "DenseNet201_B_G14_seed42_validation_classification_report.csv",
        "test_report": OUTPUT_DIR / "DenseNet201_B_G14_seed42_test_classification_report.csv",
        "validation_cm": OUTPUT_DIR / "DenseNet201_B_G14_seed42_validation_confusion_matrix.csv",
        "test_cm": OUTPUT_DIR / "DenseNet201_B_G14_seed42_test_confusion_matrix.csv",
    }
}

OUT_DISTRIBUTION = OUTPUT_DIR / "DenseNet201_AB_seed42_val_test_distribution_comparison.csv"
OUT_GAP = OUTPUT_DIR / "DenseNet201_AB_seed42_per_class_val_test_gap.csv"
OUT_CONFUSION = OUTPUT_DIR / "DenseNet201_AB_seed42_confusion_error_summary.csv"
OUT_JSON = OUTPUT_DIR / "DenseNet201_AB_seed42_validation_test_gap_analysis.json"

print("\n=== Cek File Input ===")

path_rows = []
for scenario_key, cfg in SCENARIOS.items():
    for file_type, path in cfg.items():
        if file_type in ["scenario_name", "scenario_note"]:
            continue
        path_rows.append({
            "scenario": scenario_key,
            "file_type": file_type,
            "path": str(path),
            "exists": path.exists()
        })

path_check_df = pd.DataFrame(path_rows)
display(path_check_df)

missing_files = path_check_df[path_check_df["exists"] == False]
if len(missing_files) > 0:
    print("\nAda file yang belum ditemukan. Analisis dihentikan dulu.")
    display(missing_files)
    raise FileNotFoundError("Beberapa file input belum tersedia di folder outputs.")

# ============================================================
# 2. Fungsi bantu
# ============================================================

def read_classification_report(path):
    report_df = pd.read_csv(path, index_col=0)
    report_df.index = report_df.index.astype(str)
    return report_df

def get_class_names_from_manifest(manifest_df):
    label_df = (
        manifest_df[["label_id", "class_name"]]
        .drop_duplicates()
        .sort_values("label_id")
        .reset_index(drop=True)
    )
    return label_df["class_name"].tolist()

def extract_class_rows(report_df, class_names):
    rows = []
    for class_name in class_names:
        row = report_df.loc[class_name]
        rows.append({
            "class_name": class_name,
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1-score"]),
            "support": int(row["support"])
        })
    return pd.DataFrame(rows)

def get_accuracy_from_predictions(pred_df):
    return float(pred_df["correct"].mean())

def get_distribution_table(manifest_df, scenario_key, scenario_name, scenario_note):
    dist = (
        manifest_df
        .groupby(["split", "class_name"])
        .size()
        .reset_index(name="support")
    )

    total = (
        manifest_df
        .groupby("split")
        .size()
        .reset_index(name="split_total")
    )

    dist = dist.merge(total, on="split", how="left")
    dist["support_rate"] = dist["support"] / dist["split_total"]
    dist["scenario"] = scenario_key
    dist["scenario_name"] = scenario_name
    dist["scenario_note"] = scenario_note

    return dist[[
        "scenario", "scenario_name", "scenario_note",
        "split", "class_name", "support", "split_total", "support_rate"
    ]]

def build_distribution_comparison(dist_df):
    val = dist_df[dist_df["split"] == "validation"][[
        "scenario", "scenario_name", "scenario_note", "class_name", "support", "support_rate"
    ]].rename(columns={
        "support": "val_support",
        "support_rate": "val_support_rate"
    })

    test = dist_df[dist_df["split"] == "test"][[
        "scenario", "scenario_name", "scenario_note", "class_name", "support", "support_rate"
    ]].rename(columns={
        "support": "test_support",
        "support_rate": "test_support_rate"
    })

    merged = val.merge(
        test,
        on=["scenario", "scenario_name", "scenario_note", "class_name"],
        how="inner"
    )

    merged["support_gap_test_minus_val"] = merged["test_support"] - merged["val_support"]
    merged["support_rate_gap_test_minus_val"] = merged["test_support_rate"] - merged["val_support_rate"]

    return merged

def build_gap_table(scenario_key, scenario_name, scenario_note, val_report, test_report, class_names):
    val_class = extract_class_rows(val_report, class_names)
    test_class = extract_class_rows(test_report, class_names)

    gap = val_class.merge(
        test_class,
        on="class_name",
        suffixes=("_val", "_test")
    )

    gap = gap.rename(columns={
        "precision_val": "val_precision",
        "precision_test": "test_precision",
        "recall_val": "val_recall",
        "recall_test": "test_recall",
        "f1_val": "val_f1",
        "f1_test": "test_f1",
        "support_val": "val_support",
        "support_test": "test_support"
    })

    gap["scenario"] = scenario_key
    gap["scenario_name"] = scenario_name
    gap["scenario_note"] = scenario_note

    gap["f1_gap_test_minus_val"] = gap["test_f1"] - gap["val_f1"]
    gap["recall_gap_test_minus_val"] = gap["test_recall"] - gap["val_recall"]
    gap["precision_gap_test_minus_val"] = gap["test_precision"] - gap["val_precision"]
    gap["support_gap_test_minus_val"] = gap["test_support"] - gap["val_support"]

    gap = gap[[
        "scenario", "scenario_name", "scenario_note", "class_name",
        "val_support", "test_support",
        "val_precision", "test_precision",
        "val_recall", "test_recall",
        "val_f1", "test_f1",
        "f1_gap_test_minus_val",
        "recall_gap_test_minus_val",
        "precision_gap_test_minus_val",
        "support_gap_test_minus_val"
    ]]

    return gap

def build_confusion_summary(cm_df, scenario_key, scenario_name, scenario_note, split_name):
    class_names = list(cm_df.index)
    cm = cm_df.values.astype(int)

    rows = []

    for i, true_class in enumerate(class_names):
        support = int(cm[i, :].sum())

        for j, pred_class in enumerate(class_names):
            if i == j:
                continue

            count = int(cm[i, j])
            if count > 0:
                rows.append({
                    "scenario": scenario_key,
                    "scenario_name": scenario_name,
                    "scenario_note": scenario_note,
                    "split": split_name,
                    "record_type": "confusion_pair",
                    "class_name": true_class,
                    "true_class": true_class,
                    "pred_class": pred_class,
                    "count": count,
                    "support": support,
                    "rate_within_true_class": count / support if support > 0 else np.nan,
                    "false_negatives": np.nan,
                    "false_positives": np.nan,
                    "correct": np.nan,
                    "class_error_rate": np.nan
                })

    for i, class_name in enumerate(class_names):
        support = int(cm[i, :].sum())
        correct = int(cm[i, i])
        fn = int(cm[i, :].sum() - cm[i, i])
        fp = int(cm[:, i].sum() - cm[i, i])
        error_rate = fn / support if support > 0 else np.nan

        rows.append({
            "scenario": scenario_key,
            "scenario_name": scenario_name,
            "scenario_note": scenario_note,
            "split": split_name,
            "record_type": "class_error_summary",
            "class_name": class_name,
            "true_class": class_name,
            "pred_class": "",
            "count": fn,
            "support": support,
            "rate_within_true_class": error_rate,
            "false_negatives": fn,
            "false_positives": fp,
            "correct": correct,
            "class_error_rate": error_rate
        })

    return pd.DataFrame(rows)

# ============================================================
# 3. Analisis utama
# ============================================================

distribution_tables = []
gap_tables = []
confusion_tables = []
summary_records = {}
focus_notes = {}

for scenario_key, cfg in SCENARIOS.items():
    scenario_name = cfg["scenario_name"]
    scenario_note = cfg["scenario_note"]

    print(f"\n\n=== Analisis Skenario: {scenario_name} ===")
    print("Catatan skenario:", scenario_note)

    manifest_df = pd.read_csv(cfg["manifest"])
    val_pred_df = pd.read_csv(cfg["validation_predictions"])
    test_pred_df = pd.read_csv(cfg["test_predictions"])

    val_report = read_classification_report(cfg["validation_report"])
    test_report = read_classification_report(cfg["test_report"])

    val_cm = pd.read_csv(cfg["validation_cm"], index_col=0)
    test_cm = pd.read_csv(cfg["test_cm"], index_col=0)

    class_names = get_class_names_from_manifest(manifest_df)

    # Distribusi kelas
    dist = get_distribution_table(manifest_df, scenario_key, scenario_name, scenario_note)
    dist_comp = build_distribution_comparison(dist)
    distribution_tables.append(dist_comp)

    print("\nDistribusi kelas validation vs test:")
    display(
        dist_comp.sort_values(
            "support_rate_gap_test_minus_val",
            ascending=False
        )
    )

    # Per-class gap
    gap = build_gap_table(
        scenario_key,
        scenario_name,
        scenario_note,
        val_report,
        test_report,
        class_names
    )
    gap_tables.append(gap)

    print("\nGap performa per kelas, diurutkan dari gap F1 terbesar:")
    display(
        gap.sort_values("f1_gap_test_minus_val", ascending=False)
    )

    # Confusion summary
    val_conf = build_confusion_summary(
        val_cm,
        scenario_key,
        scenario_name,
        scenario_note,
        "validation"
    )
    test_conf = build_confusion_summary(
        test_cm,
        scenario_key,
        scenario_name,
        scenario_note,
        "test"
    )

    confusion_tables.extend([val_conf, test_conf])

    print("\nConfusion terbesar pada validation:")
    display(
        val_conf[val_conf["record_type"] == "confusion_pair"]
        .sort_values("count", ascending=False)
        .head(15)
    )

    print("\nConfusion terbesar pada test:")
    display(
        test_conf[test_conf["record_type"] == "confusion_pair"]
        .sort_values("count", ascending=False)
        .head(15)
    )

    # Ringkasan metrik global
    val_accuracy = get_accuracy_from_predictions(val_pred_df)
    test_accuracy = get_accuracy_from_predictions(test_pred_df)

    val_macro_f1 = float(val_report.loc["macro avg", "f1-score"])
    test_macro_f1 = float(test_report.loc["macro avg", "f1-score"])
    val_weighted_f1 = float(val_report.loc["weighted avg", "f1-score"])
    test_weighted_f1 = float(test_report.loc["weighted avg", "f1-score"])

    val_error = (
        val_conf[val_conf["record_type"] == "class_error_summary"]
        .sort_values("class_error_rate", ascending=False)
    )

    test_error = (
        test_conf[test_conf["record_type"] == "class_error_summary"]
        .sort_values("class_error_rate", ascending=False)
    )

    positive_gap = gap.sort_values("f1_gap_test_minus_val", ascending=False)
    negative_gap = gap.sort_values("f1_gap_test_minus_val", ascending=True)

    summary_records[scenario_key] = {
        "scenario_name": scenario_name,
        "scenario_note": scenario_note,
        "validation_accuracy": val_accuracy,
        "test_accuracy": test_accuracy,
        "accuracy_gap_test_minus_val": test_accuracy - val_accuracy,
        "validation_macro_f1": val_macro_f1,
        "test_macro_f1": test_macro_f1,
        "macro_f1_gap_test_minus_val": test_macro_f1 - val_macro_f1,
        "validation_weighted_f1": val_weighted_f1,
        "test_weighted_f1": test_weighted_f1,
        "weighted_f1_gap_test_minus_val": test_weighted_f1 - val_weighted_f1,
        "largest_positive_f1_gap_classes": positive_gap[[
            "class_name", "val_support", "test_support",
            "val_precision", "test_precision",
            "val_recall", "test_recall",
            "val_f1", "test_f1",
            "f1_gap_test_minus_val"
        ]].head(5).to_dict(orient="records"),
        "largest_negative_f1_gap_classes": negative_gap[[
            "class_name", "val_support", "test_support",
            "val_precision", "test_precision",
            "val_recall", "test_recall",
            "val_f1", "test_f1",
            "f1_gap_test_minus_val"
        ]].head(5).to_dict(orient="records"),
        "hardest_validation_classes_by_error_rate": val_error[[
            "class_name", "support", "false_negatives", "false_positives", "class_error_rate"
        ]].head(5).to_dict(orient="records"),
        "hardest_test_classes_by_error_rate": test_error[[
            "class_name", "support", "false_negatives", "false_positives", "class_error_rate"
        ]].head(5).to_dict(orient="records"),
    }

# ============================================================
# 4. Gabungkan tabel dan simpan output
# ============================================================

distribution_comparison = pd.concat(distribution_tables, ignore_index=True)
per_class_gap = pd.concat(gap_tables, ignore_index=True)
confusion_summary = pd.concat(confusion_tables, ignore_index=True)

distribution_comparison.to_csv(OUT_DISTRIBUTION, index=False)
per_class_gap.to_csv(OUT_GAP, index=False)
confusion_summary.to_csv(OUT_CONFUSION, index=False)

analysis_result = {
    "created_at": datetime.now().isoformat(),
    "analysis_title": "DenseNet201 seed 42 validation-test gap analysis",
    "analysis_scope": [
        "A official",
        "B-G14 RAWSource"
    ],
    "training_used": False,
    "input_source": "saved CSV outputs from previous experiments",
    "summary": summary_records,
    "output_files": {
        "distribution_comparison": str(OUT_DISTRIBUTION),
        "per_class_gap": str(OUT_GAP),
        "confusion_error_summary": str(OUT_CONFUSION),
        "analysis_json": str(OUT_JSON)
    }
}

with open(OUT_JSON, "w") as f:
    json.dump(analysis_result, f, indent=2)

# ============================================================
# 5. Ringkasan yang mudah dibaca
# ============================================================

print("\n\n=== File Hasil Analisis Tersimpan ===")
print("Distribusi validation vs test :", OUT_DISTRIBUTION)
print("Gap performa per kelas        :", OUT_GAP)
print("Ringkasan confusion/error     :", OUT_CONFUSION)
print("Ringkasan JSON                :", OUT_JSON)

summary_df = pd.DataFrame([
    {
        "scenario": scenario_key,
        "scenario_name": item["scenario_name"],
        "validation_accuracy": item["validation_accuracy"],
        "test_accuracy": item["test_accuracy"],
        "accuracy_gap_test_minus_val": item["accuracy_gap_test_minus_val"],
        "validation_macro_f1": item["validation_macro_f1"],
        "test_macro_f1": item["test_macro_f1"],
        "macro_f1_gap_test_minus_val": item["macro_f1_gap_test_minus_val"],
        "validation_weighted_f1": item["validation_weighted_f1"],
        "test_weighted_f1": item["test_weighted_f1"],
        "weighted_f1_gap_test_minus_val": item["weighted_f1_gap_test_minus_val"],
    }
    for scenario_key, item in summary_records.items()
])

print("\n=== Ringkasan Gap Global ===")
display(summary_df)

print("\n=== Kelas dengan F1 Test Lebih Tinggi daripada Validation ===")
for scenario_key, item in summary_records.items():
    print(f"\nSkenario: {item['scenario_name']}")
    display(pd.DataFrame(item["largest_positive_f1_gap_classes"]))

print("\n=== Kelas Tersulit pada Validation ===")
for scenario_key, item in summary_records.items():
    print(f"\nSkenario: {item['scenario_name']}")
    display(pd.DataFrame(item["hardest_validation_classes_by_error_rate"]))

print("\n=== Kelas Tersulit pada Test ===")
for scenario_key, item in summary_records.items():
    print(f"\nSkenario: {item['scenario_name']}")
    display(pd.DataFrame(item["hardest_test_classes_by_error_rate"]))

print("\n=== Fokus Khusus: Orf/Cowpox pada A dan Lassa pada B-G14 ===")

focus_pairs = [
    ("A_official", "Orf"),
    ("A_official", "Cowpox"),
    ("B_G14", "Lassa")
]

focus_table = per_class_gap[
    per_class_gap.apply(
        lambda row: (row["scenario"], row["class_name"]) in focus_pairs,
        axis=1
    )
].copy()

display(focus_table)

print("\n=== Catatan Sementara ===")

for scenario_key, item in summary_records.items():
    print(f"\nSkenario: {item['scenario_name']}")
    print(f"- Gap accuracy test - validation: {item['accuracy_gap_test_minus_val']:.4f}")
    print(f"- Gap macro F1 test - validation: {item['macro_f1_gap_test_minus_val']:.4f}")
    print("- Kelas yang paling menaikkan F1 test dibanding validation:")

    for rec in item["largest_positive_f1_gap_classes"][:3]:
        print(
            f"  {rec['class_name']}: "
            f"val_f1={rec['val_f1']:.4f}, "
            f"test_f1={rec['test_f1']:.4f}, "
            f"gap={rec['f1_gap_test_minus_val']:.4f}, "
            f"val_support={rec['val_support']}, "
            f"test_support={rec['test_support']}"
        )

print("\nSel B18 selesai.")

In [ ]:
# Sel B19 — Analisis Distribusi Kelas dan RAW Source per Split

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

print("=== Sel B19: Analisis Distribusi Kelas dan RAW Source per Split ===")
print("Mode kerja: analisis struktur data dari manifest dan hasil evaluasi, tanpa training model.")

# ============================================================
# 1. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MANIFEST_B_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"

GAP_B18_PATH = OUTPUT_DIR / "DenseNet201_AB_seed42_per_class_val_test_gap.csv"

SCENARIOS = {
    "A_official": {
        "scenario_name": "A official",
        "scenario_note": "literature-comparable evaluation",
        "manifest": MANIFEST_A_PATH,
        "validation_report": OUTPUT_DIR / "DenseNet201_A_official_seed42_validation_classification_report.csv",
        "test_report": OUTPUT_DIR / "DenseNet201_A_official_seed42_test_classification_report.csv",
    },
    "B_G14": {
        "scenario_name": "B-G14 RAWSource",
        "scenario_note": "RAW source-aware leakage-controlled evaluation",
        "manifest": MANIFEST_B_PATH,
        "validation_report": OUTPUT_DIR / "DenseNet201_B_G14_seed42_validation_classification_report.csv",
        "test_report": OUTPUT_DIR / "DenseNet201_B_G14_seed42_test_classification_report.csv",
    }
}

FOCUS_CLASSES = ["Orf", "Cowpox", "Adenovirus", "Marburg", "Lassa"]

OUT_CLASS_DIST = OUTPUT_DIR / "DenseNet201_AB_seed42_class_distribution_by_split.csv"
OUT_RAW_DIVERSITY = OUTPUT_DIR / "DenseNet201_AB_seed42_raw_source_diversity_by_class_split.csv"
OUT_GAP_RAW = OUTPUT_DIR / "DenseNet201_AB_seed42_gap_with_raw_source_analysis.csv"
OUT_SUMMARY_JSON = OUTPUT_DIR / "DenseNet201_AB_seed42_structural_gap_analysis_summary.json"

# ============================================================
# 2. Cek file input
# ============================================================

print("\n=== Cek File Input ===")

path_rows = []

for scenario_key, cfg in SCENARIOS.items():
    for file_type, path in cfg.items():
        if file_type in ["scenario_name", "scenario_note"]:
            continue
        path_rows.append({
            "scenario": scenario_key,
            "file_type": file_type,
            "path": str(path),
            "exists": path.exists()
        })

path_rows.append({
    "scenario": "AB",
    "file_type": "per_class_gap_from_B18_optional",
    "path": str(GAP_B18_PATH),
    "exists": GAP_B18_PATH.exists()
})

path_check = pd.DataFrame(path_rows)
display(path_check)

missing_required = path_check[
    (path_check["exists"] == False) &
    (path_check["file_type"] != "per_class_gap_from_B18_optional")
]

if len(missing_required) > 0:
    print("\nAda file wajib yang belum ditemukan. Analisis dihentikan dulu.")
    display(missing_required)
    raise FileNotFoundError("Beberapa file wajib belum tersedia di folder outputs.")

# ============================================================
# 3. Fungsi bantu
# ============================================================

def read_report(path):
    report = pd.read_csv(path, index_col=0)
    report.index = report.index.astype(str)
    return report

def get_class_order(manifest_df):
    label_df = (
        manifest_df[["label_id", "class_name"]]
        .drop_duplicates()
        .sort_values("label_id")
        .reset_index(drop=True)
    )
    return label_df["class_name"].tolist()

def extract_per_class_metrics(report_df, class_names, split_name):
    rows = []
    for class_name in class_names:
        row = report_df.loc[class_name]
        rows.append({
            "class_name": class_name,
            f"{split_name}_precision": float(row["precision"]),
            f"{split_name}_recall": float(row["recall"]),
            f"{split_name}_f1": float(row["f1-score"]),
            f"{split_name}_support": int(row["support"])
        })
    return pd.DataFrame(rows)

def class_distribution_by_split(manifest_df, scenario_key, scenario_name, scenario_note):
    split_total = (
        manifest_df
        .groupby("split")
        .size()
        .reset_index(name="split_total")
    )

    class_dist = (
        manifest_df
        .groupby(["split", "class_name"])
        .size()
        .reset_index(name="support")
        .merge(split_total, on="split", how="left")
    )

    class_dist["support_rate"] = class_dist["support"] / class_dist["split_total"]
    class_dist["scenario"] = scenario_key
    class_dist["scenario_name"] = scenario_name
    class_dist["scenario_note"] = scenario_note

    class_dist = class_dist[[
        "scenario", "scenario_name", "scenario_note",
        "split", "class_name", "support", "split_total", "support_rate"
    ]]

    return class_dist

def raw_source_diversity(manifest_df, scenario_key, scenario_name, scenario_note):
    rows = []

    has_raw = "raw_source_id" in manifest_df.columns
    has_group = "group_id" in manifest_df.columns

    if not has_raw:
        raise ValueError(f"Manifest {scenario_name} tidak punya kolom raw_source_id.")

    for (split, class_name), part in manifest_df.groupby(["split", "class_name"]):
        raw_counts = (
            part["raw_source_id"]
            .astype(str)
            .value_counts()
        )

        n_crops = int(len(part))
        n_unique_raw = int(raw_counts.shape[0])

        if n_unique_raw > 0:
            mean_crop = float(raw_counts.mean())
            median_crop = float(raw_counts.median())
            max_crop = int(raw_counts.max())
            top_raw_source_id = str(raw_counts.index[0])
            top_raw_source_crop_count = int(raw_counts.iloc[0])
            top_raw_source_share = float(raw_counts.iloc[0] / n_crops) if n_crops > 0 else np.nan
        else:
            mean_crop = np.nan
            median_crop = np.nan
            max_crop = 0
            top_raw_source_id = ""
            top_raw_source_crop_count = 0
            top_raw_source_share = np.nan

        row = {
            "scenario": scenario_key,
            "scenario_name": scenario_name,
            "scenario_note": scenario_note,
            "split": split,
            "class_name": class_name,
            "n_crops": n_crops,
            "unique_raw_source_id": n_unique_raw,
            "mean_crop_per_raw_source": mean_crop,
            "median_crop_per_raw_source": median_crop,
            "max_crop_per_raw_source": max_crop,
            "top_raw_source_id": top_raw_source_id,
            "top_raw_source_crop_count": top_raw_source_crop_count,
            "top_raw_source_share": top_raw_source_share
        }

        if has_group:
            group_counts = (
                part["group_id"]
                .astype(str)
                .value_counts()
            )

            row["unique_group_id"] = int(group_counts.shape[0])
            row["mean_crop_per_group"] = float(group_counts.mean()) if len(group_counts) > 0 else np.nan
            row["median_crop_per_group"] = float(group_counts.median()) if len(group_counts) > 0 else np.nan
            row["max_crop_per_group"] = int(group_counts.max()) if len(group_counts) > 0 else 0
            row["top_group_id"] = str(group_counts.index[0]) if len(group_counts) > 0 else ""
            row["top_group_crop_count"] = int(group_counts.iloc[0]) if len(group_counts) > 0 else 0
            row["top_group_share"] = float(group_counts.iloc[0] / n_crops) if n_crops > 0 and len(group_counts) > 0 else np.nan
        else:
            row["unique_group_id"] = np.nan
            row["mean_crop_per_group"] = np.nan
            row["median_crop_per_group"] = np.nan
            row["max_crop_per_group"] = np.nan
            row["top_group_id"] = ""
            row["top_group_crop_count"] = np.nan
            row["top_group_share"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)

def build_gap_from_reports(scenario_key, scenario_name, scenario_note, manifest_df, val_report, test_report):
    class_names = get_class_order(manifest_df)

    val_metrics = extract_per_class_metrics(val_report, class_names, "val")
    test_metrics = extract_per_class_metrics(test_report, class_names, "test")

    gap = val_metrics.merge(test_metrics, on="class_name", how="inner")

    gap["scenario"] = scenario_key
    gap["scenario_name"] = scenario_name
    gap["scenario_note"] = scenario_note

    gap["f1_gap_test_minus_val"] = gap["test_f1"] - gap["val_f1"]
    gap["recall_gap_test_minus_val"] = gap["test_recall"] - gap["val_recall"]
    gap["precision_gap_test_minus_val"] = gap["test_precision"] - gap["val_precision"]
    gap["support_gap_test_minus_val"] = gap["test_support"] - gap["val_support"]

    gap = gap[[
        "scenario", "scenario_name", "scenario_note", "class_name",
        "val_support", "test_support",
        "val_precision", "test_precision",
        "val_recall", "test_recall",
        "val_f1", "test_f1",
        "f1_gap_test_minus_val",
        "recall_gap_test_minus_val",
        "precision_gap_test_minus_val",
        "support_gap_test_minus_val"
    ]]

    return gap

def pivot_split_metrics(df, value_cols, prefix_map=None):
    base_cols = ["scenario", "scenario_name", "scenario_note", "class_name"]
    out = None

    for split_name in ["train", "validation", "test"]:
        part = df[df["split"] == split_name][base_cols + value_cols].copy()

        rename_map = {}
        for col in value_cols:
            rename_map[col] = f"{split_name}_{col}"

        part = part.rename(columns=rename_map)

        if out is None:
            out = part
        else:
            out = out.merge(part, on=base_cols, how="outer")

    return out

def safe_float(x):
    if pd.isna(x):
        return None
    return float(x)

# ============================================================
# 4. Analisis utama
# ============================================================

all_class_dist = []
all_raw_diversity = []
all_gap = []

scenario_notes = {}

for scenario_key, cfg in SCENARIOS.items():
    scenario_name = cfg["scenario_name"]
    scenario_note = cfg["scenario_note"]

    print(f"\n\n=== Analisis Skenario: {scenario_name} ===")
    print("Catatan:", scenario_note)

    manifest_df = pd.read_csv(cfg["manifest"])
    val_report = read_report(cfg["validation_report"])
    test_report = read_report(cfg["test_report"])

    # Safety filter
    manifest_df = manifest_df[manifest_df["split"].isin(["train", "validation", "test"])].copy()
    manifest_df = manifest_df[manifest_df["class_name"] != "_EXCLUDED"].copy()
    manifest_df = manifest_df[~manifest_df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

    class_dist = class_distribution_by_split(
        manifest_df,
        scenario_key,
        scenario_name,
        scenario_note
    )

    raw_div = raw_source_diversity(
        manifest_df,
        scenario_key,
        scenario_name,
        scenario_note
    )

    gap = build_gap_from_reports(
        scenario_key,
        scenario_name,
        scenario_note,
        manifest_df,
        val_report,
        test_report
    )

    all_class_dist.append(class_dist)
    all_raw_diversity.append(raw_div)
    all_gap.append(gap)

    # Tabel ringkas distribusi kelas
    class_dist_wide = pivot_split_metrics(
        class_dist,
        ["support", "support_rate"]
    )

    print("\nDistribusi kelas per split:")
    display(
        class_dist_wide.sort_values("class_name")
    )

    # Tabel fokus raw source diversity
    focus_raw = raw_div[raw_div["class_name"].isin(FOCUS_CLASSES)].copy()

    print("\nRAW source diversity untuk kelas fokus:")
    display(
        focus_raw.sort_values(["class_name", "split"])
    )

# Gabungkan hasil
class_distribution_all = pd.concat(all_class_dist, ignore_index=True)
raw_diversity_all = pd.concat(all_raw_diversity, ignore_index=True)
gap_all = pd.concat(all_gap, ignore_index=True)

# ============================================================
# 5. Tabel gabungan gap + raw source
# ============================================================

raw_val = raw_diversity_all[raw_diversity_all["split"] == "validation"].copy()
raw_test = raw_diversity_all[raw_diversity_all["split"] == "test"].copy()

raw_val_small = raw_val[[
    "scenario", "scenario_name", "scenario_note", "class_name",
    "n_crops", "unique_raw_source_id",
    "mean_crop_per_raw_source", "median_crop_per_raw_source",
    "max_crop_per_raw_source", "top_raw_source_crop_count",
    "top_raw_source_share",
    "unique_group_id", "top_group_share"
]].rename(columns={
    "n_crops": "val_n_crops",
    "unique_raw_source_id": "val_unique_raw_source",
    "mean_crop_per_raw_source": "val_mean_crop_per_raw_source",
    "median_crop_per_raw_source": "val_median_crop_per_raw_source",
    "max_crop_per_raw_source": "val_max_crop_per_raw_source",
    "top_raw_source_crop_count": "val_top_raw_source_crop_count",
    "top_raw_source_share": "val_top_raw_source_share",
    "unique_group_id": "val_unique_group_id",
    "top_group_share": "val_top_group_share"
})

raw_test_small = raw_test[[
    "scenario", "scenario_name", "scenario_note", "class_name",
    "n_crops", "unique_raw_source_id",
    "mean_crop_per_raw_source", "median_crop_per_raw_source",
    "max_crop_per_raw_source", "top_raw_source_crop_count",
    "top_raw_source_share",
    "unique_group_id", "top_group_share"
]].rename(columns={
    "n_crops": "test_n_crops",
    "unique_raw_source_id": "test_unique_raw_source",
    "mean_crop_per_raw_source": "test_mean_crop_per_raw_source",
    "median_crop_per_raw_source": "test_median_crop_per_raw_source",
    "max_crop_per_raw_source": "test_max_crop_per_raw_source",
    "top_raw_source_crop_count": "test_top_raw_source_crop_count",
    "top_raw_source_share": "test_top_raw_source_share",
    "unique_group_id": "test_unique_group_id",
    "top_group_share": "test_top_group_share"
})

gap_with_raw = (
    gap_all
    .merge(raw_val_small, on=["scenario", "scenario_name", "scenario_note", "class_name"], how="left")
    .merge(raw_test_small, on=["scenario", "scenario_name", "scenario_note", "class_name"], how="left")
)

gap_with_raw["unique_raw_source_gap_test_minus_val"] = (
    gap_with_raw["test_unique_raw_source"] - gap_with_raw["val_unique_raw_source"]
)

gap_with_raw["top_raw_source_share_gap_test_minus_val"] = (
    gap_with_raw["test_top_raw_source_share"] - gap_with_raw["val_top_raw_source_share"]
)

gap_with_raw["mean_crop_per_raw_source_gap_test_minus_val"] = (
    gap_with_raw["test_mean_crop_per_raw_source"] - gap_with_raw["val_mean_crop_per_raw_source"]
)

# Susun kolom utama agar mudah dibaca
main_cols = [
    "scenario", "scenario_name", "scenario_note", "class_name",
    "val_support", "test_support",
    "val_unique_raw_source", "test_unique_raw_source",
    "unique_raw_source_gap_test_minus_val",
    "val_top_raw_source_share", "test_top_raw_source_share",
    "top_raw_source_share_gap_test_minus_val",
    "val_mean_crop_per_raw_source", "test_mean_crop_per_raw_source",
    "mean_crop_per_raw_source_gap_test_minus_val",
    "val_f1", "test_f1", "f1_gap_test_minus_val",
    "val_precision", "test_precision",
    "val_recall", "test_recall",
    "support_gap_test_minus_val"
]

remaining_cols = [c for c in gap_with_raw.columns if c not in main_cols]
gap_with_raw = gap_with_raw[main_cols + remaining_cols]

# ============================================================
# 6. Simpan output
# ============================================================

class_distribution_all.to_csv(OUT_CLASS_DIST, index=False)
raw_diversity_all.to_csv(OUT_RAW_DIVERSITY, index=False)
gap_with_raw.to_csv(OUT_GAP_RAW, index=False)

# ============================================================
# 7. Ringkasan struktural untuk JSON
# ============================================================

summary = {
    "created_at": datetime.now().isoformat(),
    "analysis_title": "DenseNet201 seed 42 structural validation-test gap analysis",
    "training_used": False,
    "data_modified": False,
    "focus_classes": FOCUS_CLASSES,
    "output_files": {
        "class_distribution_by_split": str(OUT_CLASS_DIST),
        "raw_source_diversity_by_class_split": str(OUT_RAW_DIVERSITY),
        "gap_with_raw_source_analysis": str(OUT_GAP_RAW),
        "summary_json": str(OUT_SUMMARY_JSON)
    },
    "scenario_summary": {}
}

for scenario_key, cfg in SCENARIOS.items():
    scenario_name = cfg["scenario_name"]

    scenario_gap = gap_with_raw[gap_with_raw["scenario"] == scenario_key].copy()
    scenario_focus = scenario_gap[scenario_gap["class_name"].isin(FOCUS_CLASSES)].copy()

    # Kelas dengan gap terbesar
    top_gap = (
        scenario_gap
        .sort_values("f1_gap_test_minus_val", ascending=False)
        .head(5)
    )

    # Kelas dengan validation F1 terendah
    low_val_f1 = (
        scenario_gap
        .sort_values("val_f1", ascending=True)
        .head(5)
    )

    # Catatan fokus kelas
    focus_records = []
    for _, row in scenario_focus.sort_values("class_name").iterrows():
        focus_records.append({
            "class_name": row["class_name"],
            "val_support": int(row["val_support"]),
            "test_support": int(row["test_support"]),
            "val_unique_raw_source": safe_float(row["val_unique_raw_source"]),
            "test_unique_raw_source": safe_float(row["test_unique_raw_source"]),
            "val_top_raw_source_share": safe_float(row["val_top_raw_source_share"]),
            "test_top_raw_source_share": safe_float(row["test_top_raw_source_share"]),
            "val_f1": safe_float(row["val_f1"]),
            "test_f1": safe_float(row["test_f1"]),
            "f1_gap_test_minus_val": safe_float(row["f1_gap_test_minus_val"]),
        })

    # Heuristik sederhana untuk catatan struktural
    structural_notes = []

    for _, row in top_gap.iterrows():
        notes = []
        if row["test_support"] < row["val_support"]:
            notes.append("test support lebih kecil daripada validation")
        elif row["test_support"] > row["val_support"]:
            notes.append("test support lebih besar daripada validation")

        if pd.notna(row["val_top_raw_source_share"]) and pd.notna(row["test_top_raw_source_share"]):
            if row["val_top_raw_source_share"] > row["test_top_raw_source_share"] + 0.05:
                notes.append("validation lebih terkonsentrasi pada raw source dominan")
            elif row["test_top_raw_source_share"] > row["val_top_raw_source_share"] + 0.05:
                notes.append("test lebih terkonsentrasi pada raw source dominan")

        if pd.notna(row["val_unique_raw_source"]) and pd.notna(row["test_unique_raw_source"]):
            if row["val_unique_raw_source"] < row["test_unique_raw_source"]:
                notes.append("validation punya raw source diversity lebih rendah")
            elif row["val_unique_raw_source"] > row["test_unique_raw_source"]:
                notes.append("validation punya raw source diversity lebih tinggi")

        structural_notes.append({
            "class_name": row["class_name"],
            "f1_gap_test_minus_val": safe_float(row["f1_gap_test_minus_val"]),
            "notes": notes
        })

    summary["scenario_summary"][scenario_key] = {
        "scenario_name": scenario_name,
        "top_f1_gap_classes": top_gap[[
            "class_name",
            "val_support", "test_support",
            "val_unique_raw_source", "test_unique_raw_source",
            "val_top_raw_source_share", "test_top_raw_source_share",
            "val_f1", "test_f1", "f1_gap_test_minus_val"
        ]].to_dict(orient="records"),
        "lowest_validation_f1_classes": low_val_f1[[
            "class_name",
            "val_support", "test_support",
            "val_unique_raw_source", "test_unique_raw_source",
            "val_top_raw_source_share", "test_top_raw_source_share",
            "val_f1", "test_f1", "f1_gap_test_minus_val"
        ]].to_dict(orient="records"),
        "focus_class_records": focus_records,
        "structural_notes_for_top_gap_classes": structural_notes
    }

with open(OUT_SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

# ============================================================
# 8. Output ringkas untuk dibaca
# ============================================================

print("\n\n=== File Hasil Analisis Tersimpan ===")
print("Distribusi kelas per split       :", OUT_CLASS_DIST)
print("Diversity RAW source per split   :", OUT_RAW_DIVERSITY)
print("Gap + RAW source analysis        :", OUT_GAP_RAW)
print("Ringkasan JSON                   :", OUT_SUMMARY_JSON)

print("\n=== Ringkasan Distribusi Kelas per Split ===")

class_dist_wide_all = pivot_split_metrics(
    class_distribution_all,
    ["support", "support_rate"]
)

display(
    class_dist_wide_all.sort_values(["scenario", "class_name"])
)

print("\n=== Ringkasan RAW Source Diversity untuk Kelas Fokus ===")

focus_raw_all = raw_diversity_all[
    raw_diversity_all["class_name"].isin(FOCUS_CLASSES)
].copy()

display(
    focus_raw_all.sort_values(["scenario", "class_name", "split"])
)

print("\n=== Gap Performa + Struktur RAW Source untuk Kelas Fokus ===")

focus_gap_raw = gap_with_raw[
    gap_with_raw["class_name"].isin(FOCUS_CLASSES)
].copy()

display(
    focus_gap_raw.sort_values(["scenario", "f1_gap_test_minus_val"], ascending=[True, False])
)

print("\n=== Kelas dengan Gap F1 Test - Validation Terbesar ===")

for scenario_key, cfg in SCENARIOS.items():
    scenario_name = cfg["scenario_name"]
    print(f"\nSkenario: {scenario_name}")

    temp = (
        gap_with_raw[gap_with_raw["scenario"] == scenario_key]
        .sort_values("f1_gap_test_minus_val", ascending=False)
        .head(8)
    )

    display(temp[[
        "class_name",
        "val_support", "test_support",
        "val_unique_raw_source", "test_unique_raw_source",
        "val_top_raw_source_share", "test_top_raw_source_share",
        "val_f1", "test_f1", "f1_gap_test_minus_val"
    ]])

print("\n=== Catatan Sementara ===")

for scenario_key, cfg in SCENARIOS.items():
    scenario_name = cfg["scenario_name"]
    temp = (
        gap_with_raw[gap_with_raw["scenario"] == scenario_key]
        .sort_values("f1_gap_test_minus_val", ascending=False)
        .head(5)
    )

    print(f"\nSkenario: {scenario_name}")
    for _, row in temp.iterrows():
        print(
            f"- {row['class_name']}: "
            f"gap F1={row['f1_gap_test_minus_val']:.4f}, "
            f"val_f1={row['val_f1']:.4f}, test_f1={row['test_f1']:.4f}, "
            f"val_support={int(row['val_support'])}, test_support={int(row['test_support'])}, "
            f"val_raw={int(row['val_unique_raw_source'])}, test_raw={int(row['test_unique_raw_source'])}, "
            f"val_top_share={row['val_top_raw_source_share']:.3f}, "
            f"test_top_share={row['test_top_raw_source_share']:.3f}"
        )

print("\nSel B19 selesai.")

In [ ]:
# Sel B20 — Pre-register Plan dan Master Experiment Log

import json
import pandas as pd
from pathlib import Path
from datetime import datetime

print("=== Sel B20: Pre-register Plan dan Master Experiment Log ===")
print("Mode kerja: menyiapkan catatan eksperimen dan rencana seed, tanpa training model.")

# ============================================================
# 1. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"
PRE_REGISTERED_PLAN_PATH = OUTPUT_DIR / "pre_registered_experiment_plan.json"

print("\n=== Path Output ===")
print("Output folder:", OUTPUT_DIR)
print("Checkpoint folder:", CHECKPOINT_DIR)
print("Master log:", MASTER_LOG_PATH)
print("Pre-registered plan:", PRE_REGISTERED_PLAN_PATH)

# ============================================================
# 2. Kolom master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id",
    "model",
    "scenario",
    "seed",
    "run_type",
    "start_time",
    "end_time",
    "input_size",
    "batch_size",
    "optimizer",
    "learning_rate",
    "weight_decay",
    "scheduler",
    "max_epoch",
    "best_epoch",
    "val_accuracy",
    "val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
    "checkpoint_path",
    "notes"
]

# ============================================================
# 3. Helper untuk membaca hasil run yang sudah selesai
# ============================================================

def safe_get(d, keys, default=""):
    current = d
    for k in keys:
        if not isinstance(current, dict) or k not in current:
            return default
        current = current[k]
    return current

def read_json_if_exists(path):
    path = Path(path)
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return None

def make_completed_run_record(
    experiment_id,
    scenario,
    seed,
    run_type,
    metrics_json_path,
    training_summary_path,
    checkpoint_path,
    notes
):
    metrics = read_json_if_exists(metrics_json_path)
    training = read_json_if_exists(training_summary_path)

    if metrics is None:
        print(f"Catatan: metrics belum ditemukan untuk {experiment_id}")
        metrics = {}

    if training is None:
        print(f"Catatan: training summary belum ditemukan untuk {experiment_id}")
        training = {}

    record = {
        "experiment_id": experiment_id,
        "model": "DenseNet201",
        "scenario": scenario,
        "seed": seed,
        "run_type": run_type,
        "start_time": "",
        "end_time": training.get("completed_at", metrics.get("evaluated_at", "")),
        "input_size": 224,
        "batch_size": 16,
        "optimizer": "AdamW",
        "learning_rate": 0.0001,
        "weight_decay": 0.0001,
        "scheduler": "ReduceLROnPlateau",
        "max_epoch": 20,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": safe_get(metrics, ["validation", "accuracy"], ""),
        "val_macro_f1": safe_get(metrics, ["validation", "macro_f1"], ""),
        "test_accuracy": safe_get(metrics, ["test", "accuracy"], ""),
        "test_macro_f1": safe_get(metrics, ["test", "macro_f1"], ""),
        "test_weighted_f1": safe_get(metrics, ["test", "weighted_f1"], ""),
        "checkpoint_path": str(checkpoint_path),
        "notes": notes
    }

    return record

# ============================================================
# 4. Buat / update master experiment log
# ============================================================

print("\n=== Menyiapkan Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    print("Master log lama ditemukan.")
    print("Jumlah baris lama:", len(master_log))

    # Pastikan semua kolom minimal ada
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

# Catat run yang sudah selesai agar log utama langsung punya baseline awal
completed_records = []

completed_records.append(
    make_completed_run_record(
        experiment_id="DenseNet201_A_official_seed42_initial",
        scenario="A_official",
        seed=42,
        run_type="initial_baseline",
        metrics_json_path=OUTPUT_DIR / "DenseNet201_A_official_seed42_evaluation_metrics_summary.json",
        training_summary_path=OUTPUT_DIR / "DenseNet201_A_official_seed42_training_summary.json",
        checkpoint_path=CHECKPOINT_DIR / "DenseNet201_A_official_seed42_best.pt",
        notes="Run pertama A official seed 42; literature-comparable evaluation; single seed; belum klaim final."
    )
)

completed_records.append(
    make_completed_run_record(
        experiment_id="DenseNet201_B_G14_seed42_initial",
        scenario="B_G14",
        seed=42,
        run_type="initial_baseline",
        metrics_json_path=OUTPUT_DIR / "DenseNet201_B_G14_seed42_evaluation_metrics_summary.json",
        training_summary_path=OUTPUT_DIR / "DenseNet201_B_G14_seed42_training_summary.json",
        checkpoint_path=CHECKPOINT_DIR / "DenseNet201_B_G14_seed42_best.pt",
        notes="Run pertama B-G14 seed 42; RAW source-aware leakage-controlled evaluation; single seed; belum klaim final."
    )
)

completed_df = pd.DataFrame(completed_records)

# Gabungkan dan hindari duplikasi experiment_id
master_log = pd.concat([master_log, completed_df], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS]
master_log = master_log.drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)

master_log.to_csv(MASTER_LOG_PATH, index=False)

print("Master log disimpan.")
print("Jumlah baris terbaru:", len(master_log))
display(master_log)

# ============================================================
# 5. Pre-registered experiment plan
# ============================================================

print("\n=== Menyiapkan Pre-registered Experiment Plan ===")

pre_registered_plan = {
    "created_at": datetime.now().isoformat(),
    "project": "TEM Virus Image Classification",
    "model_family": "DenseNet201 baseline first",
    "current_stage": "before determinism check and before multi-seed",
    "important_decisions": {
        "do_not_train_modern_models_yet": True,
        "do_not_run_C_G09_yet": True,
        "do_not_run_multi_seed_before_determinism_check": True,
        "show_validation_test_gap": True,
        "max_epoch_fixed_for_consistency": 20,
        "determinism_check_required_before_multi_seed": True
    },
    "seed_plan": {
        "A_official": {
            "scenario_name": "A official",
            "evaluation_name": "literature-comparable evaluation",
            "seeds": [42, 123, 456, 789, 1024]
        },
        "B_G14": {
            "scenario_name": "B-G14 RAWSource",
            "evaluation_name": "RAW source-aware leakage-controlled evaluation",
            "seeds": [42, 123, 456]
        },
        "C_G09": {
            "scenario_name": "C-G09 Date+Magnification",
            "evaluation_name": "stricter acquisition session-aware sensitivity",
            "seeds": [42]
        }
    },
    "determinism_check": {
        "target": "Rerun DenseNet201 A official seed 42 with same setup",
        "first_run_prefix": "DenseNet201_A_official_seed42",
        "rerun_prefix": "DenseNet201_A_official_seed42_rerun1",
        "purpose": "Compare rerun seed 42 with first seed 42 run before multi-seed",
        "first_run_reference_metrics": {
            "validation_accuracy": 0.9551,
            "validation_macro_f1": 0.9301,
            "test_accuracy": 0.9658,
            "test_macro_f1": 0.9587,
            "test_weighted_f1": 0.9657
        }
    },
    "common_training_setup": {
        "input_size": 224,
        "batch_size": 16,
        "model": "DenseNet201 pretrained ImageNet",
        "num_classes": 14,
        "optimizer": "AdamW",
        "learning_rate": 0.0001,
        "weight_decay": 0.0001,
        "scheduler": "ReduceLROnPlateau",
        "loss": "CrossEntropyLoss",
        "max_epoch": 20,
        "checkpoint_metric": "validation macro F1",
        "tie_breaker": "validation accuracy",
        "mixed_precision": True,
        "test_usage": "test set only evaluated after best checkpoint is selected from validation"
    },
    "checkpoint_policy_colab_free": {
        "required": [
            "best checkpoint",
            "last checkpoint",
            "training history per epoch",
            "training summary JSON"
        ],
        "optional_if_storage_is_safe": [
            "checkpoint every 5 epochs"
        ],
        "not_required": [
            "checkpoint every epoch unless needed for recovery"
        ],
        "notes": "Best and last checkpoint are mandatory. Extra checkpoint every 5 epochs is allowed only if Drive storage remains safe."
    },
    "master_log_path": str(MASTER_LOG_PATH),
    "notes": [
        "A official and B-G14 initial seed 42 are accepted as preliminary baselines.",
        "Current results are still single-seed and must not be overclaimed.",
        "Validation-test gap should be reported and explained, not hidden."
    ]
}

with open(PRE_REGISTERED_PLAN_PATH, "w") as f:
    json.dump(pre_registered_plan, f, indent=2)

print("Pre-registered plan disimpan.")
print(PRE_REGISTERED_PLAN_PATH)

# ============================================================
# 6. Ringkasan akhir sel
# ============================================================

print("\n=== Ringkasan Sel B20 ===")
print("Master experiment log:", MASTER_LOG_PATH)
print("Pre-registered plan:", PRE_REGISTERED_PLAN_PATH)
print("Seed A official:", pre_registered_plan["seed_plan"]["A_official"]["seeds"])
print("Seed B-G14:", pre_registered_plan["seed_plan"]["B_G14"]["seeds"])
print("Seed C-G09:", pre_registered_plan["seed_plan"]["C_G09"]["seeds"])
print("Checkpoint policy: best + last + history + summary wajib.")
print("Sel B20 selesai.")

In [ ]:
# Sel B21-R1 — Recovery Drive dan Ekstraksi Dataset dari ZIP Lokal

import os
import shutil
import zipfile
import time
from pathlib import Path

print("=== Sel B21-R1: Recovery Drive dan Dataset Lokal ===")
print("Mode kerja: memperbaiki putusnya koneksi Drive saat ekstraksi ZIP.")

# 1. Remount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# 2. Path
PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
ZIP_DRIVE = PROJECT_DIR / "data_raw" / "TEM virus dataset.zip"

LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

print("\n=== Cek Path ===")
print("ZIP Drive:", ZIP_DRIVE)
print("ZIP Drive exists:", ZIP_DRIVE.exists())
print("Local ZIP:", LOCAL_ZIP)
print("Extract root:", EXTRACT_ROOT)
print("Dataset dir:", DATASET_DIR)

if not ZIP_DRIVE.exists():
    raise FileNotFoundError(f"ZIP di Drive belum terbaca: {ZIP_DRIVE}")

# 3. Cek kapasitas disk lokal
print("\n=== Cek Ruang Disk Lokal ===")
total, used, free = shutil.disk_usage("/content")
print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used : {used / 1024**3:.2f} GB")
print(f"Free : {free / 1024**3:.2f} GB")

zip_size_gb = ZIP_DRIVE.stat().st_size / 1024**3
print(f"Ukuran ZIP Drive: {zip_size_gb:.2f} GB")

if free < ZIP_DRIVE.stat().st_size * 1.5:
    print("Peringatan: ruang lokal agak mepet. Tetap dicoba, tapi kalau gagal perlu bersihkan runtime.")

# 4. Hapus ekstraksi parsial agar tidak ada file setengah jadi
print("\n=== Bersihkan Ekstraksi Parsial ===")
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
    print("Folder ekstraksi parsial dihapus:", EXTRACT_ROOT)
else:
    print("Tidak ada folder ekstraksi lama.")

# 5. Copy ZIP dari Drive ke lokal
print("\n=== Copy ZIP ke Lokal /content ===")
if LOCAL_ZIP.exists():
    local_size = LOCAL_ZIP.stat().st_size
    drive_size = ZIP_DRIVE.stat().st_size

    if local_size == drive_size:
        print("Local ZIP sudah ada dan ukurannya sesuai. Copy dilewati.")
    else:
        print("Local ZIP ada tapi ukurannya tidak sesuai. Hapus dan copy ulang.")
        LOCAL_ZIP.unlink()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
else:
    start_copy = time.time()
    shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
    print(f"Copy selesai dalam {(time.time() - start_copy)/60:.2f} menit.")

print("Local ZIP exists:", LOCAL_ZIP.exists())
print(f"Local ZIP size: {LOCAL_ZIP.stat().st_size / 1024**3:.2f} GB")

# 6. Test ZIP lokal
print("\n=== Cek ZIP Lokal ===")
with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
    bad_file = zf.testzip()
    if bad_file is None:
        print("ZIP lokal terbaca baik.")
    else:
        raise RuntimeError(f"Ada file bermasalah di ZIP lokal: {bad_file}")

# 7. Ekstrak folder target dari ZIP lokal
print("\n=== Ekstraksi dari ZIP Lokal ===")
target_folder_name = "context_virus_1nm_256x256"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

start_extract = time.time()

with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
    members = [m for m in zf.namelist() if target_folder_name in m]
    print("Jumlah entry target:", len(members))

    for i, member in enumerate(members, start=1):
        zf.extract(member, EXTRACT_ROOT)

        if i % 3000 == 0 or i == len(members):
            print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

print("\nEkstraksi selesai.")

# 8. Verifikasi hasil
print("\n=== Verifikasi Dataset ===")
print("Dataset dir exists:", DATASET_DIR.exists())

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset folder belum terbentuk setelah ekstraksi.")

image_exts = (".tif", ".tiff", ".png", ".jpg", ".jpeg")
image_count = len([
    p for p in DATASET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
])

print("Jumlah file gambar dalam context_virus_1nm_256x256:", image_count)

for split in ["train", "validation", "test", "augmented_train"]:
    split_dir = DATASET_DIR / split
    if split_dir.exists():
        n_img = len([
            p for p in split_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        ])
        print(f"{split}: {n_img} gambar")
    else:
        print(f"{split}: folder tidak ada")

print("\nSel B21-R1 selesai. Setelah ini jalankan ulang Sel B21 dari awal.")

In [ ]:
# Sel B21 — Determinism Check DenseNet201 A Seed 42 Rerun

import os
import gc
import json
import time
import random
import zipfile
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B21: Determinism Check DenseNet201 A Official Seed 42 Rerun ===")
print("Mode kerja: rerun seed 42 dengan setup yang sama, output tidak menimpa run pertama.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount dilewati atau sudah mounted:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B21.")

device = torch.device("cuda")

print("\n=== Runtime Check ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_RAW_DIR / "TEM virus dataset.zip"
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_official_seed42_rerun1"
EXPERIMENT_ID = "DenseNet201_A_official_seed42_rerun1"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"
HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"
COMPARISON_CSV_PATH = OUTPUT_DIR / "DenseNet201_A_official_seed42_determinism_comparison.csv"
COMPARISON_JSON_PATH = OUTPUT_DIR / "DenseNet201_A_official_seed42_determinism_comparison.json"

print("\n=== Path Check ===")
print("Manifest A:", MANIFEST_A_PATH, "| exists:", MANIFEST_A_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("ZIP:", ZIP_PATH, "| exists:", ZIP_PATH.exists())
print("Best checkpoint rerun:", BEST_CKPT_PATH)

if not MANIFEST_A_PATH.exists():
    raise FileNotFoundError(f"Manifest A tidak ditemukan: {MANIFEST_A_PATH}")

# ============================================================
# 3. Recovery dataset ke /content jika perlu
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Mulai ekstraksi context_virus_1nm_256x256 dari ZIP.")
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"ZIP dataset tidak ditemukan: {ZIP_PATH}")

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset folder tetap belum ditemukan setelah recovery.")

# ============================================================
# 4. Konfigurasi setup sama seperti run pertama
# ============================================================

SEED = 42
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Sama seperti run pertama: benchmark aktif, deterministic tidak dipaksa ketat.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Rerun ===")
print("Run name:", RUN_NAME)
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Max epoch:", MAX_EPOCHS)
print("Optimizer: AdamW")
print("LR:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("cuDNN deterministic:", torch.backends.cudnn.deterministic)
print("cuDNN benchmark:", torch.backends.cudnn.benchmark)

# ============================================================
# 5. Update master log: mulai run
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

start_time = datetime.now().isoformat()

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "determinism_check_rerun",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Determinism check rerun1 untuk DenseNet201 A official seed 42; output tidak menimpa run pertama."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last")
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log sudah ditandai mulai run:")
print(MASTER_LOG_PATH)

# ============================================================
# 6. Load manifest A dan DataLoader
# ============================================================

df = pd.read_csv(MANIFEST_A_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

# Pastikan filepath ada
df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())
print("\n=== Manifest Check ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang belum ditemukan. Dataset recovery belum lengkap.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

print("Classes:", class_names_ordered)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 7. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 8. Fungsi train/eval
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)
    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true, y_pred, probs_all = [], [], []
    filepaths, filenames, true_classes = [], [], []
    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])
        true_classes.extend(batch["class_name"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 9. Training rerun
# ============================================================

print("\n=== Mulai Training Rerun ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_official",
                "run_type": "determinism_check_rerun",
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat()
            }, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        # Last checkpoint wajib
        torch.save({
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_official",
            "run_type": "determinism_check_rerun",
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat()
        }, LAST_CKPT_PATH)

        # History wajib
        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_official",
        "run_type": "determinism_check_rerun",
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat()
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training Rerun Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining rerun gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 10. Evaluasi best checkpoint rerun pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint Rerun ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)
    report_dict = classification_report(
        y_true, y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )
    report_text = classification_report(
        y_true, y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(val_pred["y_true"], val_pred["y_pred"])
test_summary, test_report_dict, test_report_text, test_cm = full_report(test_pred["y_true"], test_pred["y_pred"])

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics rerun:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics rerun:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report rerun:")
print(test_report_text)

# ============================================================
# 11. Simpan output evaluasi rerun
# ============================================================

val_pred_path = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
test_pred_path = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
val_report_csv_path = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
test_report_csv_path = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
val_cm_csv_path = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
test_cm_csv_path = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

val_pred["result_df"].to_csv(val_pred_path, index=False)
test_pred["result_df"].to_csv(test_pred_path, index=False)
pd.DataFrame(val_report_dict).T.to_csv(val_report_csv_path)
pd.DataFrame(test_report_dict).T.to_csv(test_report_csv_path)
pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(val_cm_csv_path)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(test_cm_csv_path)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_official",
    "run_type": "determinism_check_rerun",
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "files_saved": {
        "validation_predictions": str(val_pred_path),
        "test_predictions": str(test_pred_path),
        "validation_report_csv": str(val_report_csv_path),
        "test_report_csv": str(test_report_csv_path),
        "validation_confusion_matrix_csv": str(val_cm_csv_path),
        "test_confusion_matrix_csv": str(test_cm_csv_path)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 12. Tabel perbandingan run pertama vs rerun
# ============================================================

first_metrics_path = OUTPUT_DIR / "DenseNet201_A_official_seed42_evaluation_metrics_summary.json"
first_training_path = OUTPUT_DIR / "DenseNet201_A_official_seed42_training_summary.json"

with open(first_metrics_path, "r") as f:
    first_metrics = json.load(f)

with open(first_training_path, "r") as f:
    first_training = json.load(f)

first_row = {
    "run_name": "DenseNet201_A_official_seed42_initial",
    "best_epoch": first_training.get("best_epoch", first_metrics.get("best_epoch_from_checkpoint", "")),
    "val_accuracy": first_metrics["validation"]["accuracy"],
    "val_macro_f1": first_metrics["validation"]["macro_f1"],
    "test_accuracy": first_metrics["test"]["accuracy"],
    "test_macro_f1": first_metrics["test"]["macro_f1"],
    "test_weighted_f1": first_metrics["test"]["weighted_f1"],
}

rerun_row = {
    "run_name": RUN_NAME,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
}

comparison = pd.DataFrame([first_row, rerun_row])

base_test_acc = comparison.loc[0, "test_accuracy"]
base_test_f1 = comparison.loc[0, "test_macro_f1"]

comparison["delta_test_accuracy"] = comparison["test_accuracy"] - base_test_acc
comparison["delta_test_macro_f1"] = comparison["test_macro_f1"] - base_test_f1

comparison.to_csv(COMPARISON_CSV_PATH, index=False)

comparison_json = {
    "created_at": datetime.now().isoformat(),
    "purpose": "Determinism check: compare initial seed 42 run with rerun1 seed 42",
    "comparison_csv": str(COMPARISON_CSV_PATH),
    "rows": comparison.to_dict(orient="records"),
    "notes": {
        "same_seed": 42,
        "same_setup": True,
        "max_epoch": 20,
        "output_prefix": RUN_NAME,
        "interpretation": "MASTER CHAT will judge whether observed delta is acceptable before multi-seed."
    }
}

with open(COMPARISON_JSON_PATH, "w") as f:
    json.dump(comparison_json, f, indent=2)

# ============================================================
# 13. Update master experiment log: selesai run
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "determinism_check_rerun",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Determinism check rerun1 selesai; bandingkan dengan initial run sebelum multi-seed."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last")
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 14. Ringkasan akhir
# ============================================================

print("\n=== File Output Rerun Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Comparison CSV:", COMPARISON_CSV_PATH)
print("Master log:", MASTER_LOG_PATH)

print("\n=== Tabel Perbandingan Determinism Check ===")
display(comparison)

print("\n=== Ringkasan Sel B21 ===")
print("Run pertama test accuracy:", f"{first_row['test_accuracy']:.4f}")
print("Rerun test accuracy      :", f"{rerun_row['test_accuracy']:.4f}")
print("Delta test accuracy      :", f"{rerun_row['test_accuracy'] - first_row['test_accuracy']:.4f}")
print("Run pertama test macro F1:", f"{first_row['test_macro_f1']:.4f}")
print("Rerun test macro F1      :", f"{rerun_row['test_macro_f1']:.4f}")
print("Delta test macro F1      :", f"{rerun_row['test_macro_f1'] - first_row['test_macro_f1']:.4f}")
print("Sel B21 selesai.")

In [ ]:
# Sel B22 — Multi-Seed DenseNet201 A Official Seed 123

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B22: Multi-Seed DenseNet201 A Official Seed 123 ===")
print("Mode kerja: menjalankan seed 123 untuk Skenario A official. Tidak lanjut seed lain di sel ini.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B22.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_official_seed123"
EXPERIMENT_ID = "DenseNet201_A_official_seed123"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"
HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

print("\n=== Cek Path ===")
print("Manifest A:", MANIFEST_A_PATH, "| exists:", MANIFEST_A_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_A_PATH.exists():
    raise FileNotFoundError(f"Manifest A tidak ditemukan: {MANIFEST_A_PATH}")

# ============================================================
# 3. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed42_rerun1"
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 4. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 5. Setup eksperimen seed 123
# ============================================================

SEED = 123
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Sama seperti eksperimen sebelumnya.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 123 ===")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")

# ============================================================
# 6. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 A official seed 123. Hyperparameter sama dengan seed 42 initial."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai seed 123.")
print(MASTER_LOG_PATH)

# ============================================================
# 7. Load manifest dan DataLoader
# ============================================================

df = pd.read_csv(MANIFEST_A_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest A ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 8. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 9. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 10. Training seed 123
# ============================================================

print("\n=== Mulai Training Seed 123 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_official",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat()
            }, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        torch.save({
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_official",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat()
        }, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_official",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat()
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training Seed 123 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining seed 123 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 11. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint Seed 123 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics seed 123:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics seed 123:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report seed 123:")
print(test_report_text)

# ============================================================
# 12. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_official",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 13. Update master log: selesai seed 123
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 A official seed 123 selesai. Rerun1 determinism check tidak masuk statistik final multi-seed."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 14. Ringkasan akhir
# ============================================================

print("\n=== File Output Seed 123 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)

print("\n=== Ringkasan Sel B22 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)
print("Sel B22 selesai. Jangan lanjut seed 456 sebelum ada instruksi berikutnya.")

In [ ]:
# Sel B23-R1 — Cek Checkpoint Seed 456 Setelah Error Load

import os
import json
from pathlib import Path
import torch

print("=== Sel B23-R1: Cek Checkpoint Seed 456 ===")
print("Mode kerja: cek file checkpoint, belum training ulang.")

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

RUN_NAME = "DenseNet201_A_official_seed456"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"
HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

paths = {
    "best_checkpoint": BEST_CKPT_PATH,
    "last_checkpoint": LAST_CKPT_PATH,
    "history_csv": HISTORY_CSV_PATH,
    "history_json": HISTORY_JSON_PATH,
    "training_summary_json": TRAINING_SUMMARY_PATH,
}

print("\n=== File Check ===")
for name, path in paths.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1024**2 if exists else 0
    print(f"{name}: exists={exists} | size={size_mb:.2f} MB | path={path}")

def try_load_checkpoint(path, label):
    print(f"\n=== Try Load {label} ===")
    if not path.exists():
        print(f"{label} tidak ditemukan.")
        return None, False

    try:
        ckpt = torch.load(path, map_location=device, weights_only=False)
        print(f"{label} berhasil dibaca.")
        print("epoch:", ckpt.get("epoch"))
        print("best_epoch:", ckpt.get("best_epoch"))
        print("best_metric:", ckpt.get("best_metric"))
        print("best_tie_metric:", ckpt.get("best_tie_metric"))
        print("keys:", list(ckpt.keys())[:10])
        return ckpt, True
    except Exception as e:
        print(f"{label} gagal dibaca.")
        print("Error:", repr(e))
        return None, False

best_ckpt, best_ok = try_load_checkpoint(BEST_CKPT_PATH, "best checkpoint")
last_ckpt, last_ok = try_load_checkpoint(LAST_CKPT_PATH, "last checkpoint")

print("\n=== Ringkasan B23-R1 ===")
print("Best checkpoint load OK:", best_ok)
print("Last checkpoint load OK:", last_ok)

if best_ok:
    print("Keputusan sementara: best checkpoint aman. Bisa lanjut evaluasi best checkpoint.")
elif last_ok:
    print("Keputusan sementara: best checkpoint corrupt, last checkpoint masih bisa dibaca.")
    print("Catatan: last checkpoint bukan otomatis best epoch, jadi seed 456 belum boleh diterima sebagai hasil final.")
else:
    print("Keputusan sementara: best dan last checkpoint sama-sama bermasalah. Seed 456 perlu rerun.")

In [ ]:
# Cek
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

print("Output dir exists:", OUTPUT_DIR.exists())
print("Checkpoint dir exists:", CHECKPOINT_DIR.exists())
print("Jumlah file outputs:", len(list(OUTPUT_DIR.glob("*"))) if OUTPUT_DIR.exists() else 0)
print("Jumlah file checkpoints:", len(list(CHECKPOINT_DIR.glob("*"))) if CHECKPOINT_DIR.exists() else 0)

In [ ]:
# Sel B23-VERIFY — Verifikasi Output Seed 456

import json
import torch
import pandas as pd
from pathlib import Path

print("=== Sel B23-VERIFY: Verifikasi Output Seed 456 ===")

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

RUN_NAME = "DenseNet201_A_official_seed456"

BEST_CKPT = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"
EVAL_JSON = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
TRAIN_JSON = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
MASTER_LOG = OUTPUT_DIR / "experiment_master_log.csv"
COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_A_official_multiseed_temporary_comparison_until_seed456.csv"

files = {
    "best checkpoint": BEST_CKPT,
    "last checkpoint": LAST_CKPT,
    "evaluation summary": EVAL_JSON,
    "training summary": TRAIN_JSON,
    "master log": MASTER_LOG,
    "comparison csv": COMPARISON_CSV,
}

print("\n=== File Check ===")
for name, path in files.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1024**2 if exists else 0
    print(f"{name}: exists={exists} | size={size_mb:.2f} MB | {path}")

print("\n=== Checkpoint Load Check ===")

def try_load(path, label):
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        print(f"{label}: OK")
        print("  epoch:", ckpt.get("epoch"))
        print("  best_epoch:", ckpt.get("best_epoch"))
        print("  best_metric:", ckpt.get("best_metric"))
        print("  best_tie_metric:", ckpt.get("best_tie_metric"))
        return True
    except Exception as e:
        print(f"{label}: GAGAL")
        print("  Error:", repr(e))
        return False

best_ok = try_load(BEST_CKPT, "best checkpoint")
last_ok = try_load(LAST_CKPT, "last checkpoint")

print("\n=== Evaluation Summary Check ===")
eval_ok = False

if EVAL_JSON.exists():
    with open(EVAL_JSON, "r") as f:
        ev = json.load(f)

    val = ev.get("validation", {})
    test = ev.get("test", {})

    print("best_epoch:", ev.get("best_epoch_from_checkpoint"))
    print("val_accuracy:", val.get("accuracy"))
    print("val_macro_f1:", val.get("macro_f1"))
    print("test_accuracy:", test.get("accuracy"))
    print("test_macro_f1:", test.get("macro_f1"))
    print("test_weighted_f1:", test.get("weighted_f1"))
    print("stop_gate_90_percent_passed:", ev.get("stop_gate_90_percent_passed"))

    eval_ok = (
        abs(float(val.get("accuracy", -1)) - 0.9484) < 0.002 and
        abs(float(val.get("macro_f1", -1)) - 0.9234) < 0.002 and
        abs(float(test.get("accuracy", -1)) - 0.9521) < 0.002 and
        abs(float(test.get("macro_f1", -1)) - 0.9425) < 0.002 and
        abs(float(test.get("weighted_f1", -1)) - 0.9517) < 0.002
    )

    print("evaluation metrics sesuai catatan:", eval_ok)
else:
    print("Evaluation summary belum ditemukan.")

print("\n=== Master Log Seed 456 ===")
if MASTER_LOG.exists():
    log = pd.read_csv(MASTER_LOG)
    seed456_log = log[log["experiment_id"].astype(str) == "DenseNet201_A_official_seed456"]
    display(seed456_log)
else:
    print("Master log belum ditemukan.")

print("\n=== Comparison Table ===")
if COMPARISON_CSV.exists():
    comp = pd.read_csv(COMPARISON_CSV)
    display(comp)
else:
    print("Comparison CSV belum ditemukan.")

print("\n=== Keputusan B23-VERIFY ===")
print("best_ok:", best_ok)
print("last_ok:", last_ok)
print("eval_ok:", eval_ok)

if best_ok and last_ok and eval_ok:
    print("Seed 456 valid. Boleh dilaporkan ke MASTER CHAT sebagai selesai.")
else:
    print("Seed 456 belum aman. Jangan lanjut seed 789 sebelum dicek ulang.")

In [ ]:
# Sel B24 — Multi-Seed DenseNet201 A Official Seed 789

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B24: Multi-Seed DenseNet201 A Official Seed 789 ===")
print("Mode kerja: menjalankan seed 789 untuk Skenario A official. Tidak lanjut seed 1024 di sel ini.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B24.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_WORK_DIR = Path("/content/seed789_safe_work")
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_official_seed789"
EXPERIMENT_ID = "DenseNet201_A_official_seed789"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

TEMP_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_A_official_multiseed_temporary_comparison_until_seed789.csv"

print("\n=== Cek Path ===")
print("Manifest A:", MANIFEST_A_PATH, "| exists:", MANIFEST_A_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_A_PATH.exists():
    raise FileNotFoundError(f"Manifest A tidak ditemukan: {MANIFEST_A_PATH}")

# ============================================================
# 3. Fungsi save checkpoint aman
# ============================================================

def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    """
    Simpan checkpoint ke /content dulu, cek bisa dibaca,
    lalu copy ke Drive dan cek lagi.
    """
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)

    # Verifikasi lokal
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    os.replace(local_tmp, local_path)

    # Copy ke Drive via file sementara
    shutil.copy2(local_path, drive_tmp)

    # Verifikasi file sementara di Drive
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)

    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed42_rerun1",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

# Tambahkan catatan technical failure/recovery pada seed 456 jika belum jelas
mask_456 = master_log["experiment_id"].astype(str) == "DenseNet201_A_official_seed456"
if mask_456.any():
    old_note = str(master_log.loc[mask_456, "notes"].iloc[0])
    recovery_note = (
        " Technical note: B23 original failed during evaluation due to corrupt best checkpoint; "
        "final seed 456 result comes from B23-R2 recovery and was verified by B23-VERIFY."
    )
    if "B23 original failed" not in old_note and "checkpoint corrupt" not in old_note:
        master_log.loc[mask_456, "notes"] = old_note + recovery_note

master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 789
# ============================================================

SEED = 789
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Sama seperti eksperimen sebelumnya.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 789 ===")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 A official seed 789. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai seed 789.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest dan DataLoader
# ============================================================

df = pd.read_csv(MANIFEST_A_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest A ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 9. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 10. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 11. Training seed 789
# ============================================================

print("\n=== Mulai Training Seed 789 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_official",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_official",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_official",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training Seed 789 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining seed 789 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 12. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint Seed 789 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics seed 789:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics seed 789:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report seed 789:")
print(test_report_text)

# ============================================================
# 13. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_official",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 14. Update master log: selesai seed 789
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 A official seed 789 selesai. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 15. Tabel perbandingan sementara seed 42 initial, 123, 456, 789
# ============================================================

def read_metrics_for_comparison(run_prefix, experiment_id, seed, include=True):
    metrics_path = OUTPUT_DIR / f"{run_prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{run_prefix}_training_summary.json"

    if not metrics_path.exists():
        return {
            "experiment_id": experiment_id,
            "seed": seed,
            "include_in_final_multiseed": include,
            "status": "missing_metrics"
        }

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    if training_path.exists():
        with open(training_path, "r") as f:
            training = json.load(f)
    else:
        training = {}

    return {
        "experiment_id": experiment_id,
        "seed": seed,
        "include_in_final_multiseed": include,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": metrics["validation"]["accuracy"],
        "val_macro_f1": metrics["validation"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_weighted_f1": metrics["test"]["weighted_f1"],
        "status": "ok"
    }

comparison_rows = [
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed42",
        "DenseNet201_A_official_seed42_initial",
        42,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed123",
        "DenseNet201_A_official_seed123",
        123,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed456",
        "DenseNet201_A_official_seed456",
        456,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed789",
        "DenseNet201_A_official_seed789",
        789,
        True
    )
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TEMP_COMPARISON_CSV, index=False)

# ============================================================
# 16. Ringkasan akhir
# ============================================================

print("\n=== File Output Seed 789 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Tabel perbandingan sementara:", TEMP_COMPARISON_CSV)

print("\n=== Ringkasan Sel B24 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Perbandingan Sementara A Official Multi-Seed ===")
display(comparison_df)

print("\nSel B24 selesai. Jangan lanjut seed 1024 sebelum ada instruksi berikutnya.")

In [ ]:
# Sel B25 — Multi-Seed DenseNet201 A Official Seed 1024

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B25: Multi-Seed DenseNet201 A Official Seed 1024 ===")
print("Mode kerja: menjalankan seed final 1024 untuk Skenario A official.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B25.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_WORK_DIR = Path("/content/seed1024_safe_work")
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_official_seed1024"
EXPERIMENT_ID = "DenseNet201_A_official_seed1024"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

FINAL_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_A_official_multiseed_final_5seed_results.csv"
FINAL_STATS_CSV = OUTPUT_DIR / "DenseNet201_A_official_multiseed_final_5seed_summary_stats.csv"
FINAL_SUMMARY_JSON = OUTPUT_DIR / "DenseNet201_A_official_multiseed_final_5seed_summary.json"

print("\n=== Cek Path ===")
print("Manifest A:", MANIFEST_A_PATH, "| exists:", MANIFEST_A_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_A_PATH.exists():
    raise FileNotFoundError(f"Manifest A tidak ditemukan: {MANIFEST_A_PATH}")

# ============================================================
# 3. Fungsi save checkpoint aman
# ============================================================

def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    """
    Simpan checkpoint ke /content dulu, cek bisa dibaca,
    lalu copy ke Drive dan cek lagi.
    """
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)

    # Verifikasi lokal
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    os.replace(local_tmp, local_path)

    # Copy ke Drive via file sementara
    shutil.copy2(local_path, drive_tmp)

    # Verifikasi file sementara di Drive
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)

    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed42_rerun1",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 1024
# ============================================================

SEED = 1024
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Sama seperti eksperimen sebelumnya.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 1024 ===")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 A official seed 1024. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai seed 1024.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest dan DataLoader
# ============================================================

df = pd.read_csv(MANIFEST_A_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest A ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 9. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 10. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 11. Training seed 1024
# ============================================================

print("\n=== Mulai Training Seed 1024 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_official",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_official",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_official",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training Seed 1024 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining seed 1024 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 12. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint Seed 1024 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics seed 1024:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics seed 1024:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report seed 1024:")
print(test_report_text)

# ============================================================
# 13. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_official",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 14. Update master log: selesai seed 1024
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_official",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 A official seed 1024 selesai. Seed final untuk 5-seed A official."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 15. Tabel final 5-seed dan statistik mean/std
# ============================================================

def read_metrics_for_comparison(run_prefix, experiment_id, seed, include=True):
    metrics_path = OUTPUT_DIR / f"{run_prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{run_prefix}_training_summary.json"

    if not metrics_path.exists():
        return {
            "experiment_id": experiment_id,
            "seed": seed,
            "include_in_final_multiseed": include,
            "status": "missing_metrics"
        }

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    if training_path.exists():
        with open(training_path, "r") as f:
            training = json.load(f)
    else:
        training = {}

    return {
        "experiment_id": experiment_id,
        "seed": seed,
        "include_in_final_multiseed": include,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": metrics["validation"]["accuracy"],
        "val_macro_f1": metrics["validation"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_weighted_f1": metrics["test"]["weighted_f1"],
        "status": "ok"
    }

comparison_rows = [
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed42",
        "DenseNet201_A_official_seed42_initial",
        42,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed123",
        "DenseNet201_A_official_seed123",
        123,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed456",
        "DenseNet201_A_official_seed456",
        456,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed789",
        "DenseNet201_A_official_seed789",
        789,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_A_official_seed1024",
        "DenseNet201_A_official_seed1024",
        1024,
        True
    )
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(FINAL_COMPARISON_CSV, index=False)

metric_cols = [
    "val_accuracy",
    "val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1"
]

valid_final_df = comparison_df[
    (comparison_df["include_in_final_multiseed"] == True) &
    (comparison_df["status"] == "ok")
].copy()

stats_rows = []

for metric in metric_cols:
    values = valid_final_df[metric].astype(float)

    stats_rows.append({
        "metric": metric,
        "n_seed": int(values.shape[0]),
        "mean": float(values.mean()),
        "std_sample": float(values.std(ddof=1)),
        "std_population": float(values.std(ddof=0)),
        "min": float(values.min()),
        "max": float(values.max())
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(FINAL_STATS_CSV, index=False)

final_summary_json = {
    "created_at": datetime.now().isoformat(),
    "model": "DenseNet201",
    "scenario": "A_official",
    "run_type": "final_5_seed_summary",
    "included_seeds": [42, 123, 456, 789, 1024],
    "excluded_runs": [
        {
            "experiment_id": "DenseNet201_A_official_seed42_rerun1",
            "reason": "determinism check only, not included in final multi-seed statistics"
        }
    ],
    "comparison_csv": str(FINAL_COMPARISON_CSV),
    "summary_stats_csv": str(FINAL_STATS_CSV),
    "comparison_records": comparison_df.to_dict(orient="records"),
    "summary_stats": stats_df.to_dict(orient="records"),
    "benchmark_reference_2021": {
        "accuracy": 0.931,
        "f1_score": 0.921,
        "note": "Matuszewski & Sintorn 2021 benchmark reference used in current project notes"
    },
    "notes": [
        "A official is literature-comparable evaluation.",
        "Seed 42 rerun1 is excluded from final statistics.",
        "Final summary uses seed 42 initial, 123, 456, 789, and 1024 only."
    ]
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(final_summary_json, f, indent=2)

# ============================================================
# 16. Ringkasan akhir
# ============================================================

print("\n=== File Output Seed 1024 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)

print("\n=== File Final A Official 5-Seed Tersimpan ===")
print("Final 5-seed results:", FINAL_COMPARISON_CSV)
print("Final 5-seed summary stats:", FINAL_STATS_CSV)
print("Final 5-seed summary JSON:", FINAL_SUMMARY_JSON)

print("\n=== Ringkasan Sel B25 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Tabel Final A Official 5 Seed ===")
display(comparison_df)

print("\n=== Statistik Final A Official 5 Seed ===")
display(stats_df)

print("\nSel B25 selesai. Jangan lanjut ke skenario lain sebelum hasil 5-seed dilaporkan ke MASTER CHAT.")

In [ ]:
# Sel B26 — Multi-Seed DenseNet201 B-G14 Seed 123

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B26: Multi-Seed DenseNet201 B-G14 Seed 123 ===")
print("Mode kerja: menjalankan seed 123 untuk Skenario B-G14 RAWSource. Tidak lanjut seed 456 di sel ini.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B26.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_WORK_DIR = Path("/content/seed123_B_G14_safe_work")
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_B_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_B_G14_seed123"
EXPERIMENT_ID = "DenseNet201_B_G14_seed123"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

TEMP_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_temporary_comparison_until_seed123.csv"

print("\n=== Cek Path ===")
print("Manifest B-G14:", MANIFEST_B_PATH, "| exists:", MANIFEST_B_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_B_PATH.exists():
    raise FileNotFoundError(f"Manifest B-G14 tidak ditemukan: {MANIFEST_B_PATH}")

# ============================================================
# 3. Fungsi simpan checkpoint aman
# ============================================================

def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
    "DenseNet201_A_official_seed1024",
    "DenseNet201_B_G14_seed42",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 123 B-G14
# ============================================================

SEED = 123
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 123 B-G14 ===")
print("Scenario: B-G14 RAWSource")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 123. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai B-G14 seed 123.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest B-G14 dan cek split/group
# ============================================================

df = pd.read_csv(MANIFEST_B_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest B-G14 ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

for col in ["group_id", "raw_source_id"]:
    if col in df.columns:
        train_set = set(df[df["split"] == "train"][col].astype(str))
        val_set = set(df[df["split"] == "validation"][col].astype(str))
        test_set = set(df[df["split"] == "test"][col].astype(str))

        print(f"\n=== Overlap Check: {col} ===")
        print("train-validation:", len(train_set & val_set))
        print("train-test:", len(train_set & test_set))
        print("validation-test:", len(val_set & test_set))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nTrain/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 9. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 10. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 11. Training seed 123 B-G14
# ============================================================

print("\n=== Mulai Training B-G14 Seed 123 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "B_G14",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "B_G14",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "B_G14",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training B-G14 Seed 123 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining B-G14 seed 123 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 12. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint B-G14 Seed 123 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics B-G14 seed 123:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics B-G14 seed 123:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report B-G14 seed 123:")
print(test_report_text)

# ============================================================
# 13. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "B_G14",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 14. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 123 selesai. Jangan lanjut seed 456 sebelum dilaporkan."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 15. Tabel sementara B-G14 seed 42 dan 123
# ============================================================

def read_metrics_for_comparison(run_prefix, experiment_id, seed, include=True):
    metrics_path = OUTPUT_DIR / f"{run_prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{run_prefix}_training_summary.json"

    if not metrics_path.exists():
        return {
            "experiment_id": experiment_id,
            "seed": seed,
            "include_in_final_multiseed": include,
            "status": "missing_metrics"
        }

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    if training_path.exists():
        with open(training_path, "r") as f:
            training = json.load(f)
    else:
        training = {}

    return {
        "experiment_id": experiment_id,
        "seed": seed,
        "include_in_final_multiseed": include,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": metrics["validation"]["accuracy"],
        "val_macro_f1": metrics["validation"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_weighted_f1": metrics["test"]["weighted_f1"],
        "status": "ok"
    }

comparison_rows = [
    read_metrics_for_comparison(
        "DenseNet201_B_G14_seed42",
        "DenseNet201_B_G14_seed42",
        42,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_B_G14_seed123",
        "DenseNet201_B_G14_seed123",
        123,
        True
    )
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TEMP_COMPARISON_CSV, index=False)

# ============================================================
# 16. Ringkasan akhir
# ============================================================

print("\n=== File Output B-G14 Seed 123 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Tabel sementara B-G14:", TEMP_COMPARISON_CSV)

print("\n=== Ringkasan Sel B26 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Perbandingan Sementara B-G14 Multi-Seed ===")
display(comparison_df)

print("\nSel B26 selesai. Jangan lanjut seed 456 sebelum ada instruksi berikutnya.")

In [ ]:
# Sel B27 — Multi-Seed DenseNet201 B-G14 Seed 456

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B27: Multi-Seed DenseNet201 B-G14 Seed 456 ===")
print("Mode kerja: menjalankan seed final 456 untuk Skenario B-G14 RAWSource.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B27.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_WORK_DIR = Path("/content/seed456_B_G14_safe_work")
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_B_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_B_G14_seed456"
EXPERIMENT_ID = "DenseNet201_B_G14_seed456"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

FINAL_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_3seed_results.csv"
FINAL_STATS_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_3seed_summary_stats.csv"
FINAL_SUMMARY_JSON = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_3seed_summary.json"

print("\n=== Cek Path ===")
print("Manifest B-G14:", MANIFEST_B_PATH, "| exists:", MANIFEST_B_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_B_PATH.exists():
    raise FileNotFoundError(f"Manifest B-G14 tidak ditemukan: {MANIFEST_B_PATH}")

# ============================================================
# 3. Fungsi simpan checkpoint aman
# ============================================================

def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
    "DenseNet201_A_official_seed1024",
    "DenseNet201_B_G14_seed42",
    "DenseNet201_B_G14_seed123",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 456 B-G14
# ============================================================

SEED = 456
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 456 B-G14 ===")
print("Scenario: B-G14 RAWSource")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 456. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai B-G14 seed 456.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest B-G14 dan cek split/group
# ============================================================

df = pd.read_csv(MANIFEST_B_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest B-G14 ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

for col in ["group_id", "raw_source_id"]:
    if col in df.columns:
        train_set = set(df[df["split"] == "train"][col].astype(str))
        val_set = set(df[df["split"] == "validation"][col].astype(str))
        test_set = set(df[df["split"] == "test"][col].astype(str))

        print(f"\n=== Overlap Check: {col} ===")
        print("train-validation:", len(train_set & val_set))
        print("train-test:", len(train_set & test_set))
        print("validation-test:", len(val_set & test_set))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nTrain/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 9. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 10. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 11. Training seed 456 B-G14
# ============================================================

print("\n=== Mulai Training B-G14 Seed 456 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "B_G14",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "B_G14",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "B_G14",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training B-G14 Seed 456 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining B-G14 seed 456 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 12. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint B-G14 Seed 456 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics B-G14 seed 456:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics B-G14 seed 456:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report B-G14 seed 456:")
print(test_report_text)

# ============================================================
# 13. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "B_G14",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 14. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 456 selesai. Seed final untuk B-G14 3-seed."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 15. Tabel final B-G14 3 seed dan statistik
# ============================================================

def read_metrics_for_comparison(run_prefix, experiment_id, seed, include=True):
    metrics_path = OUTPUT_DIR / f"{run_prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{run_prefix}_training_summary.json"

    if not metrics_path.exists():
        return {
            "experiment_id": experiment_id,
            "seed": seed,
            "include_in_final_multiseed": include,
            "status": "missing_metrics"
        }

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    if training_path.exists():
        with open(training_path, "r") as f:
            training = json.load(f)
    else:
        training = {}

    return {
        "experiment_id": experiment_id,
        "seed": seed,
        "include_in_final_multiseed": include,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": metrics["validation"]["accuracy"],
        "val_macro_f1": metrics["validation"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_weighted_f1": metrics["test"]["weighted_f1"],
        "status": "ok"
    }

comparison_rows = [
    read_metrics_for_comparison(
        "DenseNet201_B_G14_seed42",
        "DenseNet201_B_G14_seed42",
        42,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_B_G14_seed123",
        "DenseNet201_B_G14_seed123",
        123,
        True
    ),
    read_metrics_for_comparison(
        "DenseNet201_B_G14_seed456",
        "DenseNet201_B_G14_seed456",
        456,
        True
    )
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(FINAL_COMPARISON_CSV, index=False)

metric_cols = [
    "val_accuracy",
    "val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1"
]

valid_final_df = comparison_df[
    (comparison_df["include_in_final_multiseed"] == True) &
    (comparison_df["status"] == "ok")
].copy()

stats_rows = []

for metric in metric_cols:
    values = valid_final_df[metric].astype(float)

    stats_rows.append({
        "metric": metric,
        "n_seed": int(values.shape[0]),
        "mean": float(values.mean()),
        "std_sample": float(values.std(ddof=1)),
        "std_population": float(values.std(ddof=0)),
        "min": float(values.min()),
        "max": float(values.max())
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(FINAL_STATS_CSV, index=False)

final_summary_json = {
    "created_at": datetime.now().isoformat(),
    "model": "DenseNet201",
    "scenario": "B_G14_RAWSource",
    "run_type": "final_3_seed_summary",
    "included_seeds": [42, 123, 456],
    "comparison_csv": str(FINAL_COMPARISON_CSV),
    "summary_stats_csv": str(FINAL_STATS_CSV),
    "comparison_records": comparison_df.to_dict(orient="records"),
    "summary_stats": stats_df.to_dict(orient="records"),
    "notes": [
        "B-G14 is RAW source-aware leakage-controlled evaluation.",
        "Final B-G14 summary uses seed 42, 123, and 456.",
        "Validation metrics are lower than test metrics, consistent with previous val-test gap analysis."
    ]
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(final_summary_json, f, indent=2)

# ============================================================
# 16. Ringkasan akhir
# ============================================================

print("\n=== File Output B-G14 Seed 456 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)

print("\n=== File Final B-G14 3-Seed Tersimpan ===")
print("Final 3-seed results:", FINAL_COMPARISON_CSV)
print("Final 3-seed summary stats:", FINAL_STATS_CSV)
print("Final 3-seed summary JSON:", FINAL_SUMMARY_JSON)

print("\n=== Ringkasan Sel B27 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Tabel Final B-G14 3 Seed ===")
display(comparison_df)

print("\n=== Statistik Final B-G14 3 Seed ===")
display(stats_df)

In [ ]:
# Sel B28 — Multi-Seed DenseNet201 B-G14 Seed 789 dengan Local-First Checkpoint

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B28: Multi-Seed DenseNet201 B-G14 Seed 789 ===")
print("Mode kerja: menjalankan seed 789 untuk Skenario B-G14 RAWSource.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B28.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_B_G14_seed789"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_B_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_B_G14_seed789"
EXPERIMENT_ID = "DenseNet201_B_G14_seed789"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

TEMP_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_temporary_comparison_until_seed789.csv"
CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"

print("\n=== Cek Path ===")
print("Manifest B-G14:", MANIFEST_B_PATH, "| exists:", MANIFEST_B_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)
print("Local checkpoint dir:", LOCAL_CKPT_DIR)
print("Drive checkpoint dir:", CHECKPOINT_DIR)

if not MANIFEST_B_PATH.exists():
    raise FileNotFoundError(f"Manifest B-G14 tidak ditemukan: {MANIFEST_B_PATH}")

# ============================================================
# 3. Fungsi simpan dan verifikasi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    """
    Pola aman:
    1. torch.save ke local tmp
    2. verify local tmp dengan torch.load
    3. rename local tmp ke local final
    4. copy local final ke Drive tmp
    5. verify Drive tmp dengan torch.load
    6. rename Drive tmp ke Drive final
    """
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)

    # Verifikasi local tmp
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    os.replace(local_tmp, local_path)

    # Copy ke Drive sebagai tmp
    shutil.copy2(local_path, drive_tmp)

    # Verifikasi Drive tmp
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)

    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
    "DenseNet201_A_official_seed1024",
    "DenseNet201_B_G14_seed42",
    "DenseNet201_B_G14_seed123",
    "DenseNet201_B_G14_seed456",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 789 B-G14
# ============================================================

SEED = 789
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 789 B-G14 ===")
print("Scenario: B-G14 RAWSource")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 789. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai B-G14 seed 789.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest B-G14 dan cek split/group
# ============================================================

df = pd.read_csv(MANIFEST_B_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest B-G14 ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

for col in ["group_id", "raw_source_id"]:
    if col in df.columns:
        train_set = set(df[df["split"] == "train"][col].astype(str))
        val_set = set(df[df["split"] == "validation"][col].astype(str))
        test_set = set(df[df["split"] == "test"][col].astype(str))

        print(f"\n=== Overlap Check: {col} ===")
        print("train-validation:", len(train_set & val_set))
        print("train-test:", len(train_set & test_set))
        print("validation-test:", len(val_set & test_set))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nTrain/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 9. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 10. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 11. Training seed 789 B-G14
# ============================================================

print("\n=== Mulai Training B-G14 Seed 789 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "B_G14",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "B_G14",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "B_G14",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training B-G14 Seed 789 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining B-G14 seed 789 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 12. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint B-G14 Seed 789 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics B-G14 seed 789:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics B-G14 seed 789:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report B-G14 seed 789:")
print(test_report_text)

# ============================================================
# 13. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "B_G14",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 14. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 789 selesai. Checkpoint local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 15. Tabel sementara B-G14 seed 42, 123, 456, 789
# ============================================================

def read_metrics_for_comparison(run_prefix, experiment_id, seed, include=True):
    metrics_path = OUTPUT_DIR / f"{run_prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{run_prefix}_training_summary.json"

    if not metrics_path.exists():
        return {
            "experiment_id": experiment_id,
            "seed": seed,
            "include_in_final_multiseed": include,
            "status": "missing_metrics"
        }

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    if training_path.exists():
        with open(training_path, "r") as f:
            training = json.load(f)
    else:
        training = {}

    return {
        "experiment_id": experiment_id,
        "seed": seed,
        "include_in_final_multiseed": include,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": metrics["validation"]["accuracy"],
        "val_macro_f1": metrics["validation"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_weighted_f1": metrics["test"]["weighted_f1"],
        "status": "ok"
    }

comparison_rows = [
    read_metrics_for_comparison("DenseNet201_B_G14_seed42", "DenseNet201_B_G14_seed42", 42, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed123", "DenseNet201_B_G14_seed123", 123, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed456", "DenseNet201_B_G14_seed456", 456, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed789", "DenseNet201_B_G14_seed789", 789, True),
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TEMP_COMPARISON_CSV, index=False)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Ringkasan akhir
# ============================================================

print("\n=== File Output B-G14 Seed 789 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Tabel sementara B-G14:", TEMP_COMPARISON_CSV)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)

print("\n=== Ringkasan Sel B28 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Perbandingan Sementara B-G14 Multi-Seed ===")
display(comparison_df)

print("\nSel B28 selesai. Jangan lanjut seed 1024 sebelum hasil seed 789 dilaporkan ke MASTER CHAT.")

In [ ]:
# Sel B29 — Multi-Seed DenseNet201 B-G14 Seed 1024 dengan Local-First Checkpoint

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B29: Multi-Seed DenseNet201 B-G14 Seed 1024 ===")
print("Mode kerja: menjalankan seed final 1024 untuk Skenario B-G14 RAWSource.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B29.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_B_G14_seed1024"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_B_PATH = OUTPUT_DIR / "v5_split_manifest_B_conservative_G14_RAWSource.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_B_G14_seed1024"
EXPERIMENT_ID = "DenseNet201_B_G14_seed1024"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

FINAL_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_5seed_results.csv"
FINAL_STATS_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_5seed_summary_stats.csv"
FINAL_SUMMARY_JSON = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_5seed_summary.json"
CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"

print("\n=== Cek Path ===")
print("Manifest B-G14:", MANIFEST_B_PATH, "| exists:", MANIFEST_B_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)
print("Local checkpoint dir:", LOCAL_CKPT_DIR)
print("Drive checkpoint dir:", CHECKPOINT_DIR)

if not MANIFEST_B_PATH.exists():
    raise FileNotFoundError(f"Manifest B-G14 tidak ditemukan: {MANIFEST_B_PATH}")

# ============================================================
# 3. Fungsi simpan dan verifikasi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)

    # Verifikasi local tmp
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    os.replace(local_tmp, local_path)

    # Copy ke Drive sebagai tmp
    shutil.copy2(local_path, drive_tmp)

    # Verifikasi Drive tmp
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)

    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
    "DenseNet201_A_official_seed1024",
    "DenseNet201_B_G14_seed42",
    "DenseNet201_B_G14_seed123",
    "DenseNet201_B_G14_seed456",
    "DenseNet201_B_G14_seed789",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 1024 B-G14
# ============================================================

SEED = 1024
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 1024 B-G14 ===")
print("Scenario: B-G14 RAWSource")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 1024. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai B-G14 seed 1024.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest B-G14 dan cek split/group
# ============================================================

df = pd.read_csv(MANIFEST_B_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest B-G14 ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

for col in ["group_id", "raw_source_id"]:
    if col in df.columns:
        train_set = set(df[df["split"] == "train"][col].astype(str))
        val_set = set(df[df["split"] == "validation"][col].astype(str))
        test_set = set(df[df["split"] == "test"][col].astype(str))

        print(f"\n=== Overlap Check: {col} ===")
        print("train-validation:", len(train_set & val_set))
        print("train-test:", len(train_set & test_set))
        print("validation-test:", len(val_set & test_set))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nTrain/Validation/Test:", len(train_df), len(val_df), len(test_df))

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 9. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 10. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 11. Training seed 1024 B-G14
# ============================================================

print("\n=== Mulai Training B-G14 Seed 1024 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "B_G14",
                "run_type": "multi_seed",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "B_G14",
            "run_type": "multi_seed",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "B_G14",
        "run_type": "multi_seed",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training B-G14 Seed 1024 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining B-G14 seed 1024 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 12. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint B-G14 Seed 1024 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics B-G14 seed 1024:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics B-G14 seed 1024:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report B-G14 seed 1024:")
print(test_report_text)

# ============================================================
# 13. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "B_G14",
    "run_type": "multi_seed",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 14. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "B_G14",
    "seed": SEED,
    "run_type": "multi_seed",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Multi-seed DenseNet201 B-G14 RAWSource seed 1024 selesai. Checkpoint local-first-then-Drive. Seed final B-G14 5-seed."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 15. Tabel final B-G14 5 seed dan statistik
# ============================================================

def read_metrics_for_comparison(run_prefix, experiment_id, seed, include=True):
    metrics_path = OUTPUT_DIR / f"{run_prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{run_prefix}_training_summary.json"

    if not metrics_path.exists():
        return {
            "experiment_id": experiment_id,
            "seed": seed,
            "include_in_final_multiseed": include,
            "status": "missing_metrics"
        }

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    if training_path.exists():
        with open(training_path, "r") as f:
            training = json.load(f)
    else:
        training = {}

    return {
        "experiment_id": experiment_id,
        "seed": seed,
        "include_in_final_multiseed": include,
        "best_epoch": training.get("best_epoch", metrics.get("best_epoch_from_checkpoint", "")),
        "val_accuracy": metrics["validation"]["accuracy"],
        "val_macro_f1": metrics["validation"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_weighted_f1": metrics["test"]["weighted_f1"],
        "status": "ok"
    }

comparison_rows = [
    read_metrics_for_comparison("DenseNet201_B_G14_seed42", "DenseNet201_B_G14_seed42", 42, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed123", "DenseNet201_B_G14_seed123", 123, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed456", "DenseNet201_B_G14_seed456", 456, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed789", "DenseNet201_B_G14_seed789", 789, True),
    read_metrics_for_comparison("DenseNet201_B_G14_seed1024", "DenseNet201_B_G14_seed1024", 1024, True),
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(FINAL_COMPARISON_CSV, index=False)

metric_cols = [
    "val_accuracy",
    "val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1"
]

valid_final_df = comparison_df[
    (comparison_df["include_in_final_multiseed"] == True) &
    (comparison_df["status"] == "ok")
].copy()

stats_rows = []

for metric in metric_cols:
    values = valid_final_df[metric].astype(float)

    stats_rows.append({
        "metric": metric,
        "n_seed": int(values.shape[0]),
        "mean": float(values.mean()),
        "std_sample": float(values.std(ddof=1)),
        "std_population": float(values.std(ddof=0)),
        "min": float(values.min()),
        "max": float(values.max())
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(FINAL_STATS_CSV, index=False)

final_summary_json = {
    "created_at": datetime.now().isoformat(),
    "model": "DenseNet201",
    "scenario": "B_G14_RAWSource",
    "run_type": "final_5_seed_summary",
    "included_seeds": [42, 123, 456, 789, 1024],
    "comparison_csv": str(FINAL_COMPARISON_CSV),
    "summary_stats_csv": str(FINAL_STATS_CSV),
    "comparison_records": comparison_df.to_dict(orient="records"),
    "summary_stats": stats_df.to_dict(orient="records"),
    "notes": [
        "B-G14 is RAW source-aware leakage-controlled evaluation.",
        "Final B-G14 summary uses seed 42, 123, 456, 789, and 1024.",
        "Validation metrics are lower than test metrics, consistent with previous val-test gap analysis.",
        "Checkpoint policy for seed 1024 uses local-first-then-Drive with load verification."
    ]
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(final_summary_json, f, indent=2)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Ringkasan akhir
# ============================================================

print("\n=== File Output B-G14 Seed 1024 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Master log:", MASTER_LOG_PATH)

print("\n=== File Final B-G14 5-Seed Tersimpan ===")
print("Final 5-seed results:", FINAL_COMPARISON_CSV)
print("Final 5-seed summary stats:", FINAL_STATS_CSV)
print("Final 5-seed summary JSON:", FINAL_SUMMARY_JSON)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)

print("\n=== Ringkasan Sel B29 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Tabel Final B-G14 5 Seed ===")
display(comparison_df)

print("\n=== Statistik Final B-G14 5 Seed ===")
display(stats_df)

print("\nSel B29 selesai. Jangan lanjut ke skenario lain sebelum hasil B-G14 5-seed dilaporkan ke MASTER CHAT.")

In [ ]:
# Sel B30 — Statistical Test dan Per-Class Error Analysis DenseNet201 5-Seed

import json
import math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

print("=== Sel B30: Statistical Test dan Per-Class Error Analysis DenseNet201 5-Seed ===")
print("Mode kerja: analisis non-training. Tidak memakai GPU. Tidak mengubah data.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

# ============================================================
# 1. Path dan konfigurasi
# ============================================================

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

A_RESULTS_CSV = OUTPUT_DIR / "DenseNet201_A_official_multiseed_final_5seed_results.csv"
A_STATS_CSV = OUTPUT_DIR / "DenseNet201_A_official_multiseed_final_5seed_summary_stats.csv"
B_RESULTS_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_5seed_results.csv"
B_STATS_CSV = OUTPUT_DIR / "DenseNet201_B_G14_multiseed_final_5seed_summary_stats.csv"

CI_OUT = OUTPUT_DIR / "DenseNet201_AB_5seed_confidence_interval_summary.csv"
TTEST_OUT = OUTPUT_DIR / "DenseNet201_AB_5seed_statistical_test_vs_benchmark.csv"
AB_COMP_OUT = OUTPUT_DIR / "DenseNet201_AB_5seed_A_vs_B_descriptive_comparison.csv"
A_PER_CLASS_OUT = OUTPUT_DIR / "DenseNet201_A_official_5seed_per_class_summary.csv"
B_PER_CLASS_OUT = OUTPUT_DIR / "DenseNet201_B_G14_5seed_per_class_summary.csv"
A_AGG_CM_OUT = OUTPUT_DIR / "DenseNet201_A_official_5seed_aggregate_confusion_matrix.csv"
B_AGG_CM_OUT = OUTPUT_DIR / "DenseNet201_B_G14_5seed_aggregate_confusion_matrix.csv"
SUMMARY_JSON_OUT = OUTPUT_DIR / "DenseNet201_AB_5seed_error_analysis_summary.json"

BENCHMARKS = {
    "test_accuracy": 0.931,
    "test_macro_f1": 0.921,
}

METRIC_COLS = [
    "val_accuracy",
    "val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1"
]

FOCUS_CLASSES = ["Lassa", "Orf", "Cowpox", "Adenovirus", "Marburg"]

A_SEEDS = [42, 123, 456, 789, 1024]
B_SEEDS = [42, 123, 456, 789, 1024]

A_PREFIX = {
    42: "DenseNet201_A_official_seed42",
    123: "DenseNet201_A_official_seed123",
    456: "DenseNet201_A_official_seed456",
    789: "DenseNet201_A_official_seed789",
    1024: "DenseNet201_A_official_seed1024",
}

B_PREFIX = {
    42: "DenseNet201_B_G14_seed42",
    123: "DenseNet201_B_G14_seed123",
    456: "DenseNet201_B_G14_seed456",
    789: "DenseNet201_B_G14_seed789",
    1024: "DenseNet201_B_G14_seed1024",
}

SCENARIOS = {
    "A_official": {
        "label": "A official",
        "results_csv": A_RESULTS_CSV,
        "stats_csv": A_STATS_CSV,
        "seeds": A_SEEDS,
        "prefix": A_PREFIX,
    },
    "B_G14": {
        "label": "B-G14 RAWSource",
        "results_csv": B_RESULTS_CSV,
        "stats_csv": B_STATS_CSV,
        "seeds": B_SEEDS,
        "prefix": B_PREFIX,
    }
}

print("\n=== File Check ===")
for path in [A_RESULTS_CSV, A_STATS_CSV, B_RESULTS_CSV, B_STATS_CSV]:
    print(f"{path.name}: exists={path.exists()} | path={path}")

# ============================================================
# 2. Helper statistik
# ============================================================

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    stats = None

print("\nScipy available:", SCIPY_AVAILABLE)

def t_critical_95(n):
    if n <= 1:
        return np.nan
    if SCIPY_AVAILABLE:
        return float(stats.t.ppf(0.975, df=n-1))
    # fallback khusus n=5; cukup untuk kasus kita
    fallback = {
        2: 12.706,
        3: 4.303,
        4: 3.182,
        5: 2.776,
        6: 2.571,
    }
    return fallback.get(n, 1.96)

def summarize_with_ci(values, scenario, metric):
    values = np.array(values, dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    std_sample = float(np.std(values, ddof=1)) if n > 1 else 0.0
    std_population = float(np.std(values, ddof=0)) if n > 0 else 0.0
    se = float(std_sample / math.sqrt(n)) if n > 1 else 0.0
    tcrit = t_critical_95(n)
    ci_low = float(mean - tcrit * se) if n > 1 else mean
    ci_high = float(mean + tcrit * se) if n > 1 else mean

    # Bootstrap CI berbasis seed
    rng = np.random.default_rng(42)
    if n > 1:
        boot_means = []
        for _ in range(10000):
            sample = rng.choice(values, size=n, replace=True)
            boot_means.append(np.mean(sample))
        boot_ci_low, boot_ci_high = np.percentile(boot_means, [2.5, 97.5])
    else:
        boot_ci_low, boot_ci_high = mean, mean

    return {
        "scenario": scenario,
        "metric": metric,
        "n_seed": n,
        "mean": mean,
        "std_sample": std_sample,
        "std_population": std_population,
        "standard_error": se,
        "ci_95_t_low": ci_low,
        "ci_95_t_high": ci_high,
        "bootstrap_ci_95_low": float(boot_ci_low),
        "bootstrap_ci_95_high": float(boot_ci_high),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
    }

def one_sample_ttest(values, benchmark, scenario, metric):
    values = np.array(values, dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    std_sample = float(np.std(values, ddof=1)) if n > 1 else 0.0
    mean_diff = mean - benchmark

    if SCIPY_AVAILABLE and n > 1 and std_sample > 0:
        t_stat, p_two_sided = stats.ttest_1samp(values, popmean=benchmark)
        t_stat = float(t_stat)
        p_two_sided = float(p_two_sided)

        # one-sided greater: H1 mean > benchmark
        if t_stat >= 0:
            p_greater = p_two_sided / 2
        else:
            p_greater = 1 - (p_two_sided / 2)
    else:
        t_stat = np.nan
        p_two_sided = np.nan
        p_greater = np.nan

    ci = summarize_with_ci(values, scenario, metric)

    return {
        "scenario": scenario,
        "metric": metric,
        "benchmark": benchmark,
        "n_seed": n,
        "mean": mean,
        "std_sample": std_sample,
        "mean_minus_benchmark": mean_diff,
        "ci_95_t_low": ci["ci_95_t_low"],
        "ci_95_t_high": ci["ci_95_t_high"],
        "bootstrap_ci_95_low": ci["bootstrap_ci_95_low"],
        "bootstrap_ci_95_high": ci["bootstrap_ci_95_high"],
        "t_statistic": t_stat,
        "p_value_two_sided": p_two_sided,
        "p_value_one_sided_mean_greater": p_greater,
        "ci_low_above_benchmark": bool(ci["ci_95_t_low"] > benchmark),
        "bootstrap_ci_low_above_benchmark": bool(ci["bootstrap_ci_95_low"] > benchmark),
        "interpretation": (
            "mean and 95% CI are above benchmark"
            if ci["ci_95_t_low"] > benchmark
            else "mean is above benchmark, but 95% CI crosses benchmark"
            if mean > benchmark
            else "mean is not above benchmark"
        )
    }

# ============================================================
# 3. Load hasil final 5-seed dan hitung CI
# ============================================================

all_results = {}

for scenario_key, cfg in SCENARIOS.items():
    if not cfg["results_csv"].exists():
        raise FileNotFoundError(f"File hasil tidak ditemukan: {cfg['results_csv']}")

    df = pd.read_csv(cfg["results_csv"])
    df = df[df["include_in_final_multiseed"] == True].copy()
    df = df[df["status"] == "ok"].copy()
    df["scenario"] = scenario_key
    all_results[scenario_key] = df

print("\n=== Loaded Final Results ===")
for scenario_key, df in all_results.items():
    print(scenario_key, df.shape)
    display(df)

ci_rows = []

for scenario_key, df in all_results.items():
    for metric in METRIC_COLS:
        ci_rows.append(summarize_with_ci(df[metric].values, scenario_key, metric))

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(CI_OUT, index=False)

print("\n=== Confidence Interval Summary ===")
display(ci_df)

# ============================================================
# 4. T-test terhadap benchmark 2021
# ============================================================

ttest_rows = []

for scenario_key, df in all_results.items():
    for metric, benchmark in BENCHMARKS.items():
        ttest_rows.append(one_sample_ttest(df[metric].values, benchmark, scenario_key, metric))

ttest_df = pd.DataFrame(ttest_rows)
ttest_df.to_csv(TTEST_OUT, index=False)

print("\n=== Statistical Test vs Benchmark 2021 ===")
display(ttest_df)

# ============================================================
# 5. A vs B-G14 descriptive comparison
# ============================================================

a_df = all_results["A_official"].copy()
b_df = all_results["B_G14"].copy()

a_by_seed = a_df.set_index("seed")
b_by_seed = b_df.set_index("seed")

ab_rows = []

common_seeds = sorted(set(a_by_seed.index) & set(b_by_seed.index))

for metric in METRIC_COLS:
    a_values = a_df[metric].astype(float).values
    b_values = b_df[metric].astype(float).values

    a_mean = float(np.mean(a_values))
    b_mean = float(np.mean(b_values))
    mean_diff_b_minus_a = b_mean - a_mean

    paired_deltas = []
    for seed in common_seeds:
        paired_deltas.append(float(b_by_seed.loc[seed, metric] - a_by_seed.loc[seed, metric]))

    paired_deltas = np.array(paired_deltas, dtype=float)

    if len(paired_deltas) > 1:
        delta_mean = float(np.mean(paired_deltas))
        delta_std = float(np.std(paired_deltas, ddof=1))
        delta_se = float(delta_std / math.sqrt(len(paired_deltas)))
        delta_tcrit = t_critical_95(len(paired_deltas))
        delta_ci_low = float(delta_mean - delta_tcrit * delta_se)
        delta_ci_high = float(delta_mean + delta_tcrit * delta_se)
    else:
        delta_mean = np.nan
        delta_std = np.nan
        delta_ci_low = np.nan
        delta_ci_high = np.nan

    ab_rows.append({
        "metric": metric,
        "A_official_mean": a_mean,
        "B_G14_mean": b_mean,
        "B_minus_A_mean_difference": mean_diff_b_minus_a,
        "paired_seed_delta_mean_B_minus_A": delta_mean,
        "paired_seed_delta_std": delta_std,
        "paired_seed_delta_ci_95_low": delta_ci_low,
        "paired_seed_delta_ci_95_high": delta_ci_high,
        "common_seeds": ",".join(map(str, common_seeds)),
        "interpretation_note": "Descriptive only: A official and B-G14 are different evaluation protocols."
    })

ab_comp_df = pd.DataFrame(ab_rows)
ab_comp_df.to_csv(AB_COMP_OUT, index=False)

print("\n=== A vs B-G14 Descriptive Comparison ===")
display(ab_comp_df)

# ============================================================
# 6. Per-class summary across seeds
# ============================================================

def read_classification_report_csv(path):
    df = pd.read_csv(path)

    # Jika index class_name tersimpan sebagai Unnamed: 0
    if "Unnamed: 0" in df.columns:
        df = df.rename(columns={"Unnamed: 0": "class_name"})
    elif df.columns[0] not in ["class_name", "precision"]:
        df = df.rename(columns={df.columns[0]: "class_name"})

    if "class_name" not in df.columns:
        df = df.reset_index().rename(columns={"index": "class_name"})

    return df

def collect_per_class_reports(scenario_key, split):
    cfg = SCENARIOS[scenario_key]
    rows = []

    for seed in cfg["seeds"]:
        prefix = cfg["prefix"][seed]
        report_path = OUTPUT_DIR / f"{prefix}_{split}_classification_report.csv"

        if not report_path.exists():
            print("WARNING: report tidak ditemukan:", report_path)
            continue

        rep = read_classification_report_csv(report_path)

        # Ambil hanya class rows, bukan accuracy/macro avg/weighted avg
        rep = rep[~rep["class_name"].isin(["accuracy", "macro avg", "weighted avg"])].copy()

        for _, r in rep.iterrows():
            rows.append({
                "scenario": scenario_key,
                "split": split,
                "seed": seed,
                "class_name": r["class_name"],
                "precision": float(r["precision"]),
                "recall": float(r["recall"]),
                "f1": float(r["f1-score"]),
                "support": float(r["support"]),
            })

    return pd.DataFrame(rows)

def summarize_per_class(per_seed_df):
    rows = []

    for (scenario, split, class_name), g in per_seed_df.groupby(["scenario", "split", "class_name"]):
        rows.append({
            "scenario": scenario,
            "split": split,
            "class_name": class_name,
            "precision_mean": float(g["precision"].mean()),
            "precision_std": float(g["precision"].std(ddof=1)),
            "recall_mean": float(g["recall"].mean()),
            "recall_std": float(g["recall"].std(ddof=1)),
            "f1_mean": float(g["f1"].mean()),
            "f1_std": float(g["f1"].std(ddof=1)),
            "support_mean": float(g["support"].mean()),
            "support_min": float(g["support"].min()),
            "support_max": float(g["support"].max()),
            "n_seed": int(g["seed"].nunique()),
            "is_focus_class": class_name in FOCUS_CLASSES,
        })

    return pd.DataFrame(rows).sort_values(["split", "f1_mean", "class_name"]).reset_index(drop=True)

a_per_seed_reports = pd.concat([
    collect_per_class_reports("A_official", "validation"),
    collect_per_class_reports("A_official", "test"),
], ignore_index=True)

b_per_seed_reports = pd.concat([
    collect_per_class_reports("B_G14", "validation"),
    collect_per_class_reports("B_G14", "test"),
], ignore_index=True)

a_per_class_summary = summarize_per_class(a_per_seed_reports)
b_per_class_summary = summarize_per_class(b_per_seed_reports)

a_per_class_summary.to_csv(A_PER_CLASS_OUT, index=False)
b_per_class_summary.to_csv(B_PER_CLASS_OUT, index=False)

print("\n=== A Official Per-Class Summary ===")
display(a_per_class_summary)

print("\n=== B-G14 Per-Class Summary ===")
display(b_per_class_summary)

# ============================================================
# 7. Aggregate confusion matrix across seeds
# ============================================================

def read_confusion_matrix(path):
    cm = pd.read_csv(path, index_col=0)
    cm.index = cm.index.astype(str)
    cm.columns = cm.columns.astype(str)
    return cm

def aggregate_confusion_matrix(scenario_key, split="test"):
    cfg = SCENARIOS[scenario_key]
    agg_cm = None
    used_files = []

    for seed in cfg["seeds"]:
        prefix = cfg["prefix"][seed]
        cm_path = OUTPUT_DIR / f"{prefix}_{split}_confusion_matrix.csv"

        if not cm_path.exists():
            print("WARNING: confusion matrix tidak ditemukan:", cm_path)
            continue

        cm = read_confusion_matrix(cm_path)
        used_files.append(str(cm_path))

        if agg_cm is None:
            agg_cm = cm.copy().astype(float)
        else:
            # align index/columns untuk keamanan
            agg_cm = agg_cm.add(cm.astype(float), fill_value=0)

    if agg_cm is None:
        raise FileNotFoundError(f"Tidak ada confusion matrix untuk {scenario_key} split={split}")

    agg_cm = agg_cm.astype(int)
    return agg_cm, used_files

a_agg_cm, a_cm_files = aggregate_confusion_matrix("A_official", "test")
b_agg_cm, b_cm_files = aggregate_confusion_matrix("B_G14", "test")

a_agg_cm.to_csv(A_AGG_CM_OUT)
b_agg_cm.to_csv(B_AGG_CM_OUT)

print("\n=== A Official Aggregate Test Confusion Matrix ===")
display(a_agg_cm)

print("\n=== B-G14 Aggregate Test Confusion Matrix ===")
display(b_agg_cm)

# ============================================================
# 8. Error summary: hardest classes dan top confusions
# ============================================================

def summarize_hard_classes(per_class_summary, scenario_key, split="test"):
    df = per_class_summary[
        (per_class_summary["scenario"] == scenario_key) &
        (per_class_summary["split"] == split)
    ].copy()

    lowest_f1 = df.sort_values("f1_mean").head(5)
    lowest_recall = df.sort_values("recall_mean").head(5)
    highest_f1_variation = df.sort_values("f1_std", ascending=False).head(5)

    return {
        "lowest_f1_classes": lowest_f1[[
            "class_name", "f1_mean", "f1_std", "precision_mean", "recall_mean", "support_mean"
        ]].to_dict(orient="records"),
        "lowest_recall_classes": lowest_recall[[
            "class_name", "recall_mean", "recall_std", "f1_mean", "support_mean"
        ]].to_dict(orient="records"),
        "highest_f1_variation_classes": highest_f1_variation[[
            "class_name", "f1_mean", "f1_std", "support_mean"
        ]].to_dict(orient="records"),
    }

def top_confusions_from_cm(cm, top_n=10):
    rows = []

    for true_class in cm.index:
        for pred_class in cm.columns:
            if true_class == pred_class:
                continue

            count = int(cm.loc[true_class, pred_class])
            if count > 0:
                true_total = int(cm.loc[true_class].sum())
                rows.append({
                    "true_class": true_class,
                    "predicted_class": pred_class,
                    "count": count,
                    "share_of_true_class_errors_or_total": float(count / true_total) if true_total > 0 else 0.0,
                    "true_class_total": true_total,
                })

    return sorted(rows, key=lambda x: x["count"], reverse=True)[:top_n]

a_error = summarize_hard_classes(a_per_class_summary, "A_official", "test")
b_error = summarize_hard_classes(b_per_class_summary, "B_G14", "test")

a_top_confusions = top_confusions_from_cm(a_agg_cm, top_n=10)
b_top_confusions = top_confusions_from_cm(b_agg_cm, top_n=10)

print("\n=== A Official Hard Classes ===")
print(json.dumps(a_error, indent=2))

print("\n=== B-G14 Hard Classes ===")
print(json.dumps(b_error, indent=2))

print("\n=== A Official Top Confusions ===")
display(pd.DataFrame(a_top_confusions))

print("\n=== B-G14 Top Confusions ===")
display(pd.DataFrame(b_top_confusions))

# ============================================================
# 9. Interpretasi otomatis untuk MASTER CHAT
# ============================================================

def get_metric_ci(ci_df, scenario, metric):
    row = ci_df[(ci_df["scenario"] == scenario) & (ci_df["metric"] == metric)].iloc[0]
    return row.to_dict()

a_acc_ci = get_metric_ci(ci_df, "A_official", "test_accuracy")
a_f1_ci = get_metric_ci(ci_df, "A_official", "test_macro_f1")
b_acc_ci = get_metric_ci(ci_df, "B_G14", "test_accuracy")
b_f1_ci = get_metric_ci(ci_df, "B_G14", "test_macro_f1")

def format_class_list(records, key="class_name", n=5):
    return [r[key] for r in records[:n]]

summary = {
    "created_at": datetime.now().isoformat(),
    "benchmark_2021": {
        "accuracy": 0.931,
        "f1": 0.921,
    },
    "files_saved": {
        "confidence_interval_summary": str(CI_OUT),
        "statistical_test_vs_benchmark": str(TTEST_OUT),
        "A_vs_B_descriptive_comparison": str(AB_COMP_OUT),
        "A_official_per_class_summary": str(A_PER_CLASS_OUT),
        "B_G14_per_class_summary": str(B_PER_CLASS_OUT),
        "A_official_aggregate_confusion_matrix": str(A_AGG_CM_OUT),
        "B_G14_aggregate_confusion_matrix": str(B_AGG_CM_OUT),
    },
    "confidence_interval_interpretation": {
        "A_official_test_accuracy_ci_low_above_2021_accuracy": bool(a_acc_ci["ci_95_t_low"] > 0.931),
        "A_official_test_macro_f1_ci_low_above_2021_f1": bool(a_f1_ci["ci_95_t_low"] > 0.921),
        "B_G14_test_accuracy_ci_low_above_2021_accuracy": bool(b_acc_ci["ci_95_t_low"] > 0.931),
        "B_G14_test_macro_f1_ci_low_above_2021_f1": bool(b_f1_ci["ci_95_t_low"] > 0.921),
    },
    "statistical_test_interpretation": ttest_df.to_dict(orient="records"),
    "A_official_error_summary": a_error,
    "B_G14_error_summary": b_error,
    "A_official_top_confusions": a_top_confusions,
    "B_G14_top_confusions": b_top_confusions,
    "focus_classes": FOCUS_CLASSES,
    "recommendation": {
        "baseline_strength": "DenseNet201 baseline is strong across both A official and B-G14 5-seed.",
        "next_step_priority": [
            "Use this B30 output for paper-ready statistical and error analysis tables.",
            "Proceed to formal sensitivity planning before new training.",
            "A-clean strict or C-G09 can be considered next depending on MASTER CHAT priority.",
            "ConvNeXt-Tiny should wait until baseline analysis is fully locked."
        ]
    }
}

with open(SUMMARY_JSON_OUT, "w") as f:
    json.dump(summary, f, indent=2)

# ============================================================
# 10. Ringkasan akhir
# ============================================================

print("\n=== File Output B30 Tersimpan ===")
for p in [
    CI_OUT,
    TTEST_OUT,
    AB_COMP_OUT,
    A_PER_CLASS_OUT,
    B_PER_CLASS_OUT,
    A_AGG_CM_OUT,
    B_AGG_CM_OUT,
    SUMMARY_JSON_OUT,
]:
    print(p)

print("\n=== Ringkasan untuk MASTER CHAT ===")

print("\n1. CI A official terhadap benchmark 2021")
print(f"- A test accuracy mean={a_acc_ci['mean']:.4f}, 95% CI={a_acc_ci['ci_95_t_low']:.4f}–{a_acc_ci['ci_95_t_high']:.4f}, benchmark=0.931")
print(f"- A test macro F1 mean={a_f1_ci['mean']:.4f}, 95% CI={a_f1_ci['ci_95_t_low']:.4f}–{a_f1_ci['ci_95_t_high']:.4f}, benchmark=0.921")

print("\n2. CI B-G14 terhadap benchmark 2021")
print(f"- B-G14 test accuracy mean={b_acc_ci['mean']:.4f}, 95% CI={b_acc_ci['ci_95_t_low']:.4f}–{b_acc_ci['ci_95_t_high']:.4f}, benchmark=0.931")
print(f"- B-G14 test macro F1 mean={b_f1_ci['mean']:.4f}, 95% CI={b_f1_ci['ci_95_t_low']:.4f}–{b_f1_ci['ci_95_t_high']:.4f}, benchmark=0.921")

print("\n3. Kelas tersulit A official berdasarkan test F1 mean terendah")
display(pd.DataFrame(a_error["lowest_f1_classes"]))

print("\n4. Kelas tersulit B-G14 berdasarkan test F1 mean terendah")
display(pd.DataFrame(b_error["lowest_f1_classes"]))

print("\n5. Top confusion A official")
display(pd.DataFrame(a_top_confusions))

print("\n6. Top confusion B-G14")
display(pd.DataFrame(b_top_confusions))

print("\n=== Ringkasan Sel B30 ===")
print("B30 selesai. Semua file statistik, confidence interval, per-class summary, aggregate confusion matrix, dan error summary JSON sudah tersimpan.")
print("Tidak ada training yang dilakukan.")

In [ ]:
# Sel B31 — Membuat Manifest A-clean Strict Hamming0 Duplicate Only

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

print("=== Sel B31: Membuat Manifest A-clean Strict Hamming0 Duplicate Only ===")
print("Mode kerja: membuat manifest baru saja. Tidak training. Tidak menghapus file asli.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

# ============================================================
# 1. Path utama
# ============================================================

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_A_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
ANNOTATION_V7_PATH = OUTPUT_DIR / "v7_hamming0_manual_annotation_filled.csv"

OUT_MANIFEST = OUTPUT_DIR / "v31_split_manifest_A_clean_strict_hamming0_duplicate_only.csv"
OUT_REMOVED = OUTPUT_DIR / "v31_A_clean_strict_removed_items.csv"
OUT_DIST = OUTPUT_DIR / "v31_A_clean_strict_class_distribution_before_after.csv"
OUT_DUP_CHECK = OUTPUT_DIR / "v31_A_clean_strict_duplicate_check_after_cleaning.csv"
OUT_SUMMARY = OUTPUT_DIR / "v31_A_clean_strict_summary.json"

print("\n=== Path Check ===")
print("Manifest A:", MANIFEST_A_PATH, "| exists:", MANIFEST_A_PATH.exists())
print("Annotation V7:", ANNOTATION_V7_PATH, "| exists:", ANNOTATION_V7_PATH.exists())

if not MANIFEST_A_PATH.exists():
    raise FileNotFoundError(f"Manifest A tidak ditemukan: {MANIFEST_A_PATH}")

if not ANNOTATION_V7_PATH.exists():
    candidates = sorted(OUTPUT_DIR.glob("*v7*hamming0*filled*.csv"))
    if len(candidates) == 0:
        candidates = sorted(OUTPUT_DIR.glob("*manual_annotation_filled*.csv"))
    print("\nFile V7 default tidak ditemukan. Kandidat:")
    for i, c in enumerate(candidates, start=1):
        print(f"{i}. {c.name}")
    if len(candidates) == 0:
        raise FileNotFoundError("File anotasi V7 filled tidak ditemukan.")
    ANNOTATION_V7_PATH = candidates[0]
    print("File anotasi yang dipakai:", ANNOTATION_V7_PATH)

# ============================================================
# 2. Load data
# ============================================================

manifest = pd.read_csv(MANIFEST_A_PATH)
ann = pd.read_csv(ANNOTATION_V7_PATH)

print("\n=== Loaded Data ===")
print("Manifest shape:", manifest.shape)
print("Annotation shape:", ann.shape)

print("\nKolom manifest:")
print(list(manifest.columns))

print("\nKolom annotation:")
print(list(ann.columns))

required_manifest_cols = ["filepath", "filename", "class_name", "label_id", "split"]
for col in required_manifest_cols:
    if col not in manifest.columns:
        raise ValueError(f"Kolom wajib manifest tidak ditemukan: {col}")

# ============================================================
# 3. Deteksi kolom penting
# ============================================================

cols_lower_map = {c.lower(): c for c in ann.columns}

def find_col(possible_names, contains=None):
    for name in possible_names:
        if name.lower() in cols_lower_map:
            return cols_lower_map[name.lower()]
    if contains is not None:
        for c in ann.columns:
            if contains.lower() in c.lower():
                return c
    return None

label_col = find_col(["human_label", "visual_label", "manual_label", "label"], contains="label")
hamming_col = find_col(["hamming_distance", "hamming"], contains="hamming")

split_a_col = find_col(["split_a", "split_1", "source_split_a"])
split_b_col = find_col(["split_b", "split_2", "source_split_b"])

file_a_col = find_col(["file_a", "filepath_a", "path_a", "image_a", "filename_a"])
file_b_col = find_col(["file_b", "filepath_b", "path_b", "image_b", "filename_b"])

class_col_ann = find_col(["class_name", "class", "label_name"], contains="class")

print("\n=== Deteksi Kolom Penting Annotation ===")
print("label_col:", label_col)
print("hamming_col:", hamming_col)
print("split_a_col:", split_a_col)
print("split_b_col:", split_b_col)
print("file_a_col:", file_a_col)
print("file_b_col:", file_b_col)
print("class_col_ann:", class_col_ann)

if label_col is None:
    raise ValueError("Kolom human_label/label tidak ditemukan di anotasi V7.")
if file_a_col is None or file_b_col is None:
    raise ValueError("Kolom file_a/file_b tidak ditemukan di anotasi V7.")

# Catatan revisi:
# File V7 ini sudah bernama hamming0. Jika kolom hamming_distance tidak ada,
# seluruh baris dianggap berasal dari kandidat hamming_distance = 0.
ann["_human_label_norm"] = ann[label_col].astype(str).str.strip().str.lower()

if hamming_col is None:
    print("\nCATATAN: Kolom hamming_distance tidak ada.")
    print("Karena file anotasi adalah V7 hamming0, semua baris diasumsikan hamming_distance = 0.")
    ann["_hamming_numeric"] = 0
    hamming_source_note = "assumed_zero_from_v7_hamming0_annotation_file"
else:
    ann["_hamming_numeric"] = pd.to_numeric(ann[hamming_col], errors="coerce")
    hamming_source_note = f"read_from_column_{hamming_col}"

# ============================================================
# 4. Filter strict duplicate
# ============================================================

strict_dup = ann[
    (ann["_hamming_numeric"] == 0) &
    (ann["_human_label_norm"] == "duplicate")
].copy()

print("\n=== Strict Duplicate Filter ===")
print("Total annotation rows:", len(ann))
print("Strict duplicate rows:", len(strict_dup))
print("Hamming source:", hamming_source_note)

print("\nLabel counts pada anotasi:")
display(ann["_human_label_norm"].value_counts(dropna=False).rename_axis("human_label").reset_index(name="count"))

if len(strict_dup) == 0:
    raise ValueError("Tidak ada baris dengan human_label == duplicate. Cek isi file anotasi V7.")

if len(strict_dup) != 8:
    print("\nCATATAN: jumlah strict duplicate bukan 8.")
    print("Kode tetap lanjut, tetapi angka ini perlu dilaporkan ke MASTER CHAT.")

# ============================================================
# 5. Helper matching annotation ke manifest
# ============================================================

manifest = manifest.copy()
manifest["_row_id"] = np.arange(len(manifest))
manifest["_filepath_norm"] = manifest["filepath"].astype(str).str.replace("\\", "/", regex=False)
manifest["_filename_norm"] = manifest["filename"].astype(str)
manifest["_split_norm"] = manifest["split"].astype(str).str.lower().str.strip()
manifest["_class_norm"] = manifest["class_name"].astype(str).str.lower().str.strip()

def norm_path(x):
    return str(x).replace("\\", "/").strip()

def get_basename(x):
    return Path(norm_path(x)).name

def resolve_manifest_row(file_value, split_value=None, class_value=None):
    file_str = norm_path(file_value)
    base = get_basename(file_str)

    candidates = manifest.copy()

    if split_value is not None and str(split_value).strip() != "" and str(split_value).lower() != "nan":
        split_norm = str(split_value).lower().strip()
        candidates = candidates[candidates["_split_norm"] == split_norm]

    if class_value is not None and str(class_value).strip() != "" and str(class_value).lower() != "nan":
        class_norm = str(class_value).lower().strip()
        class_candidates = candidates[candidates["_class_norm"] == class_norm]
        if len(class_candidates) > 0:
            candidates = class_candidates

    exact = candidates[candidates["_filepath_norm"] == file_str]
    if len(exact) == 1:
        return exact.iloc[0].to_dict(), "exact_filepath"

    ends = candidates[candidates["_filepath_norm"].str.endswith(file_str, na=False)]
    if len(ends) == 1:
        return ends.iloc[0].to_dict(), "endswith_path"

    fname = candidates[candidates["_filename_norm"] == base]
    if len(fname) == 1:
        return fname.iloc[0].to_dict(), "filename_with_split_class"

    contains = candidates[candidates["_filepath_norm"].str.endswith("/" + base, na=False)]
    if len(contains) == 1:
        return contains.iloc[0].to_dict(), "endswith_basename_with_split_class"

    fname_all = manifest[manifest["_filename_norm"] == base]
    if len(fname_all) == 1:
        return fname_all.iloc[0].to_dict(), "filename_unique_global"

    return None, f"not_resolved_candidates={len(fname_all)}"

# ============================================================
# 6. Tentukan baris yang dihapus sesuai policy
# ============================================================

def choose_removal_side(split_a, split_b):
    sa = str(split_a).lower().strip()
    sb = str(split_b).lower().strip()
    pair = {sa, sb}

    if pair == {"train", "test"}:
        return ("a" if sa == "train" else "b"), "train-test: remove train, keep test"

    if pair == {"train", "validation"}:
        return ("a" if sa == "train" else "b"), "train-validation: remove train, keep validation"

    if pair == {"validation", "test"}:
        return ("a" if sa == "validation" else "b"), "validation-test: remove validation, keep test"

    return None, "same split or unsupported split pair: no removal by cross-split policy"

removal_rows = []
pair_check_rows = []
remove_row_ids = set()

for idx, row in strict_dup.iterrows():
    pair_id = row["pair_id"] if "pair_id" in strict_dup.columns else idx

    split_a = row[split_a_col] if split_a_col is not None else None
    split_b = row[split_b_col] if split_b_col is not None else None
    class_value = row[class_col_ann] if class_col_ann is not None else None

    row_a, match_a = resolve_manifest_row(row[file_a_col], split_a, class_value)
    row_b, match_b = resolve_manifest_row(row[file_b_col], split_b, class_value)

    if row_a is None or row_b is None:
        pair_check_rows.append({
            "pair_id": pair_id,
            "status": "not_resolved",
            "file_a": row[file_a_col],
            "file_b": row[file_b_col],
            "split_a_annotation": split_a,
            "split_b_annotation": split_b,
            "match_a": match_a,
            "match_b": match_b,
            "removed_side": None,
            "removed_filepath": None,
            "note": "Salah satu atau kedua file tidak bisa dipetakan ke manifest."
        })
        continue

    split_a_manifest = row_a["split"]
    split_b_manifest = row_b["split"]

    remove_side, policy_note = choose_removal_side(split_a_manifest, split_b_manifest)

    if remove_side is None:
        pair_check_rows.append({
            "pair_id": pair_id,
            "status": "resolved_no_removal",
            "file_a": row[file_a_col],
            "file_b": row[file_b_col],
            "split_a_manifest": split_a_manifest,
            "split_b_manifest": split_b_manifest,
            "class_a": row_a["class_name"],
            "class_b": row_b["class_name"],
            "match_a": match_a,
            "match_b": match_b,
            "removal_policy": policy_note,
            "removed_side": None,
            "removed_filepath": None,
            "note": "Tidak ada removal karena bukan pasangan lintas split yang masuk kebijakan."
        })
        continue

    removed = row_a if remove_side == "a" else row_b
    kept = row_b if remove_side == "a" else row_a

    remove_row_ids.add(int(removed["_row_id"]))

    removal_rows.append({
        "pair_id": pair_id,
        "hamming_distance": row["_hamming_numeric"],
        "hamming_source_note": hamming_source_note,
        "human_label": row["_human_label_norm"],
        "removal_policy": policy_note,
        "removed_side": remove_side,
        "removed_row_id": int(removed["_row_id"]),
        "removed_split": removed["split"],
        "removed_class_name": removed["class_name"],
        "removed_label_id": removed["label_id"],
        "removed_filename": removed["filename"],
        "removed_filepath": removed["filepath"],
        "kept_split": kept["split"],
        "kept_class_name": kept["class_name"],
        "kept_filename": kept["filename"],
        "kept_filepath": kept["filepath"],
        "match_a": match_a,
        "match_b": match_b,
        "file_a_annotation": row[file_a_col],
        "file_b_annotation": row[file_b_col],
    })

    pair_check_rows.append({
        "pair_id": pair_id,
        "status": "resolved_removal_planned",
        "file_a": row[file_a_col],
        "file_b": row[file_b_col],
        "split_a_manifest": split_a_manifest,
        "split_b_manifest": split_b_manifest,
        "class_a": row_a["class_name"],
        "class_b": row_b["class_name"],
        "match_a": match_a,
        "match_b": match_b,
        "removal_policy": policy_note,
        "removed_side": remove_side,
        "removed_filepath": removed["filepath"],
        "kept_filepath": kept["filepath"],
        "note": "Removal direncanakan sesuai kebijakan."
    })

removed_df = pd.DataFrame(removal_rows)
pair_check_df = pd.DataFrame(pair_check_rows)

print("\n=== Removal Plan ===")
print("Jumlah pasangan strict duplicate:", len(strict_dup))
print("Jumlah removal row unik:", len(remove_row_ids))

if len(removed_df) > 0:
    display(removed_df)
else:
    print("Tidak ada item yang direncanakan untuk dihapus.")

print("\nPair check:")
display(pair_check_df)

# ============================================================
# 7. Buat manifest A-clean strict
# ============================================================

before_manifest = manifest.copy()
clean_manifest = manifest[~manifest["_row_id"].isin(remove_row_ids)].copy()

helper_cols = [c for c in clean_manifest.columns if c.startswith("_")]
clean_manifest_export = clean_manifest.drop(columns=helper_cols)

clean_manifest_export.to_csv(OUT_MANIFEST, index=False)

if len(removed_df) > 0:
    removed_unique_df = (
        removed_df
        .sort_values(["removed_row_id", "pair_id"])
        .groupby("removed_row_id", as_index=False)
        .agg({
            "pair_id": lambda x: ",".join(map(str, x)),
            "hamming_distance": "first",
            "hamming_source_note": "first",
            "human_label": "first",
            "removal_policy": "first",
            "removed_side": "first",
            "removed_split": "first",
            "removed_class_name": "first",
            "removed_label_id": "first",
            "removed_filename": "first",
            "removed_filepath": "first",
            "kept_split": "first",
            "kept_class_name": "first",
            "kept_filename": "first",
            "kept_filepath": "first",
        })
        .rename(columns={"pair_id": "source_pair_ids"})
        .reset_index(drop=True)
    )
else:
    removed_unique_df = pd.DataFrame(columns=[
        "removed_row_id", "source_pair_ids", "removed_split",
        "removed_class_name", "removed_filename", "removed_filepath"
    ])

removed_unique_df.to_csv(OUT_REMOVED, index=False)

print("\n=== Manifest A-clean Strict ===")
print("Before rows:", len(before_manifest))
print("After rows :", len(clean_manifest_export))
print("Rows removed:", len(before_manifest) - len(clean_manifest_export))
print("Manifest saved:", OUT_MANIFEST)
print("Removed items saved:", OUT_REMOVED)

# ============================================================
# 8. Distribusi sebelum dan sesudah cleaning
# ============================================================

before_counts = before_manifest.groupby(["split", "class_name"]).size().reset_index(name="count_before")
after_counts = clean_manifest.groupby(["split", "class_name"]).size().reset_index(name="count_after")

dist_df = before_counts.merge(after_counts, on=["split", "class_name"], how="outer")
dist_df["count_before"] = dist_df["count_before"].fillna(0).astype(int)
dist_df["count_after"] = dist_df["count_after"].fillna(0).astype(int)
dist_df["delta_after_minus_before"] = dist_df["count_after"] - dist_df["count_before"]

dist_df.to_csv(OUT_DIST, index=False)

split_before = before_manifest["split"].value_counts().rename_axis("split").reset_index(name="count_before")
split_after = clean_manifest["split"].value_counts().rename_axis("split").reset_index(name="count_after")
split_summary = split_before.merge(split_after, on="split", how="outer").fillna(0)
split_summary["count_before"] = split_summary["count_before"].astype(int)
split_summary["count_after"] = split_summary["count_after"].astype(int)
split_summary["delta_after_minus_before"] = split_summary["count_after"] - split_summary["count_before"]

print("\n=== Split Counts Before/After ===")
display(split_summary)

print("\n=== Class Distribution Before/After ===")
display(dist_df)

# ============================================================
# 9. Cek semua 14 kelas masih ada per split
# ============================================================

expected_classes = sorted(before_manifest["class_name"].unique().tolist())
split_class_check_rows = []

for split in ["train", "validation", "test"]:
    subset = clean_manifest[clean_manifest["split"] == split]
    classes_present = sorted(subset["class_name"].unique().tolist())
    missing_classes = sorted(set(expected_classes) - set(classes_present))

    split_class_check_rows.append({
        "split": split,
        "num_classes_present": len(classes_present),
        "missing_classes": ";".join(missing_classes) if missing_classes else "",
        "all_14_classes_present": len(missing_classes) == 0
    })

split_class_check_df = pd.DataFrame(split_class_check_rows)

print("\n=== Check 14 Kelas per Split Setelah Cleaning ===")
display(split_class_check_df)

# ============================================================
# 10. Cek duplicate strict lintas split setelah cleaning
# ============================================================

remaining_row_ids = set(clean_manifest["_row_id"].astype(int).tolist())
dup_after_rows = []

for idx, row in strict_dup.iterrows():
    pair_id = row["pair_id"] if "pair_id" in strict_dup.columns else idx

    split_a = row[split_a_col] if split_a_col is not None else None
    split_b = row[split_b_col] if split_b_col is not None else None
    class_value = row[class_col_ann] if class_col_ann is not None else None

    row_a, match_a = resolve_manifest_row(row[file_a_col], split_a, class_value)
    row_b, match_b = resolve_manifest_row(row[file_b_col], split_b, class_value)

    if row_a is None or row_b is None:
        dup_after_rows.append({
            "pair_id": pair_id,
            "status": "not_resolved",
            "file_a": row[file_a_col],
            "file_b": row[file_b_col],
            "still_both_remaining": False,
            "still_cross_split_duplicate": False,
            "note": "Tidak bisa cek karena mapping ke manifest gagal."
        })
        continue

    a_remaining = int(row_a["_row_id"]) in remaining_row_ids
    b_remaining = int(row_b["_row_id"]) in remaining_row_ids
    both_remaining = a_remaining and b_remaining
    cross_split = row_a["split"] != row_b["split"]

    dup_after_rows.append({
        "pair_id": pair_id,
        "file_a": row[file_a_col],
        "file_b": row[file_b_col],
        "split_a_manifest": row_a["split"],
        "split_b_manifest": row_b["split"],
        "class_a": row_a["class_name"],
        "class_b": row_b["class_name"],
        "row_id_a": int(row_a["_row_id"]),
        "row_id_b": int(row_b["_row_id"]),
        "a_remaining": a_remaining,
        "b_remaining": b_remaining,
        "still_both_remaining": both_remaining,
        "still_cross_split_duplicate": bool(both_remaining and cross_split),
        "status": "checked",
        "note": "OK jika still_cross_split_duplicate=False"
    })

dup_after_df = pd.DataFrame(dup_after_rows)
dup_after_df.to_csv(OUT_DUP_CHECK, index=False)

remaining_cross_split_duplicate_count = int(dup_after_df["still_cross_split_duplicate"].sum())

print("\n=== Duplicate Check After Cleaning ===")
display(dup_after_df)
print("Remaining strict cross-split duplicate pairs:", remaining_cross_split_duplicate_count)

# ============================================================
# 11. Summary JSON
# ============================================================

summary = {
    "created_at": datetime.now().isoformat(),
    "task": "A-clean strict manifest preparation using V7 hamming0 annotation and human_label=duplicate only",
    "input_files": {
        "manifest_A_official": str(MANIFEST_A_PATH),
        "annotation_V7": str(ANNOTATION_V7_PATH),
    },
    "output_files": {
        "manifest_A_clean_strict": str(OUT_MANIFEST),
        "removed_items": str(OUT_REMOVED),
        "class_distribution_before_after": str(OUT_DIST),
        "duplicate_check_after_cleaning": str(OUT_DUP_CHECK),
        "summary_json": str(OUT_SUMMARY),
    },
    "cleaning_definition": {
        "hamming_distance": 0,
        "hamming_source_note": hamming_source_note,
        "human_label": "duplicate",
        "excluded_labels": ["similar", "uncertain", "not duplicate"],
        "physical_files_deleted": False,
        "manifest_only": True,
    },
    "cleaning_policy": {
        "train_test": "remove train, keep test",
        "train_validation": "remove train, keep validation",
        "validation_test": "remove validation, keep test",
        "duplicate_file_multiple_pairs": "remove once, log transparently",
    },
    "counts": {
        "annotation_rows_total": int(len(ann)),
        "strict_duplicate_pairs_used": int(len(strict_dup)),
        "manifest_rows_before": int(len(before_manifest)),
        "manifest_rows_after": int(len(clean_manifest)),
        "unique_rows_removed": int(len(remove_row_ids)),
        "remaining_strict_cross_split_duplicate_pairs": int(remaining_cross_split_duplicate_count),
    },
    "split_counts_before_after": split_summary.to_dict(orient="records"),
    "all_14_classes_check": split_class_check_df.to_dict(orient="records"),
    "all_14_classes_present_in_each_split": bool(split_class_check_df["all_14_classes_present"].all()),
    "distribution_change": {
        "max_abs_delta_per_class_split": int(dist_df["delta_after_minus_before"].abs().max()) if len(dist_df) else 0,
        "total_removed_by_split": removed_unique_df["removed_split"].value_counts().to_dict() if len(removed_unique_df) else {},
        "total_removed_by_class": removed_unique_df["removed_class_name"].value_counts().to_dict() if len(removed_unique_df) else {},
    },
    "recommendation": {
        "manifest_ready_for_training_sensitivity": bool(
            (remaining_cross_split_duplicate_count == 0) and
            split_class_check_df["all_14_classes_present"].all()
        ),
        "next_step": "Jika MASTER CHAT menerima, lanjut training DenseNet201 A-clean strict seed 42 sebagai sensitivity analysis."
    }
}

with open(OUT_SUMMARY, "w") as f:
    json.dump(summary, f, indent=2)

# ============================================================
# 12. Ringkasan akhir
# ============================================================

print("\n=== File Output B31 Tersimpan ===")
for p in [OUT_MANIFEST, OUT_REMOVED, OUT_DIST, OUT_DUP_CHECK, OUT_SUMMARY]:
    print(p)

print("\n=== Ringkasan Sel B31 untuk MASTER CHAT ===")
print(f"Strict duplicate pairs used: {len(strict_dup)}")
print(f"Unique manifest rows removed: {len(remove_row_ids)}")
print("Removed by split:", summary["distribution_change"]["total_removed_by_split"])
print("Removed by class:", summary["distribution_change"]["total_removed_by_class"])
print("All 14 classes present in each split:", summary["all_14_classes_present_in_each_split"])
print("Remaining strict cross-split duplicate pairs:", remaining_cross_split_duplicate_count)
print("Manifest A-clean strict saved:", OUT_MANIFEST)
print("Ready for training sensitivity:", summary["recommendation"]["manifest_ready_for_training_sensitivity"])

print("\nSplit count before/after:")
display(split_summary)

print("\nRemoved items:")
display(removed_unique_df)

print("\nDuplicate check after cleaning:")
display(dup_after_df)

print("\nSel B31 selesai. Tidak ada training. Tidak ada file gambar asli yang dihapus.")

In [ ]:
# Sel B32 — DenseNet201 A-clean Strict Seed 42

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B32: DenseNet201 A-clean Strict Seed 42 ===")
print("Mode kerja: training sensitivity A-clean strict seed 42.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B32.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_A_clean_strict_seed42"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_CLEAN_PATH = OUTPUT_DIR / "v31_split_manifest_A_clean_strict_hamming0_duplicate_only.csv"
MANIFEST_A_OFFICIAL_PATH = OUTPUT_DIR / "v5_split_manifest_A_official.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_clean_strict_seed42"
EXPERIMENT_ID = "DenseNet201_A_clean_strict_seed42"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"
COMPARISON_VS_A42_CSV = OUTPUT_DIR / f"{RUN_NAME}_vs_A_official_seed42_comparison.csv"

print("\n=== Cek Path ===")
print("Manifest A-clean strict:", MANIFEST_A_CLEAN_PATH, "| exists:", MANIFEST_A_CLEAN_PATH.exists())
print("Manifest A official:", MANIFEST_A_OFFICIAL_PATH, "| exists:", MANIFEST_A_OFFICIAL_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_A_CLEAN_PATH.exists():
    raise FileNotFoundError(f"Manifest A-clean strict tidak ditemukan: {MANIFEST_A_CLEAN_PATH}")

# ============================================================
# 3. Fungsi simpan dan verifikasi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)

    # Verifikasi local tmp
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    os.replace(local_tmp, local_path)

    # Copy ke Drive sebagai tmp
    shutil.copy2(local_path, drive_tmp)

    # Verifikasi Drive tmp
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)

    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Cek master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
    "DenseNet201_A_official_seed1024",
    "DenseNet201_B_G14_seed42",
    "DenseNet201_B_G14_seed123",
    "DenseNet201_B_G14_seed456",
    "DenseNet201_B_G14_seed789",
    "DenseNet201_B_G14_seed1024",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 42 A-clean strict
# ============================================================

SEED = 42
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

A_OFFICIAL_SEED42_REF = {
    "test_accuracy": 0.9658,
    "test_macro_f1": 0.9587,
    "test_weighted_f1": 0.9657,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 42 A-clean Strict ===")
print("Scenario: A-clean strict hamming0 duplicate only")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean aggressive: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_clean_strict",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity training DenseNet201 A-clean strict seed 42. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai A-clean strict seed 42.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest A-clean strict
# ============================================================

df = pd.read_csv(MANIFEST_A_CLEAN_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest A-clean Strict ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

expected_split_counts = {"train": 5732, "validation": 2249, "test": 1900}
actual_split_counts = df["split"].value_counts().to_dict()
print("\nExpected split counts:", expected_split_counts)
print("Actual split counts:", actual_split_counts)

# Cek semua 14 kelas per split
class_check_rows = []
for split in ["train", "validation", "test"]:
    s = df[df["split"] == split]
    class_check_rows.append({
        "split": split,
        "num_classes": int(s["class_name"].nunique()),
        "all_14_classes_present": bool(s["class_name"].nunique() == 14)
    })

class_check_df = pd.DataFrame(class_check_rows)
print("\n=== Cek 14 kelas per split ===")
display(class_check_df)

if not class_check_df["all_14_classes_present"].all():
    raise ValueError("Tidak semua split memiliki 14 kelas. Cek manifest A-clean strict.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

print("\nClass names ordered:")
for i, name in enumerate(class_names_ordered):
    print(i, name)

# ============================================================
# 9. Dataset dan DataLoader
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 10. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 11. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 12. Training A-clean strict seed 42
# ============================================================

print("\n=== Mulai Training A-clean Strict Seed 42 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_clean_strict",
                "run_type": "sensitivity",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_clean_strict",
            "run_type": "sensitivity",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_clean_strict",
        "run_type": "sensitivity",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training A-clean Strict Seed 42 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining A-clean strict seed 42 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 13. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint A-clean Strict Seed 42 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics A-clean strict seed 42:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics A-clean strict seed 42:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report A-clean strict seed 42:")
print(test_report_text)

# ============================================================
# 14. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_clean_strict",
    "run_type": "sensitivity",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    },
    "comparison_reference": {
        "A_official_seed42_test_accuracy": A_OFFICIAL_SEED42_REF["test_accuracy"],
        "A_official_seed42_test_macro_f1": A_OFFICIAL_SEED42_REF["test_macro_f1"],
        "A_official_seed42_test_weighted_f1": A_OFFICIAL_SEED42_REF["test_weighted_f1"]
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 15. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_clean_strict",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity DenseNet201 A-clean strict seed 42 selesai. Manifest removes 8 train rows from strict hamming0 duplicate pairs. Checkpoint local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Bandingkan dengan A official seed 42
# ============================================================

a_clean_metrics = {
    "run_name": RUN_NAME,
    "scenario": "A_clean_strict",
    "seed": SEED,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
}

comparison_rows = [
    {
        "run_name": "DenseNet201_A_official_seed42",
        "scenario": "A_official",
        "seed": 42,
        "best_epoch": 20,
        "val_accuracy": np.nan,
        "val_macro_f1": np.nan,
        "test_accuracy": A_OFFICIAL_SEED42_REF["test_accuracy"],
        "test_macro_f1": A_OFFICIAL_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": A_OFFICIAL_SEED42_REF["test_weighted_f1"],
    },
    a_clean_metrics
]

comparison_df = pd.DataFrame(comparison_rows)

delta_row = {
    "run_name": "delta_A_clean_minus_A_official_seed42",
    "scenario": "delta",
    "seed": 42,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": a_clean_metrics["test_accuracy"] - A_OFFICIAL_SEED42_REF["test_accuracy"],
    "test_macro_f1": a_clean_metrics["test_macro_f1"] - A_OFFICIAL_SEED42_REF["test_macro_f1"],
    "test_weighted_f1": a_clean_metrics["test_weighted_f1"] - A_OFFICIAL_SEED42_REF["test_weighted_f1"],
}

comparison_df = pd.concat([comparison_df, pd.DataFrame([delta_row])], ignore_index=True)
comparison_df.to_csv(COMPARISON_VS_A42_CSV, index=False)

# ============================================================
# 18. Ringkasan akhir
# ============================================================

print("\n=== File Output A-clean Strict Seed 42 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Validation report:", VAL_REPORT_CSV_PATH)
print("Test report:", TEST_REPORT_CSV_PATH)
print("Validation CM:", VAL_CM_CSV_PATH)
print("Test CM:", TEST_CM_CSV_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)
print("Comparison vs A official seed 42:", COMPARISON_VS_A42_CSV)

print("\n=== Ringkasan Sel B32 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Delta vs A Official Seed 42 ===")
print(f"Delta test accuracy    : {delta_row['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {delta_row['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {delta_row['test_weighted_f1']:.4f}")

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Comparison Table ===")
display(comparison_df)

In [ ]:
# Sel B33 — DenseNet201 A-clean Strict Seed 123

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B33: DenseNet201 A-clean Strict Seed 123 ===")
print("Mode kerja: training sensitivity A-clean strict seed 123.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B33.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_A_clean_strict_seed123"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_CLEAN_PATH = OUTPUT_DIR / "v31_split_manifest_A_clean_strict_hamming0_duplicate_only.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_clean_strict_seed123"
EXPERIMENT_ID = "DenseNet201_A_clean_strict_seed123"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"
COMPARISON_VS_A123_CSV = OUTPUT_DIR / f"{RUN_NAME}_vs_A_official_seed123_comparison.csv"

print("\n=== Cek Path ===")
print("Manifest A-clean strict:", MANIFEST_A_CLEAN_PATH, "| exists:", MANIFEST_A_CLEAN_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_A_CLEAN_PATH.exists():
    raise FileNotFoundError(f"Manifest A-clean strict tidak ditemukan: {MANIFEST_A_CLEAN_PATH}")

# ============================================================
# 3. Fungsi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_official_seed789",
    "DenseNet201_A_official_seed1024",
    "DenseNet201_A_clean_strict_seed42",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 123 A-clean strict
# ============================================================

SEED = 123
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

# Fallback nilai dari MASTER CHAT.
# Kalau file A official seed 123 tersedia, nilai exact akan dibaca otomatis.
A_OFFICIAL_SEED123_REF = {
    "test_accuracy": 0.9674,
    "test_macro_f1": 0.9623,
    "test_weighted_f1": 0.9674,
}

A123_EVAL_JSON = OUTPUT_DIR / "DenseNet201_A_official_seed123_evaluation_metrics_summary.json"
if A123_EVAL_JSON.exists():
    try:
        with open(A123_EVAL_JSON, "r") as f:
            a123_eval = json.load(f)
        A_OFFICIAL_SEED123_REF = {
            "test_accuracy": float(a123_eval["test"]["accuracy"]),
            "test_macro_f1": float(a123_eval["test"]["macro_f1"]),
            "test_weighted_f1": float(a123_eval["test"]["weighted_f1"]),
        }
        print("\nReferensi A official seed 123 dibaca dari JSON tersimpan.")
    except Exception as e:
        print("\nGagal membaca JSON A official seed 123. Pakai nilai fallback MASTER CHAT.")
        print("Error:", e)

print("\n=== Referensi A Official Seed 123 ===")
print(A_OFFICIAL_SEED123_REF)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 123 A-clean Strict ===")
print("Scenario: A-clean strict hamming0 duplicate only")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean aggressive: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_clean_strict",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity training DenseNet201 A-clean strict seed 123. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai A-clean strict seed 123.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest A-clean strict
# ============================================================

df = pd.read_csv(MANIFEST_A_CLEAN_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest A-clean Strict ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

expected_split_counts = {"train": 5732, "validation": 2249, "test": 1900}
actual_split_counts = df["split"].value_counts().to_dict()
print("\nExpected split counts:", expected_split_counts)
print("Actual split counts:", actual_split_counts)

class_check_rows = []
for split in ["train", "validation", "test"]:
    s = df[df["split"] == split]
    class_check_rows.append({
        "split": split,
        "num_classes": int(s["class_name"].nunique()),
        "all_14_classes_present": bool(s["class_name"].nunique() == 14)
    })

class_check_df = pd.DataFrame(class_check_rows)
print("\n=== Cek 14 kelas per split ===")
display(class_check_df)

if not class_check_df["all_14_classes_present"].all():
    raise ValueError("Tidak semua split memiliki 14 kelas. Cek manifest A-clean strict.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

print("\nClass names ordered:")
for i, name in enumerate(class_names_ordered):
    print(i, name)

# ============================================================
# 9. Dataset dan DataLoader
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 10. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 11. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 12. Training A-clean strict seed 123
# ============================================================

print("\n=== Mulai Training A-clean Strict Seed 123 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_clean_strict",
                "run_type": "sensitivity",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_clean_strict",
            "run_type": "sensitivity",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_clean_strict",
        "run_type": "sensitivity",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training A-clean Strict Seed 123 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining A-clean strict seed 123 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 13. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint A-clean Strict Seed 123 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics A-clean strict seed 123:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics A-clean strict seed 123:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report A-clean strict seed 123:")
print(test_report_text)

# ============================================================
# 14. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_clean_strict",
    "run_type": "sensitivity",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    },
    "comparison_reference": {
        "A_official_seed123_test_accuracy": A_OFFICIAL_SEED123_REF["test_accuracy"],
        "A_official_seed123_test_macro_f1": A_OFFICIAL_SEED123_REF["test_macro_f1"],
        "A_official_seed123_test_weighted_f1": A_OFFICIAL_SEED123_REF["test_weighted_f1"]
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 15. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_clean_strict",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity DenseNet201 A-clean strict seed 123 selesai. Manifest removes 8 train rows from strict hamming0 duplicate pairs. Checkpoint local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Bandingkan dengan A official seed 123
# ============================================================

a_clean_metrics = {
    "run_name": RUN_NAME,
    "scenario": "A_clean_strict",
    "seed": SEED,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
}

comparison_rows = [
    {
        "run_name": "DenseNet201_A_official_seed123",
        "scenario": "A_official",
        "seed": 123,
        "best_epoch": 16,
        "val_accuracy": np.nan,
        "val_macro_f1": np.nan,
        "test_accuracy": A_OFFICIAL_SEED123_REF["test_accuracy"],
        "test_macro_f1": A_OFFICIAL_SEED123_REF["test_macro_f1"],
        "test_weighted_f1": A_OFFICIAL_SEED123_REF["test_weighted_f1"],
    },
    a_clean_metrics
]

comparison_df = pd.DataFrame(comparison_rows)

delta_row = {
    "run_name": "delta_A_clean_minus_A_official_seed123",
    "scenario": "delta",
    "seed": 123,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": a_clean_metrics["test_accuracy"] - A_OFFICIAL_SEED123_REF["test_accuracy"],
    "test_macro_f1": a_clean_metrics["test_macro_f1"] - A_OFFICIAL_SEED123_REF["test_macro_f1"],
    "test_weighted_f1": a_clean_metrics["test_weighted_f1"] - A_OFFICIAL_SEED123_REF["test_weighted_f1"],
}

comparison_df = pd.concat([comparison_df, pd.DataFrame([delta_row])], ignore_index=True)
comparison_df.to_csv(COMPARISON_VS_A123_CSV, index=False)

# ============================================================
# 18. Ringkasan akhir
# ============================================================

print("\n=== File Output A-clean Strict Seed 123 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Validation report:", VAL_REPORT_CSV_PATH)
print("Test report:", TEST_REPORT_CSV_PATH)
print("Validation CM:", VAL_CM_CSV_PATH)
print("Test CM:", TEST_CM_CSV_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)
print("Comparison vs A official seed 123:", COMPARISON_VS_A123_CSV)

print("\n=== Ringkasan Sel B33 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Delta vs A Official Seed 123 ===")
print(f"Delta test accuracy    : {delta_row['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {delta_row['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {delta_row['test_weighted_f1']:.4f}")

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Comparison Table ===")
display(comparison_df)

In [ ]:
# Sel B34 — DenseNet201 A-clean Strict Seed 456

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B34: DenseNet201 A-clean Strict Seed 456 ===")
print("Mode kerja: training sensitivity A-clean strict seed 456.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B34.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_A_clean_strict_seed456"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_A_CLEAN_PATH = OUTPUT_DIR / "v31_split_manifest_A_clean_strict_hamming0_duplicate_only.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_A_clean_strict_seed456"
EXPERIMENT_ID = "DenseNet201_A_clean_strict_seed456"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"
COMPARISON_VS_A456_CSV = OUTPUT_DIR / f"{RUN_NAME}_vs_A_official_seed456_comparison.csv"
TEMP_3SEED_COMPARISON_CSV = OUTPUT_DIR / "DenseNet201_A_clean_strict_temporary_3seed_comparison.csv"

print("\n=== Cek Path ===")
print("Manifest A-clean strict:", MANIFEST_A_CLEAN_PATH, "| exists:", MANIFEST_A_CLEAN_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_A_CLEAN_PATH.exists():
    raise FileNotFoundError(f"Manifest A-clean strict tidak ditemukan: {MANIFEST_A_CLEAN_PATH}")

# ============================================================
# 3. Fungsi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_A_official_seed123",
    "DenseNet201_A_official_seed456",
    "DenseNet201_A_clean_strict_seed42",
    "DenseNet201_A_clean_strict_seed123",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 456 A-clean strict
# ============================================================

SEED = 456
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

# Fallback dari MASTER CHAT.
# Jika JSON A official tersedia, kode akan baca nilai exact otomatis.
A_OFFICIAL_REF_FALLBACK = {
    42: {
        "test_accuracy": 0.9658,
        "test_macro_f1": 0.9587,
        "test_weighted_f1": 0.9657,
        "best_epoch": 20
    },
    123: {
        "test_accuracy": 0.9674,
        "test_macro_f1": 0.9623,
        "test_weighted_f1": 0.9674,
        "best_epoch": 16
    },
    456: {
        "test_accuracy": 0.9521,
        "test_macro_f1": 0.9425,
        "test_weighted_f1": 0.9517,
        "best_epoch": 5
    }
}

def read_official_reference(seed):
    prefix = f"DenseNet201_A_official_seed{seed}"
    metrics_path = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    training_path = OUTPUT_DIR / f"{prefix}_training_summary.json"

    ref = A_OFFICIAL_REF_FALLBACK[seed].copy()

    if metrics_path.exists():
        try:
            with open(metrics_path, "r") as f:
                metrics = json.load(f)
            ref["test_accuracy"] = float(metrics["test"]["accuracy"])
            ref["test_macro_f1"] = float(metrics["test"]["macro_f1"])
            ref["test_weighted_f1"] = float(metrics["test"]["weighted_f1"])
        except Exception as e:
            print(f"Warning: gagal baca metrics official seed {seed}, pakai fallback. Error:", e)

    if training_path.exists():
        try:
            with open(training_path, "r") as f:
                training = json.load(f)
            ref["best_epoch"] = training.get("best_epoch", ref["best_epoch"])
        except Exception:
            pass

    return ref

A_OFFICIAL_SEED456_REF = read_official_reference(456)

print("\n=== Referensi A Official Seed 456 ===")
print(A_OFFICIAL_SEED456_REF)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 456 A-clean Strict ===")
print("Scenario: A-clean strict hamming0 duplicate only")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean aggressive: tidak dipakai")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_clean_strict",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity training DenseNet201 A-clean strict seed 456. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai A-clean strict seed 456.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest A-clean strict
# ============================================================

df = pd.read_csv(MANIFEST_A_CLEAN_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest A-clean Strict ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

expected_split_counts = {"train": 5732, "validation": 2249, "test": 1900}
actual_split_counts = df["split"].value_counts().to_dict()
print("\nExpected split counts:", expected_split_counts)
print("Actual split counts:", actual_split_counts)

class_check_rows = []
for split in ["train", "validation", "test"]:
    s = df[df["split"] == split]
    class_check_rows.append({
        "split": split,
        "num_classes": int(s["class_name"].nunique()),
        "all_14_classes_present": bool(s["class_name"].nunique() == 14)
    })

class_check_df = pd.DataFrame(class_check_rows)
print("\n=== Cek 14 kelas per split ===")
display(class_check_df)

if not class_check_df["all_14_classes_present"].all():
    raise ValueError("Tidak semua split memiliki 14 kelas. Cek manifest A-clean strict.")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

print("\nClass names ordered:")
for i, name in enumerate(class_names_ordered):
    print(i, name)

# ============================================================
# 9. Dataset dan DataLoader
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 10. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 11. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 12. Training A-clean strict seed 456
# ============================================================

print("\n=== Mulai Training A-clean Strict Seed 456 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "A_clean_strict",
                "run_type": "sensitivity",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "A_clean_strict",
            "run_type": "sensitivity",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "A_clean_strict",
        "run_type": "sensitivity",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training A-clean Strict Seed 456 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining A-clean strict seed 456 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 13. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint A-clean Strict Seed 456 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics A-clean strict seed 456:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics A-clean strict seed 456:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report A-clean strict seed 456:")
print(test_report_text)

# ============================================================
# 14. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "A_clean_strict",
    "run_type": "sensitivity",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    },
    "comparison_reference": {
        "A_official_seed456_test_accuracy": A_OFFICIAL_SEED456_REF["test_accuracy"],
        "A_official_seed456_test_macro_f1": A_OFFICIAL_SEED456_REF["test_macro_f1"],
        "A_official_seed456_test_weighted_f1": A_OFFICIAL_SEED456_REF["test_weighted_f1"]
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 15. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "A_clean_strict",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity DenseNet201 A-clean strict seed 456 selesai. Manifest removes 8 train rows from strict hamming0 duplicate pairs. Checkpoint local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Bandingkan dengan A official seed 456
# ============================================================

a_clean_metrics = {
    "run_name": RUN_NAME,
    "scenario": "A_clean_strict",
    "seed": SEED,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
}

comparison_rows = [
    {
        "run_name": "DenseNet201_A_official_seed456",
        "scenario": "A_official",
        "seed": 456,
        "best_epoch": A_OFFICIAL_SEED456_REF["best_epoch"],
        "val_accuracy": np.nan,
        "val_macro_f1": np.nan,
        "test_accuracy": A_OFFICIAL_SEED456_REF["test_accuracy"],
        "test_macro_f1": A_OFFICIAL_SEED456_REF["test_macro_f1"],
        "test_weighted_f1": A_OFFICIAL_SEED456_REF["test_weighted_f1"],
    },
    a_clean_metrics
]

comparison_df = pd.DataFrame(comparison_rows)

delta_row = {
    "run_name": "delta_A_clean_minus_A_official_seed456",
    "scenario": "delta",
    "seed": 456,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": a_clean_metrics["test_accuracy"] - A_OFFICIAL_SEED456_REF["test_accuracy"],
    "test_macro_f1": a_clean_metrics["test_macro_f1"] - A_OFFICIAL_SEED456_REF["test_macro_f1"],
    "test_weighted_f1": a_clean_metrics["test_weighted_f1"] - A_OFFICIAL_SEED456_REF["test_weighted_f1"],
}

comparison_df = pd.concat([comparison_df, pd.DataFrame([delta_row])], ignore_index=True)
comparison_df.to_csv(COMPARISON_VS_A456_CSV, index=False)

# ============================================================
# 18. Tabel sementara A-clean strict 3 seed
# ============================================================

def read_eval_metrics(prefix):
    eval_path = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    train_path = OUTPUT_DIR / f"{prefix}_training_summary.json"

    if not eval_path.exists():
        return None

    with open(eval_path, "r") as f:
        eval_data = json.load(f)

    best_epoch_value = None
    if train_path.exists():
        try:
            with open(train_path, "r") as f:
                train_data = json.load(f)
            best_epoch_value = train_data.get("best_epoch")
        except Exception:
            best_epoch_value = eval_data.get("best_epoch_from_checkpoint")

    return {
        "best_epoch": best_epoch_value,
        "val_accuracy": eval_data["validation"]["accuracy"],
        "val_macro_f1": eval_data["validation"]["macro_f1"],
        "test_accuracy": eval_data["test"]["accuracy"],
        "test_macro_f1": eval_data["test"]["macro_f1"],
        "test_weighted_f1": eval_data["test"]["weighted_f1"],
    }

temp_rows = []
for seed in [42, 123, 456]:
    clean_prefix = f"DenseNet201_A_clean_strict_seed{seed}"
    clean_metrics = read_eval_metrics(clean_prefix)
    official_ref = read_official_reference(seed)

    if clean_metrics is None:
        temp_rows.append({
            "seed": seed,
            "status": "missing_A_clean_metrics"
        })
        continue

    temp_rows.append({
        "seed": seed,
        "status": "ok",
        "A_clean_best_epoch": clean_metrics["best_epoch"],
        "A_clean_val_accuracy": clean_metrics["val_accuracy"],
        "A_clean_val_macro_f1": clean_metrics["val_macro_f1"],
        "A_clean_test_accuracy": clean_metrics["test_accuracy"],
        "A_clean_test_macro_f1": clean_metrics["test_macro_f1"],
        "A_clean_test_weighted_f1": clean_metrics["test_weighted_f1"],
        "A_official_test_accuracy": official_ref["test_accuracy"],
        "A_official_test_macro_f1": official_ref["test_macro_f1"],
        "A_official_test_weighted_f1": official_ref["test_weighted_f1"],
        "delta_test_accuracy": clean_metrics["test_accuracy"] - official_ref["test_accuracy"],
        "delta_test_macro_f1": clean_metrics["test_macro_f1"] - official_ref["test_macro_f1"],
        "delta_test_weighted_f1": clean_metrics["test_weighted_f1"] - official_ref["test_weighted_f1"],
    })

temp_3seed_df = pd.DataFrame(temp_rows)
temp_3seed_df.to_csv(TEMP_3SEED_COMPARISON_CSV, index=False)

# ============================================================
# 19. Ringkasan akhir
# ============================================================

print("\n=== File Output A-clean Strict Seed 456 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Validation report:", VAL_REPORT_CSV_PATH)
print("Test report:", TEST_REPORT_CSV_PATH)
print("Validation CM:", VAL_CM_CSV_PATH)
print("Test CM:", TEST_CM_CSV_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)
print("Comparison vs A official seed 456:", COMPARISON_VS_A456_CSV)
print("Temporary 3-seed comparison:", TEMP_3SEED_COMPARISON_CSV)

print("\n=== Ringkasan Sel B34 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Delta vs A Official Seed 456 ===")
print(f"Delta test accuracy    : {delta_row['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {delta_row['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {delta_row['test_weighted_f1']:.4f}")

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Comparison Table vs A Official Seed 456 ===")
display(comparison_df)

print("\n=== Tabel Sementara A-clean Strict 3 Seed ===")
display(temp_3seed_df)

In [ ]:
# Sel B35 — Summary A-clean Strict 3-Seed dan Paired Delta

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

print("=== Sel B35: Summary A-clean Strict 3-Seed dan Paired Delta ===")
print("Mode kerja: analisis non-training. Tidak memakai GPU. Tidak mengubah data.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

# ============================================================
# 1. Path utama
# ============================================================

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")

OUT_RESULTS = OUTPUT_DIR / "DenseNet201_A_clean_strict_3seed_results.csv"
OUT_STATS = OUTPUT_DIR / "DenseNet201_A_clean_strict_3seed_summary_stats.csv"
OUT_DELTA = OUTPUT_DIR / "DenseNet201_A_clean_strict_vs_A_official_paired_delta.csv"
OUT_SUMMARY = OUTPUT_DIR / "DenseNet201_A_clean_strict_3seed_summary.json"

SEEDS = [42, 123, 456]

A_CLEAN_PREFIX = {
    42: "DenseNet201_A_clean_strict_seed42",
    123: "DenseNet201_A_clean_strict_seed123",
    456: "DenseNet201_A_clean_strict_seed456",
}

A_OFFICIAL_PREFIX = {
    42: "DenseNet201_A_official_seed42",
    123: "DenseNet201_A_official_seed123",
    456: "DenseNet201_A_official_seed456",
}

# Fallback dari hasil yang sudah disetujui MASTER CHAT.
A_OFFICIAL_FALLBACK = {
    42: {
        "best_epoch": 20,
        "test_accuracy": 0.9658,
        "test_macro_f1": 0.9587,
        "test_weighted_f1": 0.9657,
    },
    123: {
        "best_epoch": 16,
        "test_accuracy": 0.9674,
        "test_macro_f1": 0.9623,
        "test_weighted_f1": 0.9674,
    },
    456: {
        "best_epoch": 5,
        "test_accuracy": 0.9521,
        "test_macro_f1": 0.9425,
        "test_weighted_f1": 0.9517,
    },
}

BENCHMARK_2021 = {
    "accuracy": 0.931,
    "f1": 0.921,
}

print("\n=== Output Folder ===")
print(OUTPUT_DIR)

# ============================================================
# 2. Helper baca metrics
# ============================================================

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)

def read_training_summary(prefix):
    path = OUTPUT_DIR / f"{prefix}_training_summary.json"
    data = read_json(path)
    if data is None:
        return {}
    return data

def read_eval_summary(prefix):
    path = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    data = read_json(path)
    if data is None:
        return {}
    return data

def get_clean_metrics(seed):
    prefix = A_CLEAN_PREFIX[seed]
    train = read_training_summary(prefix)
    eval_data = read_eval_summary(prefix)

    if not eval_data:
        raise FileNotFoundError(f"Evaluation metrics tidak ditemukan untuk {prefix}")

    return {
        "seed": seed,
        "run_name": prefix,
        "best_epoch": train.get("best_epoch", eval_data.get("best_epoch_from_checkpoint")),
        "val_accuracy": float(eval_data["validation"]["accuracy"]),
        "val_macro_f1": float(eval_data["validation"]["macro_f1"]),
        "test_accuracy": float(eval_data["test"]["accuracy"]),
        "test_macro_f1": float(eval_data["test"]["macro_f1"]),
        "test_weighted_f1": float(eval_data["test"]["weighted_f1"]),
        "status": "ok",
    }

def get_official_metrics(seed):
    prefix = A_OFFICIAL_PREFIX[seed]
    train = read_training_summary(prefix)
    eval_data = read_eval_summary(prefix)

    fallback = A_OFFICIAL_FALLBACK[seed].copy()

    if eval_data:
        fallback["test_accuracy"] = float(eval_data["test"]["accuracy"])
        fallback["test_macro_f1"] = float(eval_data["test"]["macro_f1"])
        fallback["test_weighted_f1"] = float(eval_data["test"]["weighted_f1"])

    if train:
        fallback["best_epoch"] = train.get("best_epoch", fallback["best_epoch"])

    return {
        "seed": seed,
        "run_name": prefix,
        "best_epoch": fallback["best_epoch"],
        "test_accuracy": float(fallback["test_accuracy"]),
        "test_macro_f1": float(fallback["test_macro_f1"]),
        "test_weighted_f1": float(fallback["test_weighted_f1"]),
        "status": "ok",
    }

# ============================================================
# 3. Tabel final A-clean strict 3 seed
# ============================================================

clean_rows = []

for seed in SEEDS:
    clean_rows.append(get_clean_metrics(seed))

clean_df = pd.DataFrame(clean_rows)
clean_df = clean_df[
    [
        "seed",
        "run_name",
        "best_epoch",
        "val_accuracy",
        "val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "status",
    ]
].copy()

clean_df.to_csv(OUT_RESULTS, index=False)

print("\n=== Tabel Final A-clean Strict 3 Seed ===")
display(clean_df)

# ============================================================
# 4. Summary stats A-clean strict 3 seed
# ============================================================

metric_cols = [
    "val_accuracy",
    "val_macro_f1",
    "test_accuracy",
    "test_macro_f1",
    "test_weighted_f1",
]

stats_rows = []

for metric in metric_cols:
    values = clean_df[metric].astype(float)

    stats_rows.append({
        "metric": metric,
        "n_seed": int(values.shape[0]),
        "mean": float(values.mean()),
        "std_sample": float(values.std(ddof=1)),
        "std_population": float(values.std(ddof=0)),
        "min": float(values.min()),
        "max": float(values.max()),
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(OUT_STATS, index=False)

print("\n=== Statistik A-clean Strict 3 Seed ===")
display(stats_df)

# ============================================================
# 5. Paired comparison vs A official
# ============================================================

delta_rows = []

for seed in SEEDS:
    clean = get_clean_metrics(seed)
    official = get_official_metrics(seed)

    delta_rows.append({
        "seed": seed,

        "A_official_best_epoch": official["best_epoch"],
        "A_clean_best_epoch": clean["best_epoch"],

        "A_official_test_accuracy": official["test_accuracy"],
        "A_clean_test_accuracy": clean["test_accuracy"],
        "delta_test_accuracy": clean["test_accuracy"] - official["test_accuracy"],

        "A_official_test_macro_f1": official["test_macro_f1"],
        "A_clean_test_macro_f1": clean["test_macro_f1"],
        "delta_test_macro_f1": clean["test_macro_f1"] - official["test_macro_f1"],

        "A_official_test_weighted_f1": official["test_weighted_f1"],
        "A_clean_test_weighted_f1": clean["test_weighted_f1"],
        "delta_test_weighted_f1": clean["test_weighted_f1"] - official["test_weighted_f1"],

        "A_clean_above_2021_accuracy": clean["test_accuracy"] > BENCHMARK_2021["accuracy"],
        "A_clean_above_2021_f1": clean["test_macro_f1"] > BENCHMARK_2021["f1"],
    })

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(OUT_DELTA, index=False)

print("\n=== Paired Delta A-clean Strict vs A Official ===")
display(delta_df)

# ============================================================
# 6. Mean delta paired
# ============================================================

delta_metric_cols = [
    "delta_test_accuracy",
    "delta_test_macro_f1",
    "delta_test_weighted_f1",
]

delta_summary_rows = []

for metric in delta_metric_cols:
    values = delta_df[metric].astype(float)

    delta_summary_rows.append({
        "metric": metric,
        "n_seed": int(values.shape[0]),
        "mean_delta": float(values.mean()),
        "std_sample": float(values.std(ddof=1)),
        "std_population": float(values.std(ddof=0)),
        "min_delta": float(values.min()),
        "max_delta": float(values.max()),
    })

delta_summary_df = pd.DataFrame(delta_summary_rows)

print("\n=== Summary Delta Paired ===")
display(delta_summary_df)

# ============================================================
# 7. Interpretasi otomatis
# ============================================================

all_above_accuracy = bool(delta_df["A_clean_above_2021_accuracy"].all())
all_above_f1 = bool(delta_df["A_clean_above_2021_f1"].all())

consistent_accuracy_drop = bool((delta_df["delta_test_accuracy"] < 0).all())
consistent_macro_f1_drop = bool((delta_df["delta_test_macro_f1"] < 0).all())
consistent_weighted_f1_drop = bool((delta_df["delta_test_weighted_f1"] < 0).all())

mean_delta_accuracy = float(delta_df["delta_test_accuracy"].mean())
mean_delta_macro_f1 = float(delta_df["delta_test_macro_f1"].mean())
mean_delta_weighted_f1 = float(delta_df["delta_test_weighted_f1"].mean())

mean_test_accuracy = float(clean_df["test_accuracy"].mean())
mean_test_macro_f1 = float(clean_df["test_macro_f1"].mean())
mean_test_weighted_f1 = float(clean_df["test_weighted_f1"].mean())

if consistent_accuracy_drop and consistent_macro_f1_drop:
    drop_interpretation = "A-clean strict menunjukkan penurunan konsisten pada accuracy dan macro F1."
else:
    drop_interpretation = (
        "A-clean strict tidak menunjukkan penurunan konsisten: "
        "seed 42 turun, seed 123 turun kecil, dan seed 456 meningkat."
    )

ready_stop_3seed = bool(
    all_above_accuracy and
    all_above_f1 and
    not consistent_accuracy_drop and
    not consistent_macro_f1_drop
)

recommendation_next = (
    "Cukup stop A-clean strict di 3 seed untuk saat ini, lalu buat report sensitivity dan minta MASTER CHAT memilih antara C-G09 sensitivity atau analisis lanjutan."
    if ready_stop_3seed
    else
    "Pertimbangkan review MASTER CHAT sebelum memutuskan apakah perlu seed tambahan."
)

summary = {
    "created_at": datetime.now().isoformat(),
    "task": "Summary A-clean strict 3-seed and paired delta vs A official",
    "included_seeds": SEEDS,
    "benchmark_2021": BENCHMARK_2021,
    "output_files": {
        "A_clean_3seed_results": str(OUT_RESULTS),
        "A_clean_3seed_summary_stats": str(OUT_STATS),
        "paired_delta_vs_A_official": str(OUT_DELTA),
        "summary_json": str(OUT_SUMMARY),
    },
    "A_clean_3seed_summary": {
        "test_accuracy_mean": mean_test_accuracy,
        "test_accuracy_std_sample": float(clean_df["test_accuracy"].std(ddof=1)),
        "test_macro_f1_mean": mean_test_macro_f1,
        "test_macro_f1_std_sample": float(clean_df["test_macro_f1"].std(ddof=1)),
        "test_weighted_f1_mean": mean_test_weighted_f1,
        "test_weighted_f1_std_sample": float(clean_df["test_weighted_f1"].std(ddof=1)),
    },
    "paired_delta_summary": {
        "mean_delta_test_accuracy": mean_delta_accuracy,
        "mean_delta_test_macro_f1": mean_delta_macro_f1,
        "mean_delta_test_weighted_f1": mean_delta_weighted_f1,
        "consistent_accuracy_drop": consistent_accuracy_drop,
        "consistent_macro_f1_drop": consistent_macro_f1_drop,
        "consistent_weighted_f1_drop": consistent_weighted_f1_drop,
    },
    "benchmark_check": {
        "all_A_clean_seeds_above_2021_accuracy": all_above_accuracy,
        "all_A_clean_seeds_above_2021_f1": all_above_f1,
    },
    "interpretation": {
        "drop_interpretation": drop_interpretation,
        "ready_to_stop_at_3seed": ready_stop_3seed,
        "recommendation_next": recommendation_next,
    },
    "records": {
        "A_clean_results": clean_df.to_dict(orient="records"),
        "summary_stats": stats_df.to_dict(orient="records"),
        "paired_delta": delta_df.to_dict(orient="records"),
        "paired_delta_summary": delta_summary_df.to_dict(orient="records"),
    }
}

with open(OUT_SUMMARY, "w") as f:
    json.dump(summary, f, indent=2)

# ============================================================
# 8. Ringkasan akhir untuk MASTER CHAT
# ============================================================

print("\n=== File Output B35 Tersimpan ===")
print("A-clean 3-seed results:", OUT_RESULTS)
print("A-clean 3-seed summary stats:", OUT_STATS)
print("Paired delta vs A official:", OUT_DELTA)
print("Summary JSON:", OUT_SUMMARY)

print("\n=== Ringkasan Sel B35 untuk MASTER CHAT ===")
print(f"A-clean strict 3-seed test accuracy mean: {mean_test_accuracy:.4f}")
print(f"A-clean strict 3-seed test macro F1 mean: {mean_test_macro_f1:.4f}")
print(f"A-clean strict 3-seed test weighted F1 mean: {mean_test_weighted_f1:.4f}")

print("\nMean paired delta vs A official:")
print(f"- mean delta test accuracy: {mean_delta_accuracy:+.4f}")
print(f"- mean delta test macro F1: {mean_delta_macro_f1:+.4f}")
print(f"- mean delta test weighted F1: {mean_delta_weighted_f1:+.4f}")

print("\nConsistency check:")
print("A-clean strict causes consistent accuracy drop:", consistent_accuracy_drop)
print("A-clean strict causes consistent macro F1 drop:", consistent_macro_f1_drop)
print("All A-clean seeds above benchmark 2021 accuracy:", all_above_accuracy)
print("All A-clean seeds above benchmark 2021 F1:", all_above_f1)

print("\nInterpretasi:")
print(drop_interpretation)
print("Recommendation:", recommendation_next)

print("\nTabel final A-clean strict 3 seed:")
display(clean_df)

print("\nTabel paired delta:")
display(delta_df)

In [ ]:
# Sel B36 — DenseNet201 C-G09 Date+Magnification Seed 42

import os
import gc
import json
import time
import random
import zipfile
import shutil
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B36: DenseNet201 C-G09 Date+Magnification Seed 42 ===")
print("Mode kerja: training sensitivity C-G09 Date+Magnification seed 42.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B36.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_C_G09_seed42"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_C_G09_PATH = OUTPUT_DIR / "v5_split_manifest_B_G09_Date_Magnification.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_C_G09_seed42"
EXPERIMENT_ID = "DenseNet201_C_G09_seed42"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"
COMPARISON_VS_AB_CSV = OUTPUT_DIR / f"{RUN_NAME}_comparison_vs_A_official_B_G14_seed42.csv"

print("\n=== Cek Path ===")
print("Manifest C-G09:", MANIFEST_C_G09_PATH, "| exists:", MANIFEST_C_G09_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_C_G09_PATH.exists():
    raise FileNotFoundError(f"Manifest C-G09 tidak ditemukan: {MANIFEST_C_G09_PATH}")

# ============================================================
# 3. Fungsi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed42_initial",
    "DenseNet201_B_G14_seed42",
    "DenseNet201_A_clean_strict_seed42",
    "DenseNet201_A_clean_strict_seed123",
    "DenseNet201_A_clean_strict_seed456",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 42 C-G09
# ============================================================

SEED = 42
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

# Fallback nilai dari hasil yang sudah diterima MASTER CHAT.
A_OFFICIAL_SEED42_REF = {
    "scenario": "A_official",
    "best_epoch": 20,
    "test_accuracy": 0.9658,
    "test_macro_f1": 0.9587,
    "test_weighted_f1": 0.9657,
}

B_G14_SEED42_REF = {
    "scenario": "B_G14",
    "best_epoch": 20,
    "test_accuracy": 0.9580,
    "test_macro_f1": 0.9596,
    "test_weighted_f1": 0.9568,
}

def try_read_eval(prefix, fallback):
    eval_path = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    train_path = OUTPUT_DIR / f"{prefix}_training_summary.json"
    ref = fallback.copy()

    if eval_path.exists():
        try:
            with open(eval_path, "r") as f:
                data = json.load(f)
            ref["test_accuracy"] = float(data["test"]["accuracy"])
            ref["test_macro_f1"] = float(data["test"]["macro_f1"])
            ref["test_weighted_f1"] = float(data["test"]["weighted_f1"])
        except Exception as e:
            print(f"Warning: gagal baca eval {prefix}, pakai fallback. Error:", e)

    if train_path.exists():
        try:
            with open(train_path, "r") as f:
                tr = json.load(f)
            ref["best_epoch"] = tr.get("best_epoch", ref["best_epoch"])
        except Exception:
            pass

    return ref

A_OFFICIAL_SEED42_REF = try_read_eval("DenseNet201_A_official_seed42", A_OFFICIAL_SEED42_REF)
B_G14_SEED42_REF = try_read_eval("DenseNet201_B_G14_seed42", B_G14_SEED42_REF)

print("\n=== Reference Seed 42 ===")
print("A official seed 42:", A_OFFICIAL_SEED42_REF)
print("B-G14 seed 42:", B_G14_SEED42_REF)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 42 C-G09 ===")
print("Scenario: C-G09_Date_Magnification")
print("Function: stricter acquisition session-aware sensitivity evaluation")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai untuk C-G09")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "C_G09_Date_Magnification",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity training DenseNet201 C-G09 Date+Magnification seed 42. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai C-G09 seed 42.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest C-G09
# ============================================================

df = pd.read_csv(MANIFEST_C_G09_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest C-G09 ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Columns:", list(df.columns))
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

# Cek 14 kelas per split
class_check_rows = []
for split in ["train", "validation", "test"]:
    s = df[df["split"] == split]
    class_check_rows.append({
        "split": split,
        "num_classes": int(s["class_name"].nunique()),
        "all_14_classes_present": bool(s["class_name"].nunique() == 14)
    })

class_check_df = pd.DataFrame(class_check_rows)
print("\n=== Cek 14 kelas per split ===")
display(class_check_df)

if not class_check_df["all_14_classes_present"].all():
    raise ValueError("Tidak semua split memiliki 14 kelas. Cek manifest C-G09.")

# Cek overlap group jika kolom tersedia
possible_group_cols = [
    "group_id",
    "G09_Date_Magnification",
    "G09",
    "Date_Magnification",
    "raw_source_id"
]

print("\n=== Group/Source Overlap Check ===")
for col in possible_group_cols:
    if col in df.columns:
        train_set = set(df[df["split"] == "train"][col].astype(str))
        val_set = set(df[df["split"] == "validation"][col].astype(str))
        test_set = set(df[df["split"] == "test"][col].astype(str))

        print(f"\nKolom: {col}")
        print("Unique train:", len(train_set))
        print("Unique validation:", len(val_set))
        print("Unique test:", len(test_set))
        print("Overlap train-validation:", len(train_set & val_set))
        print("Overlap train-test:", len(train_set & test_set))
        print("Overlap validation-test:", len(val_set & test_set))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

print("\nClass names ordered:")
for i, name in enumerate(class_names_ordered):
    print(i, name)

# ============================================================
# 9. Dataset dan DataLoader
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 10. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 11. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 12. Training C-G09 seed 42
# ============================================================

print("\n=== Mulai Training C-G09 Seed 42 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "C_G09_Date_Magnification",
                "run_type": "sensitivity",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "C_G09_Date_Magnification",
            "run_type": "sensitivity",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "C_G09_Date_Magnification",
        "run_type": "sensitivity",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training C-G09 Seed 42 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining C-G09 seed 42 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 13. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint C-G09 Seed 42 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics C-G09 seed 42:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics C-G09 seed 42:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report C-G09 seed 42:")
print(test_report_text)

# ============================================================
# 14. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "C_G09_Date_Magnification",
    "run_type": "sensitivity",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "comparison_reference": {
        "A_official_seed42": A_OFFICIAL_SEED42_REF,
        "B_G14_seed42": B_G14_SEED42_REF
    },
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 15. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "C_G09_Date_Magnification",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity DenseNet201 C-G09 Date+Magnification seed 42 selesai. Checkpoint local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Bandingkan singkat dengan A official seed 42 dan B-G14 seed 42
# ============================================================

comparison_rows = [
    {
        "run_name": "DenseNet201_A_official_seed42",
        "scenario": "A_official",
        "seed": 42,
        "best_epoch": A_OFFICIAL_SEED42_REF["best_epoch"],
        "test_accuracy": A_OFFICIAL_SEED42_REF["test_accuracy"],
        "test_macro_f1": A_OFFICIAL_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": A_OFFICIAL_SEED42_REF["test_weighted_f1"],
    },
    {
        "run_name": "DenseNet201_B_G14_seed42",
        "scenario": "B_G14",
        "seed": 42,
        "best_epoch": B_G14_SEED42_REF["best_epoch"],
        "test_accuracy": B_G14_SEED42_REF["test_accuracy"],
        "test_macro_f1": B_G14_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": B_G14_SEED42_REF["test_weighted_f1"],
    },
    {
        "run_name": RUN_NAME,
        "scenario": "C_G09_Date_Magnification",
        "seed": 42,
        "best_epoch": best_epoch,
        "test_accuracy": test_summary["accuracy"],
        "test_macro_f1": test_summary["macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"],
    },
]

comparison_df = pd.DataFrame(comparison_rows)

delta_vs_a = {
    "run_name": "delta_C_G09_minus_A_official_seed42",
    "scenario": "delta_vs_A",
    "seed": 42,
    "best_epoch": "",
    "test_accuracy": test_summary["accuracy"] - A_OFFICIAL_SEED42_REF["test_accuracy"],
    "test_macro_f1": test_summary["macro_f1"] - A_OFFICIAL_SEED42_REF["test_macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"] - A_OFFICIAL_SEED42_REF["test_weighted_f1"],
}

delta_vs_b = {
    "run_name": "delta_C_G09_minus_B_G14_seed42",
    "scenario": "delta_vs_B_G14",
    "seed": 42,
    "best_epoch": "",
    "test_accuracy": test_summary["accuracy"] - B_G14_SEED42_REF["test_accuracy"],
    "test_macro_f1": test_summary["macro_f1"] - B_G14_SEED42_REF["test_macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"] - B_G14_SEED42_REF["test_weighted_f1"],
}

comparison_df = pd.concat(
    [comparison_df, pd.DataFrame([delta_vs_a, delta_vs_b])],
    ignore_index=True
)

comparison_df.to_csv(COMPARISON_VS_AB_CSV, index=False)

# ============================================================
# 18. Ringkasan akhir
# ============================================================

print("\n=== File Output C-G09 Seed 42 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Validation report:", VAL_REPORT_CSV_PATH)
print("Test report:", TEST_REPORT_CSV_PATH)
print("Validation CM:", VAL_CM_CSV_PATH)
print("Test CM:", TEST_CM_CSV_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)
print("Comparison vs A/B seed42:", COMPARISON_VS_AB_CSV)

print("\n=== Ringkasan Sel B36 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Delta vs A Official Seed 42 ===")
print(f"Delta test accuracy    : {delta_vs_a['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {delta_vs_a['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {delta_vs_a['test_weighted_f1']:.4f}")

print("\n=== Delta vs B-G14 Seed 42 ===")
print(f"Delta test accuracy    : {delta_vs_b['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {delta_vs_b['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {delta_vs_b['test_weighted_f1']:.4f}")

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Comparison Table ===")
display(comparison_df)

print("\nSel B36 selesai. Jangan lanjut seed 123 sebelum hasil seed 42 dilaporkan ke MASTER CHAT.")

In [ ]:
# Sel B36 — Resume dan Finalisasi C-G09 Seed 42

import os
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix

print("=== Sel B36: Resume dan Finalisasi C-G09 Seed 42 ===")
print("Mode kerja: resume dari checkpoint terakhir, bukan training ulang dari awal.")
print("Checkpoint policy: local-first-then-Drive.")

# ============================================================
# 1. Pastikan variable dari B36 original masih ada
# ============================================================

required_vars = [
    "RUN_NAME", "EXPERIMENT_ID", "model", "optimizer", "scheduler", "scaler",
    "train_loader", "val_loader", "test_loader", "train_one_epoch", "evaluate",
    "predict_full", "compute_metrics", "criterion", "device",
    "LOCAL_BEST_CKPT_PATH", "LOCAL_LAST_CKPT_PATH",
    "BEST_CKPT_PATH", "LAST_CKPT_PATH",
    "HISTORY_CSV_PATH", "HISTORY_JSON_PATH",
    "TRAINING_SUMMARY_PATH", "EVAL_SUMMARY_JSON_PATH", "EVAL_SUMMARY_CSV_PATH",
    "VAL_PRED_PATH", "TEST_PRED_PATH",
    "VAL_REPORT_CSV_PATH", "TEST_REPORT_CSV_PATH",
    "VAL_REPORT_TXT_PATH", "TEST_REPORT_TXT_PATH",
    "VAL_REPORT_JSON_PATH", "TEST_REPORT_JSON_PATH",
    "VAL_CM_CSV_PATH", "TEST_CM_CSV_PATH",
    "CHECKPOINT_VERIFY_JSON", "COMPARISON_VS_AB_CSV",
    "MASTER_LOG_PATH", "MASTER_LOG_COLUMNS",
    "A_OFFICIAL_SEED42_REF", "B_G14_SEED42_REF",
    "MAX_EPOCHS", "PATIENCE", "MODEL_SELECTION_METRIC", "TIE_BREAKER_METRIC",
    "NUM_CLASSES", "class_names_ordered", "id_to_class"
]

missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(
        "Variable dari B36 original tidak lengkap. Jangan restart runtime. "
        f"Variable hilang: {missing_vars}"
    )

print("Variable B36 original lengkap. Lanjut resume aman.")

# ============================================================
# 2. Helper checkpoint dan safe save
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_path.parent.mkdir(parents=True, exist_ok=True)
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    # Simpan ke local dulu
    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    # Baru copy ke Drive dan verifikasi
    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)


def safe_to_csv(df, path, index=False):
    df.to_csv(path, index=index)
    return str(path)


def safe_write_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    return str(path)


def safe_write_text(text, path):
    with open(path, "w") as f:
        f.write(text)
    return str(path)

# ============================================================
# 3. Load last checkpoint
# ============================================================

resume_path = None

if Path(LOCAL_LAST_CKPT_PATH).exists():
    resume_path = Path(LOCAL_LAST_CKPT_PATH)
elif Path(LAST_CKPT_PATH).exists():
    resume_path = Path(LAST_CKPT_PATH)
else:
    raise FileNotFoundError("Last checkpoint C-G09 seed 42 tidak ditemukan.")

print("\n=== Load Last Checkpoint ===")
print("Resume checkpoint:", resume_path)

resume_ckpt = torch.load(resume_path, map_location=device, weights_only=False)

model.load_state_dict(resume_ckpt["model_state_dict"])
optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
scheduler.load_state_dict(resume_ckpt["scheduler_state_dict"])

if resume_ckpt.get("scaler_state_dict") is not None and scaler is not None:
    scaler.load_state_dict(resume_ckpt["scaler_state_dict"])

history = resume_ckpt.get("history", [])
best_epoch = resume_ckpt.get("best_epoch")
best_metric = resume_ckpt.get("best_metric")
best_tie_metric = resume_ckpt.get("best_tie_metric")
epochs_without_improvement = resume_ckpt.get("epochs_without_improvement", 0)
last_finished_epoch = resume_ckpt.get("epoch")
start_epoch = int(last_finished_epoch) + 1

print("Last finished epoch:", last_finished_epoch)
print("Resume start epoch:", start_epoch)
print("Best epoch so far:", best_epoch)
print("Best val_macro_f1 so far:", best_metric)
print("Best val_accuracy so far:", best_tie_metric)
print("Epochs without improvement:", epochs_without_improvement)

# ============================================================
# 4. Resume training
# ============================================================

resume_start_time = time.time()

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    print("\n" + "=" * 80)
    print(f"Resume Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
    print(f"LR: {current_lr:.8f}")
    print("=" * 80)

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader, "val")

    scheduler.step(val_metrics["val_macro_f1"])

    record = {
        "epoch": epoch,
        "lr": current_lr,
        "epoch_time_min": (time.time() - epoch_start) / 60,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_precision": train_metrics["macro_precision"],
        "train_macro_recall": train_metrics["macro_recall"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_weighted_f1": train_metrics["weighted_f1"],
        **val_metrics
    }

    history.append(record)

    current_metric = float(record[MODEL_SELECTION_METRIC])
    current_tie_metric = float(record[TIE_BREAKER_METRIC])

    improved = (
        (current_metric > best_metric + 1e-8) or
        (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
    )

    if improved:
        best_metric = current_metric
        best_tie_metric = current_tie_metric
        best_epoch = epoch
        epochs_without_improvement = 0

        best_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "C_G09_Date_Magnification",
            "run_type": "sensitivity",
            "seed": 42,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive_after_resume"
        }

        safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

        print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
        print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
        print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

    last_checkpoint = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "C_G09_Date_Magnification",
        "run_type": "sensitivity",
        "seed": 42,
        "epoch": epoch,
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "epochs_without_improvement": epochs_without_improvement,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "id_to_class": {str(k): v for k, v in id_to_class.items()},
        "saved_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive_after_resume"
    }

    safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

    safe_to_csv(pd.DataFrame(history), HISTORY_CSV_PATH, index=False)
    safe_write_json(history, HISTORY_JSON_PATH)

    print("\nRingkasan epoch:")
    print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
    print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
    print(f"Best epoch sementara: {best_epoch}")

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping aktif.")
        break

resume_time_min = (time.time() - resume_start_time) / 60

training_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "C_G09_Date_Magnification",
    "run_type": "sensitivity",
    "seed": 42,
    "status": "completed_after_resume",
    "max_epochs": MAX_EPOCHS,
    "epochs_completed": len(history),
    "best_epoch": best_epoch,
    "best_metric_name": MODEL_SELECTION_METRIC,
    "best_metric": best_metric,
    "best_tie_metric_name": TIE_BREAKER_METRIC,
    "best_tie_metric": best_tie_metric,
    "resume_from_epoch": int(last_finished_epoch),
    "resume_time_min": resume_time_min,
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "last_checkpoint_path": str(LAST_CKPT_PATH),
    "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
    "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
    "history_csv_path": str(HISTORY_CSV_PATH),
    "history_json_path": str(HISTORY_JSON_PATH),
    "completed_at": datetime.now().isoformat(),
    "save_policy": "local_first_then_drive_after_resume"
}

safe_write_json(training_summary, TRAINING_SUMMARY_PATH)

print("\n=== Training Resume B36 Selesai ===")
print("Best epoch:", best_epoch)
print(f"Best val_macro_f1: {best_metric:.4f}")
print(f"Best val_accuracy: {best_tie_metric:.4f}")

# ============================================================
# 5. Evaluasi best checkpoint
# ============================================================

print("\n=== Evaluasi Best Checkpoint C-G09 Seed 42 ===")

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics C-G09 seed 42:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics C-G09 seed 42:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

# ============================================================
# 6. Simpan output evaluasi
# ============================================================

safe_to_csv(val_pred["result_df"], VAL_PRED_PATH, index=False)
safe_to_csv(test_pred["result_df"], TEST_PRED_PATH, index=False)

safe_to_csv(pd.DataFrame(val_report_dict).T, VAL_REPORT_CSV_PATH, index=True)
safe_to_csv(pd.DataFrame(test_report_dict).T, TEST_REPORT_CSV_PATH, index=True)

safe_write_text(val_report_text, VAL_REPORT_TXT_PATH)
safe_write_text(test_report_text, TEST_REPORT_TXT_PATH)

safe_write_json(val_report_dict, VAL_REPORT_JSON_PATH)
safe_write_json(test_report_dict, TEST_REPORT_JSON_PATH)

safe_to_csv(
    pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered),
    VAL_CM_CSV_PATH,
    index=True
)
safe_to_csv(
    pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered),
    TEST_CM_CSV_PATH,
    index=True
)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "C_G09_Date_Magnification",
    "run_type": "sensitivity",
    "seed": 42,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "comparison_reference": {
        "A_official_seed42": A_OFFICIAL_SEED42_REF,
        "B_G14_seed42": B_G14_SEED42_REF
    }
}

safe_write_json(eval_summary, EVAL_SUMMARY_JSON_PATH)

safe_to_csv(pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]), EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 7. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "C_G09_Date_Magnification",
    "seed": 42,
    "run_type": "sensitivity",
    "start_time": master_log.loc[
        master_log["experiment_id"].astype(str) == EXPERIMENT_ID, "start_time"
    ].iloc[-1] if (master_log["experiment_id"].astype(str) == EXPERIMENT_ID).any() else "",
    "end_time": datetime.now().isoformat(),
    "input_size": 224,
    "batch_size": 16,
    "optimizer": "AdamW",
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "C-G09 seed 42 completed after resume from interrupted epoch 12. Checkpoint stored with local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
safe_to_csv(master_log, MASTER_LOG_PATH, index=False)

# ============================================================
# 8. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
    "save_policy": "local_first_then_drive_after_resume"
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

safe_write_json(checkpoint_verification, CHECKPOINT_VERIFY_JSON)

# ============================================================
# 9. Perbandingan vs A official dan B-G14 seed 42
# ============================================================

comparison_rows = [
    {
        "run_name": "DenseNet201_A_official_seed42",
        "scenario": "A_official",
        "seed": 42,
        "best_epoch": A_OFFICIAL_SEED42_REF["best_epoch"],
        "test_accuracy": A_OFFICIAL_SEED42_REF["test_accuracy"],
        "test_macro_f1": A_OFFICIAL_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": A_OFFICIAL_SEED42_REF["test_weighted_f1"],
    },
    {
        "run_name": "DenseNet201_B_G14_seed42",
        "scenario": "B_G14",
        "seed": 42,
        "best_epoch": B_G14_SEED42_REF["best_epoch"],
        "test_accuracy": B_G14_SEED42_REF["test_accuracy"],
        "test_macro_f1": B_G14_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": B_G14_SEED42_REF["test_weighted_f1"],
    },
    {
        "run_name": RUN_NAME,
        "scenario": "C_G09_Date_Magnification",
        "seed": 42,
        "best_epoch": best_epoch,
        "test_accuracy": test_summary["accuracy"],
        "test_macro_f1": test_summary["macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"],
    },
    {
        "run_name": "delta_C_G09_minus_A_official_seed42",
        "scenario": "delta_vs_A",
        "seed": 42,
        "best_epoch": "",
        "test_accuracy": test_summary["accuracy"] - A_OFFICIAL_SEED42_REF["test_accuracy"],
        "test_macro_f1": test_summary["macro_f1"] - A_OFFICIAL_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"] - A_OFFICIAL_SEED42_REF["test_weighted_f1"],
    },
    {
        "run_name": "delta_C_G09_minus_B_G14_seed42",
        "scenario": "delta_vs_B_G14",
        "seed": 42,
        "best_epoch": "",
        "test_accuracy": test_summary["accuracy"] - B_G14_SEED42_REF["test_accuracy"],
        "test_macro_f1": test_summary["macro_f1"] - B_G14_SEED42_REF["test_macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"] - B_G14_SEED42_REF["test_weighted_f1"],
    },
]

comparison_df = pd.DataFrame(comparison_rows)
safe_to_csv(comparison_df, COMPARISON_VS_AB_CSV, index=False)

# ============================================================
# 10. Ringkasan akhir
# ============================================================

print("\n=== Ringkasan Sel B36 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Delta vs A Official Seed 42 ===")
print(f"Delta test accuracy    : {comparison_rows[3]['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {comparison_rows[3]['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {comparison_rows[3]['test_weighted_f1']:.4f}")

print("\n=== Delta vs B-G14 Seed 42 ===")
print(f"Delta test accuracy    : {comparison_rows[4]['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {comparison_rows[4]['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {comparison_rows[4]['test_weighted_f1']:.4f}")

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Comparison Table ===")
display(comparison_df)

In [ ]:
# Sel B37 — DenseNet201 C-G09 Date+Magnification Seed 123

import os, gc, json, time, random, zipfile, shutil, traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("=== Sel B37: DenseNet201 C-G09 Date+Magnification Seed 123 ===")
print("Mode kerja: training sensitivity C-G09 Date+Magnification seed 123.")
print("Checkpoint policy: local-first-then-Drive, dengan verifikasi torch.load.")

# ============================================================
# 1. Mount Drive dan cek GPU
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU sebelum menjalankan Sel B37.")

device = torch.device("cuda")

print("\n=== Cek Runtime ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. Path utama
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus")
OUTPUT_DIR = PROJECT_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
DATA_RAW_DIR = PROJECT_DIR / "data_raw"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_CKPT_ROOT = Path("/content/tem_virus_runtime_checkpoints")
LOCAL_WORK_DIR = LOCAL_RUNTIME_CKPT_ROOT / "DenseNet201_C_G09_seed123"
LOCAL_CKPT_DIR = LOCAL_WORK_DIR / "checkpoints"
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DATA_RAW_DIR / "TEM virus dataset.zip"
LOCAL_ZIP = Path("/content/TEM_virus_dataset_local.zip")
EXTRACT_ROOT = Path("/content/tem_virus_dataset_extracted")
DATASET_DIR = EXTRACT_ROOT / "TEM virus dataset" / "context_virus_1nm_256x256"

MANIFEST_C_G09_PATH = OUTPUT_DIR / "v5_split_manifest_B_G09_Date_Magnification.csv"
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

RUN_NAME = "DenseNet201_C_G09_seed123"
EXPERIMENT_ID = "DenseNet201_C_G09_seed123"

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"

LOCAL_BEST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_CKPT_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.txt"
TEST_REPORT_TXT_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.txt"
VAL_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.json"
TEST_REPORT_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.json"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"

CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"
COMPARISON_VS_AB_CSV = OUTPUT_DIR / f"{RUN_NAME}_comparison_vs_A_official_B_G14_seed123.csv"

print("\n=== Cek Path ===")
print("Manifest C-G09:", MANIFEST_C_G09_PATH, "| exists:", MANIFEST_C_G09_PATH.exists())
print("Dataset dir:", DATASET_DIR, "| exists:", DATASET_DIR.exists())
print("Master log:", MASTER_LOG_PATH, "| exists:", MASTER_LOG_PATH.exists())
print("Run name:", RUN_NAME)

if not MANIFEST_C_G09_PATH.exists():
    raise FileNotFoundError(f"Manifest C-G09 tidak ditemukan: {MANIFEST_C_G09_PATH}")

# ============================================================
# 3. Fungsi checkpoint aman
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    result = {
        "label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": 0.0,
        "load_ok": False,
        "epoch": None,
        "best_epoch": None,
        "best_metric": None,
        "best_tie_metric": None,
        "error": None
    }

    if not path.exists():
        result["error"] = "file_not_found"
        return result

    result["size_mb"] = path.stat().st_size / 1024**2

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        result["load_ok"] = True
        result["epoch"] = ckpt.get("epoch")
        result["best_epoch"] = ckpt.get("best_epoch")
        result["best_metric"] = ckpt.get("best_metric")
        result["best_tie_metric"] = ckpt.get("best_tie_metric")
    except Exception as e:
        result["error"] = repr(e)

    return result


def safe_save_checkpoint(checkpoint_obj, local_path, drive_path):
    local_path = Path(local_path)
    drive_path = Path(drive_path)

    local_path.parent.mkdir(parents=True, exist_ok=True)
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")

    if local_tmp.exists():
        local_tmp.unlink()
    if drive_tmp.exists():
        drive_tmp.unlink()

    torch.save(checkpoint_obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

# ============================================================
# 4. Master experiment log
# ============================================================

MASTER_LOG_COLUMNS = [
    "experiment_id", "model", "scenario", "seed", "run_type",
    "start_time", "end_time", "input_size", "batch_size",
    "optimizer", "learning_rate", "weight_decay", "scheduler",
    "max_epoch", "best_epoch", "val_accuracy", "val_macro_f1",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "checkpoint_path", "notes"
]

print("\n=== Cek Master Experiment Log ===")

if MASTER_LOG_PATH.exists():
    master_log = pd.read_csv(MASTER_LOG_PATH)
    for col in MASTER_LOG_COLUMNS:
        if col not in master_log.columns:
            master_log[col] = ""
else:
    master_log = pd.DataFrame(columns=MASTER_LOG_COLUMNS)
    print("Master log belum ada. File baru akan dibuat.")

expected_existing = [
    "DenseNet201_A_official_seed123",
    "DenseNet201_B_G14_seed123",
    "DenseNet201_C_G09_seed42",
]

for exp_id in expected_existing:
    found = exp_id in master_log["experiment_id"].astype(str).tolist()
    print(f"{exp_id}: {'tercatat' if found else 'belum tercatat'}")

display(master_log[master_log["experiment_id"].astype(str).isin(expected_existing)])

# ============================================================
# 5. Pastikan dataset lokal tersedia
# ============================================================

if not DATASET_DIR.exists():
    print("\nDataset lokal belum ada. Menyiapkan ulang dari ZIP lokal/Drive.")

    if not LOCAL_ZIP.exists():
        if not ZIP_DRIVE.exists():
            raise FileNotFoundError(f"ZIP Drive tidak ditemukan: {ZIP_DRIVE}")

        print("Copy ZIP dari Drive ke /content agar ekstraksi lebih stabil.")
        start_copy = time.time()
        shutil.copy2(ZIP_DRIVE, LOCAL_ZIP)
        print(f"Copy selesai dalam {(time.time() - start_copy) / 60:.2f} menit.")
    else:
        print("ZIP lokal sudah ada:", LOCAL_ZIP)

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
        print("Folder ekstraksi lama/parsial dihapus:", EXTRACT_ROOT)

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    target_folder_name = "context_virus_1nm_256x256"

    start_extract = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = [m for m in zf.namelist() if target_folder_name in m]
        print("Jumlah entry target:", len(members))

        for i, member in enumerate(members, start=1):
            zf.extract(member, EXTRACT_ROOT)
            if i % 3000 == 0 or i == len(members):
                print(f"Progress ekstraksi: {i}/{len(members)} | elapsed {(time.time()-start_extract)/60:.2f} menit")

    print("Ekstraksi selesai.")
else:
    print("\nDataset lokal sudah tersedia. Ekstraksi dilewati.")

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset lokal belum tersedia setelah recovery.")

# ============================================================
# 6. Setup eksperimen seed 123 C-G09
# ============================================================

SEED = 123
NUM_CLASSES = 14
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SELECTION_METRIC = "val_macro_f1"
TIE_BREAKER_METRIC = "val_accuracy"

A_OFFICIAL_SEED123_REF = {
    "scenario": "A_official",
    "best_epoch": 16,
    "test_accuracy": 0.9674,
    "test_macro_f1": 0.9623,
    "test_weighted_f1": 0.9674,
}

B_G14_SEED123_REF = {
    "scenario": "B_G14",
    "best_epoch": 11,
    "test_accuracy": 0.9637,
    "test_macro_f1": 0.9645,
    "test_weighted_f1": 0.9625,
}

def try_read_eval(prefix, fallback):
    eval_path = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    train_path = OUTPUT_DIR / f"{prefix}_training_summary.json"
    ref = fallback.copy()

    if eval_path.exists():
        try:
            with open(eval_path, "r") as f:
                data = json.load(f)
            ref["test_accuracy"] = float(data["test"]["accuracy"])
            ref["test_macro_f1"] = float(data["test"]["macro_f1"])
            ref["test_weighted_f1"] = float(data["test"]["weighted_f1"])
        except Exception as e:
            print(f"Warning: gagal baca eval {prefix}, pakai fallback. Error:", e)

    if train_path.exists():
        try:
            with open(train_path, "r") as f:
                tr = json.load(f)
            ref["best_epoch"] = tr.get("best_epoch", ref["best_epoch"])
        except Exception:
            pass

    return ref

A_OFFICIAL_SEED123_REF = try_read_eval("DenseNet201_A_official_seed123", A_OFFICIAL_SEED123_REF)
B_G14_SEED123_REF = try_read_eval("DenseNet201_B_G14_seed123", B_G14_SEED123_REF)

print("\n=== Reference Seed 123 ===")
print("A official seed 123:", A_OFFICIAL_SEED123_REF)
print("B-G14 seed 123:", B_G14_SEED123_REF)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("\n=== Setup Seed 123 C-G09 ===")
print("Scenario: C-G09_Date_Magnification")
print("Function: stricter acquisition session-aware sensitivity evaluation")
print("Seed:", SEED)
print("Input size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Scheduler: ReduceLROnPlateau")
print("Max epoch:", MAX_EPOCHS)
print("Checkpoint metric:", MODEL_SELECTION_METRIC)
print("Tie breaker:", TIE_BREAKER_METRIC)
print("Augmented train: tidak dipakai")
print("A-clean: tidak dipakai untuk C-G09")
print("Checkpoint save: local-first-then-Drive")

# ============================================================
# 7. Tandai start di master log
# ============================================================

start_time = datetime.now().isoformat()

start_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "C_G09_Date_Magnification",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": "",
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": "",
    "val_accuracy": "",
    "val_macro_f1": "",
    "test_accuracy": "",
    "test_macro_f1": "",
    "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity training DenseNet201 C-G09 Date+Magnification seed 123. Checkpoint memakai local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

print("\nMaster log ditandai mulai C-G09 seed 123.")
print(MASTER_LOG_PATH)

# ============================================================
# 8. Load manifest C-G09
# ============================================================

df = pd.read_csv(MANIFEST_C_G09_PATH)
df = df[df["split"].isin(["train", "validation", "test"])].copy()
df = df[df["class_name"] != "_EXCLUDED"].copy()
df = df[~df["filepath"].astype(str).str.contains("augmented", case=False, regex=False)].copy()

df["filepath_exists"] = df["filepath"].apply(lambda x: Path(str(x)).exists())
missing_count = int((~df["filepath_exists"]).sum())

print("\n=== Cek Manifest C-G09 ===")
print("Total rows:", len(df))
print("Missing filepath:", missing_count)
print("Columns:", list(df.columns))
print("Split counts:")
display(df["split"].value_counts().rename_axis("split").reset_index(name="count"))

if missing_count > 0:
    display(df[~df["filepath_exists"]].head(10))
    raise FileNotFoundError("Ada filepath yang tidak ditemukan. Cek dataset lokal dulu.")

class_check_rows = []
for split in ["train", "validation", "test"]:
    s = df[df["split"] == split]
    class_check_rows.append({
        "split": split,
        "num_classes": int(s["class_name"].nunique()),
        "all_14_classes_present": bool(s["class_name"].nunique() == 14)
    })

class_check_df = pd.DataFrame(class_check_rows)
print("\n=== Cek 14 kelas per split ===")
display(class_check_df)

if not class_check_df["all_14_classes_present"].all():
    raise ValueError("Tidak semua split memiliki 14 kelas. Cek manifest C-G09.")

print("\n=== Group/Source Overlap Check ===")
for col in ["group_id", "G09_Date_Magnification", "G09", "Date_Magnification", "raw_source_id"]:
    if col in df.columns:
        train_set = set(df[df["split"] == "train"][col].astype(str))
        val_set = set(df[df["split"] == "validation"][col].astype(str))
        test_set = set(df[df["split"] == "test"][col].astype(str))

        print(f"\nKolom: {col}")
        print("Unique train:", len(train_set))
        print("Unique validation:", len(val_set))
        print("Unique test:", len(test_set))
        print("Overlap train-validation:", len(train_set & val_set))
        print("Overlap train-test:", len(train_set & test_set))
        print("Overlap validation-test:", len(val_set & test_set))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

label_mapping = (
    df[["label_id", "class_name"]]
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)

id_to_class = dict(zip(label_mapping["label_id"].astype(int), label_mapping["class_name"]))
class_names_ordered = [id_to_class[i] for i in range(NUM_CLASSES)]

print("\nClass names ordered:")
for i, name in enumerate(class_names_ordered):
    print(i, name)

# ============================================================
# 9. Dataset dan DataLoader
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class TEMVirusDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "filepath": row["filepath"],
            "filename": row["filename"],
            "class_name": row["class_name"]
        }

train_dataset = TEMVirusDataset(train_df, train_transform)
val_dataset = TEMVirusDataset(val_df, eval_transform)
test_dataset = TEMVirusDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("\n=== DataLoader ===")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ============================================================
# 10. Model fresh DenseNet201 ImageNet
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print("\n=== Load DenseNet201 Fresh ImageNet ===")

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()

try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 11. Fungsi train/evaluasi
# ============================================================

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
    }

def train_one_epoch(model, loader):
    model.train()

    running_loss = 0.0
    y_true = []
    y_pred = []

    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs.detach(), dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(
                f"  Train batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={(time.time()-start)/60:.2f} menit"
            )

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics

@torch.no_grad()
def evaluate(model, loader, split_name):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if use_amp:
            try:
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            except TypeError:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = running_loss / len(loader.dataset)

    return {f"{split_name}_{k}": v for k, v in metrics.items()}

@torch.no_grad()
def predict_full(model, loader, split_name):
    model.eval()

    y_true = []
    y_pred = []
    probs_all = []
    filepaths = []
    filenames = []

    running_loss = 0.0

    for batch_idx, batch in enumerate(loader, start=1):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        probs_all.extend(probs.cpu().numpy().tolist())
        filepaths.extend(batch["filepath"])
        filenames.extend(batch["filename"])

        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"  {split_name} batch {batch_idx}/{len(loader)} selesai")

    result_df = pd.DataFrame({
        "split": split_name,
        "filepath": filepaths,
        "filename": filenames,
        "true_label": y_true,
        "pred_label": y_pred,
        "true_class": [id_to_class[int(x)] for x in y_true],
        "pred_class": [id_to_class[int(x)] for x in y_pred],
        "correct": [int(t == p) for t, p in zip(y_true, y_pred)]
    })

    probs_arr = np.array(probs_all)
    for class_id, class_name in id_to_class.items():
        safe_name = class_name.replace(" ", "_").replace("/", "_")
        result_df[f"prob_{class_id}_{safe_name}"] = probs_arr[:, class_id]

    return {
        "loss": running_loss / len(loader.dataset),
        "y_true": np.array(y_true),
        "y_pred": np.array(y_pred),
        "probs": probs_arr,
        "result_df": result_df
    }

# ============================================================
# 12. Training C-G09 seed 123
# ============================================================

print("\n=== Mulai Training C-G09 Seed 123 ===")

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

training_start = time.time()

try:
    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "=" * 80)
        print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME}")
        print(f"LR: {current_lr:.8f}")
        print("=" * 80)

        train_metrics = train_one_epoch(model, train_loader)
        val_metrics = evaluate(model, val_loader, "val")

        scheduler.step(val_metrics["val_macro_f1"])

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "epoch_time_min": (time.time() - epoch_start) / 60,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            **val_metrics
        }

        history.append(record)

        current_metric = float(record[MODEL_SELECTION_METRIC])
        current_tie_metric = float(record[TIE_BREAKER_METRIC])

        improved = (
            (current_metric > best_metric + 1e-8) or
            (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
        )

        if improved:
            best_metric = current_metric
            best_tie_metric = current_tie_metric
            best_epoch = epoch
            epochs_without_improvement = 0

            best_checkpoint = {
                "run_name": RUN_NAME,
                "experiment_id": EXPERIMENT_ID,
                "scenario": "C_G09_Date_Magnification",
                "run_type": "sensitivity",
                "seed": SEED,
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_metric_name": MODEL_SELECTION_METRIC,
                "best_metric": best_metric,
                "best_tie_metric_name": TIE_BREAKER_METRIC,
                "best_tie_metric": best_tie_metric,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "history": history,
                "id_to_class": {str(k): v for k, v in id_to_class.items()},
                "saved_at": datetime.now().isoformat(),
                "save_policy": "local_first_then_drive"
            }

            safe_save_checkpoint(best_checkpoint, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)

            print(f"\n-> Best checkpoint diperbarui pada epoch {epoch}")
            print(f"   val_macro_f1={best_metric:.4f}, val_accuracy={best_tie_metric:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"\n-> Belum ada improvement ({epochs_without_improvement}/{PATIENCE})")
            print(f"   current val_macro_f1={current_metric:.4f}, best={best_metric:.4f} epoch {best_epoch}")

        last_checkpoint = {
            "run_name": RUN_NAME,
            "experiment_id": EXPERIMENT_ID,
            "scenario": "C_G09_Date_Magnification",
            "run_type": "sensitivity",
            "seed": SEED,
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC,
            "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC,
            "best_tie_metric": best_tie_metric,
            "epochs_without_improvement": epochs_without_improvement,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }

        safe_save_checkpoint(last_checkpoint, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        with open(HISTORY_JSON_PATH, "w") as f:
            json.dump(history, f, indent=2)

        print("\nRingkasan epoch:")
        print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
        print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")
        print(f"Best epoch sementara: {best_epoch}")

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping aktif.")
            break

    total_training_time = (time.time() - training_start) / 60

    training_summary = {
        "run_name": RUN_NAME,
        "experiment_id": EXPERIMENT_ID,
        "scenario": "C_G09_Date_Magnification",
        "run_type": "sensitivity",
        "seed": SEED,
        "status": "completed",
        "max_epochs": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC,
        "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC,
        "best_tie_metric": best_tie_metric,
        "total_training_time_min": total_training_time,
        "best_checkpoint_path": str(BEST_CKPT_PATH),
        "last_checkpoint_path": str(LAST_CKPT_PATH),
        "local_best_checkpoint_path": str(LOCAL_BEST_CKPT_PATH),
        "local_last_checkpoint_path": str(LOCAL_LAST_CKPT_PATH),
        "history_csv_path": str(HISTORY_CSV_PATH),
        "history_json_path": str(HISTORY_JSON_PATH),
        "completed_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }

    with open(TRAINING_SUMMARY_PATH, "w") as f:
        json.dump(training_summary, f, indent=2)

    print("\n=== Training C-G09 Seed 123 Selesai ===")
    print("Best epoch:", best_epoch)
    print(f"Best val_macro_f1: {best_metric:.4f}")
    print(f"Best val_accuracy: {best_tie_metric:.4f}")
    print(f"Training time: {total_training_time:.2f} menit")

except Exception as e:
    print("\nTraining C-G09 seed 123 gagal.")
    print("Error:", e)
    print(traceback.format_exc())
    raise

# ============================================================
# 13. Evaluasi best checkpoint pada validation dan test
# ============================================================

print("\n=== Evaluasi Best Checkpoint C-G09 Seed 123 ===")

ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

def full_report(y_true, y_pred):
    summary = compute_metrics(y_true, y_pred)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    return summary, report_dict, report_text, cm

val_summary, val_report_dict, val_report_text, val_cm = full_report(
    val_pred["y_true"],
    val_pred["y_pred"]
)

test_summary, test_report_dict, test_report_text, test_cm = full_report(
    test_pred["y_true"],
    test_pred["y_pred"]
)

val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

print("\nValidation metrics C-G09 seed 123:")
for k, v in val_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics C-G09 seed 123:")
for k, v in test_summary.items():
    print(f"{k}: {v:.4f}")

print("\nTest classification report C-G09 seed 123:")
print(test_report_text)

# ============================================================
# 14. Simpan output evaluasi
# ============================================================

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)

pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)

with open(VAL_REPORT_TXT_PATH, "w") as f:
    f.write(val_report_text)

with open(TEST_REPORT_TXT_PATH, "w") as f:
    f.write(test_report_text)

with open(VAL_REPORT_JSON_PATH, "w") as f:
    json.dump(val_report_dict, f, indent=2)

with open(TEST_REPORT_JSON_PATH, "w") as f:
    json.dump(test_report_dict, f, indent=2)

pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME,
    "experiment_id": EXPERIMENT_ID,
    "scenario": "C_G09_Date_Magnification",
    "run_type": "sensitivity",
    "seed": SEED,
    "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary,
    "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
    "comparison_reference": {
        "A_official_seed123": A_OFFICIAL_SEED123_REF,
        "B_G14_seed123": B_G14_SEED123_REF
    },
    "files_saved": {
        "validation_predictions": str(VAL_PRED_PATH),
        "test_predictions": str(TEST_PRED_PATH),
        "validation_report_csv": str(VAL_REPORT_CSV_PATH),
        "test_report_csv": str(TEST_REPORT_CSV_PATH),
        "validation_report_txt": str(VAL_REPORT_TXT_PATH),
        "test_report_txt": str(TEST_REPORT_TXT_PATH),
        "validation_report_json": str(VAL_REPORT_JSON_PATH),
        "test_report_json": str(TEST_REPORT_JSON_PATH),
        "validation_confusion_matrix_csv": str(VAL_CM_CSV_PATH),
        "test_confusion_matrix_csv": str(TEST_CM_CSV_PATH)
    }
}

with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 15. Update master log
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for col in MASTER_LOG_COLUMNS:
    if col not in master_log.columns:
        master_log[col] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID,
    "model": "DenseNet201",
    "scenario": "C_G09_Date_Magnification",
    "seed": SEED,
    "run_type": "sensitivity",
    "start_time": start_time,
    "end_time": datetime.now().isoformat(),
    "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS,
    "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"],
    "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"],
    "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity DenseNet201 C-G09 Date+Magnification seed 123 selesai. Checkpoint local-first-then-Drive."
}

master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates(subset=["experiment_id"], keep="last").reset_index(drop=True)
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 16. Verifikasi checkpoint final
# ============================================================

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
    "save_policy": "local_first_then_drive"
}

checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"],
    checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"],
    checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"],
    checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"],
    checkpoint_verification["drive_last"]["load_ok"],
])

with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 17. Bandingkan singkat dengan A official seed 123 dan B-G14 seed 123
# ============================================================

comparison_rows = [
    {
        "run_name": "DenseNet201_A_official_seed123",
        "scenario": "A_official",
        "seed": 123,
        "best_epoch": A_OFFICIAL_SEED123_REF["best_epoch"],
        "test_accuracy": A_OFFICIAL_SEED123_REF["test_accuracy"],
        "test_macro_f1": A_OFFICIAL_SEED123_REF["test_macro_f1"],
        "test_weighted_f1": A_OFFICIAL_SEED123_REF["test_weighted_f1"],
    },
    {
        "run_name": "DenseNet201_B_G14_seed123",
        "scenario": "B_G14",
        "seed": 123,
        "best_epoch": B_G14_SEED123_REF["best_epoch"],
        "test_accuracy": B_G14_SEED123_REF["test_accuracy"],
        "test_macro_f1": B_G14_SEED123_REF["test_macro_f1"],
        "test_weighted_f1": B_G14_SEED123_REF["test_weighted_f1"],
    },
    {
        "run_name": RUN_NAME,
        "scenario": "C_G09_Date_Magnification",
        "seed": 123,
        "best_epoch": best_epoch,
        "test_accuracy": test_summary["accuracy"],
        "test_macro_f1": test_summary["macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"],
    },
    {
        "run_name": "delta_C_G09_minus_A_official_seed123",
        "scenario": "delta_vs_A",
        "seed": 123,
        "best_epoch": "",
        "test_accuracy": test_summary["accuracy"] - A_OFFICIAL_SEED123_REF["test_accuracy"],
        "test_macro_f1": test_summary["macro_f1"] - A_OFFICIAL_SEED123_REF["test_macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"] - A_OFFICIAL_SEED123_REF["test_weighted_f1"],
    },
    {
        "run_name": "delta_C_G09_minus_B_G14_seed123",
        "scenario": "delta_vs_B_G14",
        "seed": 123,
        "best_epoch": "",
        "test_accuracy": test_summary["accuracy"] - B_G14_SEED123_REF["test_accuracy"],
        "test_macro_f1": test_summary["macro_f1"] - B_G14_SEED123_REF["test_macro_f1"],
        "test_weighted_f1": test_summary["weighted_f1"] - B_G14_SEED123_REF["test_weighted_f1"],
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(COMPARISON_VS_AB_CSV, index=False)

# ============================================================
# 18. Ringkasan akhir
# ============================================================

print("\n=== File Output C-G09 Seed 123 Tersimpan ===")
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)
print("Training history:", HISTORY_CSV_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Evaluation summary:", EVAL_SUMMARY_JSON_PATH)
print("Validation predictions:", VAL_PRED_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Validation report:", VAL_REPORT_CSV_PATH)
print("Test report:", TEST_REPORT_CSV_PATH)
print("Validation CM:", VAL_CM_CSV_PATH)
print("Test CM:", TEST_CM_CSV_PATH)
print("Master log:", MASTER_LOG_PATH)
print("Checkpoint verification JSON:", CHECKPOINT_VERIFY_JSON)
print("Comparison vs A/B seed123:", COMPARISON_VS_AB_CSV)

print("\n=== Ringkasan Sel B37 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Delta vs A Official Seed 123 ===")
print(f"Delta test accuracy    : {comparison_rows[3]['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {comparison_rows[3]['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {comparison_rows[3]['test_weighted_f1']:.4f}")

print("\n=== Delta vs B-G14 Seed 123 ===")
print(f"Delta test accuracy    : {comparison_rows[4]['test_accuracy']:.4f}")
print(f"Delta test macro F1    : {comparison_rows[4]['test_macro_f1']:.4f}")
print(f"Delta test weighted F1 : {comparison_rows[4]['test_weighted_f1']:.4f}")

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Comparison Table ===")
display(comparison_df)

print("\nSel B37 selesai. Jangan lanjut seed 456 sebelum hasil seed 123 dilaporkan ke MASTER CHAT.")

In [ ]:
# Sel B38 — DenseNet201 C-G09 Date+Magnification Seed 456

import os, gc, json, time, random, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix

print("=== Sel B38: DenseNet201 C-G09 Date+Magnification Seed 456 ===")
print("Mode kerja: training C-G09 seed 456. Tidak mengubah data dan tidak memakai augmented_train.")

# ============================================================
# 1. Safety check
# ============================================================

required_vars = [
    "train_loader", "val_loader", "test_loader",
    "train_one_epoch", "evaluate", "predict_full", "compute_metrics",
    "class_names_ordered", "id_to_class",
    "OUTPUT_DIR", "CHECKPOINT_DIR", "MASTER_LOG_PATH", "MASTER_LOG_COLUMNS",
    "device", "NUM_CLASSES", "IMG_SIZE", "BATCH_SIZE", "MAX_EPOCHS", "PATIENCE",
    "LEARNING_RATE", "WEIGHT_DECAY", "MODEL_SELECTION_METRIC", "TIE_BREAKER_METRIC"
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Variable dari B37 tidak lengkap. Jangan restart runtime. Missing: {missing}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU belum aktif. Ubah runtime ke GPU.")

# ============================================================
# 2. Config seed 456
# ============================================================

SEED = 456
RUN_NAME = "DenseNet201_C_G09_seed456"
EXPERIMENT_ID = RUN_NAME

OUTPUT_DIR = Path(OUTPUT_DIR)
CHECKPOINT_DIR = Path(CHECKPOINT_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_WORK_DIR = Path("/content/tem_virus_runtime_checkpoints") / RUN_NAME / "checkpoints"
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
LAST_CKPT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_last.pt"
LOCAL_BEST_CKPT_PATH = LOCAL_WORK_DIR / f"{RUN_NAME}_best.pt"
LOCAL_LAST_CKPT_PATH = LOCAL_WORK_DIR / f"{RUN_NAME}_last.pt"

HISTORY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.csv"
HISTORY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_history.json"
TRAINING_SUMMARY_PATH = OUTPUT_DIR / f"{RUN_NAME}_training_summary.json"
EVAL_SUMMARY_JSON_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.json"
EVAL_SUMMARY_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_evaluation_metrics_summary.csv"

VAL_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_predictions.csv"
TEST_PRED_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_predictions.csv"
VAL_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_classification_report.csv"
TEST_REPORT_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_classification_report.csv"
VAL_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_validation_confusion_matrix.csv"
TEST_CM_CSV_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"
CHECKPOINT_VERIFY_JSON = OUTPUT_DIR / f"{RUN_NAME}_checkpoint_verification.json"

C_G09_RESULTS_CSV = OUTPUT_DIR / "DenseNet201_C_G09_3seed_results.csv"
C_G09_STATS_CSV = OUTPUT_DIR / "DenseNet201_C_G09_3seed_summary_stats.csv"
C_G09_COMPARE_CSV = OUTPUT_DIR / "DenseNet201_C_G09_vs_A_B_3seed_comparison.csv"

print("Run name:", RUN_NAME)
print("Seed:", SEED)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

# ============================================================
# 3. Helper ringkas
# ============================================================

def verify_checkpoint_file(path, label):
    path = Path(path)
    out = {"label": label, "path": str(path), "exists": path.exists(), "size_mb": 0, "load_ok": False, "error": None}
    if not path.exists():
        out["error"] = "file_not_found"
        return out
    out["size_mb"] = path.stat().st_size / 1024**2
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        out["load_ok"] = True
        out["epoch"] = ckpt.get("epoch")
        out["best_epoch"] = ckpt.get("best_epoch")
        out["best_metric"] = ckpt.get("best_metric")
    except Exception as e:
        out["error"] = repr(e)
    return out

def safe_save_checkpoint(obj, local_path, drive_path):
    local_path, drive_path = Path(local_path), Path(drive_path)
    local_tmp = local_path.with_suffix(local_path.suffix + ".tmp")
    drive_tmp = drive_path.with_suffix(drive_path.suffix + ".tmp")
    if local_tmp.exists(): local_tmp.unlink()
    if drive_tmp.exists(): drive_tmp.unlink()

    torch.save(obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)
    os.replace(local_tmp, local_path)

    shutil.copy2(local_path, drive_tmp)
    _ = torch.load(drive_tmp, map_location="cpu", weights_only=False)
    os.replace(drive_tmp, drive_path)

def read_eval(prefix, fallback=None):
    fallback = fallback or {}
    p = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    t = OUTPUT_DIR / f"{prefix}_training_summary.json"
    out = fallback.copy()

    if p.exists():
        with open(p, "r") as f:
            d = json.load(f)
        out.update({
            "val_accuracy": d.get("validation", {}).get("accuracy"),
            "val_macro_f1": d.get("validation", {}).get("macro_f1"),
            "test_accuracy": d["test"]["accuracy"],
            "test_macro_f1": d["test"]["macro_f1"],
            "test_weighted_f1": d["test"]["weighted_f1"],
        })
    if t.exists():
        with open(t, "r") as f:
            td = json.load(f)
        out["best_epoch"] = td.get("best_epoch", out.get("best_epoch"))
    return out

def full_report(y_true, y_pred):
    rep = classification_report(
        y_true, y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names_ordered,
        output_dict=True,
        zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    return rep, cm

# ============================================================
# 4. Fresh model DenseNet201
# ============================================================

gc.collect()
torch.cuda.empty_cache()

weights = models.DenseNet201_Weights.IMAGENET1K_V1
model = models.densenet201(weights=weights)
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

use_amp = torch.cuda.is_available()
try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler("cuda", enabled=use_amp)
except Exception:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler(enabled=use_amp)

print("Model siap. Total params:", f"{sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 5. Catat start ke master log
# ============================================================

start_time = datetime.now().isoformat()
master_log = pd.read_csv(MASTER_LOG_PATH) if Path(MASTER_LOG_PATH).exists() else pd.DataFrame(columns=MASTER_LOG_COLUMNS)
for c in MASTER_LOG_COLUMNS:
    if c not in master_log.columns:
        master_log[c] = ""

start_record = {
    "experiment_id": EXPERIMENT_ID, "model": "DenseNet201", "scenario": "C_G09_Date_Magnification",
    "seed": SEED, "run_type": "sensitivity", "start_time": start_time, "end_time": "",
    "input_size": IMG_SIZE, "batch_size": BATCH_SIZE, "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "scheduler": "ReduceLROnPlateau", "max_epoch": MAX_EPOCHS,
    "best_epoch": "", "val_accuracy": "", "val_macro_f1": "",
    "test_accuracy": "", "test_macro_f1": "", "test_weighted_f1": "",
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity training DenseNet201 C-G09 Date+Magnification seed 456. Checkpoint local-first-then-Drive."
}
master_log = pd.concat([master_log, pd.DataFrame([start_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates("experiment_id", keep="last")
master_log.to_csv(MASTER_LOG_PATH, index=False)

# ============================================================
# 6. Training
# ============================================================

history = []
best_metric = -1.0
best_tie_metric = -1.0
best_epoch = None
epochs_without_improvement = 0

print("\n=== Mulai Training C-G09 Seed 456 ===")

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{MAX_EPOCHS} — {RUN_NAME} | LR={current_lr:.8f}")
    print("=" * 80)

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader, "val")
    scheduler.step(val_metrics["val_macro_f1"])

    record = {
        "epoch": epoch,
        "lr": current_lr,
        "epoch_time_min": (time.time() - epoch_start) / 60,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        **val_metrics
    }
    history.append(record)

    current_metric = float(record[MODEL_SELECTION_METRIC])
    current_tie_metric = float(record[TIE_BREAKER_METRIC])

    improved = (
        current_metric > best_metric + 1e-8 or
        (abs(current_metric - best_metric) <= 1e-8 and current_tie_metric > best_tie_metric)
    )

    if improved:
        best_metric = current_metric
        best_tie_metric = current_tie_metric
        best_epoch = epoch
        epochs_without_improvement = 0

        ckpt = {
            "run_name": RUN_NAME, "experiment_id": EXPERIMENT_ID,
            "scenario": "C_G09_Date_Magnification", "run_type": "sensitivity",
            "seed": SEED, "epoch": epoch, "best_epoch": best_epoch,
            "best_metric_name": MODEL_SELECTION_METRIC, "best_metric": best_metric,
            "best_tie_metric_name": TIE_BREAKER_METRIC, "best_tie_metric": best_tie_metric,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "id_to_class": {str(k): v for k, v in id_to_class.items()},
            "saved_at": datetime.now().isoformat(),
            "save_policy": "local_first_then_drive"
        }
        safe_save_checkpoint(ckpt, LOCAL_BEST_CKPT_PATH, BEST_CKPT_PATH)
        print(f"-> Best checkpoint updated: epoch {epoch}, val_macro_f1={best_metric:.4f}, val_acc={best_tie_metric:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"-> No improvement ({epochs_without_improvement}/{PATIENCE}) | best epoch {best_epoch}")

    last_ckpt = {
        "run_name": RUN_NAME, "experiment_id": EXPERIMENT_ID,
        "scenario": "C_G09_Date_Magnification", "run_type": "sensitivity",
        "seed": SEED, "epoch": epoch, "best_epoch": best_epoch,
        "best_metric_name": MODEL_SELECTION_METRIC, "best_metric": best_metric,
        "best_tie_metric_name": TIE_BREAKER_METRIC, "best_tie_metric": best_tie_metric,
        "epochs_without_improvement": epochs_without_improvement,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "id_to_class": {str(k): v for k, v in id_to_class.items()},
        "saved_at": datetime.now().isoformat(),
        "save_policy": "local_first_then_drive"
    }
    safe_save_checkpoint(last_ckpt, LOCAL_LAST_CKPT_PATH, LAST_CKPT_PATH)

    pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
    with open(HISTORY_JSON_PATH, "w") as f:
        json.dump(history, f, indent=2)

    print(f"Train acc={record['train_accuracy']:.4f}, train macro F1={record['train_macro_f1']:.4f}")
    print(f"Val acc={record['val_accuracy']:.4f}, val macro F1={record['val_macro_f1']:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping aktif.")
        break

training_summary = {
    "run_name": RUN_NAME, "experiment_id": EXPERIMENT_ID,
    "scenario": "C_G09_Date_Magnification", "run_type": "sensitivity",
    "seed": SEED, "status": "completed", "max_epochs": MAX_EPOCHS,
    "epochs_completed": len(history), "best_epoch": best_epoch,
    "best_metric_name": MODEL_SELECTION_METRIC, "best_metric": best_metric,
    "best_tie_metric_name": TIE_BREAKER_METRIC, "best_tie_metric": best_tie_metric,
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "last_checkpoint_path": str(LAST_CKPT_PATH),
    "completed_at": datetime.now().isoformat(),
    "save_policy": "local_first_then_drive"
}
with open(TRAINING_SUMMARY_PATH, "w") as f:
    json.dump(training_summary, f, indent=2)

# ============================================================
# 7. Evaluasi best checkpoint
# ============================================================

print("\n=== Evaluasi Best Checkpoint C-G09 Seed 456 ===")

best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()

val_pred = predict_full(model, val_loader, "validation")
test_pred = predict_full(model, test_loader, "test")

val_summary = compute_metrics(val_pred["y_true"], val_pred["y_pred"])
test_summary = compute_metrics(test_pred["y_true"], test_pred["y_pred"])
val_summary["loss"] = val_pred["loss"]
test_summary["loss"] = test_pred["loss"]

val_report_dict, val_cm = full_report(val_pred["y_true"], val_pred["y_pred"])
test_report_dict, test_cm = full_report(test_pred["y_true"], test_pred["y_pred"])

val_pred["result_df"].to_csv(VAL_PRED_PATH, index=False)
test_pred["result_df"].to_csv(TEST_PRED_PATH, index=False)
pd.DataFrame(val_report_dict).T.to_csv(VAL_REPORT_CSV_PATH)
pd.DataFrame(test_report_dict).T.to_csv(TEST_REPORT_CSV_PATH)
pd.DataFrame(val_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(VAL_CM_CSV_PATH)
pd.DataFrame(test_cm, index=class_names_ordered, columns=class_names_ordered).to_csv(TEST_CM_CSV_PATH)

eval_summary = {
    "run_name": RUN_NAME, "experiment_id": EXPERIMENT_ID,
    "scenario": "C_G09_Date_Magnification", "run_type": "sensitivity",
    "seed": SEED, "evaluated_at": datetime.now().isoformat(),
    "best_checkpoint_path": str(BEST_CKPT_PATH),
    "best_epoch_from_checkpoint": best_epoch,
    "validation": val_summary, "test": test_summary,
    "stop_gate_90_percent_passed": bool(test_summary["accuracy"] >= 0.90),
}
with open(EVAL_SUMMARY_JSON_PATH, "w") as f:
    json.dump(eval_summary, f, indent=2)

pd.DataFrame([
    {"split": "validation", **val_summary},
    {"split": "test", **test_summary}
]).to_csv(EVAL_SUMMARY_CSV_PATH, index=False)

# ============================================================
# 8. Update master log dan verifikasi checkpoint
# ============================================================

master_log = pd.read_csv(MASTER_LOG_PATH)
for c in MASTER_LOG_COLUMNS:
    if c not in master_log.columns:
        master_log[c] = ""

end_record = {
    "experiment_id": EXPERIMENT_ID, "model": "DenseNet201", "scenario": "C_G09_Date_Magnification",
    "seed": SEED, "run_type": "sensitivity", "start_time": start_time,
    "end_time": datetime.now().isoformat(), "input_size": IMG_SIZE,
    "batch_size": BATCH_SIZE, "optimizer": "AdamW", "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY, "scheduler": "ReduceLROnPlateau",
    "max_epoch": MAX_EPOCHS, "best_epoch": best_epoch,
    "val_accuracy": val_summary["accuracy"], "val_macro_f1": val_summary["macro_f1"],
    "test_accuracy": test_summary["accuracy"], "test_macro_f1": test_summary["macro_f1"],
    "test_weighted_f1": test_summary["weighted_f1"],
    "checkpoint_path": str(BEST_CKPT_PATH),
    "notes": "Sensitivity DenseNet201 C-G09 Date+Magnification seed 456 selesai. Checkpoint local-first-then-Drive."
}
master_log = pd.concat([master_log, pd.DataFrame([end_record])], ignore_index=True)
master_log = master_log[MASTER_LOG_COLUMNS].drop_duplicates("experiment_id", keep="last")
master_log.to_csv(MASTER_LOG_PATH, index=False)

checkpoint_verification = {
    "run_name": RUN_NAME,
    "created_at": datetime.now().isoformat(),
    "local_best": verify_checkpoint_file(LOCAL_BEST_CKPT_PATH, "local_best"),
    "drive_best": verify_checkpoint_file(BEST_CKPT_PATH, "drive_best"),
    "local_last": verify_checkpoint_file(LOCAL_LAST_CKPT_PATH, "local_last"),
    "drive_last": verify_checkpoint_file(LAST_CKPT_PATH, "drive_last"),
}
checkpoint_verification["all_ok"] = all([
    checkpoint_verification["local_best"]["exists"], checkpoint_verification["local_best"]["load_ok"],
    checkpoint_verification["drive_best"]["exists"], checkpoint_verification["drive_best"]["load_ok"],
    checkpoint_verification["local_last"]["exists"], checkpoint_verification["local_last"]["load_ok"],
    checkpoint_verification["drive_last"]["exists"], checkpoint_verification["drive_last"]["load_ok"],
])
with open(CHECKPOINT_VERIFY_JSON, "w") as f:
    json.dump(checkpoint_verification, f, indent=2)

# ============================================================
# 9. Final C-G09 3 seed + comparison A/B
# ============================================================

def get_metrics(prefix, fallback=None):
    return read_eval(prefix, fallback or {})

fallback_A = {
    42: {"test_accuracy": 0.9658, "test_macro_f1": 0.9587, "test_weighted_f1": 0.9657},
    123: {"test_accuracy": 0.9674, "test_macro_f1": 0.9623, "test_weighted_f1": 0.9674},
    456: {"test_accuracy": 0.9521, "test_macro_f1": 0.9425, "test_weighted_f1": 0.9517},
}
fallback_B = {
    42: {"test_accuracy": 0.9580, "test_macro_f1": 0.9596, "test_weighted_f1": 0.9568},
    123: {"test_accuracy": 0.9637, "test_macro_f1": 0.9645, "test_weighted_f1": 0.9625},
    456: {"test_accuracy": 0.9590, "test_macro_f1": 0.9605, "test_weighted_f1": 0.9581},
}

c_rows, comp_rows = [], []
for s in [42, 123, 456]:
    c = get_metrics(f"DenseNet201_C_G09_seed{s}")
    a = get_metrics(f"DenseNet201_A_official_seed{s}", fallback_A[s])
    b = get_metrics(f"DenseNet201_B_G14_seed{s}", fallback_B[s])

    c_rows.append({
        "seed": s,
        "best_epoch": c.get("best_epoch"),
        "val_accuracy": c["val_accuracy"],
        "val_macro_f1": c["val_macro_f1"],
        "test_accuracy": c["test_accuracy"],
        "test_macro_f1": c["test_macro_f1"],
        "test_weighted_f1": c["test_weighted_f1"],
    })

    comp_rows.append({
        "seed": s,
        "A_official_test_accuracy": a["test_accuracy"],
        "B_G14_test_accuracy": b["test_accuracy"],
        "C_G09_test_accuracy": c["test_accuracy"],
        "delta_C_minus_A_accuracy": c["test_accuracy"] - a["test_accuracy"],
        "delta_C_minus_B_accuracy": c["test_accuracy"] - b["test_accuracy"],
        "A_official_test_macro_f1": a["test_macro_f1"],
        "B_G14_test_macro_f1": b["test_macro_f1"],
        "C_G09_test_macro_f1": c["test_macro_f1"],
        "delta_C_minus_A_macro_f1": c["test_macro_f1"] - a["test_macro_f1"],
        "delta_C_minus_B_macro_f1": c["test_macro_f1"] - b["test_macro_f1"],
        "A_official_test_weighted_f1": a["test_weighted_f1"],
        "B_G14_test_weighted_f1": b["test_weighted_f1"],
        "C_G09_test_weighted_f1": c["test_weighted_f1"],
        "delta_C_minus_A_weighted_f1": c["test_weighted_f1"] - a["test_weighted_f1"],
        "delta_C_minus_B_weighted_f1": c["test_weighted_f1"] - b["test_weighted_f1"],
    })

c_df = pd.DataFrame(c_rows)
comp_df = pd.DataFrame(comp_rows)

stats_df = c_df[["val_accuracy", "val_macro_f1", "test_accuracy", "test_macro_f1", "test_weighted_f1"]].agg(["mean", "std", "min", "max"]).T.reset_index()
stats_df = stats_df.rename(columns={"index": "metric"})

c_df.to_csv(C_G09_RESULTS_CSV, index=False)
stats_df.to_csv(C_G09_STATS_CSV, index=False)
comp_df.to_csv(C_G09_COMPARE_CSV, index=False)

# ============================================================
# 10. Print ringkasan
# ============================================================

print("\n=== Ringkasan Sel B38 ===")
print("Run name:", RUN_NAME)
print("Best epoch:", best_epoch)
print(f"Validation accuracy: {val_summary['accuracy']:.4f}")
print(f"Validation macro F1: {val_summary['macro_f1']:.4f}")
print(f"Test accuracy      : {test_summary['accuracy']:.4f}")
print(f"Test macro F1      : {test_summary['macro_f1']:.4f}")
print(f"Test weighted F1   : {test_summary['weighted_f1']:.4f}")
print("Stop-gate 90% passed:", test_summary["accuracy"] >= 0.90)

print("\n=== Checkpoint Verification Status ===")
print("All checkpoint checks OK:", checkpoint_verification["all_ok"])
print("Drive best checkpoint load OK:", checkpoint_verification["drive_best"]["load_ok"])
print("Drive last checkpoint load OK :", checkpoint_verification["drive_last"]["load_ok"])
print("Drive best size MB:", round(checkpoint_verification["drive_best"]["size_mb"], 2))
print("Drive last size MB :", round(checkpoint_verification["drive_last"]["size_mb"], 2))

print("\n=== Tabel Final C-G09 3 Seed ===")
display(c_df)

print("\n=== Statistik Final C-G09 3 Seed ===")
display(stats_df)

print("\n=== Perbandingan C-G09 vs A official dan B-G14 ===")
display(comp_df)

In [ ]:
# Sel B39 — Final Sensitivity Summary dan Comparison Report DenseNet201

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

print("=== Sel B39: Final Sensitivity Summary dan Comparison Report DenseNet201 ===")
print("Mode kerja: non-training. Tidak memakai GPU. Tidak mengubah data.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive sudah tersedia atau mount dilewati:", e)

# ============================================================
# 1. Path output
# ============================================================

OUTPUT_DIR = Path("/content/drive/MyDrive/Riset_TEM_Virus/outputs")
MASTER_LOG_PATH = OUTPUT_DIR / "experiment_master_log.csv"

OUT_ALL_SUMMARY = OUTPUT_DIR / "DenseNet201_all_scenarios_final_summary.csv"
OUT_ACLEAN_DELTA = OUTPUT_DIR / "DenseNet201_Aclean_vs_Aofficial_paired_delta_final.csv"
OUT_CG09_A_DELTA = OUTPUT_DIR / "DenseNet201_CG09_vs_Aofficial_paired_delta_final.csv"
OUT_CG09_B_DELTA = OUTPUT_DIR / "DenseNet201_CG09_vs_BG14_paired_delta_final.csv"
OUT_JSON = OUTPUT_DIR / "DenseNet201_final_sensitivity_interpretation.json"
OUT_REPORT = OUTPUT_DIR / "LEAF_REPORT_FINAL_DenseNet201_All_Scenarios.md"

BENCH_ACC = 0.931
BENCH_F1 = 0.921

print("Output dir:", OUTPUT_DIR)

# ============================================================
# 2. Helper
# ============================================================

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)

def get_seed_metrics(prefix, seed, scenario_name):
    eval_path = OUTPUT_DIR / f"{prefix}_evaluation_metrics_summary.json"
    train_path = OUTPUT_DIR / f"{prefix}_training_summary.json"

    eval_data = read_json(eval_path)
    train_data = read_json(train_path)

    if eval_data is None:
        raise FileNotFoundError(f"File evaluasi tidak ditemukan: {eval_path}")

    best_epoch = None
    if train_data is not None:
        best_epoch = train_data.get("best_epoch")
    if best_epoch is None:
        best_epoch = eval_data.get("best_epoch_from_checkpoint")

    return {
        "scenario": scenario_name,
        "seed": seed,
        "run_name": prefix,
        "best_epoch": best_epoch,
        "val_accuracy": float(eval_data["validation"]["accuracy"]),
        "val_macro_f1": float(eval_data["validation"]["macro_f1"]),
        "test_accuracy": float(eval_data["test"]["accuracy"]),
        "test_macro_f1": float(eval_data["test"]["macro_f1"]),
        "test_weighted_f1": float(eval_data["test"]["weighted_f1"]),
    }

def build_records(scenario_name, seeds, prefix_template):
    rows = []
    for seed in seeds:
        prefix = prefix_template.format(seed=seed)
        rows.append(get_seed_metrics(prefix, seed, scenario_name))
    return pd.DataFrame(rows)

def summarize_scenario(df, scenario_name, scenario_function):
    metrics = [
        "val_accuracy",
        "val_macro_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
    ]

    row = {
        "scenario": scenario_name,
        "scenario_function": scenario_function,
        "n_seed": int(len(df)),
    }

    for m in metrics:
        row[f"{m}_mean"] = float(df[m].mean())
        row[f"{m}_std"] = float(df[m].std(ddof=1)) if len(df) > 1 else 0.0

    row["mean_above_benchmark_accuracy_2021"] = bool(row["test_accuracy_mean"] > BENCH_ACC)
    row["mean_above_benchmark_f1_2021"] = bool(row["test_macro_f1_mean"] > BENCH_F1)
    row["all_seeds_above_benchmark_accuracy_2021"] = bool((df["test_accuracy"] > BENCH_ACC).all())
    row["all_seeds_above_benchmark_f1_2021"] = bool((df["test_macro_f1"] > BENCH_F1).all())

    return row

def paired_delta(df_left, df_right, left_name, right_name):
    left = df_left.set_index("seed")
    right = df_right.set_index("seed")
    common = sorted(set(left.index) & set(right.index))

    rows = []
    for seed in common:
        rows.append({
            "seed": seed,
            f"{left_name}_test_accuracy": left.loc[seed, "test_accuracy"],
            f"{right_name}_test_accuracy": right.loc[seed, "test_accuracy"],
            "delta_test_accuracy": left.loc[seed, "test_accuracy"] - right.loc[seed, "test_accuracy"],

            f"{left_name}_test_macro_f1": left.loc[seed, "test_macro_f1"],
            f"{right_name}_test_macro_f1": right.loc[seed, "test_macro_f1"],
            "delta_test_macro_f1": left.loc[seed, "test_macro_f1"] - right.loc[seed, "test_macro_f1"],

            f"{left_name}_test_weighted_f1": left.loc[seed, "test_weighted_f1"],
            f"{right_name}_test_weighted_f1": right.loc[seed, "test_weighted_f1"],
            "delta_test_weighted_f1": left.loc[seed, "test_weighted_f1"] - right.loc[seed, "test_weighted_f1"],
        })

    out = pd.DataFrame(rows)

    mean_row = {"seed": "MEAN"}
    for col in ["delta_test_accuracy", "delta_test_macro_f1", "delta_test_weighted_f1"]:
        mean_row[col] = float(out[col].mean())

    out = pd.concat([out, pd.DataFrame([mean_row])], ignore_index=True)
    return out

def round_df(df, digits=4):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].round(digits)
    return out

def df_to_md(df):
    try:
        return round_df(df).to_markdown(index=False)
    except Exception:
        return round_df(df).to_string(index=False)

def records_for_json(df):
    return json.loads(df.to_json(orient="records"))

# ============================================================
# 3. Load semua skenario
# ============================================================

A_df = build_records(
    scenario_name="A_official",
    seeds=[42, 123, 456, 789, 1024],
    prefix_template="DenseNet201_A_official_seed{seed}"
)

B_df = build_records(
    scenario_name="B_G14",
    seeds=[42, 123, 456, 789, 1024],
    prefix_template="DenseNet201_B_G14_seed{seed}"
)

Aclean_df = build_records(
    scenario_name="A_clean_strict",
    seeds=[42, 123, 456],
    prefix_template="DenseNet201_A_clean_strict_seed{seed}"
)

CG09_df = build_records(
    scenario_name="C_G09_Date_Magnification",
    seeds=[42, 123, 456],
    prefix_template="DenseNet201_C_G09_seed{seed}"
)

print("\n=== Data loaded ===")
print("A official:", A_df.shape)
print("B-G14:", B_df.shape)
print("A-clean strict:", Aclean_df.shape)
print("C-G09:", CG09_df.shape)

# ============================================================
# 4. Final summary semua skenario
# ============================================================

summary_rows = [
    summarize_scenario(
        A_df,
        "A_official",
        "literature-comparable evaluation"
    ),
    summarize_scenario(
        B_df,
        "B_G14",
        "RAW source-aware leakage-controlled evaluation"
    ),
    summarize_scenario(
        Aclean_df,
        "A_clean_strict",
        "strict duplicate hamming-0 sensitivity"
    ),
    summarize_scenario(
        CG09_df,
        "C_G09_Date_Magnification",
        "stricter acquisition session-aware sensitivity evaluation"
    ),
]

all_summary_df = pd.DataFrame(summary_rows)
all_summary_df.to_csv(OUT_ALL_SUMMARY, index=False)

print("\n=== Tabel Final Semua Skenario DenseNet201 ===")
display(round_df(all_summary_df))

# ============================================================
# 5. Paired delta
# ============================================================

A_3 = A_df[A_df["seed"].isin([42, 123, 456])].copy()
B_3 = B_df[B_df["seed"].isin([42, 123, 456])].copy()

aclean_vs_a_df = paired_delta(
    Aclean_df,
    A_3,
    "A_clean_strict",
    "A_official"
)

cg09_vs_a_df = paired_delta(
    CG09_df,
    A_3,
    "C_G09",
    "A_official"
)

cg09_vs_b_df = paired_delta(
    CG09_df,
    B_3,
    "C_G09",
    "B_G14"
)

aclean_vs_a_df.to_csv(OUT_ACLEAN_DELTA, index=False)
cg09_vs_a_df.to_csv(OUT_CG09_A_DELTA, index=False)
cg09_vs_b_df.to_csv(OUT_CG09_B_DELTA, index=False)

print("\n=== Paired Delta: A-clean strict vs A official ===")
display(round_df(aclean_vs_a_df))

print("\n=== Paired Delta: C-G09 vs A official ===")
display(round_df(cg09_vs_a_df))

print("\n=== Paired Delta: C-G09 vs B-G14 ===")
display(round_df(cg09_vs_b_df))

# ============================================================
# 6. Benchmark check
# ============================================================

benchmark_rows = []

for scenario_name, df in [
    ("A_official", A_df),
    ("B_G14", B_df),
    ("A_clean_strict", Aclean_df),
    ("C_G09_Date_Magnification", CG09_df),
]:
    benchmark_rows.append({
        "scenario": scenario_name,
        "n_seed": len(df),
        "test_accuracy_mean": float(df["test_accuracy"].mean()),
        "test_macro_f1_mean": float(df["test_macro_f1"].mean()),
        "mean_accuracy_above_2021": bool(df["test_accuracy"].mean() > BENCH_ACC),
        "mean_macro_f1_above_2021": bool(df["test_macro_f1"].mean() > BENCH_F1),
        "all_seed_accuracy_above_2021": bool((df["test_accuracy"] > BENCH_ACC).all()),
        "all_seed_macro_f1_above_2021": bool((df["test_macro_f1"] > BENCH_F1).all()),
        "below_benchmark_f1_seeds": df.loc[df["test_macro_f1"] <= BENCH_F1, "seed"].astype(str).tolist(),
    })

benchmark_df = pd.DataFrame(benchmark_rows)

print("\n=== Benchmark 2021 Check ===")
display(round_df(benchmark_df))

# ============================================================
# 7. Interpretasi final
# ============================================================

mean_delta_aclean_acc = float(aclean_vs_a_df.loc[aclean_vs_a_df["seed"] == "MEAN", "delta_test_accuracy"].iloc[0])
mean_delta_aclean_f1 = float(aclean_vs_a_df.loc[aclean_vs_a_df["seed"] == "MEAN", "delta_test_macro_f1"].iloc[0])

mean_delta_cg09_a_acc = float(cg09_vs_a_df.loc[cg09_vs_a_df["seed"] == "MEAN", "delta_test_accuracy"].iloc[0])
mean_delta_cg09_a_f1 = float(cg09_vs_a_df.loc[cg09_vs_a_df["seed"] == "MEAN", "delta_test_macro_f1"].iloc[0])

mean_delta_cg09_b_acc = float(cg09_vs_b_df.loc[cg09_vs_b_df["seed"] == "MEAN", "delta_test_accuracy"].iloc[0])
mean_delta_cg09_b_f1 = float(cg09_vs_b_df.loc[cg09_vs_b_df["seed"] == "MEAN", "delta_test_macro_f1"].iloc[0])

cg09_seed456_f1 = float(CG09_df.loc[CG09_df["seed"] == 456, "test_macro_f1"].iloc[0])
cg09_seed456_below_f1 = bool(cg09_seed456_f1 < BENCH_F1)

interpretation = {
    "created_at": datetime.now().isoformat(),
    "benchmark_2021": {
        "accuracy": BENCH_ACC,
        "f1": BENCH_F1,
    },
    "scenario_roles": {
        "A_official": "literature-comparable evaluation",
        "B_G14": "RAW source-aware leakage-controlled evaluation",
        "A_clean_strict": "strict duplicate hamming-0 sensitivity",
        "C_G09_Date_Magnification": "stricter acquisition session-aware sensitivity evaluation",
    },
    "main_findings": {
        "A_official": "DenseNet201 kuat pada protokol pembanding literatur resmi.",
        "B_G14": "DenseNet201 tetap kuat pada split yang mengontrol overlap RAW source.",
        "A_clean_strict": (
            "Penghapusan 8 strict duplicate hamming-0 dari train tidak menyebabkan penurunan konsisten; "
            "mean delta terhadap A official sangat kecil."
        ),
        "C_G09_Date_Magnification": (
            "C-G09 lebih sulit daripada A official dan B-G14; penurunan paling jelas terjadi pada macro F1, "
            "tetapi mean tetap kompetitif terhadap benchmark 2021."
        ),
    },
    "paired_delta_mean": {
        "A_clean_vs_A_official": {
            "mean_delta_test_accuracy": mean_delta_aclean_acc,
            "mean_delta_test_macro_f1": mean_delta_aclean_f1,
        },
        "C_G09_vs_A_official": {
            "mean_delta_test_accuracy": mean_delta_cg09_a_acc,
            "mean_delta_test_macro_f1": mean_delta_cg09_a_f1,
        },
        "C_G09_vs_B_G14": {
            "mean_delta_test_accuracy": mean_delta_cg09_b_acc,
            "mean_delta_test_macro_f1": mean_delta_cg09_b_f1,
        },
    },
    "benchmark_notes": {
        "scenarios_mean_above_benchmark": benchmark_df[
            (benchmark_df["mean_accuracy_above_2021"]) &
            (benchmark_df["mean_macro_f1_above_2021"])
        ]["scenario"].tolist(),
        "scenarios_all_seed_above_benchmark": benchmark_df[
            (benchmark_df["all_seed_accuracy_above_2021"]) &
            (benchmark_df["all_seed_macro_f1_above_2021"])
        ]["scenario"].tolist(),
        "c_g09_seed456_macro_f1": cg09_seed456_f1,
        "c_g09_seed456_macro_f1_below_benchmark": cg09_seed456_below_f1,
    },
    "recommendation": {
        "denseNet201_main_model_status": "DenseNet201 sudah cukup kuat sebagai model utama sementara.",
        "convnext_status": (
            "ConvNeXt belum wajib langsung dijalankan. Lebih aman diskusi dengan MASTER CHAT/Claude dulu, "
            "karena DenseNet201 sudah kuat dan sensitivity analysis sudah menunjukkan pola yang jelas."
        ),
        "next_step": (
            "Buat report final hasil DenseNet201 dan error analysis/per-class discussion sebelum memutuskan "
            "apakah perlu ConvNeXt sebagai model modern pembanding."
        ),
    },
    "tables": {
        "all_scenarios_summary": records_for_json(all_summary_df),
        "benchmark_check": records_for_json(benchmark_df),
        "Aclean_vs_Aofficial_delta": records_for_json(aclean_vs_a_df),
        "CG09_vs_Aofficial_delta": records_for_json(cg09_vs_a_df),
        "CG09_vs_BG14_delta": records_for_json(cg09_vs_b_df),
    }
}

with open(OUT_JSON, "w") as f:
    json.dump(interpretation, f, indent=2)

# ============================================================
# 8. Markdown LEAF report
# ============================================================

report_md = f"""# LEAF REPORT FINAL — DenseNet201 All Scenarios TEM Virus

## 1. Nama LEAF

LEAF Final Sensitivity Summary dan Comparison Report DenseNet201.

## 2. Fokus Pekerjaan

LEAF ini merangkum seluruh hasil eksperimen DenseNet201 pada empat skenario evaluasi:

1. **A official** sebagai *literature-comparable evaluation*.
2. **B-G14** sebagai *RAW source-aware leakage-controlled evaluation*.
3. **A-clean strict** sebagai *strict duplicate hamming-0 sensitivity*.
4. **C-G09 Date+Magnification** sebagai *stricter acquisition session-aware sensitivity evaluation*.

Tidak ada training baru pada B39. Analisis hanya memakai output eksperimen yang sudah tersimpan.

## 3. Tabel Final Semua Skenario

{df_to_md(all_summary_df)}

## 4. Paired Comparison A-clean Strict vs A Official

A-clean strict dibandingkan dengan A official pada seed yang sama: 42, 123, dan 456.

{df_to_md(aclean_vs_a_df)}

Interpretasi:
- A-clean strict tidak menyebabkan penurunan konsisten.
- Mean delta test accuracy = {mean_delta_aclean_acc:+.4f}.
- Mean delta test macro F1 = {mean_delta_aclean_f1:+.4f}.
- Penghapusan 8 strict duplicate hamming-0 dari train tidak membuat performa DenseNet201 runtuh.

## 5. Paired Comparison C-G09 vs A Official

C-G09 dibandingkan dengan A official pada seed yang sama: 42, 123, dan 456.

{df_to_md(cg09_vs_a_df)}

Interpretasi:
- C-G09 selalu lebih rendah dari A official pada seed 42, 123, dan 456.
- Mean delta test accuracy = {mean_delta_cg09_a_acc:+.4f}.
- Mean delta test macro F1 = {mean_delta_cg09_a_f1:+.4f}.
- Penurunan paling kuat terjadi pada macro F1.

## 6. Paired Comparison C-G09 vs B-G14

C-G09 dibandingkan dengan B-G14 pada seed yang sama: 42, 123, dan 456.

{df_to_md(cg09_vs_b_df)}

Interpretasi:
- C-G09 juga lebih rendah dari B-G14 pada seed 42, 123, dan 456.
- Mean delta test accuracy = {mean_delta_cg09_b_acc:+.4f}.
- Mean delta test macro F1 = {mean_delta_cg09_b_f1:+.4f}.
- Hal ini mendukung bahwa C-G09 merupakan protokol evaluasi yang lebih ketat.

## 7. Benchmark 2021

Benchmark utama 2021:
- Accuracy: **{BENCH_ACC:.3f}**
- F1-score: **{BENCH_F1:.3f}**

{df_to_md(benchmark_df)}

Catatan penting:
- Semua skenario memiliki mean test accuracy dan mean test macro F1 di atas benchmark 2021.
- A official, B-G14, dan A-clean strict memiliki semua seed di atas benchmark accuracy dan F1.
- C-G09 memiliki mean di atas benchmark, tetapi seed 456 memiliki test macro F1 = **{cg09_seed456_f1:.4f}**, yaitu di bawah benchmark F1 2021.
- Karena itu, klaim C-G09 harus hati-hati: secara rata-rata masih kompetitif, tetapi tidak semua seed unggul pada macro F1.

## 8. Interpretasi Metodologis

### A official

A official berfungsi sebagai evaluasi yang paling sebanding dengan literatur utama. Hasil 5-seed menunjukkan DenseNet201 kuat dan stabil pada protokol ini.

### B-G14

B-G14 mengontrol overlap berdasarkan RAW source. Hasil 5-seed tetap kuat, sehingga performa DenseNet201 tidak hanya bergantung pada split official.

### A-clean strict

A-clean strict menghapus hanya strict duplicate hamming-0 yang diberi label manual `duplicate`. Hasil 3-seed menunjukkan penghapusan duplicate ketat tidak menyebabkan penurunan konsisten.

### C-G09 Date+Magnification

C-G09 adalah sensitivity analysis paling ketat sejauh ini karena mempertimbangkan acquisition session berbasis Date + Magnification. Hasil 3-seed menunjukkan performa turun, terutama pada macro F1, tetapi tidak runtuh.

## 9. Kesimpulan LEAF

DenseNet201 sudah cukup kuat sebagai model utama sementara. Hasilnya:
- kuat pada A official;
- tetap kuat pada B-G14;
- robust terhadap A-clean strict;
- masih kompetitif pada C-G09 meskipun C-G09 jelas lebih menantang.

## 10. Rekomendasi LEAF

1. DenseNet201 layak dijadikan model utama sementara.
2. Jangan langsung menjalankan ConvNeXt sebelum MASTER CHAT menilai report final ini.
3. ConvNeXt dapat dipertimbangkan sebagai model modern pembanding, tetapi bukan kebutuhan mendesak karena DenseNet201 sudah kuat.
4. Langkah berikutnya yang lebih penting adalah menyusun narasi hasil dan pembahasan sensitivity analysis.
5. Diskusi dengan Claude disarankan sebelum memutuskan apakah ConvNeXt perlu dijalankan.

## 11. File Output

- `DenseNet201_all_scenarios_final_summary.csv`
- `DenseNet201_Aclean_vs_Aofficial_paired_delta_final.csv`
- `DenseNet201_CG09_vs_Aofficial_paired_delta_final.csv`
- `DenseNet201_CG09_vs_BG14_paired_delta_final.csv`
- `DenseNet201_final_sensitivity_interpretation.json`
- `LEAF_REPORT_FINAL_DenseNet201_All_Scenarios.md`
"""

with open(OUT_REPORT, "w") as f:
    f.write(report_md)

# ============================================================
# 9. Ringkasan akhir
# ============================================================

print("\n=== File Output B39 Tersimpan ===")
print("All scenarios summary:", OUT_ALL_SUMMARY)
print("A-clean vs A official paired delta:", OUT_ACLEAN_DELTA)
print("C-G09 vs A official paired delta:", OUT_CG09_A_DELTA)
print("C-G09 vs B-G14 paired delta:", OUT_CG09_B_DELTA)
print("Interpretation JSON:", OUT_JSON)
print("LEAF report MD:", OUT_REPORT)

print("\n=== Ringkasan Sel B39 untuk MASTER CHAT ===")
print("A official 5-seed test accuracy mean:", round(float(A_df["test_accuracy"].mean()), 4))
print("A official 5-seed test macro F1 mean:", round(float(A_df["test_macro_f1"].mean()), 4))
print("B-G14 5-seed test accuracy mean:", round(float(B_df["test_accuracy"].mean()), 4))
print("B-G14 5-seed test macro F1 mean:", round(float(B_df["test_macro_f1"].mean()), 4))
print("A-clean strict 3-seed test accuracy mean:", round(float(Aclean_df["test_accuracy"].mean()), 4))
print("A-clean strict 3-seed test macro F1 mean:", round(float(Aclean_df["test_macro_f1"].mean()), 4))
print("C-G09 3-seed test accuracy mean:", round(float(CG09_df["test_accuracy"].mean()), 4))
print("C-G09 3-seed test macro F1 mean:", round(float(CG09_df["test_macro_f1"].mean()), 4))

print("\nMean paired delta:")
print("A-clean strict vs A official accuracy:", round(mean_delta_aclean_acc, 4))
print("A-clean strict vs A official macro F1:", round(mean_delta_aclean_f1, 4))
print("C-G09 vs A official accuracy:", round(mean_delta_cg09_a_acc, 4))
print("C-G09 vs A official macro F1:", round(mean_delta_cg09_a_f1, 4))
print("C-G09 vs B-G14 accuracy:", round(mean_delta_cg09_b_acc, 4))
print("C-G09 vs B-G14 macro F1:", round(mean_delta_cg09_b_f1, 4))

print("\nBenchmark note:")
print("Benchmark 2021 accuracy:", BENCH_ACC)
print("Benchmark 2021 F1:", BENCH_F1)
print("C-G09 seed 456 macro F1 below benchmark:", cg09_seed456_below_f1)

print("\nRecommendation:")
print(interpretation["recommendation"]["denseNet201_main_model_status"])
print(interpretation["recommendation"]["convnext_status"])

print("\nSel B39 selesai. Tidak ada training. Tidak memakai GPU.")